In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import duckdb
from pathlib import Path
import re
import seaborn as sns

#### Define directories

In [ ]:
ROOT = Path.cwd().resolve()

DATA = (ROOT / "data").resolve()

eda_df = pd.read_parquet(DATA/"eda_dataset.parquet")

#### Read the data in

In [ ]:
eda_df.columns

In [ ]:
eda_df.Year.value_counts()

#### Modeling target decision

In [ ]:
target_decision = pd.DataFrame({
    "Decision": [
        "Outcome for reporting",
        "Modeling target",
        "Transformation method",
        "Justification"
    ],
    "Value": [
        "stdzd_amt_per_service (raw, $/service)",
        "log1p(stdzd_amt_per_service) (stored as log_stdzd_amt_per_service)",
        "TTR with log1p and expm1.",
        "Heavy-tailed raw outcome. Log reduces tail dominance and improves stability."
    ]
})
target_decision

#### Training inclusion filter

In [ ]:
# Define training inclusion mask explicitly (even if eda_df is already filtered)
train_mask = (
    (eda_df["services"] >= 11) &
    (eda_df["stdzd_amt_per_service"].notna()) &
    (eda_df["stdzd_amt_per_service"] >= 0)
)

train_filter_summary = pd.DataFrame({
    "Metric": [
        "Rows in eda_df",
        "Rows meeting training filter",
        "Share kept",
        "Unique NPIs (all)",
        "Unique NPIs (kept)",
        "Years present (all)",
        "Years present (kept)",
    ],
    "Value": [
        len(eda_df),
        int(train_mask.sum()),
        float(train_mask.mean()),
        eda_df["Rndrng_NPI"].nunique(),
        eda_df.loc[train_mask, "Rndrng_NPI"].nunique(),
        sorted(eda_df["Year"].unique().tolist()),
        sorted(eda_df.loc[train_mask, "Year"].unique().tolist()),
    ]
})

train_filter_summary

#### Define feature list:

In [ ]:
# Categorical features (to encode)
cat_features = [
    "rbcs_family_desc",    # or use "RBCS_FamNumb" instead, but pick one consistently
    "Place_Of_Srvc",
    "provider_type",
    "state",
    "ruca_bucket",
]

# Numeric features
num_features = [
    "bene_avg_risk_score",
    "years_since_enumeration",
    "log_services",
    "log_benes",
    "p_cancer6", "p_diabetes", "p_ckd", "p_copd", "p_htn",
]

# Final target
target_col = "stdzd_amt_per_service"

# Exclusions (documented)
excluded = [
    # raw cost outcomes besides target
    "log_stdzd_amt_per_service", "allowed_amt_per_service", "payment_amt_per_service", "submitted_charge_per_service",
    # totals derived from outcomes (spend columns)
    "stdzd_spend", "allowed_spend", "payment_spend", "submitted_spend",
    # flags/buckets used for slicing, not for training features
    "is_top_1pct_stdzd_amt_per_service", "svc_bucket", "services_bins", "services_custom", "services_custom2",
    # provider-year totals (often avoided to prevent scale leakage; can revisit intentionally later)
    "tot_mdcr_stdzd_amt",
]

features_table = pd.DataFrame({
    "Type": (["categorical"] * len(cat_features)) + (["numeric"] * len(num_features)) + (["target"] * 1),
    "Column": cat_features + num_features + [target_col]
})

features_table

This is our intended modeling feature set.

Categorical features
- `rbcs_family_desc`
- `Place_Of_Srvc`
- `provider_type`
- `state`
- `ruca_bucket`

Interpretation:
- These explain systematic price differences due to:
    - what service is being delivered,
    - where it is delivered,
    - what specialty is delivering it,
    - geography and rurality.

Numeric features
- risk and experience:
    - `bene_avg_risk_score`
    - `years_since_enumeration`
- exposure/intensity controls (log):
    - `log_services`
    - `log_benes`
- case mix proportions:
    - `p_cancer6`, `p_diabetes`, `p_ckd`, `p_copd`, `p_htn`

Interpretation:
- This is a classic “risk adjustment + context” set.
- We are not leaking the target because you excluded spend totals and other cost measures.

Modeling implication
- We picked `rbcs_family_desc` over `RBCS_FamNumb`. Although ID is often cleaner, desc is often fine too.

#### Quick availability check:

In [ ]:
missing_cols = [c for c in (cat_features + num_features + [target_col]) if c not in eda_df.columns]
missing_cols

`missing_cols` gives empty list `[]`

Interpretation
- Everything we plan to model exists in our dataframe.
- This prevents “modeling notebook surprises.”

#### Create split masks

In [ ]:
train_years = [2020, 2021, 2022]
test_years = [2023]

mask_train = eda_df["Year"].isin(train_years)
mask_test = eda_df["Year"].isin(test_years)

split_counts = pd.DataFrame({
    "Split": ["train", "test"],
    "Years": [train_years, test_years],
    "Rows": [int(mask_train.sum()), int(mask_test.sum())],
    "Unique NPIs": [eda_df.loc[mask_train, "Rndrng_NPI"].nunique(), eda_df.loc[mask_test, "Rndrng_NPI"].nunique()],
    "Spend share": [
        float(eda_df.loc[mask_train, "stdzd_spend"].sum() / eda_df["stdzd_spend"].sum()),
        float(eda_df.loc[mask_test, "stdzd_spend"].sum() / eda_df["stdzd_spend"].sum()),
    ],
})
split_counts

The output
- Train:
    - 213,296 rows
    - 19,838 NPIs
    - 66.6% spend
- Test:
    - 105,026 rows
    - 19,226 NPIs
    - 33.4% spend

Interpretation
- YoWeu have a healthy split. About one-third of dollars are in the test year.
- The train and test have similar provider counts, which is good for generalization tests.

Modeling implication
- This is a realistic production-like test: learn patterns from earlier years, apply to the next year.


#### Provider overlap (leakage / generalization check):

In [ ]:
npi_train = set(eda_df.loc[mask_train, "Rndrng_NPI"].unique().tolist())
npi_test = set(eda_df.loc[mask_test, "Rndrng_NPI"].unique().tolist())

overlap = npi_train.intersection(npi_test)
test_only = npi_test - npi_train

provider_overlap_tbl = pd.DataFrame({
    "Metric": ["Train NPIs", "Test NPIs", "Overlap NPIs", "Test-only NPIs"],
    "Value": [len(npi_train), len(npi_test), len(overlap), len(test_only)],
    "Share of test NPIs": [
        np.nan,
        1.0,
        len(overlap)/len(npi_test) if len(npi_test) else np.nan,
        len(test_only)/len(npi_test) if len(npi_test) else np.nan,
    ]
})
provider_overlap_tbl

The `provider_overlap_tbl` output table:
- `Test NPIs`: 19,226
- `Overlap with train`: 18,145 (94.38%)
- `Test-only`: 1,081 (5.62%)

Interpretation
- Most test providers were seen in train. That means your evaluation is mostly:
    - “new year for known providers” (easier)
- But you still have a meaningful cold-start set:
    - 1,081 providers

Modeling implication
- You should report performance separately for:
    - seen providers (in train)
    - unseen providers (test-only)

Because those are different deployment realities.

#### Planned slice keys table:

In [ ]:
eval_plan = pd.DataFrame({
    "Category": [
        "Primary metrics",
        "Target scale",
        "Core slices (report)",
        "Stability slices (flagging)",
        "Tail diagnostic"
    ],
    "Plan": [
        "MAE, RMSE",
        "log of stdzd_amt_per_service via TTR (log1p)",
        "provider_type, Place_Of_Srvc, ruca_bucket, state",
        "svc_bucket and services>=50 / >=100 subsets",
        "Compare tail vs non-tail behavior without changing labels"
    ]
})
eval_plan

#### Modeling readiness summary table (final contract):

In [ ]:
readiness_contract = pd.DataFrame({
    "Decision Area": [
        "Dataset grain",
        "Final modeling target",
        "Training inclusion",
        "Tail handling",
        "Visualization-only clipping",
        "Post-model flagging threshold",
        "Features (categorical)",
        "Features (numeric)",
        "Split plan",
        "Evaluation slices"
    ],
    "Final Choice": [
        "provider-year-RBCS family-place of service",
        "log of stdzd_amt_per_service via TTR = log1p(stdzd_amt_per_service)",
        "services >= 11; stdzd_amt_per_service not null and >= 0",
        "Keep tail; do not winsorize labels; use tail flag for diagnostics",
        "Allow p99 clipping for readability in plots only",
        "Flagging candidates evaluated on services >= 50 (and >= 100 sensitivity)",
        ", ".join(cat_features),
        ", ".join(num_features),
        "Train 2020–2022, Test 2023",
        "provider_type, Place_Of_Srvc, ruca_bucket, svc_bucket, tail flag (diagnostic)"
    ],
    "Why defensible": [
        "Matches EDA grain and planned use case",
        "Controls heavy tail while preserving signal",
        "Reliability threshold reduces denominator noise",
        "Tail appears real and interpretable (not pure error). Log target handles skew",
        "Prevents misleading plots without altering training distribution",
        "Reduces false positives from low-volume instability",
        "Captures key context (service, POS, specialty, geography, rurality)",
        "Captures case-mix + intensity + experience + comorbidity composition",
        "Temporal generalization is the real-world test",
        "Ensures interpretability and stability across key segments"
    ]
})

readiness_contract

#### Create data frames for modeling

In [ ]:
model_cols = cat_features + num_features + [target_col, "Year", "Rndrng_NPI", "services", "log_stdzd_amt_per_service"]

model_df = eda_df.loc[train_mask, model_cols].copy()

train_df = model_df[model_df["Year"].isin(train_years)].copy()
test_df = model_df[model_df["Year"].isin(test_years)].copy()

train_df.shape, test_df.shape

#### Missingness in features used in modeling

In [ ]:
feature_missing = (
    model_df[cat_features + num_features + [target_col]]
    .isna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
    .rename("pct_missing")
    .reset_index()
    .rename(columns={"index": "column"})
)
feature_missing

The `feature_missing` is the one we should pay attention to before training.

We have missingness in:
- `p_copd`: 4.59%
- `p_ckd`: 2.05%
- `p_diabetes`: 1.15%
- `p_cancer6`: 0.86%
- `years_since_enumeration`: 0.45%
- `p_htn`: 0.05%
Everything else: 0%

Interpret each variable’s missingness and what it implies

- `p_copd` (4.59% missing)
    - This is the highest missingness among our features.
    - Likely causes:
        - certain provider-year records lack the COPD percentage due to suppression, reporting rules, or merge gaps.
    - Modeling risk:
        - dropping rows would throw away ~4.6% of our data for one feature.
    - Recommended handling:
        - impute missing with a neutral value (commonly 0) plus a missingness indicator, or
        - impute with median and add indicator.
    - Why indicator matters:
        - missingness might correlate with provider type or geography, so the “missing” itself can carry signal.

- `p_ckd` (2.05% missing)
    - Similar logic, lower magnitude.
    - We should handle it the same way as p_copd for consistency.

- `p_diabetes` (1.15% missing)
    - Small but non-trivial.
    - Same treatment.

- `p_cancer6` (0.86% missing)
    - Small. Same treatment.
    - This one is especially sensitive conceptually in oncology, so do not silently drop rows.

- `years_since_enumeration` (0.45% missing)
    - This is “provider experience proxy.”
    - Missingness likely means:
        - NPI enumeration date missing upstream, or mapping failed.
    - Modeling risk:
        - leaving it missing can break some models.
    - Handling:
        - impute median and add missingness flag is the safest.

- `p_htn` (0.05% missing)
    - Very small. Still handle systematically (same imputation pattern).
    - Consistency matters. We do not want special-case logic for one feature.

- All categoricals have 0% missing
    - That is excellent. It means our slicing features are complete.
    - Especially important for:
        - `provider_type`, `Place_Of_Srvc`, `ruca_bucket`.

- `bene_avg_risk_score` has 0% missing
    - This is great because it is typically our primary adjustment feature.

Modeling implication
- We need a missingness strategy as part of Notebook 10 or the first modeling notebook.
- The best practice approach here is:

1.	For each numeric feature with missing:

- create `is_missing_<feature>` indicator

2.	Impute missing values:

- either 0 (for percentage fields) or median

3.	Keep the indicator in the model

This preserves rows, avoids bias from dropping, and allows missingness patterns to be learned.


#### Build X/y

In [ ]:
# Target is log target (your modeling target)
y_train = train_df[target_col].copy()
y_test  = test_df[target_col].copy()

# Keep these for analysis, but not as model predictors
id_cols = ["Rndrng_NPI", "Year"]

# Also exclude raw cost outcome (you do not want leakage)
exclude_from_X = [target_col, "log_stdzd_amt_per_service"] + id_cols

X_train = train_df.drop(columns=exclude_from_X).copy()
X_test  = test_df.drop(columns=exclude_from_X).copy()

Now X_train contains exactly `cat_features + num_features`. 

#### Preprocess with `ColumnTransformer`

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder

cat_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="None"))
])

cat_pipe_ohe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="None")),
    ("ohe", OneHotEncoder(handle_unknown="ignore", drop="first"))
])

num_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median", add_indicator=True))
])

preprocess = ColumnTransformer(
    transformers=[
        ("cat", cat_pipe, cat_features),
        ("num", num_pipe, num_features),
    ],
    remainder="drop"
)

preprocess_ohe = ColumnTransformer(
    transformers=[
        ("cat_ohe", cat_pipe_ohe, cat_features),
        ("num", num_pipe, num_features),
    ],
    remainder="drop"
)

#### Fit-transform train, transform test

In [ ]:
X_train_proc      = preprocess.fit_transform(X_train)
X_test_proc       = preprocess.transform(X_test)

X_train_proc_ohe  = preprocess_ohe.fit_transform(X_train)
X_test_proc_ohe   = preprocess_ohe.transform(X_test)

feat_names_ohe = preprocess_ohe.get_feature_names_out()
feat_names_ohe[:10]

In [ ]:
print(X_train_proc.shape)
print(X_test_proc.shape)
print(X_train_proc_ohe.shape)
print(X_test_proc_ohe.shape)

Note: We used `get_feature_names_out()` for debugging and SHAP later.

#### Inspect the reference categories for each categorical variable, i.e., `cat_features`:

Here, we need to 

1. reach into the `preprocess_ohe`, which is the our `ColumnTransformer` and 
2. grab the `cat_ohe` step, the `cat_pipe_ohe`, which is our `Pipeline`, 
3. then we reach into the ``cat_pipe_ohe` `Pipeline`, and 
4. grab the `ohe` step, which is our `OneHotEncoder(...)`

In [ ]:
# A list to store our reference categories
refs = []

# Get the fitted OHE object of the cat_pipe_ohe Pipeline
ohe = preprocess_ohe.named_transformers_["cat_ohe"].named_steps["ohe"]

# categories_ is a list alighed to cat_features
# each entry is an array of the learned categories for that feature 
for col, cats in zip(cat_features, ohe.categories_):
    refs.append({
        "feature": col,
        "reference_dropped":cats[0], # dropped because drop="first"
        "all_categories": list(cats), #optional, can be long for rbcs_family_desc
        "n_categories": len(cats)
    })

ref_table = pd.DataFrame(refs)[["feature","reference_dropped", "n_categories"]]
ref_table

Now we have:
- `X_train_proc`: imputed features
- `X_test_proc`: same columns, same encoder mapping, no leakage

- `X_train_proc_ohe`: one-hot encoded and imputed features
- `X_test_proc_ohe`: same columns, same encoder mapping, no leakage

#### Explain `UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros warnings.warn(msg, UserWarning)`

- During `fit_transform(X_train)`, the encoder learns the set of categories seen in **train** for each categorical feature.
- During `transform(X_test)`, it encountered at least one category in **column [0]** (that is your first categorical feature, `rbcs_family_desc`) that was **not present in train**.
- Because we set `handle_unknown="ignore"`, sklearn does not crash. It encodes those unseen categories as **all zeros across the OHE columns for that feature**.

So the model effectively treats “unseen category” as “none of the known categories”.

Let's add a small check so we know how often it happens (this tells us what fraction of test rows have unseen `rbcs_family_desc`):

In [ ]:
ohe = preprocess_ohe.named_transformers_["cat_ohe"].named_steps["ohe"]
cats = ohe.categories_[0]  # categories for first categorical feature
n_unknown = (~X_test[cat_features[0]].astype(str).isin(cats)).sum()
share_unknown = n_unknown / len(X_test)
n_unknown, share_unknown

The result basically says:

- **Only 2 rows in your entire 2023 test set** have an `rbcs_family_desc` value that never appeared in 2020–2022 training.
- That is **0.0019% of test rows** (about 1 in 52,500 rows).

So the warning is totally benign here.

In [ ]:
from preprocessing import CorrelationThreshold

# ==========================================
# RE-SYNC & PLOT (RUN THIS WHOLE BLOCK)
# ==========================================

# 0. Build a dense DataFrame with correct column names, then compute correlation
import scipy.sparse as sp

# X_train_proc_ohe is csr_matrix, feat_names_ohe length = 193
X_train_proc_ohe_dense = X_train_proc_ohe.toarray() if sp.issparse(X_train_proc_ohe) else np.asarray(X_train_proc_ohe)
X_train_proc_ohe_df = pd.DataFrame(X_train_proc_ohe_dense, columns=feat_names_ohe)

# 1. RE-CALCULATE DROPS (Ensure the list is fresh!)
corr_selector = CorrelationThreshold(threshold=0.9)
corr_selector.fit(X_train_proc_ohe_df)
dropped_cols = corr_selector.to_drop_

# 2. RE-CALCULATE SUBSET DATA
# We need to rebuild the subset_corr to match the fresh dropped_cols list
if len(dropped_cols) > 0:
    # Find partners again
    corr_matrix = X_train_proc_ohe_df.corr().abs()
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    partners = []
    for dropped in dropped_cols:
        # Find the feature kept
        partner_match = upper.index[upper[dropped] > 0.9].tolist()
        if partner_match:
            partners.append(partner_match[0])
            
    # Combine lists
    features_to_plot = list(set(dropped_cols + partners))
    subset_corr = X_train_proc_ohe_df[features_to_plot].corr()

    # 3. PLOT
    plt.figure(figsize=(16, 14))
    ax = plt.gca()

    # Heatmap
    mask = np.triu(np.ones_like(subset_corr, dtype=bool))
    sns.heatmap(
        subset_corr,
        mask=mask,
        cmap='coolwarm',
        center=0,
        square=True,
        linewidths=.5,
        cbar_kws={"shrink": .5},
        annot=False, 
        ax=ax
    )

    # 4. HIGHLIGHTING LOOP
    # Fix X-axis labels
    new_x_labels = []
    for label in ax.get_xticklabels():
        text = label.get_text()
        if text in dropped_cols:
            label.set_color('red')
            label.set_weight('bold')
            label.set_text(f"[DROP] {text}") 
        new_x_labels.append(label)
    ax.set_xticklabels(new_x_labels)

    # Fix Y-axis labels
    new_y_labels = []
    for label in ax.get_yticklabels():
        text = label.get_text()
        if text in dropped_cols:
            label.set_color('red')
            label.set_weight('bold')
            label.set_text(f"[DROP] {text}")
        new_y_labels.append(label)
    ax.set_yticklabels(new_y_labels)

    plt.title(f"Redundancy Audit: Features marked with [DROP] will be removed ({len(dropped_cols)} total)")
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

else:
    print("Zero redundancies found. Nothing to plot!")

In [ ]:
X_train_proc_ohe.shape, len(feat_names_ohe), [c for c in feat_names_ohe if "ruca_bucket" in c]

In [ ]:
# What categories are actually present in train for ruca_bucket?
X_train["ruca_bucket"].astype(str).value_counts(dropna=False).head(10)

In [ ]:
ohe = preprocess_ohe.named_transformers_["cat_ohe"].named_steps["ohe"]
# index of ruca_bucket within cat_features
ruca_idx = cat_features.index("ruca_bucket")
ohe.categories_[ruca_idx]

In [ ]:
dropped_cols

#### Extract the feature names for the `X_train_proc`

Remember that `X_train_proc` is a matrix, so it does not have "column names". 

We need to extract the column names from the `Pipeline` called `cat_pipe`:

In [ ]:
feat_names = preprocess.get_feature_names_out()
feat_names[:10]

In [ ]:
len(feat_names)

#### Create a pandas dataframe from the `X_train_proc` which is a sparse `csr_matrix` 

First, we need to turn it into a dense matrix

Second, we turn the dense matrix into a Pandas DataFrame

In [ ]:
# X_train_proc_ohe is csr_matrix, feat_names length = 193
X_train_proc_dense = X_train_proc.toarray() if sp.issparse(X_train_proc) else np.asarray(X_train_proc)
X_train_proc_df = pd.DataFrame(X_train_proc_dense, columns=feat_names)

In [ ]:
X_train_proc_df.head()

> Now we can inspect the `X_train_proc_df` just like the `X_train_proc_ohe_df`, if desired. We could apply the same logic to the `X_test_proc` and `X_test_proc_ohe` also. 

### Audit correlations on the numeric block only

#### Extract the transformed numeric matrix (with missingness indicators)

In [ ]:
num_features

In [ ]:
# Grab the fitted numeric pipeline (the num_pipe)
num_pipe_fitted = preprocess_ohe.named_transformers_["num"]

# Transform only the numeric columns 
X_train_num_proc = num_pipe_fitted.transform(X_train[num_features]) # numpy array, small width

#### Get the numeric feature names (including indicators)

In [ ]:
num_feat_names = num_pipe_fitted.get_feature_names_out(num_features)
num_feat_names

#### Run your CorrelationThreshold on the numeric DataFrame

In [ ]:
X_train_num_df = pd.DataFrame(X_train_num_proc, columns=num_feat_names, index=X_train.index)

corr_selector_num = CorrelationThreshold(threshold=0.9)
corr_selector_num.fit(X_train_num_df)

dropped_num_cols = corr_selector_num.to_drop_
dropped_num_cols

### Feature Extractor

Here I define a feature extraction function that systematically checks a model's attributes to identify the object, peel off any wrappers, grab the actual model, extract the coefficients or feature importances, construct a dataframe ready to for the next function to plot. 

In [ ]:
import numpy as np
import pandas as pd

def extract_model_features(model_object, feat_names):
    """
    Extract feature names and weights (coefficients or importances).

    Parameters
    ----------
    model_object : fitted estimator
        Can be Pipeline, GridSearchCV, TransformedTargetRegressor, or a plain estimator.
    feat_names : array-like
        Feature names that align with the model's final input space (post-preprocessing).
        Example: preprocess_ohe.get_feature_names_out()

    Returns
    -------
    pd.DataFrame with columns:
      Feature, Coefficient/Importance, Abs_Weight
    """

    # 1) unwrap GridSearchCV
    obj = model_object
    if hasattr(obj, "best_estimator_"):
        obj = obj.best_estimator_

    # 2) unwrap TransformedTargetRegressor
    if hasattr(obj, "regressor_"):
        obj = obj.regressor_

    # 3) identify final fitted estimator
    # If Pipeline, final step is last named step
    if hasattr(obj, "named_steps"):
        final_step_name = list(obj.named_steps.keys())[-1]
        final_model = obj.named_steps[final_step_name]
    else:
        final_model = obj

    current_features = np.array(feat_names)

    # 4) extract weights
    metric_name = None
    weights = None

    if hasattr(final_model, "coef_"):
        weights = final_model.coef_
        metric_name = "Coefficient"

        # Handle shape (1, n_features) or (n_targets, n_features)
        weights = np.asarray(weights)
        if weights.ndim == 2:
            # common single-target 2D shapes
            if 1 in weights.shape:
                weights = weights.ravel()
            else:
                raise ValueError(
                    f"coef_ is 2D with shape {weights.shape}. "
                    "This looks like multioutput. Decide which target to plot."
                )

        # Optional: drop exact/near zeros (useful for Lasso/ElasticNet)
        mask = np.abs(weights) > 1e-5
        current_features = current_features[mask]
        weights = weights[mask]

    elif hasattr(final_model, "feature_importances_"):
        weights = np.asarray(final_model.feature_importances_)
        metric_name = "Importance"

    elif hasattr(final_model, "get_feature_importance"):
        weights = np.asarray(final_model.get_feature_importance())
        metric_name = "Importance"

    elif hasattr(final_model, "estimators_"):
        print("VotingRegressor has no single weight vector. Plot base estimators individually.")
        return pd.DataFrame()

    else:
        raise TypeError(f"Unsupported model type for extraction: {type(final_model)}")

    # 5) sanity check alignment
    if len(current_features) != len(weights):
        raise ValueError(
            f"Feature name mismatch. len(features)={len(current_features)} "
            f"but len(weights)={len(weights)}. "
            "Make sure feat_names matches the model's final input space."
        )

    df = pd.DataFrame({
        "Feature": current_features,
        metric_name: weights
    })
    df["Abs_Weight"] = df[metric_name].abs()
    return df.sort_values("Abs_Weight", ascending=False)

### Feature Impact Visuzlization Function

Here I defined a plotting function that takes in the dataframe containing a model's coefficients or feature importance, sorts the top 20 features by the absolute values of their coefficients or importances (i.e., weights), then plots them as horizontal bar plot.

In [ ]:
# ==========================================
# FEATURE IMPACT VISUALIZATION FUNCTION
# ==========================================

def plot_feature_impact(df, title="Feature Impact", top_n=20):
    if df is None or df.empty:
        print("No data to plot.")
        return

    # Pick metric column
    if "Coefficient" in df.columns:
        metric_col = "Coefficient"
        is_coef = True
    elif "Importance" in df.columns:
        metric_col = "Importance"
        is_coef = False
    else:
        raise ValueError("df must contain either 'Coefficient' or 'Importance' column.")

    # Ensure sorted by absolute weight, then take top_n
    if "Abs_Weight" not in df.columns:
        df = df.copy()
        df["Abs_Weight"] = df[metric_col].abs()

    plot_df = (
        df.sort_values("Abs_Weight", ascending=False)
          .head(top_n)
          .sort_values("Abs_Weight", ascending=True)  # for horizontal bar readability
    )

    plt.figure(figsize=(10, 8))

    if is_coef:
        colors = ["green" if x > 0 else "red" for x in plot_df[metric_col]]
        xlabel = "Impact on log1p(stdzd_amt_per_service) (TTR space)"
    else:
        colors = "skyblue"
        xlabel = "Feature importance"

    plt.barh(plot_df["Feature"], plot_df[metric_col], color=colors)

    if is_coef:
        plt.axvline(x=0, color="black", linestyle="--", linewidth=0.8)

    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel("Features")
    plt.tight_layout()
    plt.show()

# Start modeling

## OLS

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.compose import TransformedTargetRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, mean_squared_error

- Define the inner OLS pipe

In [ ]:
ols_inner_pipe = Pipeline(steps=[
    ("preprocess", preprocess_ohe),
    ("scaler", StandardScaler(with_mean=False)),  # sparse safe
    ("model", LinearRegression())
])

- wrap the inner pipe inside a `TransformedTargetRegressor()` (TTR)

In [ ]:
ols_full_model = TransformedTargetRegressor(
    regressor=ols_inner_pipe,
    func=np.log1p,
    inverse_func=np.expm1
)

- Fit the model

In [ ]:
ols_full_model.fit(X_train, y_train)

- Diagnostics

In [ ]:
pred_test = ols_full_model.predict(X_test)

mae = mean_absolute_error(y_test, pred_test)
rmse = root_mean_squared_error(y_test, pred_test)

print("OLS (Raw y + TTR(log1p))")
print("Train R2:", ols_full_model.score(X_train, y_train))
print("Test  R2:", ols_full_model.score(X_test, y_test))
print("MAE ($):", mae)
print("RMSE ($):", rmse)

1. It's likely he relationship is not well captured by a single global linear surface in this feature space (even after the log transform)
2. The RMSE being much larger than the MAE is a classic sign that a small fraction of predictions are very wrong (heavy tail, hard categories, or rare combinations). That is consistent with our EDA.

#### Investigate the reason for poor performance

- Let's make sure we did not accidentally score on the wrong target scale.

In [ ]:
from sklearn.metrics import r2_score

pred_test = ols_full_model.predict(X_test)

r2_dollars = r2_score(y_test, pred_test)

r2_log = r2_score(np.log1p(y_test), np.log1p(pred_test.clip(min=0)))

r2_dollars, r2_log

A) The OLS model is much better at ranking and relative cost than at matching dollars

An **R² of ~0.77 in log space** says the linear model is capturing a lot of the systematic structure in **log1p(cost)**.

But **R² of ~0.22 in dollars** says that once we convert back to dollars, the remaining errors (especially for high-cost cases) explode in magnitude and dominate the variance.

This is exactly what heavy-tailed outcomes do: a small number of expensive rows contribute a huge fraction of dollar variance.

B) The log transform changes the loss geometry

In log space, being off by (say) 30% and being off by 2x are “closer” than they look in dollars.

In dollars, those same misses can be hundreds or thousands of dollars and they dominate SSE, so dollar-scale R² drops.

C) The MAE and RMSE already hinted at this

MAE ~$32 but RMSE ~$152 means “most points are okay, but a few are very wrong in dollars”. Those few are usually the tail.

In [ ]:
pred_test = ols_full_model.predict(X_test)

test_eval = test_df.copy()
test_eval["pred"] = pred_test
test_eval["abs_err"] = (test_eval[target_col] - test_eval["pred"]).abs()
test_eval["sq_err"]  = (test_eval[target_col] - test_eval["pred"])**2

# Bring back tail flag (and optionally svc_bucket) from eda_df using the shared index
test_eval = test_eval.join(
    eda_df.loc[:, ["is_top_1pct_stdzd_amt_per_service", "svc_bucket"]],
    how="left"
)

tail_summary = (
    test_eval.groupby("is_top_1pct_stdzd_amt_per_service", dropna=False)
    .agg(
        n=("pred", "size"),
        mae=("abs_err", "mean"),
        rmse=("sq_err", lambda s: np.sqrt(s.mean())),
        y_mean=(target_col, "mean"),
        pred_mean=("pred", "mean"),
    )
)

tail_summary

**Non-tail (False)**

- n = 104,107 test rows
- MAE ≈ $27.93
- RMSE ≈ $50.95
- Mean actual y (y_mean) ≈ $87.04
- Mean prediction (pred_mean) ≈ $85.14

Interpretation:

- On the bulk of the data, the model is roughly centered correctly (mean prediction close to mean actual).
- Error levels are moderate relative to the mean, and consistent with our earlier “R2 in dollars is low but R2 in log space is high” observation.

**Tail (True, top 1% cost per service)**

- n = 919 test rows
- MAE ≈ $514.68
- RMSE ≈ $1532.28
- Mean actual y (y_mean) ≈ $885.80
- Mean prediction (pred_mean) ≈ $372.09

Interpretation:

- The model **massively underpredicts** the tail on average.
- The mean prediction is **about 42%** of the mean actual.

A quick calculation we can do mentally:

- Bias in tail mean ≈ $885.8 − $372.1 ≈ **$513.7**, which is basically the MAE.
- That’s a strong sign the dominant error mode in the tail is “systematic underprediction,” not just noisy scatter.

#### Two small follow-up diagnostics that will help us confirm the story

**1. Tail mean ratio and bias**

In [ ]:
tail = tail_summary.loc[True]
ratio = tail["pred_mean"] / tail["y_mean"]
bias = tail["pred_mean"] - tail["y_mean"]
ratio, bias

- **Ratio = 0.4201**
    - On average, in the tail the OLS model predicts only **42%** of the true cost per service.
- **Bias = −$513.71**
    - On average, it’s under by about **$514** per row in the tail.

This aligns almost exactly with the tail MAE we saw (**~$514.68**). That’s not a coincidence. It means the dominant error mode in the tail is a consistent downward bias, not random noise.

**2. Tail share of total squared error (how much tail dominates RMSE)**

In [ ]:
err_share = (
    test_eval.groupby("is_top_1pct_stdzd_amt_per_service")["sq_err"]
    .sum()
    .pipe(lambda s: s / s.sum())
)
err_share

We found:

- **Tail rows (True) account for 88.87% of total squared error**
- **Non-tail rows (False) account for 11.13%**

This is the key takeaway:

Even though the tail is a tiny slice of rows (919 out of 105,026, under 1%), it contributes almost **9 out of every 10 “RMSE dollars”** because squared error explodes when we miss large values.

That means:

- Our **overall RMSE in dollars is basically a tail metric**.
- Improvements that help the bulk (non-tail) may barely move RMSE if the tail remains underpredicted.
- A model can look “fine” on non-tail MAE and still look “terrible” overall due to tail.

**3. Quick sanity check (very informative): This will tell us whether the tail errors are mostly “underpredict” vs “overpredict”:**

In [ ]:
tail_rows = test_eval["is_top_1pct_stdzd_amt_per_service"]
signed_err = (test_eval.loc[tail_rows, "pred"] - test_eval.loc[tail_rows, target_col])

signed_err.describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9, 0.99])

The tail errors are overwhelmingly **systematic underprediction**, with a few extreme misses that dominate RMSE.

**What each line tells us**

A) Median and percentiles confirm “almost always under”

- **50% (median) = −308.69**
- **75% = −145.17**
- **90% = −76.42**
- Even at the 90th percentile, the error is still negative. That means **at least 90% of tail rows are underpredicted**.

A quick inference we can safely state: **Underprediction is the norm, not an occasional issue.**

B) Only a tiny fraction overpredict

- **max = +125.48**
    
    So the worst overprediction in the tail is only +$125, while the worst underprediction is enormous (see below). That asymmetry is telling.

C) The mean matches the bias

- **mean = −513.71**, exactly what we computed before.
    
    So the “tail bias” number is not a fluke. It’s literally the average signed error.

D) RMSE is being crushed by a few catastrophic misses

- **min = −41,766.47**
- **std = 1,444.38**

That one line explains why tail RMSE is so huge. Squared error makes a single −$41k miss count like thousands of “normal” misses.

**4. What share of tail rows are underpredicted?**

In [ ]:
tail = test_eval.loc[test_eval["is_top_1pct_stdzd_amt_per_service"]]
under_rate = (tail["pred"] < tail[target_col]).mean()
under_rate

**Underprediction rate = 99.13% (tail)**

0.9913 means **911 out of 919** tail rows are underpredicted (roughly). So the tail problem is not “high variance”. It is a **systematic downward bias** in the tail regime.

This matches everything we saw earlier:

- tail mean ratio ≈ 0.42
- tail mean bias ≈ −$514
- tail error percentiles mostly negative

**5. How many “catastrophic” misses are driving tail SSE?**

In [ ]:
tail = test_eval.loc[test_eval["is_top_1pct_stdzd_amt_per_service"]].copy()
tail["sq_err"] = (tail["pred"] - tail[target_col])**2

# fraction of tail SSE explained by top k worst rows
for k in [1, 5, 10, 25, 50]:
    share = tail["sq_err"].nlargest(k).sum() / tail["sq_err"].sum()
    print(k, float(share))

**Tail SSE is dominated by a single catastrophic miss**

The SSE concentration is extreme:

- **Top 1 tail row explains 80.85% of tail SSE**
- Top 5 explains 83.64%
- Top 10 explains 84.98%
- Top 50 explains 90.53%

So when we report RMSE in dollars, we are mostly measuring “how bad is the single worst tail miss,” not the typical performance.

That also explains why:

- Dollar R² is low (because SSE is huge from a few points).
- Log-space R² looks strong (because the log compresses the effect of that outlier).

**6. Identify the single worst tail row and inspect it**

In [ ]:
tail = test_eval.loc[test_eval["is_top_1pct_stdzd_amt_per_service"]].copy()
tail["err"] = tail["pred"] - tail[target_col]
tail["abs_err"] = tail["err"].abs()
tail["sq_err"] = tail["err"]**2

worst = tail.sort_values("sq_err", ascending=False).head(1)
worst[["Rndrng_NPI","Year","rbcs_family_desc","Place_Of_Srvc","provider_type","state","ruca_bucket","services",target_col,"pred","err","abs_err"]]

**Worst tail row**

- `rbcs_family_desc` = `Chemotherapeutic Agent`
- `provider_type` = `Radiation Oncology`
- `Place_Of_Srvc` = `O`
- `services` = `53`
- **actual** `stdzd_amt_per_service` = `41,967`
- **predicted** ~`201`
- `err`or `-41,766`

>So the model is behaving like “Chemotherapeutic Agent in this context should cost a few hundred per service,” but the data says “it is forty thousand per service.”

That combination is either:

1. **A real but extremely rare regime** our linear model cannot express from the current feature set (interactions, nonlinearities), or
2. **A coding / mapping / aggregation artifact** (less common, but worth ruling out because the magnitude is so extreme).

Either way, this single point dominating SSE is why our dollar-RMSE looks disastrous while log-space metrics look decent.

Let's check whether it is:

- a weird combination (rare RBCS family + unusual POS + tiny services just above threshold), or
- a data quality oddity (e.g., denominator effect, miscoding), or
- a genuinely extreme but real provider-year outlier.

**7. Quantify “typical tail error” with a robust metric**

In [ ]:
tail = test_eval.loc[test_eval["is_top_1pct_stdzd_amt_per_service"]].copy()
tail["abs_err"] = (tail["pred"] - tail[target_col]).abs()

tail_abs_summary = tail["abs_err"].describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99])
tail_abs_summary

**Our tail abs error summary says**

From our tail `abs_err` distribution:

- Median tail miss: **~$309**
- 90th percentile: **~$962**
- 99th percentile: **~$2,370**
- Max: **~$41,766** (the monster)

So for 99 percent of tail rows, our error is in the hundreds to low thousands. Then one row is off by forty thousand and it blows up RMSE and SSE.

**8. Pull the underlying spend totals and other per-service fields for that exact row**

In [ ]:
row_idx = worst.index[0]

eda_df.loc[row_idx, [
    "Rndrng_NPI","Year","rbcs_family_desc","Place_Of_Srvc","provider_type","state","ruca_bucket",
    "services","benes",
    "stdzd_amt_per_service","stdzd_spend",
    "allowed_amt_per_service","allowed_spend",
    "payment_amt_per_service","payment_spend",
    "submitted_charge_per_service","submitted_spend"
]]

**9. Is this NPI consistently extreme, or is 2023 a one-off spike?**

In [ ]:
npi = worst["Rndrng_NPI"].iloc[0]

eda_df.loc[
    (eda_df["Rndrng_NPI"] == npi) & (eda_df["rbcs_family_desc"] == "Chemotherapeutic Agent"),
    ["Year","services","stdzd_amt_per_service","stdzd_spend","Place_Of_Srvc","provider_type","state"]
].sort_values("Year")

In [ ]:
npi

The “catastrophic miss” is a **real, stable, learnable pattern in the data**, not a one-off glitch.

A) It is not a construction artifact

Our per-service and total fields line up:

- `stdzd_spend` ≈ `services * stdzd_amt_per_service`
    
    `53` * `41,967.405094` ≈ `2,224,272.47` (matches our `stdzd_spend`)
    
- `payment_amt_per_service` == `stdzd_amt_per_service` and `payment_spend` == `stdzd_spend`
    
    So the standardized and payment views are consistent.
    
- `allowed_amt_per_service` is even higher (~52.7k), and `submitted_charge_per_service` is higher still (98k).
    
    That “submitted > allowed > paid/standardized” ordering is typical and also internally consistent.
    

So this is a genuine extremely high-cost provider-year-service bucket.

B) It is not a 2023 spike. It is stable across years for this NPI

Same NPI, same service family, same POS, same provider type, same state:

- 2021: ~41,862 per service
- 2022: ~41,705 per service
- 2023: ~41,967 per service

That stability is exactly what we want to see if this is “real behavior” rather than noise.

**10. How extreme is this row relative to its peer group?**

In [ ]:
peer = train_df.loc[
    (train_df["rbcs_family_desc"] == "Chemotherapeutic Agent") &
    (train_df["provider_type"] == "Radiation Oncology") &
    (train_df["Place_Of_Srvc"] == "O"),
    ["stdzd_amt_per_service","services","state","ruca_bucket"]
]

peer["stdzd_amt_per_service"].describe(percentiles=[0.5,0.9,0.95,0.99])

In [ ]:
peer

The peer group is bimodal. Most rows are cheap, a few are ultra-expensive. 

Our peer group is only 18 rows, and the distribution screams “two regimes”:

- median: **~$39.78 per service**
- 90th percentile: **~$12,540**
- 95th percentile: **~$41,729**
- max: **~$41,863**

So in the exact same coarse slice (Chemotherapeutic Agent + Radiation Oncology + POS=O), there are rows clustered around tens of dollars, and a small number clustered around ~42k.

That also explains why our OLS prediction is ~200. With the current feature set, the model mostly learns the dominant “low-cost mode,” and it has no reliable signal to identify which rows belong to the “ultra-expensive mode.”

The next most informative thing to compute is this, using the `test_eval`:
- For that `peer` slice, compare feature values (the numeric covariates) between the ultra-high rows and the low-cost rows. If they are indistinguishable, then we have strong evidence we need a more granular categorical feature (for example `rbcs_cat_subcat`) or a provider-history feature to capture the regime.

1. Let's create a new dataset called `peer_test_eval` from the `test_eval` where we get `rbcs_family_desc` = `"Chemotherapeutic Agent"`, `provider_type` = `"Radiation Oncology"`, `Place_Of_Srvc` = `"O Agent"`:

**1.A. Let's first make a copy of the sliced `test_eval` dataset:**

In [ ]:
peer_test_eval = test_eval.loc[
    (test_eval["rbcs_family_desc"] == "Chemotherapeutic Agent") &
    (test_eval["provider_type"] == "Radiation Oncology") &
    (test_eval["Place_Of_Srvc"] == "O")
].copy()

**1.B. Let's add a a new column `is_worst` that indicates the rows that belong to `npi` from `worst`.**

In [ ]:
peer_test_eval["is_worst"] = peer_test_eval["Rndrng_NPI"].eq(npi)

2. Let's define “ultra-high” vs “low-cost” within the peer slice

This is usually better than using the global top 1% flag, because our peer slice is already narrow.

In [ ]:
num_cols = num_features  # our list

peer = peer_test_eval.copy()

# Define ultra-high and low-cost within this peer slice
hi_cut = peer[target_col].quantile(0.90)   # top 10% within peer
lo_cut = peer[target_col].quantile(0.50)   # bottom 50% within peer

peer["cost_group"] = np.select(
    [peer[target_col] >= hi_cut, peer[target_col] <= lo_cut],
    ["ultra_high", "low_cost"],
    default="middle"
)

peer["cost_group"].value_counts(dropna=False)

3. Let's compare numeric covariates between groups

This produces a compact “are they distinguishable?” table for numeric features:

3.A. Mean/median comparison table

In [ ]:
compare_groups = peer.loc[peer["cost_group"].isin(["ultra_high", "low_cost"])].copy()

summary = (
    compare_groups
    .groupby("cost_group")[num_cols]
    .agg(["mean", "median", "std"])
)

summary

3.B. Add standardized mean difference (best quick signal)

This gives us a single “effect size” number per feature. If SMD is near 0, the groups are basically indistinguishable on that feature.

In [ ]:
def one_vs_group_z(x, group):
    x = float(x)
    g = np.asarray(group, dtype=float)
    mu = np.nanmean(g)
    sd = np.nanstd(g, ddof=1)  # ok because low_cost has n=4 here
    return (x - mu) / sd if sd > 0 else np.nan

hi = peer.loc[peer["cost_group"] == "ultra_high"]
lo = peer.loc[peer["cost_group"] == "low_cost"]

z_tbl = pd.DataFrame({
    "feature": num_cols,
    "ultra_high_value": [hi[c].iloc[0] for c in num_cols],
    "low_cost_mean":    [lo[c].mean() for c in num_cols],
    "low_cost_sd":      [lo[c].std(ddof=1) for c in num_cols],
    "z_vs_low_cost":    [one_vs_group_z(hi[c].iloc[0], lo[c]) for c in num_cols],
}).sort_values("z_vs_low_cost", key=lambda s: s.abs(), ascending=False)

z_tbl

The key interpretation rule:

- **Negative z**: `ultra_high` value is **below** the `low_cost` mean.
- **Positive z**: `ultra_high` value is **above** the `low_cost` mean.
- **Magnitude**:
    - |z| ≈ 0 to 1: not very different
    - |z| ≈ 2: pretty different
    - |z| ≥ 3: extremely different (especially with only 4 `low_cost` rows, this is a strong signal that this point sits far from that group on that feature)

Now our table:

1) `log_services`: z = -7.37 (huge)

- `ultra_high_value` = 3.988984
- `low_cost_mean` = 9.559847
- `low_cost_sd` = 0.756033
- `z_vs_low_cost` = (3.99 - 9.56) / 0.756 ≈ -7.37

Interpretation:

- Within this peer slice, the `ultra_high` row has **much lower `log_services`** than the `low_cost` rows.
- Since `log_services` is log-transformed, this is a massive difference on the original services scale.
- This is a red flag that the `ultra_high` cost-per-service case might be associated with a very different volume regime (even though our `services` column for the worst row was `53`, the `low_cost` rows in this peer slice likely have much higher services if their `log_services` mean is `9.56`, which is extremely large). That suggests we should sanity-check how `log_services` was defined in this dataset.

This single row is telling us: “I’m expensive per service, but I do not have high service volume relative to these `low_cost` rows.”

2) `bene_avg_risk_score`: z = -3.57

- `ultra_high` row’s beneficiaries are **lower risk** than `low_cost` mean by ~3.6 SDs.
- If this holds up, it suggests the extreme cost-per-service is not explained by higher risk score, at least not relative to these low-cost peers.

3) `p_copd`: z = -2.22 and `p_ckd`: z = -1.93

- `ultra_high` row has **lower COPD and CKD prevalence** than `low_cost` peers, relative to the `low_cost` variation.
- Again, this pushes against “this is just sicker patients” as the explanation.

4) `p_cancer6`: z = +1.54

- `ultra_high` has somewhat higher cancer prevalence than `low_cost`, but only ~1.5 SD.
- Not nothing, but not nearly as extreme as the service-volume signal.

5) `log_benes`: z = -1.21

- `ultra_high` row has fewer beneficiaries (or whatever `log_benes` captures) than `low_cost` peers, by ~1.2 SD.

6) `p_htn`, `p_diabetes`, `years_since_enumeration`: z near 0

- These look basically similar between `ultra_high` and `low_cost` within this peer slice.

Within that very narrow peer slice, the ultra-high cost-per-service row is not “high” because the numeric covariates scream “complex population.” Instead it looks like:

- **Lower volume signals (`log_services`, `log_benes`)**
- Some comorbidity rates are lower, not higher
- Cancer prevalence is a bit higher, but not enough to explain a 40k per service situation

That supports the hypothesis we mentioned earlier: **our feature set cannot represent the regime that creates ultra-high per-service costs**, because it is likely driven by something categorical or structural we are not encoding at the right granularity (or by a special pricing/HCPCS subcategory, drug, setting nuance, etc.).

- Just for sanity check, let's look at the `services`, `log_services`, `stdzd_amt_per_service` columns of the `peer` dataframe and calculate services from `log_services` column named `services_from_log`:

In [ ]:
peer.loc[peer["cost_group"] == "low_cost", ["services", "log_services", target_col]] \
    .assign(services_from_log=lambda d: np.expm1(d["log_services"])) \
    .sort_values("services", ascending=False)

#### OLS performance and diagnostics summary:

- Dollar space (business impact): MAE, RMSE, dollar R²
- Log space (relative error, stability): MAE/RMSE on log1p(y) and log R²

In [ ]:
pred_test = ols_full_model.predict(X_test)
pred_test_clip = pred_test.clip(min=0)

mae = mean_absolute_error(y_test, pred_test)
rmse = root_mean_squared_error(y_test, pred_test)

log_pred_test = ols_full_model.regressor_.predict(X_test)  # predictions in transformed target space
r2_log_true = r2_score(np.log1p(y_test), log_pred_test)
mae_log_true = mean_absolute_error(np.log1p(y_test), log_pred_test)
rmse_log_true = root_mean_squared_error(np.log1p(y_test), log_pred_test)

print("OLS (Raw y + TTR(log1p))")
print("Train R2 ($):", ols_full_model.score(X_train, y_train))
print("Test  R2 ($):", ols_full_model.score(X_test, y_test))
print("MAE ($):", mae)
print("RMSE ($):", rmse)
print("Test  R2 (log $):", r2_log_true)
print("MAE (log $):", r2_log_true)
print("RMSE (log $):", rmse_log_true)

Let's prettify the prints:

In [ ]:
print("OLS (Raw y + TTR(log1p))")
print(f"Train R2 ($):     {ols_full_model.score(X_train, y_train):.3f}")
print(f"Test  R2 ($):     {ols_full_model.score(X_test, y_test):.3f}")
print(f"MAE ($):          {mae:.3f}")
print(f"RMSE ($):         {rmse:.3f}")
print(f"Test  R2 (log $): {r2_log_true:.3f}")
print(f"MAE (log $):      {r2_log_true:.3f}")
print(f"RMSE (log $):     {rmse_log_true:.3f}")

- Tail calibration: for `is_top_1pct_stdzd_amt_per_service` rows, track:
    - pred_mean / y_mean ratio
    - underprediction rate
    - tail share of SSE

In [ ]:
tail_summary_1 = tail_summary.copy()
tail_summary_1["pred_to_y_ratio"] = tail_summary_1["pred_mean"] / tail_summary_1["y_mean"]

In [ ]:
tail_summary_2 = (test_eval
 .assign(is_under_predicted = lambda d: d["pred"]<d[target_col])
 .groupby("is_top_1pct_stdzd_amt_per_service")
 .agg(under_prediction_rate = ("is_under_predicted", "mean"),
      sse = ("sq_err","sum"))
 .assign(share_of_total_sse = lambda d: d["sse"]/d["sse"].sum()))
tail_summary_2

In [ ]:
eval_tail = pd.concat([tail_summary_1,tail_summary_2], axis=1)
eval_tail["bias"] = eval_tail["pred_mean"] - eval_tail["y_mean"]
eval_tail

Let's prettify the output of `eval_tail`:

In [ ]:
eval_tail_pretty = eval_tail.round(3)
eval_tail_pretty

## Ridge

In [ ]:
from sklearn.linear_model import Ridge

ridge_inner_pipe = Pipeline(steps=[
    ("preprocess", preprocess_ohe),
    ("scaler", StandardScaler(with_mean=False)),
    ("model", Ridge())
])

In [ ]:
ridge_full_model = TransformedTargetRegressor(
    regressor=ridge_inner_pipe,
    func=np.log1p,
    inverse_func=np.expm1
)

In [ ]:
ridge_full_model.fit(X_train,y_train)

In [ ]:
pred_test = ridge_full_model.predict(X_test)
pred_test_clip = pred_test.clip(min=0)

mae = mean_absolute_error(y_test, pred_test)
rmse = root_mean_squared_error(y_test, pred_test)

log_pred_test = ridge_full_model.regressor_.predict(X_test)  # predictions in transformed target space
r2_log_true = r2_score(np.log1p(y_test), log_pred_test)
mae_log_true = mean_absolute_error(np.log1p(y_test), log_pred_test)
rmse_log_true = root_mean_squared_error(np.log1p(y_test), log_pred_test)

print("Ridge (Raw y + TTR(log1p))")
print("Train R2 ($):", ridge_full_model.score(X_train, y_train))
print("Test  R2 ($):", ridge_full_model.score(X_test, y_test))
print("MAE ($):", mae)
print("RMSE ($):", rmse)
print("Test  R2 (log $):", r2_log_true)
print("MAE (log $):", mae_log_true)
print("RMSE (log $):", rmse_log_true)

## XGBoost

In [ ]:
from xgboost import XGBRegressor

xgb_inner_pipe = Pipeline(steps=[
    ("preprocess", preprocess_ohe),
    ("model", XGBRegressor(
        objective="reg:squarederror",
        tree_method="hist",
        n_jobs=1,
        random_state=0
    ))
])

In [ ]:
xgb_full_model = TransformedTargetRegressor(
    regressor=xgb_inner_pipe,
    func=np.log1p,
    inverse_func=np.expm1
)

In [ ]:
xgb_full_model.fit(X_train,y_train)

In [ ]:
pred_test = xgb_full_model.predict(X_test)
pred_test_clip = pred_test.clip(min=0)

mae = mean_absolute_error(y_test, pred_test)
rmse = root_mean_squared_error(y_test, pred_test)

log_pred_test = xgb_full_model.regressor_.predict(X_test)  # predictions in transformed target space
r2_log_true = r2_score(np.log1p(y_test), log_pred_test)
mae_log_true = mean_absolute_error(np.log1p(y_test), log_pred_test)
rmse_log_true = root_mean_squared_error(np.log1p(y_test), log_pred_test)

print("XGBoost (Raw y + TTR(log1p))")
print("Train R2 ($):", xgb_full_model.score(X_train, y_train))
print("Test  R2 ($):", xgb_full_model.score(X_test, y_test))
print("MAE ($):", mae)
print("RMSE ($):", rmse)
print("Test  R2 (log $):", r2_log_true)
print("MAE (log $):", mae_log_true)
print("RMSE (log $):", rmse_log_true)

In [ ]:
pred_test = xgb_full_model.predict(X_test)

test_eval = test_df.copy()
test_eval["pred"] = pred_test
test_eval["abs_err"] = (test_eval[target_col] - test_eval["pred"]).abs()
test_eval["sq_err"]  = (test_eval[target_col] - test_eval["pred"])**2

# Bring back tail flag (and optionally svc_bucket) from eda_df using the shared index
test_eval = test_eval.join(
    eda_df.loc[:, ["is_top_1pct_stdzd_amt_per_service", "svc_bucket"]],
    how="left"
)

tail_summary = (
    test_eval.groupby("is_top_1pct_stdzd_amt_per_service", dropna=False)
    .agg(
        n=("pred", "size"),
        mae=("abs_err", "mean"),
        rmse=("sq_err", lambda s: np.sqrt(s.mean())),
        y_mean=(target_col, "mean"),
        pred_mean=("pred", "mean"),
    )
)

tail_summary

In [ ]:
tail_summary_1 = tail_summary.copy()
tail_summary_1["pred_to_y_ratio"] = tail_summary_1["pred_mean"] / tail_summary_1["y_mean"]

In [ ]:
tail_summary_2 = (test_eval
 .assign(is_under_predicted = lambda d: d["pred"]<d[target_col])
 .groupby("is_top_1pct_stdzd_amt_per_service")
 .agg(under_prediction_rate = ("is_under_predicted", "mean"),
      sse = ("sq_err","sum"))
 .assign(share_of_total_sse = lambda d: d["sse"]/d["sse"].sum()))
tail_summary_2

In [ ]:
eval_tail = pd.concat([tail_summary_1,tail_summary_2], axis=1)
eval_tail["bias"] = eval_tail["pred_mean"] - eval_tail["y_mean"]
eval_tail

## **Increase granularity of the train and test dataset**

### Redefine `cat_features`

In [ ]:
# Categorical features (to encode)
cat_features = [
    "rbcs_cat_subcat",    # replaced "rbcs_family_desc" for more granularity
    "Place_Of_Srvc",
    "provider_type",
    "state",
    "ruca_bucket",
]

# BELOW STAYS THE SAME AS BEFORE

# # Numeric features
# num_features = [
#     "bene_avg_risk_score",
#     "years_since_enumeration",
#     "log_services",
#     "log_benes",
#     "p_cancer6", "p_diabetes", "p_ckd", "p_copd", "p_htn",
# ]

# # Final target
# target_col = "stdzd_amt_per_service"

# # Exclusions (documented)
# excluded = [
#     # raw cost outcomes besides target
#     "log_stdzd_amt_per_service", "allowed_amt_per_service", "payment_amt_per_service", "submitted_charge_per_service",
#     # totals derived from outcomes (spend columns)
#     "stdzd_spend", "allowed_spend", "payment_spend", "submitted_spend",
#     # flags/buckets used for slicing, not for training features
#     "is_top_1pct_stdzd_amt_per_service", "svc_bucket", "services_bins", "services_custom", "services_custom2",
#     # provider-year totals (often avoided to prevent scale leakage; can revisit intentionally later)
#     "tot_mdcr_stdzd_amt",
# ]

# features_table = pd.DataFrame({
#     "Type": (["categorical"] * len(cat_features)) + (["numeric"] * len(num_features)) + (["target"] * 1),
#     "Column": cat_features + num_features + [target_col]
# })

# features_table

In [ ]:
# Reprint the features_table as sanity-check
features_table = pd.DataFrame({
    "Type": (["categorical"] * len(cat_features)) + (["numeric"] * len(num_features)) + (["target"] * 1),
    "Column": cat_features + num_features + [target_col]
})

features_table

#### Recreate dataframes for modeling

In [ ]:
model_cols = cat_features + num_features + [target_col, "Year", "Rndrng_NPI", "services", "log_stdzd_amt_per_service"]

model_df = eda_df.loc[train_mask, model_cols].copy()

train_df = model_df[model_df["Year"].isin(train_years)].copy()
test_df = model_df[model_df["Year"].isin(test_years)].copy()

train_df.shape, test_df.shape

#### Rebuild X/y

In [ ]:
# Target is log target (your modeling target)
y_train = train_df[target_col].copy()
y_test  = test_df[target_col].copy()

# Keep these for analysis, but not as model predictors
id_cols = ["Rndrng_NPI", "Year"]

# Also exclude raw cost outcome (you do not want leakage)
exclude_from_X = [target_col, "log_stdzd_amt_per_service"] + id_cols

X_train = train_df.drop(columns=exclude_from_X).copy()
X_test  = test_df.drop(columns=exclude_from_X).copy()

#### Redefine the `preprocess` and `preprocess_ohe` (so they start with clean slate; nothing fitted them)

In [ ]:
cat_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="None"))
])

cat_pipe_ohe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="None")),
    ("ohe", OneHotEncoder(handle_unknown="ignore", drop="first"))
])

num_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median", add_indicator=True))
])

preprocess = ColumnTransformer(
    transformers=[
        ("cat", cat_pipe, cat_features),
        ("num", num_pipe, num_features),
    ],
    remainder="drop"
)

preprocess_ohe = ColumnTransformer(
    transformers=[
        ("cat_ohe", cat_pipe_ohe, cat_features),
        ("num", num_pipe, num_features),
    ],
    remainder="drop"
)

#### Re fit-transform train and transform test using `preprocess` and `preprocess_ohe` (as defined before)

In [ ]:
X_train_proc      = preprocess.fit_transform(X_train)
X_test_proc       = preprocess.transform(X_test)

X_train_proc_ohe  = preprocess_ohe.fit_transform(X_train)
X_test_proc_ohe   = preprocess_ohe.transform(X_test)

feat_names_ohe = preprocess_ohe.get_feature_names_out()
print(feat_names_ohe[:10])

feat_names = preprocess.get_feature_names_out()
print(feat_names)

#### Introducing weights to the train data

> Here, we have to ensure we are giving higher weights to the actual tail rows because we want the model to care more about those data points in the tail. They are NOT OUTLIERS. If they were, we would give them lower weight to tell the model to not care about them as much. 

In [ ]:
is_tail_train = eda_df.loc[X_train.index, "is_top_1pct_stdzd_amt_per_service"].astype(int)
w_tail = 25
sample_w = 1 + (w_tail - 1) * is_tail_train  # non-tail=1, tail=w_tail

#### Rebuild the `xgb_inner_pipe`

In [ ]:
xgb_inner_pipe = Pipeline(steps=[
    ("preprocess", preprocess_ohe),
    ("model", XGBRegressor(
        objective="reg:squarederror",
        tree_method="hist",
        n_jobs=1,
        random_state=0
    ))
])

#### Rebuild `xgb_full_model` with updated `xgb_inner_pipe`:

In [ ]:
xgb_full_model = TransformedTargetRegressor(
    regressor=xgb_inner_pipe,
    func=np.log1p,
    inverse_func=np.expm1
)

#### Fit the model with `model__sample_weight = sample_w`

In [ ]:
xgb_full_model.fit(
    X_train, y_train, model__sample_weight=sample_w
)

#### Redo model diagnostics

In [ ]:
pred_test = xgb_full_model.predict(X_test)
pred_train = xgb_full_model.predict(X_train)
pred_test_clip = pred_test.clip(min=0)

mae = mean_absolute_error(y_test, pred_test)
rmse = root_mean_squared_error(y_test, pred_test)

log_pred_train = xgb_full_model.regressor_.predict(X_train)  # predictions in transformed target space
log_pred_test = xgb_full_model.regressor_.predict(X_test)  # predictions in transformed target space

r2_log_true_train = r2_score(np.log1p(y_train), log_pred_train)
r2_log_true_test = r2_score(np.log1p(y_test), log_pred_test)

mae_log_true = mean_absolute_error(np.log1p(y_test), log_pred_test)
rmse_log_true = root_mean_squared_error(np.log1p(y_test), log_pred_test)

# Create weights for the test set (following same logic as train)
is_tail_test = eda_df.loc[X_test.index, "is_top_1pct_stdzd_amt_per_service"].astype(int)
sample_w_test = 1 + (w_tail - 1) * is_tail_test

r2_train_w = r2_score(y_train, pred_train, sample_weight=sample_w)
r2_test_w = r2_score(y_test, pred_test, sample_weight=sample_w_test)
mae_w = mean_absolute_error(y_test, pred_test, sample_weight=sample_w_test)
rmse_w = root_mean_squared_error(y_test, pred_test, sample_weight=sample_w_test)

print("XGBoost (Raw y + TTR(log1p))")
print(50*"-")
print("Train R2 ($) (Unweighted: Average Case):", xgb_full_model.score(X_train, y_train))
print("Test  R2 ($) (Unweighted: Average Case):", xgb_full_model.score(X_test, y_test))
print("MAE      ($) (Unweighted: Average Case):", mae)
print("RMSE     ($) (Unweighted: Average Case):", rmse)
print(50*"-")
print("Train R2 (log $) (Unweighted: Average Case):", r2_log_true_train)
print("Test  R2 (log $) (Unweighted: Average Case):", r2_log_true_test)
print("MAE      (log $) (Unweighted: Average Case):", mae_log_true)
print("RMSE     (log $) (Unweighted: Average Case):", rmse_log_true)
print(50*"-")
print("Train R2 ($) (Weighted: Eval Metrics):", r2_train_w)
print("Test  R2 ($) (Weighted: Eval Metrics):", r2_test_w)
print("MAE      ($) (Weighted: Eval Metrics):", mae_w)
print("RMSE     ($) (Weighted: Eval Metrics):", rmse_w)


In [ ]:
pred_test = xgb_full_model.predict(X_test)

test_eval = test_df.copy()
test_eval["pred"] = pred_test
test_eval["abs_err"] = (test_eval[target_col] - test_eval["pred"]).abs()
test_eval["sq_err"]  = (test_eval[target_col] - test_eval["pred"])**2

# Bring back tail flag (and optionally svc_bucket) from eda_df using the shared index
test_eval = test_eval.join(
    eda_df.loc[:, ["is_top_1pct_stdzd_amt_per_service", "svc_bucket"]],
    how="left"
)

tail_summary = (
    test_eval.groupby("is_top_1pct_stdzd_amt_per_service", dropna=False)
    .agg(
        n=("pred", "size"),
        mae=("abs_err", "mean"),
        rmse=("sq_err", lambda s: np.sqrt(s.mean())),
        y_mean=(target_col, "mean"),
        pred_mean=("pred", "mean"),
    )
)

tail_summary_1 = tail_summary.copy()
tail_summary_1["pred_to_y_ratio"] = tail_summary_1["pred_mean"] / tail_summary_1["y_mean"]

tail_summary_2 = (test_eval
 .assign(is_under_predicted = lambda d: d["pred"]<d[target_col])
 .groupby("is_top_1pct_stdzd_amt_per_service")
 .agg(under_prediction_rate = ("is_under_predicted", "mean"),
      sse = ("sq_err","sum"))
 .assign(share_of_total_sse = lambda d: d["sse"]/d["sse"].sum()))

eval_tail = pd.concat([tail_summary_1,tail_summary_2], axis=1)
eval_tail["bias"] = eval_tail["pred_mean"] - eval_tail["y_mean"]
eval_tail

#### Fit the model WITHOUT `model__sample_weight = sample_w` (UNWEIGHTED BASELINE)

In [ ]:
xgb_inner_pipe = Pipeline(steps=[
    ("preprocess", preprocess_ohe),
    ("model", XGBRegressor(
        objective="reg:squarederror",
        tree_method="hist",
        n_jobs=1,
        random_state=0
    ))
])

xgb_full_model = TransformedTargetRegressor(
    regressor=xgb_inner_pipe,
    func=np.log1p,
    inverse_func=np.expm1
)

xgb_full_model.fit(
    X_train, y_train
)

In [ ]:
pred_test = xgb_full_model.predict(X_test)
pred_train = xgb_full_model.predict(X_train)
pred_test_clip = pred_test.clip(min=0)

mae = mean_absolute_error(y_test, pred_test)
rmse = root_mean_squared_error(y_test, pred_test)

log_pred_train = xgb_full_model.regressor_.predict(X_train)  # predictions in transformed target space
log_pred_test = xgb_full_model.regressor_.predict(X_test)  # predictions in transformed target space

r2_log_true_train = r2_score(np.log1p(y_train), log_pred_train)
r2_log_true_test = r2_score(np.log1p(y_test), log_pred_test)

mae_log_true = mean_absolute_error(np.log1p(y_test), log_pred_test)
rmse_log_true = root_mean_squared_error(np.log1p(y_test), log_pred_test)

print("XGBoost (Raw y + TTR(log1p))")
print(50*"-")
print("Train R2 ($) (Unweighted: Average Case):", xgb_full_model.score(X_train, y_train))
print("Test  R2 ($) (Unweighted: Average Case):", xgb_full_model.score(X_test, y_test))
print("MAE      ($) (Unweighted: Average Case):", mae)
print("RMSE     ($) (Unweighted: Average Case):", rmse)
print(50*"-")
print("Train R2 (log $) (Unweighted: Average Case):", r2_log_true_train)
print("Test  R2 (log $) (Unweighted: Average Case):", r2_log_true_test)
print("MAE      (log $) (Unweighted: Average Case):", mae_log_true)
print("RMSE     (log $) (Unweighted: Average Case):", rmse_log_true)


In [ ]:
pred_test = xgb_full_model.predict(X_test)

test_eval = test_df.copy()
test_eval["pred"] = pred_test
test_eval["abs_err"] = (test_eval[target_col] - test_eval["pred"]).abs()
test_eval["sq_err"]  = (test_eval[target_col] - test_eval["pred"])**2

# Bring back tail flag (and optionally svc_bucket) from eda_df using the shared index
test_eval = test_eval.join(
    eda_df.loc[:, ["is_top_1pct_stdzd_amt_per_service", "svc_bucket"]],
    how="left"
)

tail_summary = (
    test_eval.groupby("is_top_1pct_stdzd_amt_per_service", dropna=False)
    .agg(
        n=("pred", "size"),
        mae=("abs_err", "mean"),
        rmse=("sq_err", lambda s: np.sqrt(s.mean())),
        y_mean=(target_col, "mean"),
        pred_mean=("pred", "mean"),
    )
)

tail_summary_1 = tail_summary.copy()
tail_summary_1["pred_to_y_ratio"] = tail_summary_1["pred_mean"] / tail_summary_1["y_mean"]

tail_summary_2 = (test_eval
 .assign(is_under_predicted = lambda d: d["pred"]<d[target_col])
 .groupby("is_top_1pct_stdzd_amt_per_service")
 .agg(under_prediction_rate = ("is_under_predicted", "mean"),
      sse = ("sq_err","sum"))
 .assign(share_of_total_sse = lambda d: d["sse"]/d["sse"].sum()))

eval_tail = pd.concat([tail_summary_1,tail_summary_2], axis=1)
eval_tail["bias"] = eval_tail["pred_mean"] - eval_tail["y_mean"]
eval_tail

In [ ]:
weights = [1,2,5,10,15,25]
r2_train_lst = []
r2_test_lst = []
mae_lst = []
rmse_lst = []
r2_train_log_lst = []
r2_test_log_lst = []
mae_log_lst = []
rmse_log_lst = []
r2_train_w_lst = []
r2_test_w_lst = []
mae_w_lst = []
rmse_w_lst = []
non_tail_pred_to_y_ratio_lst = []
tail_pred_to_y_ratio_lst = []
non_tail_sse_lst = []
tail_sse_lst = []
non_tail_share_of_total_sse_lst = []
tail_share_of_total_sse_lst = []
non_tail_under_prediction_rate_lst = []
tail_under_prediction_rate_lst = []
non_tail_bias_lst = []
tail_bias_lst = []

for w in weights:

    # define the training weights 
    is_tail_train = eda_df.loc[X_train.index, "is_top_1pct_stdzd_amt_per_service"].astype(int)
    w_tail = w
    sample_w = 1 + (w_tail - 1) * is_tail_train  # non-tail=1, tail=w_tail

    # define the inner pipe
    xgb_inner_pipe = Pipeline(steps=[
    ("preprocess", preprocess_ohe),
    ("model", XGBRegressor(objective="reg:squarederror", tree_method = "hist", n_jobs = 1, random_state = 0))
    ])

    # define the full model 
    xgb_full_model = TransformedTargetRegressor(
    regressor=xgb_inner_pipe,
    func=np.log1p,
    inverse_func=np.expm1
    )

    # Fit the model
    xgb_full_model.fit(
    X_train, y_train, model__sample_weight=sample_w
    )

    # Run diagnostics

    # get train and test predictions and clip test predictions 
    pred_test = xgb_full_model.predict(X_test)
    pred_train = xgb_full_model.predict(X_train)
    pred_test_clip = pred_test.clip(min=0)

    # get weighted-train unweighted/average-case metrics
    r2_train = xgb_full_model.score(X_train, y_train)
    r2_test = xgb_full_model.score(X_test, y_test)
    mae = mean_absolute_error(y_test, pred_test)
    rmse = root_mean_squared_error(y_test, pred_test)

    # get weighted-train unweighted/average-case metrics in log space
    log_pred_train = xgb_full_model.regressor_.predict(X_train)  # predictions in transformed target space
    log_pred_test = xgb_full_model.regressor_.predict(X_test)  # predictions in transformed target space
    
    r2_log_true_train = r2_score(np.log1p(y_train), log_pred_train)
    r2_log_true_test = r2_score(np.log1p(y_test), log_pred_test)
    mae_log_true = mean_absolute_error(np.log1p(y_test), log_pred_test)
    rmse_log_true = root_mean_squared_error(np.log1p(y_test), log_pred_test)

    # get weighted-train weighted/eval metrics
    
    # Create weights for the test set (following same logic as train)
    is_tail_test = eda_df.loc[X_test.index, "is_top_1pct_stdzd_amt_per_service"].astype(int)
    sample_w_test = 1 + (w_tail - 1) * is_tail_test

    r2_train_w = r2_score(y_train, pred_train, sample_weight=sample_w)
    r2_test_w = r2_score(y_test, pred_test, sample_weight=sample_w_test)
    mae_w = mean_absolute_error(y_test, pred_test, sample_weight=sample_w_test)
    rmse_w = root_mean_squared_error(y_test, pred_test, sample_weight=sample_w_test)

    # Calculate tail evals
    test_eval = test_df.copy().join(
    eda_df.loc[:, ["is_top_1pct_stdzd_amt_per_service"]],
    how="left"
    )
    test_eval["pred"] = pred_test
    test_eval["abs_err"] = (test_eval[target_col] - test_eval["pred"]).abs()
    test_eval["sq_err"]  = (test_eval[target_col] - test_eval["pred"])**2

    non_tail_pred_to_y_ratio = test_eval.loc[~test_eval["is_top_1pct_stdzd_amt_per_service"],"pred"].mean() / test_eval.loc[~test_eval["is_top_1pct_stdzd_amt_per_service"],target_col].mean()
    tail_pred_to_y_ratio = test_eval.loc[test_eval["is_top_1pct_stdzd_amt_per_service"],"pred"].mean() / test_eval.loc[test_eval["is_top_1pct_stdzd_amt_per_service"],target_col].mean()

    non_tail_sse = test_eval.loc[~test_eval["is_top_1pct_stdzd_amt_per_service"],"sq_err"].sum()
    tail_sse = test_eval.loc[test_eval["is_top_1pct_stdzd_amt_per_service"],"sq_err"].sum()

    non_tail_share_of_total_sse = non_tail_sse / (non_tail_sse + tail_sse)
    tail_share_of_total_sse = tail_sse / (non_tail_sse + tail_sse)

    non_tail_under_prediction_rate = ((test_eval.loc[~test_eval["is_top_1pct_stdzd_amt_per_service"],"pred"]) < test_eval.loc[~test_eval["is_top_1pct_stdzd_amt_per_service"],target_col]).mean()
    tail_under_prediction_rate = ((test_eval.loc[test_eval["is_top_1pct_stdzd_amt_per_service"],"pred"]) < test_eval.loc[test_eval["is_top_1pct_stdzd_amt_per_service"],target_col]).mean()

    non_tail_bias = test_eval.loc[~test_eval["is_top_1pct_stdzd_amt_per_service"],"pred"].mean() - test_eval.loc[~test_eval["is_top_1pct_stdzd_amt_per_service"],target_col].mean()
    tail_bias = test_eval.loc[test_eval["is_top_1pct_stdzd_amt_per_service"],"pred"].mean() - test_eval.loc[test_eval["is_top_1pct_stdzd_amt_per_service"],target_col].mean()


    r2_train_lst.append(r2_train)
    r2_test_lst.append(r2_test)
    mae_lst.append(mae)
    rmse_lst.append(rmse)
    r2_train_log_lst.append(r2_log_true_train)
    r2_test_log_lst.append(r2_log_true_test)
    mae_log_lst.append(mae_log_true)
    rmse_log_lst.append(rmse_log_true)
    r2_train_w_lst.append(r2_train_w)
    r2_test_w_lst.append(r2_test_w)
    mae_w_lst.append(mae_w)
    rmse_w_lst.append(rmse_w)
    non_tail_pred_to_y_ratio_lst.append(non_tail_pred_to_y_ratio)
    tail_pred_to_y_ratio_lst.append(tail_pred_to_y_ratio)
    non_tail_sse_lst.append(non_tail_sse)
    tail_sse_lst.append(tail_sse)
    non_tail_share_of_total_sse_lst.append(non_tail_share_of_total_sse)
    tail_share_of_total_sse_lst.append(tail_share_of_total_sse)
    non_tail_under_prediction_rate_lst.append(non_tail_under_prediction_rate)
    tail_under_prediction_rate_lst.append(tail_under_prediction_rate)
    non_tail_bias_lst.append(non_tail_bias)
    tail_bias_lst.append(tail_bias)


In [ ]:

weighted_train_performance_metrics_test_eval = pd.DataFrame({
    "train_sample_weights":[1,2,5,10,15,25],
    "r2_train": r2_train_lst,
    "r2_test": r2_test_lst,
    "mae": mae_lst,
    "rmse": rmse_lst,
    "r2_train_log": r2_train_log_lst,
    "r2_test_log": r2_test_log_lst,
    "mae_log": mae_log_lst,
    "rmse_log": rmse_log_lst,
    "r2_train_w": r2_train_w_lst,
    "r2_test_w": r2_test_w_lst,
    "mae_w": mae_w_lst,
    "rmse_w": rmse_w_lst,
    "non_tail_pred_to_y_ratio": non_tail_pred_to_y_ratio_lst,
    "tail_pred_to_y_ratio": tail_pred_to_y_ratio_lst,
    "non_tail_sse": non_tail_sse_lst,
    "tail_sse": tail_sse_lst,
    "non_tail_share_of_total_sse": non_tail_share_of_total_sse_lst,
    "tail_share_of_total_sse": tail_share_of_total_sse_lst,
    "non_tail_under_prediction_rate": non_tail_under_prediction_rate_lst,
    "tail_under_prediction_rate": tail_under_prediction_rate_lst,
    "non_tail_bias": non_tail_bias_lst,
    "tail_bias": tail_bias_lst
})

weighted_train_performance_metrics_test_eval

The `weighted_train_performance_metrics_test_eval` table shows a very clean tradeoff curve between “average-case performance” and “tail calibration”.

A. As we increase `w_tail`, the model shifts attention toward the tail

You can see this in 4 tail-specific columns that move in the “right” direction as `w_tail` increases:

Tail calibration improves
•	`tail_pred_to_y_ratio`: `0.518` → `0.776` (we go from predicting ~52% of tail mean to ~78% of tail mean)`
•	`tail_bias`: `-427` → `-199` (tail underprediction shrinks by ~$228 on average)

Tail underprediction rate improves
•	tail_under_prediction_rate: `0.904` → `0.705`
•	Still underpredicting most tail rows, but much less extreme.

Tail dominance over `sse` improves
•	`tail_share_of_total_sse`: `0.914` → `0.775`
•	Tail still dominates total squared error, but less so.

This is exactly what tail upweighting is supposed to do.

⸻

B. But average-case performance gets worse when you upweight too hard

Look at the “standard metrics”:
•	`r2_test` peaks at `w=10`:
•	`w=1`: `0.257`
•	`w=2`: `0.276`
•	`w=5`: `0.272`
•	`w=10`: `0.290` (best)
•	`w=15`: `0.254`
•	`w=25`: `0.213`
•	`mae` and `rmse` worsen for large weights:
•	RMSE: `147.97` (`w=1`) → `152.26` (`w=25`)
•	MAE: `28.33` (`w=1`) → `33.97` (`w=25`)

This is the classic tradeoff: once you force the model to chase rare, high-cost points, it starts sacrificing fit for the majority.

⸻

C. Non-tail bias flips sign as weight grows

This is a really important diagnostic: `non_tail_bias` makes it obvious:
•	`non_tail_bias`: `-7.26` → `+1.30`
•	At low weight, you slightly underpredict non-tail on average.
•	By `w=25`, you overpredict non-tail on average.

This also matches:
•	`non_tail_pred_to_y_ratio`: `0.917` → `1.015`

So the model is “lifting” predictions overall to reduce tail underprediction, and that causes slight overprediction for the bulk.

⸻

D. The “weighted evaluation” metrics are not what we should optimize here

`r2_test_w`, `mae_w`, `rmse_w` will often look ugly when tail points are huge, because once we weight them 10x or 25x, our evaluation function is basically saying:

“I mostly care about the tail, and tail errors are still enormous.”

So it is totally normal that:
•	`rmse_w` explodes as `w` increases.

Those weighted metrics are still useful, but mainly as a “tail pain index”, not as the primary model selection metric.

⸻

2) What’s the best`w_tail` from this run?

It depends on what we want to optimize.

If our goal is best overall predictive accuracy (business average case)

Pick `w=10`.
•	Best `r2_test` (`0.290`)
•	Best `rmse` among weighted runs (`144.64`)
•	Still meaningful tail improvements:
•	`tail_bias`: `-246` (vs `-427` baseline)
•	`tail_pred_to_y_ratio`: `0.723` (vs `0.518`)
•	`tail_share_of_total_sse`: `0.841` (vs `0.914`)

If our goal is “improve tail calibration as much as possible without going insane”

Pick `w=15`.
•	Tail bias improves further (`-216`)
•	Tail ratio improves (`0.756`)
•	But overall test R2 drops (`0.254`)

If our goal is “tail calibration first, accept average-case pain”

Pick `w=25`.
•	Best tail calibration in our grid:
•	tail ratio `0.776`
•	tail bias `-199`
•	tail underprediction rate `0.705`
•	But overall metrics deteriorate noticeably, and non-tail bias flips positive.

⸻

3) Why weighting alone still cannot fix the tail

Even at `w=25`, tail bias is still `-199` and tail SSE is still `77.5%` of total SSE.

That strongly suggests: the model still lacks features that separate the high-cost tail regime, so it can only “lift” predictions globally.

This matches what we discovered earlier: ultra-high cost per service lines are legitimate but extremely rare. Without regime-identifying predictors, the best the model can do is compromise.

⸻

4) Next steps that are most likely to actually move the needle

Step 2. Keep weighting (probably w=10 or w=15), but stop using default XGBoost settings

Right now we are comparing different training weight schemes, but our model is still basically “stock XGB”.

Do a small hyperparameter search with a fixed w_tail (start with 10). Focus on parameters that affect generalization and tail handling:
•	`n_estimators` (try 500–3000)
•	`learning_rate` (0.02–0.1)
•	`max_depth` (3–8)
•	`min_child_weight` (1–20)
•	`subsample`, `colsample_bytree` (0.6–1.0)
•	`reg_alpha`, `reg_lambda` (L1/L2 regularization)

If we do this, we will usually get much better log-space fit and often slightly better dollar performance, even before feature work.

⸻

Step 3. Change the loss to something better for heavy tails

Right now we are effectively doing squared error in log space (because TTR transforms y then we fit `reg:squarederror`).

Try an objective that is more robust to extreme residuals:
•	`objective="reg:pseudohubererror"` (great for “few catastrophic misses”)
•	Or try `objective="reg:gamma"` (only for strictly positive targets, which we have)
•	Or `objective:"reg:tweedie"` (often good for skewed positive outcomes)

These objectives can reduce the incentive to massively underpredict rare huge values.

⸻

Step 4. Add “regime” features that we can use at inference time

***This is the biggest lever.***

We already saw that ultra-high is not explained by comorbidity or risk score signals. That screams “missing categorical granularity or provider-history”.

Most effective additions (and all can be done without leakage if we do them carefully):

A) Provider-history features (lagged, prior-years only)
For each `Rndrng_NPI` (and optionally within `rbcs_cat_subcat`), compute on *TRAIN years only*:
•	prior-year mean of `stdzd_amt_per_service`
•	trailing mean over years
•	trailing percentile rank of provider within category
•	log of provider total services (prior year)

Then merge those into train/test by NPI-year with proper lagging.

This will likely capture “this provider is consistently in the expensive regime”.

B) Interaction features
Tree models learn interactions, but only if the split structure can find them. Sometimes explicit cross features help:
•	`rbcs_cat_subcat` × `provider_type`
•	`rbcs_cat_subcat` × `Place_Of_Srvc`
•	`provider_type` × `Place_Of_Srvc`

With OHE, this can explode dimensionality, so this is where CatBoost becomes attractive.

⸻

Step 5. Try CatBoost instead of OHE for these high-card categoricals

We have high-card categorical fields (especially `rbcs_cat_subcat`). OHE + XGB can work, but CatBoost often wins on exactly this kind of problem because it handles categorical encodings natively and learns smoother category effects.

If we try CatBoost, keep the same evaluation framework we built. Compare tail bias, tail ratio, and tail SSE share directly.

⸻

5) Should we build two models, tail vs non-tail?

Not yet.

Two-model systems create a new hard problem: ***how do we decide tail membership at inference time without using y?***

You would need a classifier that predicts “tail-like” based only on features, and with 1% prevalence it will be brittle.

A better “two-stage” system (if we go there) is:
1.	Stage 1 predicts the expected log cost (our current model).
2.	Stage 2 predicts an uplift factor or residual correction for cases likely to be in high-cost regimes, using provider-history and fine-grain categories.

That is more stable than hard gating into two separate regressors.

⸻

6) Should we remove ultra-high rows?

I would not remove them from the dataset if they are legitimate and we care about predicting them.

But we can do one of these safer alternatives:
•	Use a robust objective (pseudohuber) so one extreme row does not dominate.
•	Winsorize y during training only (cap at p99.9), then evaluate on uncapped. This stabilizes training but keeps evaluation honest.
•	Add provider-history features so the model can actually learn why those rows are extreme.

⸻

My recommended next experiment sequence
1. Pick w_tail = 10 as our default tradeoff point.
2. Tune XGB hyperparameters (small grid) with fixed w=10.
3. Try objective="reg:pseudohubererror" with the tuned-ish setup.
4. Add lagged provider-history features (done correctly with year lagging).
5. Run CatBoost with the same feature set and compare tail metrics.

Let's execute this plan! 



### Tuning the XGBoost model

In [ ]:
is_tail_train = eda_df.loc[X_train.index, "is_top_1pct_stdzd_amt_per_service"].astype(int)
w_tail = 10
sample_w = 1 + (w_tail - 1) * is_tail_train
sample_w = np.asarray(sample_w, dtype=float)
print(len(sample_w), X_train.shape[0])

#### 1. `GridSearchCV` for the weighted XGBoost model (with `objective:"reg:squarederror"`)

In [ ]:
from sklearn.model_selection import GridSearchCV, GroupKFold
from pathlib import Path
from joblib import dump, load

# 1. Setup Paths
ROOT = Path.cwd().resolve()
MODELS = (ROOT / "models").resolve()
MODELS.mkdir(parents=True, exist_ok=True)

# Point to the model file
model_path = MODELS / "old_forecast_model_wt10_without_lags.joblib"

# 2. Check/Load or Train/Save
if model_path.exists():
    print(f"Loading {model_path.name} from disk... (Skipping training)")
    old_forecast_model_wt10_without_lags = load(model_path)

else:
    print(f"File not found. Preparing data and training...")

    is_tail_train = eda_df.loc[X_train.index, "is_top_1pct_stdzd_amt_per_service"].astype(int)
    w_tail = 10
    sample_w = 1 + (w_tail - 1) * is_tail_train
    sample_w = np.asarray(sample_w, dtype=float)
    print(len(sample_w), X_train.shape[0])

    xgb_inner_pipe = Pipeline(steps=[
        ("preprocess", preprocess_ohe),
        ("model", XGBRegressor(
            objective="reg:squarederror",
            tree_method="hist",
            n_jobs=1,
            random_state=0
        ))
    ])

    xgb_full_model = TransformedTargetRegressor(
        regressor=xgb_inner_pipe,
        func=np.log1p,
        inverse_func=np.expm1
    )

    param_grid_xgb = {
        "regressor__model__n_estimators": [800, 1600],
        "regressor__model__learning_rate": [0.03, 0.07],
        "regressor__model__max_depth": [3, 5],
        "regressor__model__min_child_weight": [1, 5, 10],
        "regressor__model__subsample": [0.7, 0.9],
        "regressor__model__colsample_bytree": [0.7, 0.9],
        "regressor__model__reg_lambda": [1, 10], # L2 (Ridge) 
        "regressor__model__reg_alpha": [0, 0.1], # L1 (Lasso)
    }

    groups = train_df.loc[X_train.index, "Rndrng_NPI"].to_numpy()
    sample_w = sample_w.to_numpy() if hasattr(sample_w, "to_numpy") else sample_w
    cv = GroupKFold(n_splits=3)

    search = GridSearchCV(
        xgb_full_model,
        param_grid_xgb,
        cv=cv,
        scoring="r2",
        n_jobs=-1,
        verbose=1
    )

    # --- MISSING LINES ADDED HERE ---
    # 1. Run the grid search
    search.fit(X_train, y_train, model__sample_weight=sample_w, groups=groups)
    
    # 2. Extract best estimator
    old_forecast_model_wt10_without_lags = search.best_estimator_
    
    # --------------------------------

    # --- SAVE ---
    # 3. Save the trained model
    dump(old_forecast_model_wt10_without_lags, model_path)
    print(f"Training complete. Model saved to {model_path}")

#### Fit the best model, `old_forecast_model_wt10_without_lags` to `X_train` and `y_train` with `sample_w`

In [ ]:
old_forecast_model_wt10_without_lags.fit(X_train, y_train, model__sample_weight = sample_w)

#### Run the `tail_eval` with the `best_model` 

In [ ]:
# Run diagnostics

# get train and test predictions and clip test predictions 
pred_test = old_forecast_model_wt10_without_lags.predict(X_test)
pred_train = old_forecast_model_wt10_without_lags.predict(X_train)

# get weighted-train unweighted/average-case metrics
r2_train = old_forecast_model_wt10_without_lags.score(X_train, y_train)
r2_test = old_forecast_model_wt10_without_lags.score(X_test, y_test)
mae = mean_absolute_error(y_test, pred_test)
rmse = root_mean_squared_error(y_test, pred_test)

# get weighted-train unweighted/average-case metrics in log space
log_pred_train = old_forecast_model_wt10_without_lags.regressor_.predict(X_train)  # predictions in transformed target space
log_pred_test = old_forecast_model_wt10_without_lags.regressor_.predict(X_test)  # predictions in transformed target space

r2_log_true_train = r2_score(np.log1p(y_train), log_pred_train)
r2_log_true_test = r2_score(np.log1p(y_test), log_pred_test)
mae_log_true = mean_absolute_error(np.log1p(y_test), log_pred_test)
rmse_log_true = root_mean_squared_error(np.log1p(y_test), log_pred_test)

# get weighted-train weighted/eval metrics

# Create weights for the test set (following same logic as train)
is_tail_test = eda_df.loc[X_test.index, "is_top_1pct_stdzd_amt_per_service"].astype(int)
sample_w_test = 1 + (w_tail - 1) * is_tail_test

r2_train_w = r2_score(y_train, pred_train, sample_weight=sample_w)
r2_test_w = r2_score(y_test, pred_test, sample_weight=sample_w_test)
mae_w = mean_absolute_error(y_test, pred_test, sample_weight=sample_w_test)
rmse_w = root_mean_squared_error(y_test, pred_test, sample_weight=sample_w_test)

# Calculate tail evals
test_eval = test_df.copy().join(
eda_df.loc[:, ["is_top_1pct_stdzd_amt_per_service"]],
how="left"
)
test_eval["pred"] = pred_test
test_eval["abs_err"] = (test_eval[target_col] - test_eval["pred"]).abs()
test_eval["sq_err"]  = (test_eval[target_col] - test_eval["pred"])**2

non_tail_pred_to_y_ratio = test_eval.loc[~test_eval["is_top_1pct_stdzd_amt_per_service"],"pred"].mean() / test_eval.loc[~test_eval["is_top_1pct_stdzd_amt_per_service"],target_col].mean()
tail_pred_to_y_ratio = test_eval.loc[test_eval["is_top_1pct_stdzd_amt_per_service"],"pred"].mean() / test_eval.loc[test_eval["is_top_1pct_stdzd_amt_per_service"],target_col].mean()

non_tail_sse = test_eval.loc[~test_eval["is_top_1pct_stdzd_amt_per_service"],"sq_err"].sum()
tail_sse = test_eval.loc[test_eval["is_top_1pct_stdzd_amt_per_service"],"sq_err"].sum()

non_tail_share_of_total_sse = non_tail_sse / (non_tail_sse + tail_sse)
tail_share_of_total_sse = tail_sse / (non_tail_sse + tail_sse)

non_tail_under_prediction_rate = ((test_eval.loc[~test_eval["is_top_1pct_stdzd_amt_per_service"],"pred"]) < test_eval.loc[~test_eval["is_top_1pct_stdzd_amt_per_service"],target_col]).mean()
tail_under_prediction_rate = ((test_eval.loc[test_eval["is_top_1pct_stdzd_amt_per_service"],"pred"]) < test_eval.loc[test_eval["is_top_1pct_stdzd_amt_per_service"],target_col]).mean()

non_tail_bias = test_eval.loc[~test_eval["is_top_1pct_stdzd_amt_per_service"],"pred"].mean() - test_eval.loc[~test_eval["is_top_1pct_stdzd_amt_per_service"],target_col].mean()
tail_bias = test_eval.loc[test_eval["is_top_1pct_stdzd_amt_per_service"],"pred"].mean() - test_eval.loc[test_eval["is_top_1pct_stdzd_amt_per_service"],target_col].mean()

weighted_train_performance_metrics_test_eval = pd.DataFrame({
    "train_sample_weights":[10],
    "r2_train": r2_train,
    "r2_test": r2_test,
    "mae": mae,
    "rmse": rmse,
    "r2_train_log": r2_log_true_train,
    "r2_test_log": r2_log_true_test,
    "mae_log": mae_log_true,
    "rmse_log": rmse_log_true,
    "r2_train_w": r2_train_w,
    "r2_test_w": r2_test_w,
    "mae_w": mae_w,
    "rmse_w": rmse_w,
    "non_tail_pred_to_y_ratio": non_tail_pred_to_y_ratio,
    "tail_pred_to_y_ratio": tail_pred_to_y_ratio,
    "non_tail_sse": non_tail_sse,
    "tail_sse": tail_sse,
    "non_tail_share_of_total_sse": non_tail_share_of_total_sse,
    "tail_share_of_total_sse": tail_share_of_total_sse,
    "non_tail_under_prediction_rate": non_tail_under_prediction_rate,
    "tail_under_prediction_rate": tail_under_prediction_rate,
    "non_tail_bias": non_tail_bias,
    "tail_bias": tail_bias
})

weighted_train_performance_metrics_test_eval


**Compare tuned vs untuned at w_tail = 10**

Let’s compare your tuned row to the earlier default hyperparameter row for w_tail=10 (row index 3 in your for-loop table).

A. Overall unweighted test performance (average-case)

Default (w=10):
- Test R² ($): 0.2900
- MAE ($): 30.832
- RMSE ($): 144.642

Tuned (w=10):
- Test R² ($): 0.2866
- MAE ($): 29.375
- RMSE ($): 144.985

Interpretation:
- Dollar R² is basically the same (tiny worse).
- MAE improved a bit.
- RMSE basically unchanged (tiny worse).
- Net: tuning didn’t meaningfully improve global generalization, but it didn’t break it either.

B. Log-space generalization (relative-error view)

Default (w=10):
- Test R² (log): 0.7771
- MAE (log): 0.3810
- RMSE (log): 0.5847

Tuned (w=10):
- Test R² (log): 0.7923
- MAE (log): 0.3683
- RMSE (log): 0.5644

Interpretation:
- This is a meaningful improvement. In the space the model is trained in, it is fitting better and generalizing better.

C. Weighted evaluation (tail-focused objective)

Default (w=10):
- Test R² weighted ($): 0.2287
- MAE weighted ($): 49.034
- RMSE weighted ($): 407.767

Tuned (w=10):
- Test R² weighted ($): 0.2099
- MAE weighted ($): 47.426
- RMSE weighted ($): 412.703

Interpretation:
- Slightly better weighted MAE.
- Slightly worse weighted R² and RMSE.
- This usually means you reduced typical tail error a little but did not reduce the largest explosions (RMSE and R² are dominated by those).

⸻

**Tail calibration metrics. This is where the tune helped**

These are the most actionable improvements.

Tail pred_to_y_ratio (calibration)
- Default w=10: 0.7227
- Tuned w=10: 0.7217

Basically identical (good. You didn’t lose calibration).

Tail underprediction rate
- Default w=10: 0.7378
- Tuned w=10: 0.7639

This is actually worse (more underprediction), but not crazy.

Tail SSE share
- Default w=10: 0.8415
- Tuned w=10: 0.8601

Also worse (tail dominates SSE a bit more).

Tail bias (mean pred minus mean actual)
- Default w=10: -245.60
- Tuned w=10: -246.48

Essentially unchanged.

Non-tail bias
- Default w=10: -2.21
- Tuned w=10: -2.76

Slightly worse but still tiny relative to the mean.

Interpretation:
- The tuning helped log-space fit, but did not materially improve the tail failure mode in dollars. The biggest tail misses are still dominating.

That is consistent with what you already observed. A tiny number of ultra-high cost-per-service points create massive squared errors. Hyperparameter tuning with a squared-loss objective often improves the “bulk” first, and struggles to fix a tiny extreme pocket without either (a) more signal features, (b) a different loss emphasis, or (c) structural modeling changes.


## Revisiting the worst offender and its peers (`peer` dataframe) using `rbcs_cat_subcat` (instead of `rbcs_family_desc`)

In [ ]:
row_idx = worst.index[0]

eda_df.loc[row_idx, [
    "Rndrng_NPI","Year","rbcs_family_desc","Place_Of_Srvc","provider_type","state","ruca_bucket",
    "services","benes",
    "stdzd_amt_per_service","stdzd_spend",
    "allowed_amt_per_service","allowed_spend",
    "payment_amt_per_service","payment_spend",
    "submitted_charge_per_service","submitted_spend","rbcs_cat_subcat"
]]

In [ ]:
worst

In [ ]:
npi = worst["Rndrng_NPI"].iloc[0]

eda_df.loc[
    (eda_df["Rndrng_NPI"] == npi) & (eda_df["rbcs_family_desc"] == "Chemotherapeutic Agent"),
    ["Year","services","stdzd_amt_per_service","stdzd_spend","Place_Of_Srvc","provider_type","state", "rbcs_cat_subcat", "ruca_bucket"]
].sort_values("Year")

In [ ]:
eda_df["rbcs_cat_subcat"].value_counts()[:10]

In [ ]:
peer = train_df.loc[
    (train_df["rbcs_cat_subcat"] == "RH") &
    (train_df["provider_type"] == "Radiation Oncology") &
    (train_df["Place_Of_Srvc"] == "O"),
    ["stdzd_amt_per_service","services","state","ruca_bucket"]
]

peer.sort_values("stdzd_amt_per_service", ascending=False)

> We cannot separate ultra-high cost per service lines from their peers at this level of granularity. 

***We must change strategy. See the findings in the EDA notebook at the bottom of the "Data by Provider and Service" section***

> Key insight from EDA notebook: The “ultra-high” tail is not some mysterious provider behavior at the RBCS-family level. It is mostly “this provider billed a very specific HCPCS (or a NOC HCPCS) that is inherently expensive”. If you do not include HCPCS-level signal (or a proxy for it), the model is forced to average over fundamentally different things. That is why it keeps regressing those cases toward the mean.

In [ ]:
eda_hcpcs_df = pd.read_parquet(DATA/"eda_dataset_hcpcs.parquet")

In [ ]:
eda_hcpcs_df.columns

## Streamlining generation of `train_df`, `test_df`, `X_train`, `target_col`, `cat_features`, `num_features`, `feature_cols`, `y_train`, `X_test`, `y_test`, `sample_w_train`, `sample_w_test`, `groups_train`, `cv` (`PredefinedSplit()` or `GroupKFold()`), `search` (`GridSearchCV()`) based on `approach` (either `"forecast"`, or `"new_providers"`)

### 1. Create the `PreparedModelingObjects` Class

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Any, Dict, List, Optional, Tuple, Union

import numpy as np
import pandas as pd
from scipy.stats import loguniform, randint, uniform

from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.impute import SimpleImputer
from sklearn.model_selection import (
    GridSearchCV,
    GroupKFold,
    GroupShuffleSplit,
    PredefinedSplit,
    RandomizedSearchCV,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

from xgboost import XGBRegressor


# -----------------------------------------------------------------------------
# Container for everything the notebook typically needs downstream
# -----------------------------------------------------------------------------
@dataclass
class PreparedModelingObjects:
    """
    Convenience bundle returned by prepare_xgb_* helpers.

    Why this exists:
    - Keeps the split dataframes, X/y matrices, sample weights, CV object, and the
      hyperparameter search object together so the notebook stays clean.
    - Makes it easy to log/print the configuration and to reuse objects later.

    Notes:
    - `cv` is either:
        - PredefinedSplit for forecast (explicit year-based validation)
        - GroupKFold for new_providers (provider-disjoint validation)
    - `search` is ready to call `.fit(...)`.
    """

    # High-level scenario
    approach: str
    target_col: str

    # Feature configuration actually used (after filtering to existing columns)
    cat_features: List[str]
    num_features: List[str]
    feature_cols: List[str]

    # The split dataframes used to build X/y
    train_df: pd.DataFrame
    test_df: pd.DataFrame

    # Matrices/vectors for modeling
    X_train: pd.DataFrame
    y_train: pd.Series
    X_test: pd.DataFrame
    y_test: pd.Series

    # Sample weights (computed within each split)
    sample_w_train: np.ndarray
    sample_w_test: np.ndarray

    # Group vector used only for group CV (new_providers). None otherwise.
    groups_train: Optional[np.ndarray]

    # CV strategy and hyperparameter search object
    cv: Any
    search: Union[GridSearchCV, RandomizedSearchCV]

### 2. Create the `prepare_xgb_gridsearch` function

In [ ]:
def prepare_xgb_gridsearch(
    df: pd.DataFrame,
    approach: str,
    target_col: str = "avg_mdcr_stdzd_amt",
    *,
    # Tail weighting (emphasize “tail” rows during training/evaluation)
    w_tail: int = 10,
    tail_flag_col: str = "is_top_1pct_avg_mdcr_stdzd_amt",
    # Forecast-only: include or exclude 2020 rows in the final fit dataset
    incl_2020_in_final_fit: bool = True,
    # Splits
    test_year: int = 2023,
    train_years: Tuple[int, ...] = (2020, 2021, 2022),
    group_col: str = "Rndrng_NPI",
    test_size: float = 0.20,
    random_state: int = 0,
    n_splits: int = 3,
    # Features
    include_lags: bool = True,
    # Model search
    param_grid: Optional[Dict[str, List[Any]]] = None,
    scoring: str = "r2",
    n_jobs: int = -1,
    verbose: int = 1,
    # if target is log_delta_cost, do not use TTR
    use_ttr: bool = True,
    # XGB defaults
    xgb_fixed_params: Optional[Dict[str, Any]] = None,
) -> PreparedModelingObjects:
    """
    Prepare XGBoost + preprocessing + TransformedTargetRegressor, and return a
    leakage-aware GridSearchCV plus the exact split data used.

    Supported approaches
    --------------------
    1) approach="forecast"
       - Final test set is `test_year` (default 2023).
       - Inner CV is a PredefinedSplit that always validates on the last training year (default 2022).
       - Training years are `train_years` (default 2020-2022).
       - If `incl_2020_in_final_fit=True`, 2020 rows are included in X_train/y_train for final fitting,
         but CV validation remains 2022 and “always-train” includes 2020 and 2021.
       - If `incl_2020_in_final_fit=False`, X_train/y_train used by search contain only 2021-2022.

       Practical intent:
       - CV checks: “learn on 2020/2021 and validate on 2022” (or “learn on 2021 and validate on 2022”)
         while keeping the final holdout year 2023 untouched.

    2) approach="new_providers"
       - Group holdout by `group_col` (default Rndrng_NPI): providers in test never appear in train.
       - Inner CV is GroupKFold on the training portion.

    Returns
    -------
    PreparedModelingObjects
        Bundle containing split dataframes, X/y, sample weights, cv, and a GridSearchCV object.

    Notes
    -----
    - Uses TransformedTargetRegressor with log1p/expm1 by default (stabilizes heavy-tailed targets).
    - Numeric features: median impute + missing indicators.
    - Lag numeric features (if present): constant=0 impute + missing indicators.
    - Categorical features: most_frequent impute + one-hot encoding.
    - Sample weights: tail rows get weight `w_tail`, others weight 1.0.
    """

    # -------------------------------------------------------------------------
    # Validation
    # -------------------------------------------------------------------------
    if approach not in {"forecast", "new_providers"}:
        raise ValueError("approach must be one of: {'forecast', 'new_providers'}")

    if target_col not in df.columns:
        raise ValueError(f"target_col '{target_col}' not found in df.columns")

    # -------------------------------------------------------------------------
    # 1) Define splits and CV strategy
    # -------------------------------------------------------------------------
    if approach == "forecast":
        # Final holdout (never used in CV or fitting inside search)
        test_df = df[df["Year"] == test_year].copy()

        # All training years available to the forecast workflow
        train_df_all = df[df["Year"].isin(train_years)].copy()

        years_sorted = sorted(set(train_years))
        if len(years_sorted) < 3:
            raise ValueError(
                "forecast expects 3 training years in train_years (e.g., (2020, 2021, 2022)) "
                "so we can do CV train=2021, val=2022 with lags available."
            )

        # The “validation year” is always the last year in train_years.
        # The year before it is the main “train year” for the CV story.
        train_year = years_sorted[-2]  # typically 2021
        val_year = years_sorted[-1]    # typically 2022

        # CV is conceptually “validate on 2022”.
        # We keep the CV design stable even if we include 2020 rows in the final fit.
        train_df_cv = train_df_all[train_df_all["Year"].isin([train_year, val_year])].copy()

        # The dataset actually used by GridSearchCV for fitting its estimator:
        # - if True: include 2020-2022 rows (more training signal)
        # - if False: use only 2021-2022 rows (closer to original design)
        train_df = train_df_all.copy() if incl_2020_in_final_fit else train_df_cv.copy()

        # PredefinedSplit expects an array aligned to train_df rows.
        # 0 denotes the validation fold (here: 2022 rows).
        # -1 denotes “always in train” (here: 2020/2021 rows, and also 2021-only mode).
        fold = train_df["Year"].map({val_year: 0}).fillna(-1).astype(int).to_numpy()
        cv = PredefinedSplit(test_fold=fold)

        groups_train = None

    else:
        # Group holdout so test providers never appear in training
        groups_all = df[group_col].to_numpy()
        gss = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=random_state)
        train_idx, test_idx = next(gss.split(df, groups=groups_all))

        train_df = df.iloc[train_idx].copy()
        test_df = df.iloc[test_idx].copy()

        groups_train = train_df[group_col].to_numpy()
        cv = GroupKFold(n_splits=n_splits)

    # -------------------------------------------------------------------------
    # 2) Define feature lists (then filter to only existing columns)
    # -------------------------------------------------------------------------
    # Categorical features to one-hot encode
    cat_features = [
        "HCPCS_Cd",
        "Place_Of_Srvc",
        "provider_type",
        "state",
        "ruca_bucket",
        "rbcs_family_desc",
        "hcpcs_drug_ind",
        "is_core_scope",
    ]

    # Numeric features (plus optional lag features)
    num_features = [
        "services",
        "benes",
        "bene_day_services",
        "years_since_enumeration",
        "bene_avg_risk_score",
        "p_cancer6",
        "p_diabetes",
        "p_ckd",
        "p_copd",
        "p_htn",
    ]

    if include_lags:
        num_features += ["lag1_avg_amt", "lag1_tot_srvcs", "lag1_spend"]

    # Safety: allow experimentation across datasets without breaking if some columns are absent
    cat_features = [c for c in cat_features if c in train_df.columns]
    num_features = [c for c in num_features if c in train_df.columns]

    # Treat lag variables differently (common choice: fill missing lag with 0)
    lag_candidates = ["lag1_avg_amt", "lag1_tot_srvcs", "lag1_spend"]
    lag_features = [c for c in lag_candidates if c in num_features]
    base_num_features = [c for c in num_features if c not in lag_features]

    feature_cols = cat_features + num_features

    # -------------------------------------------------------------------------
    # 3) Build X/y (train and test)
    # -------------------------------------------------------------------------
    X_train = train_df[feature_cols].copy()
    y_train = train_df[target_col].astype(float)

    X_test = test_df[feature_cols].copy()
    y_test = test_df[target_col].astype(float)

    # -------------------------------------------------------------------------
    # 4) Sample weights (computed strictly within each split)
    # -------------------------------------------------------------------------
    if tail_flag_col not in train_df.columns:
        raise ValueError(f"tail_flag_col '{tail_flag_col}' not found in df.columns")

    is_tail_train = train_df[tail_flag_col].astype(int).to_numpy()
    is_tail_test = test_df[tail_flag_col].astype(int).to_numpy()

    # Non-tail => 1.0, tail => w_tail
    sample_w_train = (1 + (w_tail - 1) * is_tail_train).astype(float)
    sample_w_test = (1 + (w_tail - 1) * is_tail_test).astype(float)

    # -------------------------------------------------------------------------
    # 5) Preprocessing + model + target transform
    # -------------------------------------------------------------------------
    # Categorical pipeline: impute then one-hot
    cat_pipe = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=True)),
        ]
    )

    # Numeric pipeline: median impute + missing flags
    num_pipe = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
        ]
    )

    # Lag pipeline: treat missing lag as 0 (plus missing flags)
    lag_pipe = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="constant", fill_value=0.0, add_indicator=True)),
        ]
    )

    transformers = [
        ("cat", cat_pipe, cat_features),
        ("num", num_pipe, base_num_features),
    ]
    if lag_features:
        transformers.append(("lag", lag_pipe, lag_features))

    preprocess_ohe = ColumnTransformer(
        transformers=transformers,
        remainder="drop",
        sparse_threshold=0.3,
    )

    # XGBoost defaults: keep model single-threaded because CV/search is parallelized
    if xgb_fixed_params is None:
        xgb_fixed_params = dict(
            objective="reg:squarederror",
            tree_method="hist",
            n_jobs=1,
            random_state=random_state,
        )

    xgb_inner_pipe = Pipeline(
        steps=[
            ("preprocess", preprocess_ohe),
            ("model", XGBRegressor(**xgb_fixed_params)),
        ]
    )

    if use_ttr:
        # Transform target: model learns in log-space, predictions are returned in original scale
        xgb_full_model = TransformedTargetRegressor(
            regressor=xgb_inner_pipe,
            func=np.log1p,
            inverse_func=np.expm1,
        )
    else:
        xgb_full_model = xgb_inner_pipe # not TTR

    # -------------------------------------------------------------------------
    # 6) Parameter grid
    # -------------------------------------------------------------------------
    if use_ttr:
        if param_grid is None:
            param_grid = {
                "regressor__model__n_estimators": [800, 1600],
                "regressor__model__learning_rate": [0.03, 0.07],
                "regressor__model__max_depth": [3, 5],
                "regressor__model__min_child_weight": [1, 5, 10],
                "regressor__model__subsample": [0.7, 0.9],
                "regressor__model__colsample_bytree": [0.7, 0.9],
                "regressor__model__reg_lambda": [1, 10],
                "regressor__model__reg_alpha": [0, 0.1],
            }
    else:
        if param_grid is None:
            param_grid = {
                "model__n_estimators": [800, 1600],
                "model__learning_rate": [0.03, 0.07],
                "model__max_depth": [3, 5],
                "model__min_child_weight": [1, 5, 10],
                "model__subsample": [0.7, 0.9],
                "model__colsample_bytree": [0.7, 0.9],
                "model__reg_lambda": [1, 10],
                "model__reg_alpha": [0, 0.1],
            }

    # -------------------------------------------------------------------------
    # 7) GridSearchCV (ready to fit)
    # -------------------------------------------------------------------------
    search = GridSearchCV(
        estimator=xgb_full_model,
        param_grid=param_grid,
        cv=cv,
        scoring=scoring,
        n_jobs=n_jobs,
        verbose=verbose,
    )

    return PreparedModelingObjects(
        approach=approach,
        target_col=target_col,
        cat_features=cat_features,
        num_features=num_features,
        feature_cols=feature_cols,
        train_df=train_df,
        test_df=test_df,
        X_train=X_train,
        y_train=y_train,
        X_test=X_test,
        y_test=y_test,
        sample_w_train=sample_w_train,
        sample_w_test=sample_w_test,
        groups_train=groups_train,
        cv=cv,
        search=search,
    )

### 3. Create the `prepare_xgb_randomizedsearch` function

In [ ]:
def prepare_xgb_randomizedsearch(
    df: pd.DataFrame,
    approach: str,
    target_col: str = "avg_mdcr_stdzd_amt",
    *,
    # Tail weighting (emphasize “tail” rows during training/evaluation)
    w_tail: int = 10,
    tail_flag_col: str = "is_top_1pct_avg_mdcr_stdzd_amt",
    # Forecast-only: include or exclude 2020 rows in the final fit dataset
    incl_2020_in_final_fit: bool = True,
    # Splits
    test_year: int = 2023,
    train_years: Tuple[int, ...] = (2020, 2021, 2022),
    group_col: str = "Rndrng_NPI",
    test_size: float = 0.20,
    random_state: int = 0,
    n_splits: int = 3,
    # Features
    include_lags: bool = True,
    # Model search
    param_distributions: Optional[Dict[str, Any]] = None,
    n_iter: int = 50,
    scoring: str = "r2",
    n_jobs: int = -1,
    verbose: int = 1,
    # if target is log_delta_cost, do not use TTR
    use_ttr: bool = True,
    # XGB defaults
    xgb_fixed_params: Optional[Dict[str, Any]] = None,
) -> PreparedModelingObjects:
    """
    Same as `prepare_xgb_gridsearch`, but uses RandomizedSearchCV instead of GridSearchCV.

    The key behavior you asked for is preserved:
    - Forecast:
        - CV validates on 2022 via PredefinedSplit.
        - `incl_2020_in_final_fit` controls whether 2020 is included in the dataset that the
          hyperparameter search fits on (while still validating only on 2022 rows).
        - Final test set is 2023.
    - New providers:
        - provider-disjoint holdout split
        - GroupKFold for inner CV

    Returns
    -------
    PreparedModelingObjects
        Bundle containing split dataframes, X/y, sample weights, cv, and a RandomizedSearchCV object.
    """

    # -------------------------------------------------------------------------
    # Validation
    # -------------------------------------------------------------------------
    if approach not in {"forecast", "new_providers"}:
        raise ValueError("approach must be one of: {'forecast', 'new_providers'}")

    if target_col not in df.columns:
        raise ValueError(f"target_col '{target_col}' not found in df.columns")

    # -------------------------------------------------------------------------
    # 1) Define splits and CV strategy
    # -------------------------------------------------------------------------
    if approach == "forecast":
        test_df = df[df["Year"] == test_year].copy()
        train_df_all = df[df["Year"].isin(train_years)].copy()

        years_sorted = sorted(set(train_years))
        if len(years_sorted) < 3:
            raise ValueError(
                "forecast expects 3 training years in train_years (e.g., (2020, 2021, 2022)) "
                "so we can do CV train=2021, val=2022 with lags available."
            )

        train_year = years_sorted[-2]  # typically 2021
        val_year = years_sorted[-1]    # typically 2022

        # CV always focuses on 2021-2022, independent of final-fit choice
        train_df_cv = train_df_all[train_df_all["Year"].isin([train_year, val_year])].copy()

        # Dataset used for fitting inside the search
        train_df = train_df_all.copy() if incl_2020_in_final_fit else train_df_cv.copy()

        # Validate only on 2022 rows
        fold = train_df["Year"].map({val_year: 0}).fillna(-1).astype(int).to_numpy()
        cv = PredefinedSplit(test_fold=fold)

        groups_train = None

    else:
        groups_all = df[group_col].to_numpy()
        gss = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=random_state)
        train_idx, test_idx = next(gss.split(df, groups=groups_all))

        train_df = df.iloc[train_idx].copy()
        test_df = df.iloc[test_idx].copy()

        groups_train = train_df[group_col].to_numpy()
        cv = GroupKFold(n_splits=n_splits)

    # -------------------------------------------------------------------------
    # 2) Define feature lists (then filter to only existing columns)
    # -------------------------------------------------------------------------
    cat_features = [
        "HCPCS_Cd",
        "Place_Of_Srvc",
        "provider_type",
        "state",
        "ruca_bucket",
        "rbcs_family_desc",
        "hcpcs_drug_ind",
        "is_core_scope",
    ]

    num_features = [
        "services",
        "benes",
        "bene_day_services",
        "years_since_enumeration",
        "bene_avg_risk_score",
        "p_cancer6",
        "p_diabetes",
        "p_ckd",
        "p_copd",
        "p_htn",
    ]

    if include_lags:
        num_features += ["lag1_avg_amt", "lag1_tot_srvcs", "lag1_spend"]

    cat_features = [c for c in cat_features if c in train_df.columns]
    num_features = [c for c in num_features if c in train_df.columns]

    lag_candidates = ["lag1_avg_amt", "lag1_tot_srvcs", "lag1_spend"]
    lag_features = [c for c in lag_candidates if c in num_features]
    base_num_features = [c for c in num_features if c not in lag_features]

    feature_cols = cat_features + num_features

    # -------------------------------------------------------------------------
    # 3) Build X/y (train and test)
    # -------------------------------------------------------------------------
    X_train = train_df[feature_cols].copy()
    y_train = train_df[target_col].astype(float)

    X_test = test_df[feature_cols].copy()
    y_test = test_df[target_col].astype(float)

    # -------------------------------------------------------------------------
    # 4) Sample weights
    # -------------------------------------------------------------------------
    if tail_flag_col not in train_df.columns:
        raise ValueError(f"tail_flag_col '{tail_flag_col}' not found in df.columns")

    is_tail_train = train_df[tail_flag_col].astype(int).to_numpy()
    is_tail_test = test_df[tail_flag_col].astype(int).to_numpy()

    sample_w_train = (1 + (w_tail - 1) * is_tail_train).astype(float)
    sample_w_test = (1 + (w_tail - 1) * is_tail_test).astype(float)

    # -------------------------------------------------------------------------
    # 5) Preprocessing + model + target transform
    # -------------------------------------------------------------------------
    cat_pipe = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=True)),
        ]
    )

    num_pipe = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
        ]
    )

    lag_pipe = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="constant", fill_value=0.0, add_indicator=True)),
        ]
    )

    transformers = [
        ("cat", cat_pipe, cat_features),
        ("num", num_pipe, base_num_features),
    ]
    if lag_features:
        transformers.append(("lag", lag_pipe, lag_features))

    preprocess_ohe = ColumnTransformer(
        transformers=transformers,
        remainder="drop",
        sparse_threshold=0.3,
    )

    if xgb_fixed_params is None:
        xgb_fixed_params = dict(
            objective="reg:squarederror",
            tree_method="hist",
            n_jobs=1,
            random_state=random_state,
        )

    xgb_inner_pipe = Pipeline(
        steps=[
            ("preprocess", preprocess_ohe),
            ("model", XGBRegressor(**xgb_fixed_params)),
        ]
    )
    if use_ttr:
        xgb_full_model = TransformedTargetRegressor(
            regressor=xgb_inner_pipe,
            func=np.log1p,
            inverse_func=np.expm1,
        )
    else:
        xgb_full_model = xgb_inner_pipe # no TTR

    # -------------------------------------------------------------------------
    # 6) Parameter distributions (random search space)
    # -------------------------------------------------------------------------
    if use_ttr:
        if param_distributions is None:
            param_distributions = {
                "regressor__model__n_estimators": randint(800, 1601),
                "regressor__model__max_depth": randint(2, 5),
                "regressor__model__min_child_weight": randint(10, 81),
                "regressor__model__max_delta_step": randint(0, 6),
                "regressor__model__subsample": uniform(0.6, 0.35),
                "regressor__model__colsample_bytree": uniform(0.6, 0.35),
                "regressor__model__gamma": uniform(0.0, 2.0),
                "regressor__model__learning_rate": loguniform(0.01, 0.08),
                "regressor__model__reg_lambda": loguniform(1.0, 300.0),
                "regressor__model__reg_alpha": loguniform(1e-6, 1.0),
                "regressor__model__max_bin": [256, 512],
            }
    else:
        if param_distributions is None:
            param_distributions = {
                "model__n_estimators": randint(800, 1601),
                "model__max_depth": randint(2, 5),
                "model__min_child_weight": randint(10, 81),
                "model__max_delta_step": randint(0, 6),
                "model__subsample": uniform(0.6, 0.35),
                "model__colsample_bytree": uniform(0.6, 0.35),
                "model__gamma": uniform(0.0, 2.0),
                "model__learning_rate": loguniform(0.01, 0.08),
                "model__reg_lambda": loguniform(1.0, 300.0),
                "model__reg_alpha": loguniform(1e-6, 1.0),
                "model__max_bin": [256, 512],
            }

    # -------------------------------------------------------------------------
    # 7) RandomizedSearchCV (ready to fit)
    # -------------------------------------------------------------------------
    search = RandomizedSearchCV(
        estimator=xgb_full_model,
        param_distributions=param_distributions,
        n_iter=n_iter,
        cv=cv,
        scoring=scoring,
        n_jobs=n_jobs,
        verbose=verbose,
        random_state=random_state,
    )

    return PreparedModelingObjects(
        approach=approach,
        target_col=target_col,
        cat_features=cat_features,
        num_features=num_features,
        feature_cols=feature_cols,
        train_df=train_df,
        test_df=test_df,
        X_train=X_train,
        y_train=y_train,
        X_test=X_test,
        y_test=y_test,
        sample_w_train=sample_w_train,
        sample_w_test=sample_w_test,
        groups_train=groups_train,
        cv=cv,
        search=search,
    )

## 1. Forecast approach: Training a model that predicts the expected average cost per service for 2023 using 2021 and 2022 using `GridSearchCV()`

In [ ]:
from pathlib import Path
from joblib import dump, load

# 1. Setup Paths
ROOT = Path.cwd().resolve()
MODELS = (ROOT / "models").resolve()
MODELS.mkdir(parents=True, exist_ok=True)

# We now point to a package file instead of just the model file
pkg_path = MODELS / "forecast_pkg_wt10_with_lags_no_2020.joblib"

# 2. Check/Load or Train/Save
if pkg_path.exists():
    print(f"Loading {pkg_path.name} from disk... (Skipping data prep and training)")
    # Load the entire package (data + fitted search object)
    forecast_pkg_wt10_with_lags_no_2020 = load(pkg_path)
    
    # Extract the model so downstream code works as expected
    best_forecast_model_wt10_with_lags_no_2020 = forecast_pkg_wt10_with_lags_no_2020.search.best_estimator_

else:
    print(f"File not found. Preparing data and training...")

    # --- PREPARE DATA ---
    forecast_pkg_wt10_with_lags_no_2020 = prepare_xgb_gridsearch(
        eda_hcpcs_df,
        approach="forecast",
        target_col="avg_mdcr_stdzd_amt",
        w_tail=10,
        include_lags=True,
        incl_2020_in_final_fit=False
    )

    # --- TRAIN ---
    if forecast_pkg_wt10_with_lags_no_2020.groups_train is None:
        forecast_pkg_wt10_with_lags_no_2020.search.fit(
            forecast_pkg_wt10_with_lags_no_2020.X_train,
            forecast_pkg_wt10_with_lags_no_2020.y_train,
            model__sample_weight=forecast_pkg_wt10_with_lags_no_2020.sample_w_train,
        )
    else:
        forecast_pkg_wt10_with_lags_no_2020.search.fit(
            forecast_pkg_wt10_with_lags_no_2020.X_train,
            forecast_pkg_wt10_with_lags_no_2020.y_train,
            groups=forecast_pkg_wt10_with_lags_no_2020.groups_train,
            model__sample_weight=forecast_pkg_wt10_with_lags_no_2020.sample_w_train,
        )
        
    best_forecast_model_wt10_with_lags_no_2020 = forecast_pkg_wt10_with_lags_no_2020.search.best_estimator_

    # --- SAVE ---
    # 3. Save the ENTIRE package
    dump(forecast_pkg_wt10_with_lags_no_2020, pkg_path)
    print(f"Training complete. Package and model saved to {pkg_path}")

#### Model diagnostics (weighted training including lag variables)

In [ ]:
# Run diagnostics

# get train and test predictions and clip test predictions
pred_test = best_forecast_model_wt10_with_lags_no_2020.predict(forecast_pkg_wt10_with_lags_no_2020.X_test)
pred_train = best_forecast_model_wt10_with_lags_no_2020.predict(forecast_pkg_wt10_with_lags_no_2020.X_train)

# get weighted-train unweighted/average-case metrics
r2_train = best_forecast_model_wt10_with_lags_no_2020.score(forecast_pkg_wt10_with_lags_no_2020.X_train,forecast_pkg_wt10_with_lags_no_2020.y_train)
r2_test = best_forecast_model_wt10_with_lags_no_2020.score(forecast_pkg_wt10_with_lags_no_2020.X_test, forecast_pkg_wt10_with_lags_no_2020.y_test)
mae = mean_absolute_error(forecast_pkg_wt10_with_lags_no_2020.y_test, pred_test)
rmse = root_mean_squared_error(forecast_pkg_wt10_with_lags_no_2020.y_test, pred_test)

# get weighted-train unweighted/average-case metrics in log space
log_pred_train = best_forecast_model_wt10_with_lags_no_2020.regressor_.predict(forecast_pkg_wt10_with_lags_no_2020.X_train)  # predictions in transformed target space
log_pred_test = best_forecast_model_wt10_with_lags_no_2020.regressor_.predict(forecast_pkg_wt10_with_lags_no_2020.X_test)  # predictions in transformed target space

r2_log_true_train = r2_score(np.log1p(forecast_pkg_wt10_with_lags_no_2020.y_train), log_pred_train)
r2_log_true_test = r2_score(np.log1p(forecast_pkg_wt10_with_lags_no_2020.y_test), log_pred_test)
mae_log_true = mean_absolute_error(np.log1p(forecast_pkg_wt10_with_lags_no_2020.y_test), log_pred_test)
rmse_log_true = root_mean_squared_error(np.log1p(forecast_pkg_wt10_with_lags_no_2020.y_test), log_pred_test)

# get weighted-train weighted/eval metrics

r2_train_w = r2_score(forecast_pkg_wt10_with_lags_no_2020.y_train, pred_train, sample_weight=forecast_pkg_wt10_with_lags_no_2020.sample_w_train)
r2_test_w = r2_score(forecast_pkg_wt10_with_lags_no_2020.y_test, pred_test, sample_weight=forecast_pkg_wt10_with_lags_no_2020.sample_w_test)
mae_w = mean_absolute_error(forecast_pkg_wt10_with_lags_no_2020.y_test, pred_test, sample_weight=forecast_pkg_wt10_with_lags_no_2020.sample_w_test)
rmse_w = root_mean_squared_error(forecast_pkg_wt10_with_lags_no_2020.y_test, pred_test, sample_weight=forecast_pkg_wt10_with_lags_no_2020.sample_w_test)

# Calculate tail evals
target_col = forecast_pkg_wt10_with_lags_no_2020.target_col
test_eval = forecast_pkg_wt10_with_lags_no_2020.test_df.copy()

test_eval["pred"] = pred_test
test_eval["abs_err"] = (test_eval[target_col] - test_eval["pred"]).abs()
test_eval["sq_err"]  = (test_eval[target_col] - test_eval["pred"])**2

non_tail_pred_to_y_ratio = test_eval.loc[~test_eval["is_top_1pct_avg_mdcr_stdzd_amt"],"pred"].mean() / test_eval.loc[~test_eval["is_top_1pct_avg_mdcr_stdzd_amt"],target_col].mean()
tail_pred_to_y_ratio = test_eval.loc[test_eval["is_top_1pct_avg_mdcr_stdzd_amt"],"pred"].mean() / test_eval.loc[test_eval["is_top_1pct_avg_mdcr_stdzd_amt"],target_col].mean()

non_tail_sse = test_eval.loc[~test_eval["is_top_1pct_avg_mdcr_stdzd_amt"],"sq_err"].sum()
tail_sse = test_eval.loc[test_eval["is_top_1pct_avg_mdcr_stdzd_amt"],"sq_err"].sum()

non_tail_share_of_total_sse = non_tail_sse / (non_tail_sse + tail_sse)
tail_share_of_total_sse = tail_sse / (non_tail_sse + tail_sse)

non_tail_under_prediction_rate = ((test_eval.loc[~test_eval["is_top_1pct_avg_mdcr_stdzd_amt"],"pred"]) < test_eval.loc[~test_eval["is_top_1pct_avg_mdcr_stdzd_amt"],target_col]).mean()
tail_under_prediction_rate = ((test_eval.loc[test_eval["is_top_1pct_avg_mdcr_stdzd_amt"],"pred"]) < test_eval.loc[test_eval["is_top_1pct_avg_mdcr_stdzd_amt"],target_col]).mean()

non_tail_bias = test_eval.loc[~test_eval["is_top_1pct_avg_mdcr_stdzd_amt"],"pred"].mean() - test_eval.loc[~test_eval["is_top_1pct_avg_mdcr_stdzd_amt"],target_col].mean()
tail_bias = test_eval.loc[test_eval["is_top_1pct_avg_mdcr_stdzd_amt"],"pred"].mean() - test_eval.loc[test_eval["is_top_1pct_avg_mdcr_stdzd_amt"],target_col].mean()

forecast_weight10_lag_train_performance_metrics_test_eval = pd.DataFrame({
    "train_sample_weights":[10],
    "r2_train": r2_train,
    "r2_test": r2_test,
    "mae": mae,
    "rmse": rmse,
    "r2_train_log": r2_log_true_train,
    "r2_test_log": r2_log_true_test,
    "mae_log": mae_log_true,
    "rmse_log": rmse_log_true,
    "r2_train_w": r2_train_w,
    "r2_test_w": r2_test_w,
    "mae_w": mae_w,
    "rmse_w": rmse_w,
    "non_tail_pred_to_y_ratio": non_tail_pred_to_y_ratio,
    "tail_pred_to_y_ratio": tail_pred_to_y_ratio,
    "non_tail_sse": non_tail_sse,
    "tail_sse": tail_sse,
    "non_tail_share_of_total_sse": non_tail_share_of_total_sse,
    "tail_share_of_total_sse": tail_share_of_total_sse,
    "non_tail_under_prediction_rate": non_tail_under_prediction_rate,
    "tail_under_prediction_rate": tail_under_prediction_rate,
    "non_tail_bias": non_tail_bias,
    "tail_bias": tail_bias
})

forecast_weight10_lag_train_performance_metrics_test_eval

## Finding baseline: Predict `2023` `avg_mdcr_stdzd_amt` using either "lag only" values or "lag + backoff mean"

### **Baseline 1: Lag-only (no training)**

For each 2023 row:
- Prediction = `lag1_avg_amt`
    - This only works where `lag1_avg_amt` is not null.
    - Rows with `null` lag are “can’t predict” for this baseline (or we can drop them when scoring this baseline).

In [ ]:
from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error

target_col = "avg_mdcr_stdzd_amt"
tail_flag = "is_top_1pct_avg_mdcr_stdzd_amt"

# Updated to use the correctly named package variable
test_df = forecast_pkg_wt10_with_lags_no_2020.test_df.copy()   # should already be 2023 only
y_true = test_df[target_col]

# Lag-only predictions
pred_lag = test_df["lag1_avg_amt"]

# Score only rows where lag exists
mask = pred_lag.notna()
y_true_lag = y_true[mask]
pred_lag_valid = pred_lag[mask]

r2_lag = r2_score(y_true_lag, pred_lag_valid)
mae_lag = mean_absolute_error(y_true_lag, pred_lag_valid)
rmse_lag = root_mean_squared_error(y_true_lag, pred_lag_valid)

print("Lag-only baseline (lag present rows only)")
print("rows:", mask.sum(), "of", len(test_df))
print("r2:", r2_lag, "mae:", mae_lag, "rmse:", rmse_lag)

### **Baseline 2: Lag + backoff mean (uses training data only)**
- Here we fill missing lags with a group mean computed from train years only (2021–2022).
    - Mean over (`provider_type`, `HCPCS_Cd`, `Place_Of_Srvc`)

In [ ]:
# Added this line to ensure test_df is pulled from the correctly named package
test_df = forecast_pkg_wt10_with_lags_no_2020.test_df.copy()  
train_df = forecast_pkg_wt10_with_lags_no_2020.train_df.copy()  # should be 2021-2022 only

group_cols = ["provider_type", "HCPCS_Cd", "Place_Of_Srvc"]
train_means = (
    train_df.groupby(group_cols)[target_col]
    .mean()
    .rename("mean_train_grp")
    .reset_index()
)

test_with_means = test_df.merge(train_means, on=group_cols, how="left")

# Optional fallback if group mean missing (new combo in 2023): mean by HCPCS only
hcpcs_means = (
    train_df.groupby(["HCPCS_Cd"])[target_col]
    .mean()
    .rename("mean_train_hcpcs")
    .reset_index()
)

test_with_means = test_with_means.merge(hcpcs_means, on="HCPCS_Cd", how="left")

global_mean = train_df[target_col].mean()

# Build final baseline pred:
# 1) use lag if present
# 2) else group mean (provider_type, HCPCS, POS)
# 3) else HCPCS mean
# 4) else global mean
pred_backoff = test_with_means["lag1_avg_amt"].copy()
pred_backoff = pred_backoff.fillna(test_with_means["mean_train_grp"])
pred_backoff = pred_backoff.fillna(test_with_means["mean_train_hcpcs"])
pred_backoff = pred_backoff.fillna(global_mean)

y_true = test_with_means[target_col]

r2_b = r2_score(y_true, pred_backoff)
mae_b = mean_absolute_error(y_true, pred_backoff)
rmse_b = root_mean_squared_error(y_true, pred_backoff)

print("Lag + backoff baseline (all rows)")
print("r2:", r2_b, "mae:", mae_b, "rmse:", rmse_b)

#### **Compare against the XGB model on the same metrics**

We already have `pred_test` from XGB. Now compute the same summary for:

- XGB predictions
- lag-only (lag-present rows)
- lag+backoff (all rows)

**1. Tail-only metrics**

In [ ]:
def tail_metrics(df, y_col, pred, tail_flag_col):
    is_tail = df[tail_flag_col].astype(bool)
    out = {}
    for name, m in [("non_tail", ~is_tail), ("tail", is_tail)]:
        yy = df.loc[m, y_col]
        pp = pred.loc[m] if isinstance(pred, pd.Series) else pd.Series(pred, index=df.index).loc[m]
        out[f"{name}_mae"] = mean_absolute_error(yy, pp)
        out[f"{name}_rmse"] = root_mean_squared_error(yy, pp)
        out[f"{name}_pred_to_y_ratio"] = pp.mean() / yy.mean()
        out[f"{name}_bias"] = pp.mean() - yy.mean()
    return out

# XGB
# Updated the index reference to use forecast_pkg_wt10_with_lags
pred_xgb = pd.Series(pred_test, index=forecast_pkg_wt10_with_lags_no_2020.X_test.index)
xgb_tail = tail_metrics(test_df, target_col, pred_xgb, tail_flag)

# Backoff baseline
pred_backoff = pd.Series(pred_backoff.values, index=test_df.index)
b_tail = tail_metrics(test_df, target_col, pred_backoff, tail_flag)

print("XGB tail metrics:", xgb_tail)
print("Baseline tail metrics:", b_tail)

**Results we got:**

- XGB
    - non-tail MAE: 8.31
    - non-tail RMSE: 53.79
    - tail MAE: 171.12
    - tail RMSE: 1219.92
    - tail bias: +51.74

- Baseline (lag + backoff)
    - non-tail MAE: 5.28
    - non-tail RMSE: 33.35
    - tail MAE: 78.95
    - tail RMSE: 271.61
    - tail bias: 11.76

**Interpretation**
- Our simple baseline is dramatically better than XGB, especially in the tail.
- That means, in our current setup, the model is not learning something useful beyond “use the past.”
- Instead it is sometimes producing wild predictions that explode errors.

**2. Worst-case rows**

In [ ]:
# Explicitly pull the test dataframe from your updated package
worst = forecast_pkg_wt10_with_lags_no_2020.test_df.copy()

worst["pred"] = pred_xgb  # using the pred_xgb calculated in the previous step
worst["sq_err"] = (worst[target_col] - worst["pred"])**2

cols = [
    "Rndrng_NPI", "Year", "HCPCS_Cd", "Place_Of_Srvc", "provider_type", "state",
    "services", "lag1_avg_amt", target_col, "pred", "sq_err", tail_flag
]

# Display the 20 rows with the largest squared errors
worst.sort_values("sq_err", ascending=False).head(20)[cols]

**What the worst rows show:**

- The top 5 are all:
    - `HCPCS_Cd` = `J3590`
    - `CA` (California)
    - `Medical Oncology` or `Hematology-Oncology`
    - True value around 3600
    - Lag value around 3633
    - XGB predicts around 28,000 to 31,000

*That is catastrophic overprediction.*

And it explains why our tail RMSE is enormous.

***What this strongly suggests (directly)***
- Our model is sometimes taking a code that is “normally $3,600” and predicting it as if it belongs to the “$30,000” regime.
- The baseline, using lag, is basically perfect here (lag is ~3633). So any sane model should stay near 3633 unless there is a strong reason not to. Yet XGB is jumping to 30k.

**That implies one of these is happening:**
1. The model is not actually using lag features effectively, or lag is getting lost in preprocessing.
2. Some other features are pushing it hard upward (like `services`, `stdzd_spend`, provider context) and the model is overreacting.
3. Train/test regime mismatch for what we are trying to predict, so it learns relationships that do not hold in 2023.
4. Feature leakage-like behavior inside the model inputs (not the classic kind from the future, but “proxy leakage” where a feature almost encodes the target differently in train vs test).
5. Objective mismatch with our evaluation goal, for example training in log space but evaluating in dollar space can create weird incentives unless handled carefully.

Given our baseline performance, the strongest immediate hypothesis is (1) or (2).

#### **Let's quantify “insane jumps” vs lag**

Thresholds like “more than 5x or more than $1000” can be useful. Let's do both.

**1. Run on rows where lag exists: percentage of cases insane jumps happen**

Let's also compute how often those insane jumps are actually helpful

In [ ]:
# Explicitly pull the test dataframe from your updated package
df = forecast_pkg_wt10_with_lags_no_2020.test_df.copy()

mask = df["lag1_avg_amt"].notna()

df = df.loc[mask].copy()
df["abs_jump"] = (pred_xgb.loc[df.index] - df["lag1_avg_amt"]).abs()
df["ratio_jump"] = df["abs_jump"] / (df["lag1_avg_amt"].abs() + 1e-9)

jump_1000 = (df["abs_jump"] > 1000).mean()
jump_5x = (df["ratio_jump"] > 5).mean()

print("Share with abs jump > 1000:", jump_1000)
print("Share with ratio jump > 5x:", jump_5x)

**2. Let's also quantify the MAE: XGB vs LAG (in general and where jumps are huge):**

In [ ]:
df["abs_err_xgb"] = (df[target_col] - pred_xgb.loc[df.index]).abs()
df["abs_err_lag"] = (df[target_col] - df["lag1_avg_amt"]).abs()

print("MAE XGB:", df["abs_err_xgb"].mean())
print("MAE LAG:", df["abs_err_lag"].mean())

# On insane-jump rows only
insane = (df["abs_jump"] > 1000) | (df["ratio_jump"] > 5)
print("MAE XGB insane:", df.loc[insane, "abs_err_xgb"].mean())
print("MAE LAG insane:", df.loc[insane, "abs_err_lag"].mean())

#### **Why J3590 specifically is showing up**

From our table, J3590 appears in California, oncology provider types, POS O, and is flagged as top 1 percent. So it is a tail-associated code in training.

Our model likely learned a branch like:
“If HCPCS is J3590 and provider_type is oncology and state is CA and POS is O, predict huge.”

That branch might be correct for some training cases, but it is clearly wrong for these particular NPIs because lag is telling you the truth: they were around $3600 and stayed around $3600.

So the fix is to force the model to respect lag, either by delta modeling or by explicit guardrails.

In [ ]:
forecast_pkg_wt10_with_lags_no_2020.cat_features

In [ ]:
forecast_pkg_wt10_with_lags_no_2020.num_features

In [ ]:
forecast_pkg_wt10_with_lags_no_2020.search.best_params_

## 1. Changing the modeling target to `log_delta_cost` (difference between `avg_mdcr_stdzd_amt` and `lag1_avg_amt`) and using `prepare_xgb_gridsearch` for modeling package

#### Adding `log_delta_cost` to the `eda_hcpcs_df` dataframe

In [ ]:
# Keep full data intact
eda_hcpcs_full_df = eda_hcpcs_df.copy()

# Lag-present mask
mask_lag = (
    eda_hcpcs_full_df["lag1_avg_amt"].notna()
    & (eda_hcpcs_full_df["lag1_avg_amt"] >= 0)
)

# Subset only for residual-model training
eda_hcpcs_lags_present_df = eda_hcpcs_full_df.loc[mask_lag].copy()

# Residual target (log space delta)
eda_hcpcs_lags_present_df["log_delta_cost"] = (
    np.log1p(eda_hcpcs_lags_present_df["avg_mdcr_stdzd_amt"])
    - np.log1p(eda_hcpcs_lags_present_df["lag1_avg_amt"])
)

In [ ]:
from pathlib import Path
from joblib import dump, load

# 1. Setup Paths
ROOT = Path.cwd().resolve()
MODELS = (ROOT / "models").resolve()
MODELS.mkdir(parents=True, exist_ok=True)

# We now point to a package file instead of just the model file
pkg_path = MODELS / "forecast_pkg_wt10_with_lags_no_2020_delta_trgt.joblib"

# 2. Check/Load or Train/Save
if pkg_path.exists():
    print(f"Loading {pkg_path.name} from disk... (Skipping data prep and training)")
    # Load the entire package (data + fitted search object)
    forecast_pkg_wt10_with_lags_no_2020_delta_trgt = load(pkg_path)
    
    # Extract the model so downstream code works as expected
    best_forecast_model_wt10_with_lags_no_2020_delta_trgt = forecast_pkg_wt10_with_lags_no_2020_delta_trgt.search.best_estimator_

else:
    print(f"File not found. Preparing data and training...")

    # --- PREPARE DATA ---
    forecast_pkg_wt10_with_lags_no_2020_delta_trgt = prepare_xgb_gridsearch(
        eda_hcpcs_lags_present_df,
        approach="forecast",
        target_col="log_delta_cost",
        w_tail=10,
        include_lags=True,
        incl_2020_in_final_fit=False,
        use_ttr=False
    )

    # --- TRAIN ---
    if forecast_pkg_wt10_with_lags_no_2020_delta_trgt.groups_train is None:
        forecast_pkg_wt10_with_lags_no_2020_delta_trgt.search.fit(
            forecast_pkg_wt10_with_lags_no_2020_delta_trgt.X_train,
            forecast_pkg_wt10_with_lags_no_2020_delta_trgt.y_train,
            model__sample_weight=forecast_pkg_wt10_with_lags_no_2020_delta_trgt.sample_w_train,
        )
    else:
        forecast_pkg_wt10_with_lags_no_2020_delta_trgt.search.fit(
            forecast_pkg_wt10_with_lags_no_2020_delta_trgt.X_train,
            forecast_pkg_wt10_with_lags_no_2020_delta_trgt.y_train,
            groups=forecast_pkg_wt10_with_lags_no_2020_delta_trgt.groups_train,
            model__sample_weight=forecast_pkg_wt10_with_lags_no_2020_delta_trgt.sample_w_train,
        )
        
    best_forecast_model_wt10_with_lags_no_2020_delta_trgt = forecast_pkg_wt10_with_lags_no_2020_delta_trgt.search.best_estimator_

    # --- SAVE ---
    # 3. Save the ENTIRE package
    dump(forecast_pkg_wt10_with_lags_no_2020_delta_trgt, pkg_path)
    print(f"Training complete. Package and model saved to {pkg_path}")

### Hybrid approach for predictions: use model for lag-present rows and use lag + backoff for lag-absent rows. Use "Baseline 2 fallback ladder": `mean_train_grp` --> `mean_train_hcpcs` --> `global_mean`.

In [ ]:
import numpy as np
import pandas as pd

target_col = "avg_mdcr_stdzd_amt"
tail_flag_col = "is_top_1pct_avg_mdcr_stdzd_amt"
group_cols = ["provider_type", "HCPCS_Cd", "Place_Of_Srvc"]

# 1) Full 2023 test set (all rows, lag and no-lag)
test_df_full_2023 = eda_hcpcs_full_df.loc[eda_hcpcs_full_df["Year"] == 2023].copy()

# 2) Choose which years define the backoff means
# If you want to match the package that trained on 2021-2022 only:
train_years_for_means = [2021, 2022]
# If you intentionally want more history for means, use:
# train_years_for_means = [2020, 2021, 2022]

train_df_for_means = eda_hcpcs_full_df.loc[
    eda_hcpcs_full_df["Year"].isin(train_years_for_means)
].copy()

# 3) Compute backoff mean at (provider_type, HCPCS, POS)
train_means_grp = (
    train_df_for_means.groupby(group_cols)[target_col]
    .mean()
    .rename("mean_train_grp")
    .reset_index()
)

# Optional fallback: HCPCS-only mean
train_means_hcpcs = (
    train_df_for_means.groupby(["HCPCS_Cd"])[target_col]
    .mean()
    .rename("mean_train_hcpcs")
    .reset_index()
)

global_mean = float(train_df_for_means[target_col].mean())

# 4) Attach means to the FULL 2023 test set (preserve original index)
test_with_means = test_df_full_2023.copy()
test_with_means["_orig_idx"] = test_with_means.index

test_with_means = test_with_means.merge(train_means_grp, on=group_cols, how="left")
test_with_means = test_with_means.merge(train_means_hcpcs, on="HCPCS_Cd", how="left")

# Restore original index
test_with_means = test_with_means.set_index("_orig_idx", drop=True)
test_with_means = test_with_means.loc[test_df.index]  # keep same order

# 5) Split lag vs no-lag for residual model
mask_2023_lag = test_with_means["lag1_avg_amt"].notna() & (test_with_means["lag1_avg_amt"] >= 0)
test_2023_lag = test_with_means.loc[mask_2023_lag].copy()
test_2023_nolag = test_with_means.loc[~mask_2023_lag].copy()

# 6) Residual model prediction for lag-present rows
feature_cols = forecast_pkg_wt10_with_lags_no_2020_delta_trgt.feature_cols
X_2023_lag = test_2023_lag[feature_cols]

log_delta_hat = best_forecast_model_wt10_with_lags_no_2020_delta_trgt.predict(X_2023_lag)

# predictions for the lag-present rows: take lag1_avg_amt and add the model-predicted log_delta_hat
pred_level_lag = np.expm1(np.log1p(test_2023_lag["lag1_avg_amt"].values) + log_delta_hat)

# 7) Backoff prediction for no-lag rows (group mean -> HCPCS mean -> global)
pred_level_nolag = (
    test_2023_nolag["mean_train_grp"]
    .fillna(test_2023_nolag["mean_train_hcpcs"])
    .fillna(global_mean)
    .to_numpy()
)

# 8) Combine into a full prediction vector aligned to original 2023 test index
pred_full = pd.Series(index=test_df_full_2023.index, dtype=float)
pred_full.loc[test_2023_lag.index] = pred_level_lag
pred_full.loc[test_2023_nolag.index] = pred_level_nolag

# Rebuild pred_backoff

pred_backoff = test_with_means["lag1_avg_amt"].copy()
pred_backoff = pred_backoff.fillna(test_with_means["mean_train_grp"])
pred_backoff = pred_backoff.fillna(test_with_means["mean_train_hcpcs"])
pred_backoff = pred_backoff.fillna(global_mean)

### Calculate model metrics and tail performance

In [ ]:
from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# Inputs you already have from your hybrid prediction cell:
# - test_df_full_2023  (full 2023 rows)
# - pred_full          (full 2023 predictions aligned to test_df_full_2023.index)
# ------------------------------------------------------------

target_col = "avg_mdcr_stdzd_amt"
tail_flag_col = "is_top_1pct_avg_mdcr_stdzd_amt"

y_true = test_df_full_2023[target_col].astype(float)
pred_test = pred_full.astype(float)

# Basic sanity check
assert pred_test.index.equals(test_df_full_2023.index)

# ------------------------------------------------------------
# 1) Overall metrics (2023 full)
# ------------------------------------------------------------
r2_test = r2_score(y_true, pred_test)
mae = mean_absolute_error(y_true, pred_test)
rmse = root_mean_squared_error(y_true, pred_test)

# "log space" metrics on LEVELS (not on delta), if you still want them:
r2_log_true_test = r2_score(np.log1p(y_true), np.log1p(pred_test.clip(lower=0)))
mae_log_true = mean_absolute_error(np.log1p(y_true), np.log1p(pred_test.clip(lower=0)))
rmse_log_true = root_mean_squared_error(np.log1p(y_true), np.log1p(pred_test.clip(lower=0)))

# ------------------------------------------------------------
# 2) Weighted (tail-emphasized) metrics on 2023 full
#    Use the SAME w_tail you used for weighting (10 here)
# ------------------------------------------------------------
w_tail = 10
is_tail_test = test_df_full_2023[tail_flag_col].astype(int).to_numpy()
sample_w_test = (1 + (w_tail - 1) * is_tail_test).astype(float)

r2_test_w = r2_score(y_true, pred_test, sample_weight=sample_w_test)
mae_w = mean_absolute_error(y_true, pred_test, sample_weight=sample_w_test)
rmse_w = root_mean_squared_error(y_true, pred_test, sample_weight=sample_w_test)

# ------------------------------------------------------------
# 3) Tail analysis table (same style as before)
# ------------------------------------------------------------
test_eval = test_df_full_2023.copy()
test_eval["pred"] = pred_test
test_eval["abs_err"] = (test_eval[target_col] - test_eval["pred"]).abs()
test_eval["sq_err"] = (test_eval[target_col] - test_eval["pred"]) ** 2

non_tail_pred_to_y_ratio = (
    test_eval.loc[~test_eval[tail_flag_col], "pred"].mean()
    / test_eval.loc[~test_eval[tail_flag_col], target_col].mean()
)
tail_pred_to_y_ratio = (
    test_eval.loc[test_eval[tail_flag_col], "pred"].mean()
    / test_eval.loc[test_eval[tail_flag_col], target_col].mean()
)

non_tail_sse = test_eval.loc[~test_eval[tail_flag_col], "sq_err"].sum()
tail_sse = test_eval.loc[test_eval[tail_flag_col], "sq_err"].sum()

non_tail_share_of_total_sse = non_tail_sse / (non_tail_sse + tail_sse)
tail_share_of_total_sse = tail_sse / (non_tail_sse + tail_sse)

non_tail_under_prediction_rate = (
    (test_eval.loc[~test_eval[tail_flag_col], "pred"] < test_eval.loc[~test_eval[tail_flag_col], target_col]).mean()
)
tail_under_prediction_rate = (
    (test_eval.loc[test_eval[tail_flag_col], "pred"] < test_eval.loc[test_eval[tail_flag_col], target_col]).mean()
)

non_tail_bias = (
    test_eval.loc[~test_eval[tail_flag_col], "pred"].mean()
    - test_eval.loc[~test_eval[tail_flag_col], target_col].mean()
)
tail_bias = (
    test_eval.loc[test_eval[tail_flag_col], "pred"].mean()
    - test_eval.loc[test_eval[tail_flag_col], target_col].mean()
)

forecast_hybrid_metrics_2023 = pd.DataFrame({
    "w_tail":[w_tail],
    "r2_test": [r2_test],
    "mae": [mae],
    "rmse": [rmse],
    "r2_test_log": [r2_log_true_test],
    "mae_log": [mae_log_true],
    "rmse_log": [rmse_log_true],
    "r2_test_w": [r2_test_w],
    "mae_w": [mae_w],
    "rmse_w": [rmse_w],
    "non_tail_pred_to_y_ratio": [non_tail_pred_to_y_ratio],
    "tail_pred_to_y_ratio": [tail_pred_to_y_ratio],
    "non_tail_sse": [non_tail_sse],
    "tail_sse": [tail_sse],
    "non_tail_share_of_total_sse": [non_tail_share_of_total_sse],
    "tail_share_of_total_sse": [tail_share_of_total_sse],
    "non_tail_under_prediction_rate": [non_tail_under_prediction_rate],
    "tail_under_prediction_rate": [tail_under_prediction_rate],
    "non_tail_bias": [non_tail_bias],
    "tail_bias": [tail_bias],
})

forecast_hybrid_metrics_2023

## 2. Recalculate "baseline 2" with all the columns as `forecast_hybrid_metrics_2023`: lag + backoff mean (using train data only) 

In [ ]:
from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error
import numpy as np
import pandas as pd

tail_flag_col = "is_top_1pct_avg_mdcr_stdzd_amt"
target_col = "avg_mdcr_stdzd_amt"
w_tail = 10

eval_df = test_with_means.copy()
pred = pred_backoff.astype(float)   # aligned to test_with_means
y_true = eval_df[target_col].astype(float)

# local weights aligned to THIS evaluation frame
sample_w = (
    1 + (w_tail - 1) * eval_df[tail_flag_col].astype(int).to_numpy()
).astype(float)

eval_df["pred"] = pred
is_tail = eval_df[tail_flag_col].astype(bool)
is_non_tail = ~is_tail

eval_df["abs_err"] = (eval_df[target_col] - eval_df["pred"]).abs()
eval_df["sq_err"] = (eval_df[target_col] - eval_df["pred"]) ** 2

# overall
r2 = r2_score(y_true, pred)
mae = mean_absolute_error(y_true, pred)
rmse = root_mean_squared_error(y_true, pred)

# log-level (optional)
pred_clip = pred.clip(lower=0)
r2_log = r2_score(np.log1p(y_true), np.log1p(pred_clip))
mae_log = mean_absolute_error(np.log1p(y_true), np.log1p(pred_clip))
rmse_log = root_mean_squared_error(np.log1p(y_true), np.log1p(pred_clip))

# weighted
r2_w = r2_score(y_true, pred, sample_weight=sample_w)
mae_w = mean_absolute_error(y_true, pred, sample_weight=sample_w)
rmse_w = root_mean_squared_error(y_true, pred, sample_weight=sample_w)

non_tail_pred_to_y_ratio = eval_df.loc[is_non_tail, "pred"].mean() / eval_df.loc[is_non_tail, target_col].mean()
tail_pred_to_y_ratio = eval_df.loc[is_tail, "pred"].mean() / eval_df.loc[is_tail, target_col].mean()

non_tail_sse = eval_df.loc[is_non_tail, "sq_err"].sum()
tail_sse = eval_df.loc[is_tail, "sq_err"].sum()

non_tail_share = non_tail_sse / (non_tail_sse + tail_sse)
tail_share = tail_sse / (non_tail_sse + tail_sse)

non_tail_upr = (eval_df.loc[is_non_tail, "pred"] < eval_df.loc[is_non_tail, target_col]).mean()
tail_upr = (eval_df.loc[is_tail, "pred"] < eval_df.loc[is_tail, target_col]).mean()

non_tail_bias = eval_df.loc[is_non_tail, "pred"].mean() - eval_df.loc[is_non_tail, target_col].mean()
tail_bias = eval_df.loc[is_tail, "pred"].mean() - eval_df.loc[is_tail, target_col].mean()

baseline_metrics_2023 = pd.DataFrame({
    "w_tail":[w_tail],
    "r2_test":[r2],
    "mae":[mae],
    "rmse":[rmse],
    "r2_test_log":[r2_log],
    "mae_log":[mae_log],
    "rmse_log":[rmse_log],
    "r2_test_w":[r2_w],
    "mae_w":[mae_w],
    "rmse_w":[rmse_w],
    "non_tail_pred_to_y_ratio":[non_tail_pred_to_y_ratio],
    "tail_pred_to_y_ratio":[tail_pred_to_y_ratio],
    "non_tail_sse":[non_tail_sse],
    "tail_sse":[tail_sse],
    "non_tail_share_of_total_sse":[non_tail_share],
    "tail_share_of_total_sse":[tail_share],
    "non_tail_under_prediction_rate":[non_tail_upr],
    "tail_under_prediction_rate":[tail_upr],
    "non_tail_bias":[non_tail_bias],
    "tail_bias":[tail_bias],
})

baseline_metrics_2023

> **Baseline 2 (lag + backoff mean) beats our most sophisticated model!**

## 1. Changing the modeling target to `log_delta_cost` (difference between `avg_mdcr_stdzd_amt` and `lag1_avg_amt`) and using `prepare_xgb_randomizedsearch` for modeling package

In [ ]:
from pathlib import Path
from joblib import dump, load

# 1. Setup Paths
ROOT = Path.cwd().resolve()
MODELS = (ROOT / "models").resolve()
MODELS.mkdir(parents=True, exist_ok=True)

# We now point to a package file instead of just the model file
pkg_path = MODELS / "forecast_pkg_wt10_with_lags_no_2020_delta_trgt_rs50.joblib"

# 2. Check/Load or Train/Save
if pkg_path.exists():
    print(f"Loading {pkg_path.name} from disk... (Skipping data prep and training)")
    # Load the entire package (data + fitted search object)
    forecast_pkg_wt10_with_lags_no_2020_delta_trgt_rs50 = load(pkg_path)
    
    # Extract the model so downstream code works as expected
    best_forecast_model_wt10_with_lags_no_2020_delta_trgt_rs50 = forecast_pkg_wt10_with_lags_no_2020_delta_trgt_rs50.search.best_estimator_

else:
    print(f"File not found. Preparing data and training...")

    # --- PREPARE DATA ---
    forecast_pkg_wt10_with_lags_no_2020_delta_trgt_rs50 = prepare_xgb_randomizedsearch(
        eda_hcpcs_lags_present_df,
        approach="forecast",
        target_col="log_delta_cost",
        w_tail=10,
        include_lags=True,
        incl_2020_in_final_fit=False,
        use_ttr=False
    )

    # --- TRAIN ---
    if forecast_pkg_wt10_with_lags_no_2020_delta_trgt_rs50.groups_train is None:
        forecast_pkg_wt10_with_lags_no_2020_delta_trgt_rs50.search.fit(
            forecast_pkg_wt10_with_lags_no_2020_delta_trgt_rs50.X_train,
            forecast_pkg_wt10_with_lags_no_2020_delta_trgt_rs50.y_train,
            model__sample_weight=forecast_pkg_wt10_with_lags_no_2020_delta_trgt_rs50.sample_w_train,
        )
    else:
        forecast_pkg_wt10_with_lags_no_2020_delta_trgt_rs50.search.fit(
            forecast_pkg_wt10_with_lags_no_2020_delta_trgt_rs50.X_train,
            forecast_pkg_wt10_with_lags_no_2020_delta_trgt_rs50.y_train,
            groups=forecast_pkg_wt10_with_lags_no_2020_delta_trgt_rs50.groups_train,
            model__sample_weight=forecast_pkg_wt10_with_lags_no_2020_delta_trgt_rs50.sample_w_train,
        )
        
    best_forecast_model_wt10_with_lags_no_2020_delta_trgt_rs50 = forecast_pkg_wt10_with_lags_no_2020_delta_trgt_rs50.search.best_estimator_

    # --- SAVE ---
    # 3. Save the ENTIRE package
    dump(forecast_pkg_wt10_with_lags_no_2020_delta_trgt_rs50, pkg_path)
    print(f"Training complete. Package and model saved to {pkg_path}")

### Hybrid approach for predictions: use model for lag-present rows and use lag + backoff for lag-absent rows: Use "Baseline 2" for filling in the lag-missing values of the test target

In [ ]:
import numpy as np
import pandas as pd

target_col = "avg_mdcr_stdzd_amt"
tail_flag_col = "is_top_1pct_avg_mdcr_stdzd_amt"
group_cols = ["provider_type", "HCPCS_Cd", "Place_Of_Srvc"]

# 1) Full 2023 test set (all rows, lag and no-lag)
test_df_full_2023 = eda_hcpcs_full_df.loc[eda_hcpcs_full_df["Year"] == 2023].copy()

# 2) Choose which years define the backoff means
# If you want to match the package that trained on 2021-2022 only:
train_years_for_means = [2021, 2022]
# If you intentionally want more history for means, use:
# train_years_for_means = [2020, 2021, 2022]

train_df_for_means = eda_hcpcs_full_df.loc[
    eda_hcpcs_full_df["Year"].isin(train_years_for_means)
].copy()

# 3) Compute backoff mean at (provider_type, HCPCS, POS)
train_means_grp = (
    train_df_for_means.groupby(group_cols)[target_col]
    .mean()
    .rename("mean_train_grp")
    .reset_index()
)

# Optional fallback: HCPCS-only mean
train_means_hcpcs = (
    train_df_for_means.groupby(["HCPCS_Cd"])[target_col]
    .mean()
    .rename("mean_train_hcpcs")
    .reset_index()
)

global_mean = float(train_df_for_means[target_col].mean())

# 4) Attach means to the FULL 2023 test set (preserve original index)
test_with_means = test_df_full_2023.copy()
test_with_means["_orig_idx"] = test_with_means.index

test_with_means = test_with_means.merge(train_means_grp, on=group_cols, how="left")
test_with_means = test_with_means.merge(train_means_hcpcs, on="HCPCS_Cd", how="left")

# Restore original index
test_with_means = test_with_means.set_index("_orig_idx", drop=True)
test_with_means = test_with_means.loc[test_df.index]  # keep same order

# 5) Split lag vs no-lag for residual model
mask_2023_lag = test_with_means["lag1_avg_amt"].notna() & (test_with_means["lag1_avg_amt"] >= 0)
test_2023_lag = test_with_means.loc[mask_2023_lag].copy()
test_2023_nolag = test_with_means.loc[~mask_2023_lag].copy()

# 6) Residual model prediction for lag-present rows
feature_cols = forecast_pkg_wt10_with_lags_no_2020_delta_trgt_rs50.feature_cols
X_2023_lag = test_2023_lag[feature_cols]

log_delta_hat = best_forecast_model_wt10_with_lags_no_2020_delta_trgt_rs50.predict(X_2023_lag)

pred_level_lag = np.expm1(np.log1p(test_2023_lag["lag1_avg_amt"].values) + log_delta_hat)

# 7) Backoff prediction for no-lag rows (group mean -> HCPCS mean -> global)
pred_level_nolag = (
    test_2023_nolag["mean_train_grp"]
    .fillna(test_2023_nolag["mean_train_hcpcs"])
    .fillna(global_mean)
    .to_numpy()
)

# 8) Combine into a full prediction vector aligned to original 2023 test index
pred_full = pd.Series(index=test_df_full_2023.index, dtype=float)
pred_full.loc[test_2023_lag.index] = pred_level_lag
pred_full.loc[test_2023_nolag.index] = pred_level_nolag

# Rebuild pred_backoff

pred_backoff = test_with_means["lag1_avg_amt"].copy()
pred_backoff = pred_backoff.fillna(test_with_means["mean_train_grp"])
pred_backoff = pred_backoff.fillna(test_with_means["mean_train_hcpcs"])
pred_backoff = pred_backoff.fillna(global_mean)

### Calculate model metrics and tail performance

In [ ]:
from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# Inputs you already have from your hybrid prediction cell:
# - test_df_full_2023  (full 2023 rows)
# - pred_full          (full 2023 predictions aligned to test_df_full_2023.index)
# ------------------------------------------------------------

target_col = "avg_mdcr_stdzd_amt"
tail_flag_col = "is_top_1pct_avg_mdcr_stdzd_amt"

y_true = test_df_full_2023[target_col].astype(float)
pred_test = pred_full.astype(float)

# Basic sanity check
assert pred_test.index.equals(test_df_full_2023.index)

# ------------------------------------------------------------
# 1) Overall metrics (2023 full)
# ------------------------------------------------------------
r2_test = r2_score(y_true, pred_test)
mae = mean_absolute_error(y_true, pred_test)
rmse = root_mean_squared_error(y_true, pred_test)

# "log space" metrics on LEVELS (not on delta), if you still want them:
r2_log_true_test = r2_score(np.log1p(y_true), np.log1p(pred_test.clip(lower=0)))
mae_log_true = mean_absolute_error(np.log1p(y_true), np.log1p(pred_test.clip(lower=0)))
rmse_log_true = root_mean_squared_error(np.log1p(y_true), np.log1p(pred_test.clip(lower=0)))

# ------------------------------------------------------------
# 2) Weighted (tail-emphasized) metrics on 2023 full
#    Use the SAME w_tail you used for weighting (10 here)
# ------------------------------------------------------------
w_tail = 10
is_tail_test = test_df_full_2023[tail_flag_col].astype(int).to_numpy()
sample_w_test = (1 + (w_tail - 1) * is_tail_test).astype(float)

r2_test_w = r2_score(y_true, pred_test, sample_weight=sample_w_test)
mae_w = mean_absolute_error(y_true, pred_test, sample_weight=sample_w_test)
rmse_w = root_mean_squared_error(y_true, pred_test, sample_weight=sample_w_test)

# ------------------------------------------------------------
# 3) Tail analysis table (same style as before)
# ------------------------------------------------------------
test_eval = test_df_full_2023.copy()
test_eval["pred"] = pred_test
test_eval["abs_err"] = (test_eval[target_col] - test_eval["pred"]).abs()
test_eval["sq_err"] = (test_eval[target_col] - test_eval["pred"]) ** 2

non_tail_pred_to_y_ratio = (
    test_eval.loc[~test_eval[tail_flag_col], "pred"].mean()
    / test_eval.loc[~test_eval[tail_flag_col], target_col].mean()
)
tail_pred_to_y_ratio = (
    test_eval.loc[test_eval[tail_flag_col], "pred"].mean()
    / test_eval.loc[test_eval[tail_flag_col], target_col].mean()
)

non_tail_sse = test_eval.loc[~test_eval[tail_flag_col], "sq_err"].sum()
tail_sse = test_eval.loc[test_eval[tail_flag_col], "sq_err"].sum()

non_tail_share_of_total_sse = non_tail_sse / (non_tail_sse + tail_sse)
tail_share_of_total_sse = tail_sse / (non_tail_sse + tail_sse)

non_tail_under_prediction_rate = (
    (test_eval.loc[~test_eval[tail_flag_col], "pred"] < test_eval.loc[~test_eval[tail_flag_col], target_col]).mean()
)
tail_under_prediction_rate = (
    (test_eval.loc[test_eval[tail_flag_col], "pred"] < test_eval.loc[test_eval[tail_flag_col], target_col]).mean()
)

non_tail_bias = (
    test_eval.loc[~test_eval[tail_flag_col], "pred"].mean()
    - test_eval.loc[~test_eval[tail_flag_col], target_col].mean()
)
tail_bias = (
    test_eval.loc[test_eval[tail_flag_col], "pred"].mean()
    - test_eval.loc[test_eval[tail_flag_col], target_col].mean()
)

forecast_hybrid_metrics_2023_rs50 = pd.DataFrame({
    "w_tail":[w_tail],
    "r2_test": [r2_test],
    "mae": [mae],
    "rmse": [rmse],
    "r2_test_log": [r2_log_true_test],
    "mae_log": [mae_log_true],
    "rmse_log": [rmse_log_true],
    "r2_test_w": [r2_test_w],
    "mae_w": [mae_w],
    "rmse_w": [rmse_w],
    "non_tail_pred_to_y_ratio": [non_tail_pred_to_y_ratio],
    "tail_pred_to_y_ratio": [tail_pred_to_y_ratio],
    "non_tail_sse": [non_tail_sse],
    "tail_sse": [tail_sse],
    "non_tail_share_of_total_sse": [non_tail_share_of_total_sse],
    "tail_share_of_total_sse": [tail_share_of_total_sse],
    "non_tail_under_prediction_rate": [non_tail_under_prediction_rate],
    "tail_under_prediction_rate": [tail_under_prediction_rate],
    "non_tail_bias": [non_tail_bias],
    "tail_bias": [tail_bias],
})

forecast_hybrid_metrics_2023_rs50

In [ ]:
forecast_pkg_wt10_with_lags_no_2020_delta_trgt_rs50.search.best_params_

## Narrowing the `param_distributions` to generate a more strinctly tuned model: using `prepare_xgb_randomizedsearch()`

In [ ]:
from pathlib import Path
from joblib import dump, load

# 1. Setup Paths
ROOT = Path.cwd().resolve()
MODELS = (ROOT / "models").resolve()
MODELS.mkdir(parents=True, exist_ok=True)

# We now point to a package file instead of just the model file
pkg_path = MODELS / "forecast_pkg_wt10_with_lags_no_2020_delta_trgt_rs50_retune_v1.joblib"

# 2. Check/Load or Train/Save
if pkg_path.exists():
    print(f"Loading {pkg_path.name} from disk... (Skipping data prep and training)")
    # Load the entire package (data + fitted search object)
    forecast_pkg_wt10_with_lags_no_2020_delta_trgt_rs50_retune_v1 = load(pkg_path)
    
    # Extract the model so downstream code works as expected
    best_forecast_model_wt10_with_lags_no_2020_delta_trgt_rs50_retune_v1 = forecast_pkg_wt10_with_lags_no_2020_delta_trgt_rs50_retune_v1.search.best_estimator_

else:
    print(f"File not found. Preparing data and training...")

    # --- PREPARE DATA ---
    forecast_pkg_wt10_with_lags_no_2020_delta_trgt_rs50_retune_v1 = prepare_xgb_randomizedsearch(
        eda_hcpcs_lags_present_df,
        approach="forecast",
        target_col="log_delta_cost",
        w_tail=10,
        include_lags=True,
        incl_2020_in_final_fit=False,
        use_ttr=False,
        param_distributions = {
            "model__n_estimators": randint(500, 1300),
            "model__learning_rate": loguniform(0.01, 0.05),

            "model__max_depth": randint(1, 4),              # cap depth
            "model__min_child_weight": randint(30, 151),    # even more conservative
            "model__gamma": uniform(0.0, 3.0),

            "model__subsample": uniform(0.6, 0.35),
            "model__colsample_bytree": uniform(0.6, 0.35),

            "model__reg_lambda": loguniform(10.0, 500.0),   # stronger L2
            "model__reg_alpha": loguniform(1e-6, 0.1),

            "model__max_delta_step": randint(0, 11),
            "model__max_bin": [256, 512],
        }
    )

    # --- TRAIN ---
    if forecast_pkg_wt10_with_lags_no_2020_delta_trgt_rs50_retune_v1.groups_train is None:
        forecast_pkg_wt10_with_lags_no_2020_delta_trgt_rs50_retune_v1.search.fit(
            forecast_pkg_wt10_with_lags_no_2020_delta_trgt_rs50_retune_v1.X_train,
            forecast_pkg_wt10_with_lags_no_2020_delta_trgt_rs50_retune_v1.y_train,
            model__sample_weight=forecast_pkg_wt10_with_lags_no_2020_delta_trgt_rs50_retune_v1.sample_w_train,
        )
    else:
        forecast_pkg_wt10_with_lags_no_2020_delta_trgt_rs50_retune_v1.search.fit(
            forecast_pkg_wt10_with_lags_no_2020_delta_trgt_rs50_retune_v1.X_train,
            forecast_pkg_wt10_with_lags_no_2020_delta_trgt_rs50_retune_v1.y_train,
            groups=forecast_pkg_wt10_with_lags_no_2020_delta_trgt_rs50_retune_v1.groups_train,
            model__sample_weight=forecast_pkg_wt10_with_lags_no_2020_delta_trgt_rs50_retune_v1.sample_w_train,
        )
        
    best_forecast_model_wt10_with_lags_no_2020_delta_trgt_rs50_retune_v1 = forecast_pkg_wt10_with_lags_no_2020_delta_trgt_rs50_retune_v1.search.best_estimator_

    # --- SAVE ---
    # 3. Save the ENTIRE package
    dump(forecast_pkg_wt10_with_lags_no_2020_delta_trgt_rs50_retune_v1, pkg_path)
    print(f"Training complete. Package and model saved to {pkg_path}")

### Hybrid approach for predictions: use model for lag-present rows and use lag + backoff for lag-absent rows (`..._retune_v1`)

In [ ]:
import numpy as np
import pandas as pd

target_col = "avg_mdcr_stdzd_amt"
tail_flag_col = "is_top_1pct_avg_mdcr_stdzd_amt"
group_cols = ["provider_type", "HCPCS_Cd", "Place_Of_Srvc"]

# 1) Full 2023 test set (all rows, lag and no-lag)
test_df_full_2023 = eda_hcpcs_full_df.loc[eda_hcpcs_full_df["Year"] == 2023].copy()

# 2) Choose which years define the backoff means
# If you want to match the package that trained on 2021-2022 only:
train_years_for_means = [2021, 2022]
# If you intentionally want more history for means, use:
# train_years_for_means = [2020, 2021, 2022]

train_df_for_means = eda_hcpcs_full_df.loc[
    eda_hcpcs_full_df["Year"].isin(train_years_for_means)
].copy()

# 3) Compute backoff mean at (provider_type, HCPCS, POS)
train_means_grp = (
    train_df_for_means.groupby(group_cols)[target_col]
    .mean()
    .rename("mean_train_grp")
    .reset_index()
)

# Optional fallback: HCPCS-only mean
train_means_hcpcs = (
    train_df_for_means.groupby(["HCPCS_Cd"])[target_col]
    .mean()
    .rename("mean_train_hcpcs")
    .reset_index()
)

global_mean = float(train_df_for_means[target_col].mean())

# 4) Attach means to the FULL 2023 test set (preserve original index)
test_with_means = test_df_full_2023.copy()
test_with_means["_orig_idx"] = test_with_means.index

test_with_means = test_with_means.merge(train_means_grp, on=group_cols, how="left")
test_with_means = test_with_means.merge(train_means_hcpcs, on="HCPCS_Cd", how="left")

# Restore original index
test_with_means = test_with_means.set_index("_orig_idx", drop=True)
test_with_means = test_with_means.loc[test_df.index]  # keep same order

# 5) Split lag vs no-lag for residual model
mask_2023_lag = test_with_means["lag1_avg_amt"].notna() & (test_with_means["lag1_avg_amt"] >= 0)
test_2023_lag = test_with_means.loc[mask_2023_lag].copy()
test_2023_nolag = test_with_means.loc[~mask_2023_lag].copy()

# 6) Residual model prediction for lag-present rows
feature_cols = forecast_pkg_wt10_with_lags_no_2020_delta_trgt_rs50_retune_v1.feature_cols
X_2023_lag = test_2023_lag[feature_cols]

log_delta_hat = best_forecast_model_wt10_with_lags_no_2020_delta_trgt_rs50_retune_v1.predict(X_2023_lag)

pred_level_lag = np.expm1(np.log1p(test_2023_lag["lag1_avg_amt"].values) + log_delta_hat)

# 7) Backoff prediction for no-lag rows (group mean -> HCPCS mean -> global)
pred_level_nolag = (
    test_2023_nolag["mean_train_grp"]
    .fillna(test_2023_nolag["mean_train_hcpcs"])
    .fillna(global_mean)
    .to_numpy()
)

# 8) Combine into a full prediction vector aligned to original 2023 test index
pred_full = pd.Series(index=test_df_full_2023.index, dtype=float)
pred_full.loc[test_2023_lag.index] = pred_level_lag
pred_full.loc[test_2023_nolag.index] = pred_level_nolag

# Rebuild pred_backoff

pred_backoff = test_with_means["lag1_avg_amt"].copy()
pred_backoff = pred_backoff.fillna(test_with_means["mean_train_grp"])
pred_backoff = pred_backoff.fillna(test_with_means["mean_train_hcpcs"])
pred_backoff = pred_backoff.fillna(global_mean)

### Calculate model metrics and tail performance (`..._retune_v1`)

In [ ]:
from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# Inputs you already have from your hybrid prediction cell:
# - test_df_full_2023  (full 2023 rows)
# - pred_full          (full 2023 predictions aligned to test_df_full_2023.index)
# ------------------------------------------------------------

target_col = "avg_mdcr_stdzd_amt"
tail_flag_col = "is_top_1pct_avg_mdcr_stdzd_amt"

y_true = test_df_full_2023[target_col].astype(float)
pred_test = pred_full.astype(float)

# Basic sanity check
assert pred_test.index.equals(test_df_full_2023.index)

# ------------------------------------------------------------
# 1) Overall metrics (2023 full)
# ------------------------------------------------------------
r2_test = r2_score(y_true, pred_test)
mae = mean_absolute_error(y_true, pred_test)
rmse = root_mean_squared_error(y_true, pred_test)

# "log space" metrics on LEVELS (not on delta), if you still want them:
r2_log_true_test = r2_score(np.log1p(y_true), np.log1p(pred_test.clip(lower=0)))
mae_log_true = mean_absolute_error(np.log1p(y_true), np.log1p(pred_test.clip(lower=0)))
rmse_log_true = root_mean_squared_error(np.log1p(y_true), np.log1p(pred_test.clip(lower=0)))

# ------------------------------------------------------------
# 2) Weighted (tail-emphasized) metrics on 2023 full
#    Use the SAME w_tail you used for weighting (10 here)
# ------------------------------------------------------------
w_tail = 10
is_tail_test = test_df_full_2023[tail_flag_col].astype(int).to_numpy()
sample_w_test = (1 + (w_tail - 1) * is_tail_test).astype(float)

r2_test_w = r2_score(y_true, pred_test, sample_weight=sample_w_test)
mae_w = mean_absolute_error(y_true, pred_test, sample_weight=sample_w_test)
rmse_w = root_mean_squared_error(y_true, pred_test, sample_weight=sample_w_test)

# ------------------------------------------------------------
# 3) Tail analysis table (same style as before)
# ------------------------------------------------------------
test_eval = test_df_full_2023.copy()
test_eval["pred"] = pred_test
test_eval["abs_err"] = (test_eval[target_col] - test_eval["pred"]).abs()
test_eval["sq_err"] = (test_eval[target_col] - test_eval["pred"]) ** 2

non_tail_pred_to_y_ratio = (
    test_eval.loc[~test_eval[tail_flag_col], "pred"].mean()
    / test_eval.loc[~test_eval[tail_flag_col], target_col].mean()
)
tail_pred_to_y_ratio = (
    test_eval.loc[test_eval[tail_flag_col], "pred"].mean()
    / test_eval.loc[test_eval[tail_flag_col], target_col].mean()
)

non_tail_sse = test_eval.loc[~test_eval[tail_flag_col], "sq_err"].sum()
tail_sse = test_eval.loc[test_eval[tail_flag_col], "sq_err"].sum()

non_tail_share_of_total_sse = non_tail_sse / (non_tail_sse + tail_sse)
tail_share_of_total_sse = tail_sse / (non_tail_sse + tail_sse)

non_tail_under_prediction_rate = (
    (test_eval.loc[~test_eval[tail_flag_col], "pred"] < test_eval.loc[~test_eval[tail_flag_col], target_col]).mean()
)
tail_under_prediction_rate = (
    (test_eval.loc[test_eval[tail_flag_col], "pred"] < test_eval.loc[test_eval[tail_flag_col], target_col]).mean()
)

non_tail_bias = (
    test_eval.loc[~test_eval[tail_flag_col], "pred"].mean()
    - test_eval.loc[~test_eval[tail_flag_col], target_col].mean()
)
tail_bias = (
    test_eval.loc[test_eval[tail_flag_col], "pred"].mean()
    - test_eval.loc[test_eval[tail_flag_col], target_col].mean()
)

forecast_hybrid_metrics_2023_rs50_retune_v1 = pd.DataFrame({
    "w_tail":[w_tail],
    "r2_test": [r2_test],
    "mae": [mae],
    "rmse": [rmse],
    "r2_test_log": [r2_log_true_test],
    "mae_log": [mae_log_true],
    "rmse_log": [rmse_log_true],
    "r2_test_w": [r2_test_w],
    "mae_w": [mae_w],
    "rmse_w": [rmse_w],
    "non_tail_pred_to_y_ratio": [non_tail_pred_to_y_ratio],
    "tail_pred_to_y_ratio": [tail_pred_to_y_ratio],
    "non_tail_sse": [non_tail_sse],
    "tail_sse": [tail_sse],
    "non_tail_share_of_total_sse": [non_tail_share_of_total_sse],
    "tail_share_of_total_sse": [tail_share_of_total_sse],
    "non_tail_under_prediction_rate": [non_tail_under_prediction_rate],
    "tail_under_prediction_rate": [tail_under_prediction_rate],
    "non_tail_bias": [non_tail_bias],
    "tail_bias": [tail_bias],
})

forecast_hybrid_metrics_2023_rs50_retune_v1

## Systematic investigation of underperformance of `best_forecast_model_wt10_with_lags_no_2020_delta_trgt_rs50`

### 1. Build a single “explain table” for 2023 with all components

In [ ]:
import numpy as np
import pandas as pd

target_col = "avg_mdcr_stdzd_amt"
tail_flag_col = "is_top_1pct_avg_mdcr_stdzd_amt"

# Full 2023 rows (same as your evaluation set)
df23 = eda_hcpcs_full_df.loc[eda_hcpcs_full_df["Year"] == 2023].copy()

# Identify lag-present rows (eligible for residual model)
mask_lag = df23["lag1_avg_amt"].notna() & (df23["lag1_avg_amt"] >= 0)

# Residual truth for lag-present rows (in log space)
df23.loc[mask_lag, "log_delta_true"] = (
    np.log1p(df23.loc[mask_lag, target_col]) - np.log1p(df23.loc[mask_lag, "lag1_avg_amt"])
)

# Residual model prediction on lag-present rows
feature_cols = forecast_pkg_wt10_with_lags_no_2020_delta_trgt_rs50.feature_cols
X23_lag = df23.loc[mask_lag, feature_cols]
df23.loc[mask_lag, "log_delta_hat"] = best_forecast_model_wt10_with_lags_no_2020_delta_trgt_rs50.predict(X23_lag)

# Convert to level prediction for lag-present rows
df23.loc[mask_lag, "pred_level_lag"] = np.expm1(
    np.log1p(df23.loc[mask_lag, "lag1_avg_amt"].to_numpy()) 
    + df23.loc[mask_lag, "log_delta_hat"].to_numpy()
)

# Explicitly attach no-lag predictions from pred_full
df23["pred_level_nolag"] = np.nan
df23.loc[~mask_lag, "pred_level_nolag"] = pred_full.reindex(df23.index).loc[~mask_lag].to_numpy()

# Final prediction column (use pred_full if you already computed it correctly)
df23["pred"] = pred_full.reindex(df23.index).astype(float)

# Error decomposition
df23["err"] = df23[target_col] - df23["pred"]
df23["abs_err"] = df23["err"].abs()
df23["sq_err"] = df23["err"] ** 2

# Helpful ratios
df23["pred_to_y"] = df23["pred"] / (df23[target_col] + 1e-9)
df23["lag_to_y"] = df23["lag1_avg_amt"] / (df23[target_col] + 1e-9)
df23["y_to_lag"] = (df23[target_col] + 1e-9) / (df23["lag1_avg_amt"] + 1e-9)

Now we can directly answer: is the failure coming from the residual piece (`log_delta_hat`), or from backoff?

### 2. Separate “catastrophic underprediction” into root causes

In [ ]:
# Example definition. Tweak thresholds to your taste.
catastrophic = (df23[target_col] > 1000) & (df23["pred"] < 200)
share_nolag_within_cat = ((~mask_lag) & catastrophic).sum() / max(1, catastrophic.sum())
df23.loc[catastrophic, ["lag1_avg_amt", target_col, "pred", "log_delta_true", "log_delta_hat"]].head()
print("Catastrophic count:", catastrophic.sum())
print("Share no-lag inside catastrophic:", share_nolag_within_cat)

- If **many catastrophic rows are no-lag**, the model is not “routing expensive to cheap”. Our backoff is. Fix is a better no-lag policy (richer grouping, per-HCPCS quantiles, or a classifier to detect high-cost risk).

- If **catastrophic rows have lag present**, then the residual model is predicting a too-negative delta, or lag itself is not informative in those regimes.

### 3 Split catastrophic by mechanism: no-lag vs lag-present

In [ ]:
cat_no_lag = catastrophic & (~mask_lag)
cat_lag = catastrophic & mask_lag

print("Catastrophic no-lag:", cat_no_lag.sum())
print("Catastrophic lag-present:", cat_lag.sum())

### 4. Quantify whether 2023 deltas are out-of-distribution vs training

In [ ]:
train_df = forecast_pkg_wt10_with_lags_no_2020_delta_trgt_rs50.train_df.copy()  # should be 2021-2022 lag-present
train_delta = train_df["log_delta_cost"].astype(float)

test_delta = df23.loc[mask_lag, "log_delta_true"].dropna().astype(float)

print("Train delta quantiles:", train_delta.quantile([0.001,0.01,0.1,0.5,0.9,0.99,0.999]))
print("Test  delta quantiles:", test_delta.quantile([0.001,0.01,0.1,0.5,0.9,0.99,0.999]))

# Where does catastrophic underprediction live in delta space?
print(df23.loc[catastrophic & mask_lag, "log_delta_true"].describe())
print(df23.loc[catastrophic & mask_lag, "log_delta_hat"].describe())

- If catastrophic rows have **true delta far outside training support**, the model will regress toward `0` and fail.

- If `log_delta_hat` has a narrow range while `log_delta_true` is wide, you are seeing “shrinkage to the mean” due to regularization, depth limits, `min_child_weight`, or simply lack of signal.

### 5. Check support. Are these combinations “rare or new”?

For catastrophic lag-present rows, check if their categorical combinations are sparse in training.

If we see lots of `n_train == 0` or tiny counts, then “routing” is just the model being forced to generalize across unseen combos. In that case, the lag-only baseline winning is expected.

In [ ]:
key_cols = ["provider_type", "HCPCS_Cd", "Place_Of_Srvc", "state"]
train_key = train_df[key_cols].copy()
train_key["n_train"] = 1
train_counts = train_key.groupby(key_cols)["n_train"].sum().reset_index()

bad = df23.loc[catastrophic & mask_lag, key_cols + ["lag1_avg_amt", target_col, "pred"]].copy()
bad = bad.merge(train_counts, on=key_cols, how="left")
bad["n_train"] = bad["n_train"].fillna(0).astype(int)

bad.sort_values("n_train").head(20)
print("Median n_train among catastrophic:", bad["n_train"].median())
print("Share with n_train==0:", (bad["n_train"] == 0).mean())

### 6. Compare residual model vs “do nothing” residual (delta = 0)

This isolates whether the residual model is adding value or harming.

If the residual model is worse than delta=0 a lot of the time in the tail, then it is “overcorrecting” the lag. That’s our mechanism.

In [ ]:
# For lag-present rows:
y = df23.loc[mask_lag, target_col].astype(float)
lag = df23.loc[mask_lag, "lag1_avg_amt"].astype(float)

# Delta=0 baseline prediction is just lag (in level space)
pred_delta0 = lag

# Residual model prediction (in level space)
pred_resid = df23.loc[mask_lag, "pred_level_lag"].astype(float)

# Where is residual model worse than delta=0?
worse = (y - pred_resid).abs() > (y - pred_delta0).abs()
print("Share residual worse than delta=0:", worse.mean())

# Specifically among catastrophic underpredictions
cat_lag = catastrophic & mask_lag
worse_cat = worse.reindex(df23.index, fill_value=False) & cat_lag
print("Share catastrophic where residual worse than lag:", worse_cat.mean())

### 7. Explain individual failures with local attribution

In [ ]:
import shap

best_pipe = best_forecast_model_wt10_with_lags_no_2020_delta_trgt_rs50  # Pipeline(preprocess, model)

pre = best_pipe.named_steps["preprocess"]
xgb = best_pipe.named_steps["model"]

X_trans = pre.transform(X23_lag)
feat_names = pre.get_feature_names_out()

explainer = shap.TreeExplainer(xgb)
# Use a small sample first to keep it fast
lag_err = df23.loc[mask_lag].copy()
lag_err["abs_err_lagmodel"] = (lag_err[target_col] - lag_err["pred_level_lag"]).abs()

idx_bad = lag_err.sort_values("abs_err_lagmodel", ascending=False).head(50).index

X_bad = pre.transform(df23.loc[idx_bad, feature_cols])
shap_values = explainer.shap_values(X_bad)

# Show one example
i = 0
shap.plots.waterfall(shap.Explanation(values=shap_values[i], base_values=explainer.expected_value, data=X_bad[i].toarray()[0] if hasattr(X_bad[i], "toarray") else X_bad[i], feature_names=feat_names))

### 8. Explain the backoff model to explain the true catastrophic failures

> Let's refine the backoff ladder and then evaluate the improved ladder's performance in terms of whether the same catastrophic observations we saw earlier will be reduced with the new strategy.

In [ ]:
import numpy as np
import pandas as pd

target_col = "avg_mdcr_stdzd_amt"
group_cols = ["provider_type", "HCPCS_Cd", "Place_Of_Srvc"]

# Rebuild df23 fresh
df23 = eda_hcpcs_full_df.loc[eda_hcpcs_full_df["Year"] == 2023].copy()

# Training years used for the backoff means
train_years_for_means = [2021, 2022]
train_df_for_means = eda_hcpcs_full_df.loc[
    eda_hcpcs_full_df["Year"].isin(train_years_for_means)
].copy()

# Mean tables
train_means_grp = (
    train_df_for_means.groupby(group_cols)[target_col]
    .mean()
    .rename("mean_train_grp")
    .reset_index()
)

train_means_hcpcs = (
    train_df_for_means.groupby(["HCPCS_Cd"])[target_col]
    .mean()
    .rename("mean_train_hcpcs")
    .reset_index()
)

global_mean = float(train_df_for_means[target_col].mean())

# Preserve original index while merging
df23["_orig_idx"] = df23.index
df23 = df23.merge(train_means_grp, on=group_cols, how="left")
df23 = df23.merge(train_means_hcpcs, on="HCPCS_Cd", how="left")
df23 = df23.set_index("_orig_idx", drop=True)

# Reattach final hybrid prediction
df23["pred"] = pred_full.reindex(df23.index).astype(float)

# Identify lag rows
mask_lag = df23["lag1_avg_amt"].notna() & (df23["lag1_avg_amt"] >= 0)

# Build a simple backoff-source label consistent with the old ladder
df23["backoff_source"] = "lag_or_residual"
df23.loc[~mask_lag, "backoff_source"] = np.where(
    df23.loc[~mask_lag, "mean_train_grp"].notna(),
    "grp_mean",
    np.where(
        df23.loc[~mask_lag, "mean_train_hcpcs"].notna(),
        "hcpcs_mean",
        "global_mean"
    )
)

In [ ]:
catastrophic = (df23[target_col] > 1000) & (df23["pred"] < 200)

df23.loc[catastrophic, [
    "backoff_source",
    "mean_train_grp",
    "mean_train_hcpcs",
    "lag1_avg_amt",
    target_col,
    "pred"
]]

### QC for alignment

Let's do a sanity/alignment check:

After rebuilding df23 and attaching predictions (`"pred"`) to it, we need to confirm two things:

A. Does pred_full actually correspond to the rows in df23?
If the indices do not line up, then the “catastrophic failures” might be fake, caused by predictions being assigned to the wrong rows.

B. Did any predictions disappear during reindexing?
This checks whether any pred values became missing when attached to df23


That matters because if pred had NaNs, then:
- catastrophic counts could be wrong
- mean errors could be wrong
- some rows might silently disappear from later logic


In [ ]:
print("pred_full covers df23 index:", pred_full.index.isin(df23.index).mean())
print("pred NaN rate in df23:", df23["pred"].isna().mean())

 ### Direct row-level inspection of the catastrophic failures

 What do these rows actually look like?

A. Was lag missing?

If `lag1_avg_amt` is `NaN`, that means the residual model could not operate and the row fell into fallback logic.

B. Did grouped support exist?

If `mean_train_grp` is `NaN`, then the detailed `(provider_type, HCPCS_Cd, Place_Of_Service)` combination was unseen in the training window.

C. Did HCPCS-level support exist?

If `mean_train_hcpcs` is also `NaN`, then even the HCPCS-only fallback failed.

D. What fallback source actually got used?

If `backoff_source == "global_mean"`, then the model had no real contextual support and had to use a generic global average.


In [ ]:
bad_idx = df23.loc[catastrophic].index

cols = [
    "HCPCS_Cd", "provider_type", "Place_Of_Srvc", "state",
    "lag1_avg_amt", target_col, "pred",
    "mean_train_grp", "mean_train_hcpcs", "backoff_source"
]
df23.loc[bad_idx, cols]

### Support-existence test

After seeing that `mean_train_hcpcs` is missing for catastrophic rows, the natural follow-up question is:

- Are these HCPCS codes absent from the 2021-2022 training window entirely?

For the HCPCS codes belonging to catastrophic rows:
- do any records with those HCPCS codes exist in training years 2021 and 2022?

If the result is an empty Series, that means:
- none of those bad HCPCS codes appear in the backoff training window at all


In [ ]:
train_years_for_means = [2021, 2022]
train_df_for_means = eda_hcpcs_full_df.loc[eda_hcpcs_full_df["Year"].isin(train_years_for_means)].copy()

bad_hcpcs = df23.loc[bad_idx, "HCPCS_Cd"]

train_df_for_means.loc[train_df_for_means["HCPCS_Cd"].isin(bad_hcpcs), ["Year","HCPCS_Cd"]].value_counts().head(20)

### Historical robustness check (Year 2020)

Once we found those HCPCS codes are missing in 2021-2022, the next question becomes:

- Maybe they were present in 2020, and we just lost them because our backoff window only used 2021-2022?

In [ ]:
train_years_for_means = [2020, 2021, 2022]
train_df_for_means = eda_hcpcs_full_df.loc[eda_hcpcs_full_df["Year"].isin(train_years_for_means)].copy()

bad_hcpcs = df23.loc[bad_idx, "HCPCS_Cd"]

train_df_for_means.loc[train_df_for_means["HCPCS_Cd"].isin(bad_hcpcs), ["Year","HCPCS_Cd"]].value_counts().head(20)

## Let's try to improve the fallback ladder systematically

### 1. The first improved backoff ladder

The first fallback redesign experiment.

By that point, we had learned:
- catastrophic failures were no-lag rows
- their HCPCS codes were unseen historically
- therefore the old ladder had no useful HCPCS-specific support
- so it often fell all the way to global mean

That suggested a new question:

“Can we design a smarter fallback ladder that uses broader but still clinically relevant context?”

So you introduced:
- `mean_pt_pos`
= `mean_pt`

These are broader context levels than HCPCS.

If HCPCS is unseen, we still may know:
- what provider type this is
- what place of service this is


In [ ]:
import numpy as np
import pandas as pd

target_col = "avg_mdcr_stdzd_amt"
group_cols = ["provider_type", "HCPCS_Cd", "Place_Of_Srvc"]
pt_pos_cols = ["provider_type", "Place_Of_Srvc"]
pt_cols = ["provider_type"]
tail_flag_col = "is_top_1pct_avg_mdcr_stdzd_amt"

# Full 2023 test
df23 = eda_hcpcs_full_df.loc[eda_hcpcs_full_df["Year"] == 2023].copy()

# Means training window (keep as 2021-2022 to match your baseline intent)
train_years_for_means = [2021, 2022]
train_df = eda_hcpcs_full_df.loc[eda_hcpcs_full_df["Year"].isin(train_years_for_means)].copy()

global_mean = float(train_df[target_col].mean())

# Compute multiple mean tables
means_grp = train_df.groupby(group_cols)[target_col].mean().rename("mean_grp").reset_index()
means_hcpcs = train_df.groupby(["HCPCS_Cd"])[target_col].mean().rename("mean_hcpcs").reset_index()
means_pt_pos = train_df.groupby(pt_pos_cols)[target_col].mean().rename("mean_pt_pos").reset_index()
means_pt = train_df.groupby(pt_cols)[target_col].mean().rename("mean_pt").reset_index()

# Merge while preserving original index
df23["_orig_idx"] = df23.index
df23 = df23.merge(means_grp, on=group_cols, how="left")
df23 = df23.merge(means_hcpcs, on="HCPCS_Cd", how="left")
df23 = df23.merge(means_pt_pos, on=pt_pos_cols, how="left")
df23 = df23.merge(means_pt, on=pt_cols, how="left")
df23 = df23.set_index("_orig_idx", drop=True)

# Identify lag-present
mask_lag = df23["lag1_avg_amt"].notna() & (df23["lag1_avg_amt"] >= 0)

# Build backoff prediction for NO-LAG rows only
pred_nolag = df23["mean_grp"]
source = pd.Series(index=df23.index, dtype="object")

source[pred_nolag.notna()] = "grp_mean"
pred_nolag = pred_nolag.fillna(df23["mean_hcpcs"])
source[(source.isna()) & (df23["mean_hcpcs"].notna())] = "hcpcs_mean"

pred_nolag = pred_nolag.fillna(df23["mean_pt_pos"])
source[(source.isna()) & (df23["mean_pt_pos"].notna())] = "pt_pos_mean"

pred_nolag = pred_nolag.fillna(df23["mean_pt"])
source[(source.isna()) & (df23["mean_pt"].notna())] = "pt_mean"

pred_nolag = pred_nolag.fillna(global_mean)
source[source.isna()] = "global_mean"

# Final pred_backoff: use lag when available, else the improved backoff
pred_backoff = df23["lag1_avg_amt"].copy()
pred_backoff.loc[~mask_lag] = pred_nolag.loc[~mask_lag]

df23["pred_backoff"] = pred_backoff.astype(float)
df23["backoff_source"] = "lag"
df23.loc[~mask_lag, "backoff_source"] = source.loc[~mask_lag]

# Now inspect the catastrophic rows again
catastrophic = (df23[target_col] > 1000) & (df23["pred_backoff"] < 200)
df23.loc[catastrophic, ["HCPCS_Cd","provider_type","Place_Of_Srvc","state",
                        "lag1_avg_amt", target_col, "pred_backoff", "backoff_source",
                        "mean_grp","mean_hcpcs","mean_pt_pos","mean_pt"]]

### 2. The second improved backoff ladder

This is the next refinement of the same idea.

Let's ask a sharper question: Can we make fallback even more geographically and contextually specific?

So, let's introduced:
- `mean_pt_pos_state`
- `median_pt_pos_state`

Why state is added: Because costs can vary materially by geography. A provider-type/POS mean across the whole country may still be too coarse.

Why median is added: Mean can be distorted by outliers. For sparse, unstable groups, median can sometimes be more robust.

Again a design experiment: Can a richer fallback ladder reduce catastrophic misses for unseen-HCPCS, no-lag rows?

In [ ]:
import numpy as np
import pandas as pd

target_col = "avg_mdcr_stdzd_amt"
group_cols = ["provider_type", "HCPCS_Cd", "Place_Of_Srvc"]
pt_pos_state_cols = ["provider_type", "Place_Of_Srvc", "state"]
pt_pos_cols = ["provider_type", "Place_Of_Srvc"]
pt_cols = ["provider_type"]
tail_flag_col = "is_top_1pct_avg_mdcr_stdzd_amt"

# Full 2023 test
df23 = eda_hcpcs_full_df.loc[eda_hcpcs_full_df["Year"] == 2023].copy()

# Means training window (keep as 2021-2022 to match your baseline intent)
train_years_for_means = [2021, 2022]
train_df = eda_hcpcs_full_df.loc[eda_hcpcs_full_df["Year"].isin(train_years_for_means)].copy()

global_mean = float(train_df[target_col].mean())

# Compute multiple mean tables
means_grp = train_df.groupby(group_cols)[target_col].mean().rename("mean_grp").reset_index()
means_hcpcs = train_df.groupby(["HCPCS_Cd"])[target_col].mean().rename("mean_hcpcs").reset_index()
means_pt_pos_state = train_df.groupby(pt_pos_state_cols)[target_col].mean().rename("mean_pt_pos_state").reset_index()
medians_pt_pos_state = train_df.groupby(pt_pos_state_cols)[target_col].median().rename("median_pt_pos_state").reset_index()
means_pt_pos = train_df.groupby(pt_pos_cols)[target_col].mean().rename("mean_pt_pos").reset_index()
means_pt = train_df.groupby(pt_cols)[target_col].mean().rename("mean_pt").reset_index()

# Merge while preserving original index
df23["_orig_idx"] = df23.index
df23 = df23.merge(means_grp, on=group_cols, how="left")
df23 = df23.merge(means_hcpcs, on="HCPCS_Cd", how="left")
df23 = df23.merge(means_pt_pos_state, on=pt_pos_state_cols, how="left")
df23 = df23.merge(medians_pt_pos_state, on=pt_pos_state_cols, how="left")
df23 = df23.merge(means_pt_pos, on=pt_pos_cols, how="left")
df23 = df23.merge(means_pt, on=pt_cols, how="left")
df23 = df23.set_index("_orig_idx", drop=True)

# Identify lag-present
mask_lag = df23["lag1_avg_amt"].notna() & (df23["lag1_avg_amt"] >= 0)

# Build backoff prediction for NO-LAG rows only
pred_nolag = df23["mean_grp"]
source = pd.Series(index=df23.index, dtype="object")

source[pred_nolag.notna()] = "grp_mean"
pred_nolag = pred_nolag.fillna(df23["mean_hcpcs"])
source[(source.isna()) & (df23["mean_hcpcs"].notna())] = "hcpcs_mean"

pred_nolag = pred_nolag.fillna(df23["mean_pt_pos_state"])
source[(source.isna()) & (df23["mean_pt_pos_state"].notna())] = "pt_pos_state_mean"

pred_nolag = pred_nolag.fillna(df23["median_pt_pos_state"])
source[(source.isna()) & (df23["median_pt_pos_state"].notna())] = "pt_pos_state_median"

pred_nolag = pred_nolag.fillna(df23["mean_pt_pos"])
source[(source.isna()) & (df23["mean_pt_pos"].notna())] = "pt_pos_mean"

pred_nolag = pred_nolag.fillna(df23["mean_pt"])
source[(source.isna()) & (df23["mean_pt"].notna())] = "pt_mean"

pred_nolag = pred_nolag.fillna(global_mean)
source[source.isna()] = "global_mean"

# Final pred_backoff: use lag when available, else the improved backoff
pred_backoff = df23["lag1_avg_amt"].copy()
pred_backoff.loc[~mask_lag] = pred_nolag.loc[~mask_lag]

df23["pred_backoff"] = pred_backoff.astype(float)
df23["backoff_source"] = "lag"
df23.loc[~mask_lag, "backoff_source"] = source.loc[~mask_lag]

# Now inspect the catastrophic rows again
catastrophic = (df23[target_col] > 1000) & (df23["pred_backoff"] < 200)
df23.loc[catastrophic, [
    "HCPCS_Cd", "provider_type", "Place_Of_Srvc", "state",
    "lag1_avg_amt", target_col, "pred_backoff", "backoff_source",
    "mean_grp", "mean_hcpcs", "mean_pt_pos_state", "median_pt_pos_state",
    "mean_pt_pos", "mean_pt"
]]

So the next question is not really “is the code correct?”

It is “do we want a stronger no-lag unseen-HCPCS fallback policy?”

A very practical next step would be to quantify how many no-lag rows use each fallback source and how each source performs. For example:

### Population-size check

Since fallback policy is central to the story, we need to know:
- “How many rows are actually lag-present versus no-lag?”

Because the importance of fallback depends on how often fallback gets used.

This tells us the population split:
- how many rows can use the lag/residual path
- how many rows must use fallback

If no-lag rows are tiny, fallback would matter less. But if there are tens of thousands of no-lag rows, then fallback is a major part of the overall forecasting system. 

In [ ]:
df23.value_counts(mask_lag)

### Fallback-routing distribution check

With the richer ladder, you want to know:

“How often is each fallback rung actually being used?”

This cell counts how many no-lag rows were routed to:
- `grp_mean`
- `hcpcs_mean`
- `pt_pos_state_mean`
- `pt_pos_state_median`
- `pt_pos_mean`
- `pt_mean`
- `global_mean`

In [ ]:
df23.loc[~mask_lag, "backoff_source"].value_counts(dropna=False)

### Fallback performance audit

> This is very important to understand! 

After learning how many rows route to each fallback source, the next question is: How well does each fallback source actually perform?

This cell gives source-level performance.

- `n`: How many no-lag rows used that fallback source
- `mae`: Average absolute error for rows using that source
- `rmse`: Root mean squared error for rows using that source
- `mean_true`: Average actual cost for rows using that source
- `mean_pred`: Average predicted cost for rows using that source

In [ ]:
df23.loc[~mask_lag].groupby("backoff_source").apply(
    lambda g: pd.Series({
        "n": len(g),
        "mae": (g[target_col] - g["pred_backoff"]).abs().mean(),
        "rmse": np.sqrt(((g[target_col] - g["pred_backoff"]) ** 2).mean()),
        "mean_true": g[target_col].mean(),
        "mean_pred": g["pred_backoff"].mean(),
    })
)

## Let's restrict evaluation to the subset where `lag1_avg_amt` exists, then compare:
- model-based lag-row predictions from `best_forecast_model_wt10_with_lags_no_2020_delta_trgt_rs50`
- naive lag prediction using `lag1_avg_amt` directly

That gives us a clean answer to:

On the rows where the residual model is actually allowed to operate, did it improve on simply carrying lag forward?

What this table tells us

This is the cleanest apples-to-apples test for the residual model.
- If the residual model has better R² / lower MAE / lower RMSE than `lag1_avg_amt_baseline_on_lag_rows`, then it is adding value on lag-present rows.

- If it is worse, then the residual model is not beating the simpler “just use lag” rule on the rows it was specifically designed for.

In [ ]:
from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error
import numpy as np
import pandas as pd

target_col = "avg_mdcr_stdzd_amt"
tail_flag_col = "is_top_1pct_avg_mdcr_stdzd_amt"
w_tail = 10

# ------------------------------------------------------------
# 1) Build the lag-present evaluation subset
# ------------------------------------------------------------
df_lag = eda_hcpcs_full_df.loc[eda_hcpcs_full_df["Year"] == 2023].copy()
mask_lag = df_lag["lag1_avg_amt"].notna() & (df_lag["lag1_avg_amt"] >= 0)
df_lag = df_lag.loc[mask_lag].copy()

# ------------------------------------------------------------
# 2) Generate model predictions on lag-present rows only
# ------------------------------------------------------------
feature_cols = forecast_pkg_wt10_with_lags_no_2020_delta_trgt_rs50.feature_cols
X_lag = df_lag[feature_cols]

log_delta_hat = best_forecast_model_wt10_with_lags_no_2020_delta_trgt_rs50.predict(X_lag)

pred_model = np.expm1(np.log1p(df_lag["lag1_avg_amt"].to_numpy()) + log_delta_hat)
pred_lag = df_lag["lag1_avg_amt"].to_numpy()

y_true = df_lag[target_col].astype(float).to_numpy()

# ------------------------------------------------------------
# 3) Tail weights on this lag-present subset
# ------------------------------------------------------------
is_tail = df_lag[tail_flag_col].astype(int).to_numpy()
sample_w = (1 + (w_tail - 1) * is_tail).astype(float)

# ------------------------------------------------------------
# 4) Helper to compute the same metrics table style
# ------------------------------------------------------------
def eval_metrics_table(df, y_true, pred, sample_w, model_label, target_col, tail_flag_col):
    df_eval = df.copy()
    df_eval["pred"] = pred
    df_eval["abs_err"] = np.abs(y_true - pred)
    df_eval["sq_err"] = (y_true - pred) ** 2

    is_tail = df_eval[tail_flag_col].astype(bool)
    is_non_tail = ~is_tail

    r2_test = r2_score(y_true, pred)
    mae = mean_absolute_error(y_true, pred)
    rmse = root_mean_squared_error(y_true, pred)

    pred_clip = np.clip(pred, a_min=0, a_max=None)
    r2_test_log = r2_score(np.log1p(y_true), np.log1p(pred_clip))
    mae_log = mean_absolute_error(np.log1p(y_true), np.log1p(pred_clip))
    rmse_log = root_mean_squared_error(np.log1p(y_true), np.log1p(pred_clip))

    r2_test_w = r2_score(y_true, pred, sample_weight=sample_w)
    mae_w = mean_absolute_error(y_true, pred, sample_weight=sample_w)
    rmse_w = root_mean_squared_error(y_true, pred, sample_weight=sample_w)

    non_tail_pred_to_y_ratio = (
        df_eval.loc[is_non_tail, "pred"].mean() /
        df_eval.loc[is_non_tail, target_col].mean()
    )
    tail_pred_to_y_ratio = (
        df_eval.loc[is_tail, "pred"].mean() /
        df_eval.loc[is_tail, target_col].mean()
    )

    non_tail_sse = df_eval.loc[is_non_tail, "sq_err"].sum()
    tail_sse = df_eval.loc[is_tail, "sq_err"].sum()

    non_tail_share_of_total_sse = non_tail_sse / (non_tail_sse + tail_sse)
    tail_share_of_total_sse = tail_sse / (non_tail_sse + tail_sse)

    non_tail_under_prediction_rate = (
        (df_eval.loc[is_non_tail, "pred"] < df_eval.loc[is_non_tail, target_col]).mean()
    )
    tail_under_prediction_rate = (
        (df_eval.loc[is_tail, "pred"] < df_eval.loc[is_tail, target_col]).mean()
    )

    non_tail_bias = (
        df_eval.loc[is_non_tail, "pred"].mean() -
        df_eval.loc[is_non_tail, target_col].mean()
    )
    tail_bias = (
        df_eval.loc[is_tail, "pred"].mean() -
        df_eval.loc[is_tail, target_col].mean()
    )

    return pd.DataFrame({
        "model": [model_label],
        "rows_eval": [len(df_eval)],
        "w_tail": [w_tail],
        "r2_test": [r2_test],
        "mae": [mae],
        "rmse": [rmse],
        "r2_test_log": [r2_test_log],
        "mae_log": [mae_log],
        "rmse_log": [rmse_log],
        "r2_test_w": [r2_test_w],
        "mae_w": [mae_w],
        "rmse_w": [rmse_w],
        "non_tail_pred_to_y_ratio": [non_tail_pred_to_y_ratio],
        "tail_pred_to_y_ratio": [tail_pred_to_y_ratio],
        "non_tail_sse": [non_tail_sse],
        "tail_sse": [tail_sse],
        "non_tail_share_of_total_sse": [non_tail_share_of_total_sse],
        "tail_share_of_total_sse": [tail_share_of_total_sse],
        "non_tail_under_prediction_rate": [non_tail_under_prediction_rate],
        "tail_under_prediction_rate": [tail_under_prediction_rate],
        "non_tail_bias": [non_tail_bias],
        "tail_bias": [tail_bias],
    })

# ------------------------------------------------------------
# 5) Build side-by-side comparison table
# ------------------------------------------------------------
lag_subset_model_metrics = eval_metrics_table(
    df=df_lag,
    y_true=y_true,
    pred=pred_model,
    sample_w=sample_w,
    model_label="residual_model_on_lag_rows",
    target_col=target_col,
    tail_flag_col=tail_flag_col,
)

lag_subset_naive_metrics = eval_metrics_table(
    df=df_lag,
    y_true=y_true,
    pred=pred_lag,
    sample_w=sample_w,
    model_label="lag1_avg_amt_baseline_on_lag_rows",
    target_col=target_col,
    tail_flag_col=tail_flag_col,
)

lag_rows_comparison_metrics = pd.concat(
    [lag_subset_model_metrics, lag_subset_naive_metrics],
    ignore_index=True,
)

lag_rows_comparison_metrics

### Let's explicitly subtract baseline from model for the main metrics

In [ ]:
compare = lag_rows_comparison_metrics.set_index("model")

delta_summary = pd.DataFrame({
    "metric": ["r2_test", "mae", "rmse", "r2_test_w", "mae_w", "rmse_w"],
    "model_minus_baseline": [
        compare.loc["residual_model_on_lag_rows", "r2_test"] - compare.loc["lag1_avg_amt_baseline_on_lag_rows", "r2_test"],
        compare.loc["residual_model_on_lag_rows", "mae"] - compare.loc["lag1_avg_amt_baseline_on_lag_rows", "mae"],
        compare.loc["residual_model_on_lag_rows", "rmse"] - compare.loc["lag1_avg_amt_baseline_on_lag_rows", "rmse"],
        compare.loc["residual_model_on_lag_rows", "r2_test_w"] - compare.loc["lag1_avg_amt_baseline_on_lag_rows", "r2_test_w"],
        compare.loc["residual_model_on_lag_rows", "mae_w"] - compare.loc["lag1_avg_amt_baseline_on_lag_rows", "mae_w"],
        compare.loc["residual_model_on_lag_rows", "rmse_w"] - compare.loc["lag1_avg_amt_baseline_on_lag_rows", "rmse_w"],
    ]
})

delta_summary

# Summarize hybrid model (residual model (`best_forecast_model_wt10_with_lags_no_2020_delta_trgt_rs50`) plus fallback ladder) vs lag baseline (lag + fallback ladder)

In [ ]:
from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error
import numpy as np
import pandas as pd

target_col = "avg_mdcr_stdzd_amt"
tail_flag_col = "is_top_1pct_avg_mdcr_stdzd_amt"
w_tail = 10

# -------------------------------------------------------------------
# Assumes you already have:
# 1) df23 with:
#    - lag1_avg_amt
#    - pred_backoff
#    - backoff_source
#    - target_col
#    - tail_flag_col
# 2) fitted residual model:
#    - best_forecast_model_wt10_with_lags_no_2020_delta_trgt_rs50
#    - forecast_pkg_wt10_with_lags_no_2020_delta_trgt_rs50.feature_cols
# -------------------------------------------------------------------

# If not already present, define lag mask on df23
mask_lag = df23["lag1_avg_amt"].notna() & (df23["lag1_avg_amt"] >= 0)
mask_nolag = ~mask_lag

# -------------------------------------------------------------------
# Rebuild residual-model predictions on lag-present rows
# -------------------------------------------------------------------
feature_cols = forecast_pkg_wt10_with_lags_no_2020_delta_trgt_rs50.feature_cols
X_lag = df23.loc[mask_lag, feature_cols]

log_delta_hat = best_forecast_model_wt10_with_lags_no_2020_delta_trgt_rs50.predict(X_lag)

pred_residual_lag = np.expm1(
    np.log1p(df23.loc[mask_lag, "lag1_avg_amt"].to_numpy()) + log_delta_hat
)

# Baseline on lag-present rows is simply lag1_avg_amt
pred_lag_baseline_lag = df23.loc[mask_lag, "lag1_avg_amt"].to_numpy()

# On no-lag rows, both methods use the same fallback ladder
pred_nolag_shared = df23.loc[mask_nolag, "pred_backoff"].to_numpy()

# -------------------------------------------------------------------
# Build full-row predictions for both strategies
# -------------------------------------------------------------------
pred_all_lag_baseline = df23["pred_backoff"].astype(float).to_numpy()

pred_all_hybrid = df23["pred_backoff"].astype(float).copy()
pred_all_hybrid[mask_lag.to_numpy()] = pred_residual_lag

# -------------------------------------------------------------------
# Helper to compute a one-row metrics summary
# -------------------------------------------------------------------
def summarize_segment(
    df_segment: pd.DataFrame,
    pred: np.ndarray,
    segment_name: str,
    strategy_name: str,
    target_col: str,
    tail_flag_col: str,
    w_tail: int,
    fallback_note: str = ""
) -> pd.DataFrame:
    y_true = df_segment[target_col].astype(float).to_numpy()
    pred = np.asarray(pred, dtype=float)

    sample_w = (
        1 + (w_tail - 1) * df_segment[tail_flag_col].astype(int).to_numpy()
    ).astype(float)

    df_eval = df_segment.copy()
    df_eval["pred"] = pred
    df_eval["abs_err"] = np.abs(y_true - pred)
    df_eval["sq_err"] = (y_true - pred) ** 2

    is_tail = df_eval[tail_flag_col].astype(bool)
    is_non_tail = ~is_tail

    r2_test = r2_score(y_true, pred)
    mae = mean_absolute_error(y_true, pred)
    rmse = root_mean_squared_error(y_true, pred)

    pred_clip = np.clip(pred, a_min=0, a_max=None)
    r2_test_log = r2_score(np.log1p(y_true), np.log1p(pred_clip))
    mae_log = mean_absolute_error(np.log1p(y_true), np.log1p(pred_clip))
    rmse_log = root_mean_squared_error(np.log1p(y_true), np.log1p(pred_clip))

    r2_test_w = r2_score(y_true, pred, sample_weight=sample_w)
    mae_w = mean_absolute_error(y_true, pred, sample_weight=sample_w)
    rmse_w = root_mean_squared_error(y_true, pred, sample_weight=sample_w)

    non_tail_pred_to_y_ratio = (
        df_eval.loc[is_non_tail, "pred"].mean()
        / df_eval.loc[is_non_tail, target_col].mean()
        if is_non_tail.sum() > 0 else np.nan
    )
    tail_pred_to_y_ratio = (
        df_eval.loc[is_tail, "pred"].mean()
        / df_eval.loc[is_tail, target_col].mean()
        if is_tail.sum() > 0 else np.nan
    )

    non_tail_sse = df_eval.loc[is_non_tail, "sq_err"].sum()
    tail_sse = df_eval.loc[is_tail, "sq_err"].sum()
    total_sse = non_tail_sse + tail_sse

    non_tail_share_of_total_sse = non_tail_sse / total_sse if total_sse > 0 else np.nan
    tail_share_of_total_sse = tail_sse / total_sse if total_sse > 0 else np.nan

    non_tail_under_prediction_rate = (
        (df_eval.loc[is_non_tail, "pred"] < df_eval.loc[is_non_tail, target_col]).mean()
        if is_non_tail.sum() > 0 else np.nan
    )
    tail_under_prediction_rate = (
        (df_eval.loc[is_tail, "pred"] < df_eval.loc[is_tail, target_col]).mean()
        if is_tail.sum() > 0 else np.nan
    )

    non_tail_bias = (
        df_eval.loc[is_non_tail, "pred"].mean()
        - df_eval.loc[is_non_tail, target_col].mean()
        if is_non_tail.sum() > 0 else np.nan
    )
    tail_bias = (
        df_eval.loc[is_tail, "pred"].mean()
        - df_eval.loc[is_tail, target_col].mean()
        if is_tail.sum() > 0 else np.nan
    )

    return pd.DataFrame({
        "segment": [segment_name],
        "prediction_strategy": [strategy_name],
        "rows_eval": [len(df_segment)],
        "tail_rows": [int(is_tail.sum())],
        "non_tail_rows": [int(is_non_tail.sum())],
        "w_tail": [w_tail],
        "fallback_note": [fallback_note],
        "r2_test": [r2_test],
        "mae": [mae],
        "rmse": [rmse],
        "r2_test_log": [r2_test_log],
        "mae_log": [mae_log],
        "rmse_log": [rmse_log],
        "r2_test_w": [r2_test_w],
        "mae_w": [mae_w],
        "rmse_w": [rmse_w],
        "non_tail_pred_to_y_ratio": [non_tail_pred_to_y_ratio],
        "tail_pred_to_y_ratio": [tail_pred_to_y_ratio],
        "non_tail_sse": [non_tail_sse],
        "tail_sse": [tail_sse],
        "non_tail_share_of_total_sse": [non_tail_share_of_total_sse],
        "tail_share_of_total_sse": [tail_share_of_total_sse],
        "non_tail_under_prediction_rate": [non_tail_under_prediction_rate],
        "tail_under_prediction_rate": [tail_under_prediction_rate],
        "non_tail_bias": [non_tail_bias],
        "tail_bias": [tail_bias],
    })

# -------------------------------------------------------------------
# Build the four segment rows
# -------------------------------------------------------------------

# 1) Lag-present, lag baseline
tbl_lag_baseline_lag = summarize_segment(
    df_segment=df23.loc[mask_lag].copy(),
    pred=pred_lag_baseline_lag,
    segment_name="lag_present",
    strategy_name="lag_baseline",
    target_col=target_col,
    tail_flag_col=tail_flag_col,
    w_tail=w_tail,
    fallback_note="uses lag1_avg_amt directly"
)

# 2) Lag-present, residual model
tbl_residual_lag = summarize_segment(
    df_segment=df23.loc[mask_lag].copy(),
    pred=pred_residual_lag,
    segment_name="lag_present",
    strategy_name="residual_model",
    target_col=target_col,
    tail_flag_col=tail_flag_col,
    w_tail=w_tail,
    fallback_note="uses residual model on top of lag1_avg_amt"
)

# 3) No-lag, lag baseline path (really fallback ladder)
tbl_lag_baseline_nolag = summarize_segment(
    df_segment=df23.loc[mask_nolag].copy(),
    pred=pred_nolag_shared,
    segment_name="no_lag",
    strategy_name="lag_baseline",
    target_col=target_col,
    tail_flag_col=tail_flag_col,
    w_tail=w_tail,
    fallback_note="fallback ladder: grp -> hcpcs -> pt_pos_state_mean -> pt_pos_state_median -> pt_pos -> pt -> global"
)

# 4) No-lag, residual model path (same fallback ladder)
tbl_residual_nolag = summarize_segment(
    df_segment=df23.loc[mask_nolag].copy(),
    pred=pred_nolag_shared,
    segment_name="no_lag",
    strategy_name="residual_model",
    target_col=target_col,
    tail_flag_col=tail_flag_col,
    w_tail=w_tail,
    fallback_note="same fallback ladder as lag baseline"
)

# -------------------------------------------------------------------
# Build two all-rows rows
# -------------------------------------------------------------------

# 5) All rows, lag baseline + fallback
tbl_lag_baseline_all = summarize_segment(
    df_segment=df23.copy(),
    pred=pred_all_lag_baseline,
    segment_name="all_rows",
    strategy_name="lag_baseline",
    target_col=target_col,
    tail_flag_col=tail_flag_col,
    w_tail=w_tail,
    fallback_note="lag1_avg_amt when present, else fallback ladder"
)

# 6) All rows, hybrid = residual on lag rows + fallback on no-lag rows
tbl_hybrid_all = summarize_segment(
    df_segment=df23.copy(),
    pred=pred_all_hybrid,
    segment_name="all_rows",
    strategy_name="hybrid",
    target_col=target_col,
    tail_flag_col=tail_flag_col,
    w_tail=w_tail,
    fallback_note="residual model on lag-present rows, same fallback ladder on no-lag rows"
)

# -------------------------------------------------------------------
# Final comprehensive summary table
# -------------------------------------------------------------------
lag_vs_residual_segment_summary = pd.concat([
    tbl_lag_baseline_lag,
    tbl_residual_lag,
    tbl_lag_baseline_nolag,
    tbl_residual_nolag,
    tbl_lag_baseline_all,
    tbl_hybrid_all,
], ignore_index=True)

lag_vs_residual_segment_summary

## Forecast Model Comparison Summary

The segment-level comparison tells a clear and internally consistent story about how the residual model behaves relative to the naive lag baseline.

---

### Overall Conclusion

The **residual model is not uniformly better** than the lag baseline. Its benefit depends on which performance criterion matters most:

* It **improves R² and RMSE** on lag-present rows and on the full 2023 evaluation set.
* It **worsens MAE, weighted metrics, and tail calibration** relative to the simpler lag-based baseline.
* On **no-lag rows**, both approaches are identical because both use the same fallback ladder.

> **Key Takeaway:** The residual model is helping mostly on **average squared-error fit**, but it is not improving the aspects of performance that matter most for **robustness, tail behavior, or practical reliability**.

---

### 1. Performance on lag-present rows

Among the **250,090 rows** where `lag1_avg_amt` exists, the naive lag baseline is already extremely strong. This confirms that year-over-year lag is a very powerful predictor in this dataset. For most providers and services, cost per service is highly persistent from one year to the next.

| Metric | Lag Baseline | Residual Model | Difference |
| :--- | :--- | :--- | :--- |
| **R²** | 0.972676 | 0.976289 | Improved by ~0.0036 |
| **MAE** | 4.413878 | 4.613294 | Worsened by ~0.20 |
| **RMSE** | 29.008265 | 27.021981 | Improved by ~2.0 |

This indicates that the residual model is improving fit in a **squared-error sense**, but not in an **absolute-error sense**. In other words, it helps enough moderate-to-large errors to lower RMSE, but it also introduces enough additional deviation from lag to make typical absolute errors slightly worse.

#### Tail-specific behavior on lag-present rows

The lag-present row comparison is especially important because this is the only segment where the residual model is actually making learned adjustments. Compared with the lag baseline, the residual model showed:

* Worse weighted R²
* Worse weighted MAE
* Worse weighted RMSE
* Stronger underprediction in the tail
* More negative tail bias

**For example:**
* **Tail prediction-to-actual ratio:** Fell from about 1.0002 under the lag baseline to 0.9883 under the residual model.
* **Tail bias:** Moved from about +0.27 to about -15.60.

This means that although the residual model improves average squared-error fit, it is also **less well calibrated in high-cost regimes**. The model is pushing predictions away from lag in ways that help some rows overall, but can hurt the most important expensive rows.

---

### 2. Performance on no-lag rows

Among the **41,919 rows** where lag is missing, both approaches are identical because both use the same fallback ladder. 

**Metrics:**
* **R²:** 0.748200
* **MAE:** 14.540462
* **RMSE:** 83.642306

This is materially worse than lag-present performance, which is expected. These rows are inherently harder because the strongest signal, prior-year lag, is absent.

Still, the fallback ladder performs reasonably given the difficulty of the problem. It provides a workable systematic strategy for no-lag rows, but clearly these rows remain the hardest segment of the forecasting problem. Because the lag baseline and residual-model system use the **same no-lag fallback strategy**, there is nothing to compare between them on this segment unless the fallback ladder itself is changed.

---

### 3. Full 2023 performance across all rows

When evaluated on all **292,009 rows** in 2023, the tradeoff seen on lag-present rows carries through to the full end-to-end system:

| Metric | Lag Baseline + Fallback | Hybrid Model |
| :--- | :--- | :--- |
| **R²** | 0.943189 | 0.946328 |
| **MAE** | 5.867587 | 6.038377 |
| **RMSE** | 41.532990 | 40.369217 |

* The **hybrid model** improves R² and RMSE.
* The **lag baseline** retains better MAE.

So the hybrid system is better if the goal is to optimize **variance explained** or **squared-error loss**, but the simpler lag baseline remains better if the goal is **absolute accuracy** and **predictive stability**.

---

### 4. Tail behavior across all rows

The tail metrics are especially important here because the most expensive rows often matter most analytically and operationally.

| Metric | Lag Baseline + Fallback | Hybrid Model |
| :--- | :--- | :--- |
| **Tail prediction-to-actual ratio** | 0.997044 | 0.986999 |
| **Tail bias** | -3.901280 | -17.156784 |
| **Tail underprediction rate** | 0.311879 | 0.601849 |

**This is one of the most important findings in the notebook.**

The lag baseline is **much better calibrated in the tail**. Its tail ratio is very close to 1, its bias is only mildly negative, and it underpredicts tail rows much less often.

The hybrid model, by contrast, has a much stronger tendency to **underpredict tail observations**. Even though it improves global R² and RMSE, it does so at the cost of **more tail underestimation and worse tail-weighted behavior**. This is a strong sign that the residual model is not conservative enough relative to the already-powerful lag anchor.

---

### 5. Interpretation of model behavior

These results suggest the following behavioral pattern:

* **Lag is already an exceptionally strong predictor** for rows where it exists.
* The residual model learns corrections that sometimes improve fit enough to reduce RMSE.
* However, because lag is already so strong, the model has limited room to add value.
* As a result, some learned corrections improve average squared-error performance while still making predictions **less reliable in the tail**.
* On no-lag rows, the system depends entirely on the fallback ladder, so performance there is driven by the quality of the backoff policy rather than the residual model.

In practical terms, the residual model behaves more like a **challenger model** that can improve average fit, but not yet like a clearly superior replacement for the simpler lag-based system.

---

### 6. Practical takeaway

If the goal is to build the **strongest practical forecasting system**, the evidence currently favors the **lag baseline + fallback ladder**.

**Why:**
* It is simpler.
* It is easier to explain.
* It is extremely strong on lag-present rows.
* It is better calibrated in the tail.
* It performs better on MAE and weighted metrics.
* It appears more robust and production-safe.

If the goal is instead to optimize **R² or RMSE**, then the **hybrid residual approach** is a legitimate challenger because it does improve those metrics. But based on the current evidence, the hybrid model is best viewed as interesting and technically competitive, but **not yet superior in practical forecasting terms**.

---

### Final Summary

* **On lag-present rows:** The residual model improved R² and RMSE relative to the naive lag baseline, but worsened MAE and degraded tail-weighted performance.
* **On no-lag rows:** Both approaches were identical because both used the same fallback ladder.
* **Across all 2023 rows:** The hybrid model improved R² (0.946 vs 0.943) and RMSE (40.37 vs 41.53), but the lag baseline retained better MAE (5.87 vs 6.04), better weighted metrics, and much better tail calibration.
* **Overall:** The simpler **lag + fallback** approach appears to be the stronger practical forecasting strategy, while the **residual model + fallback** remains a useful challenger for improving average squared-error fit.

# Quick experimental section below (out of curiosity tested the without lag model)

## Rebuild `forecast_pkg_wt10_with_lags_no_2020_delta_trgt_rs50` but without lag features: build `forecast_pkg_wt10_without_lags_no_2020_delta_trgt_rs50`

In [ ]:
from pathlib import Path
from joblib import dump, load

# 1. Setup Paths
ROOT = Path.cwd().resolve()
MODELS = (ROOT / "models").resolve()
MODELS.mkdir(parents=True, exist_ok=True)

# We now point to a package file instead of just the model file
pkg_path = MODELS / "forecast_pkg_wt10_without_lags_no_2020_delta_trgt_rs50.joblib"

# 2. Check/Load or Train/Save
if pkg_path.exists():
    print(f"Loading {pkg_path.name} from disk... (Skipping data prep and training)")
    # Load the entire package (data + fitted search object)
    forecast_pkg_wt10_without_lags_no_2020_delta_trgt_rs50 = load(pkg_path)
    
    # Extract the model so downstream code works as expected
    best_forecast_model_wt10_without_lags_no_2020_delta_trgt_rs50 = forecast_pkg_wt10_without_lags_no_2020_delta_trgt_rs50.search.best_estimator_

else:
    print(f"File not found. Preparing data and training...")

    # --- PREPARE DATA ---
    forecast_pkg_wt10_without_lags_no_2020_delta_trgt_rs50 = prepare_xgb_randomizedsearch(
        eda_hcpcs_lags_present_df,
        approach="forecast",
        target_col="log_delta_cost",
        w_tail=10,
        include_lags=False,
        incl_2020_in_final_fit=False,
        use_ttr=False
    )

    # --- TRAIN ---
    if forecast_pkg_wt10_without_lags_no_2020_delta_trgt_rs50.groups_train is None:
        forecast_pkg_wt10_without_lags_no_2020_delta_trgt_rs50.search.fit(
            forecast_pkg_wt10_without_lags_no_2020_delta_trgt_rs50.X_train,
            forecast_pkg_wt10_without_lags_no_2020_delta_trgt_rs50.y_train,
            model__sample_weight=forecast_pkg_wt10_without_lags_no_2020_delta_trgt_rs50.sample_w_train,
        )
    else:
        forecast_pkg_wt10_without_lags_no_2020_delta_trgt_rs50.search.fit(
            forecast_pkg_wt10_without_lags_no_2020_delta_trgt_rs50.X_train,
            forecast_pkg_wt10_without_lags_no_2020_delta_trgt_rs50.y_train,
            groups=forecast_pkg_wt10_without_lags_no_2020_delta_trgt_rs50.groups_train,
            model__sample_weight=forecast_pkg_wt10_without_lags_no_2020_delta_trgt_rs50.sample_w_train,
        )
        
    best_forecast_model_wt10_without_lags_no_2020_delta_trgt_rs50 = forecast_pkg_wt10_without_lags_no_2020_delta_trgt_rs50.search.best_estimator_

    # --- SAVE ---
    # 3. Save the ENTIRE package
    dump(forecast_pkg_wt10_without_lags_no_2020_delta_trgt_rs50, pkg_path)
    print(f"Training complete. Package and model saved to {pkg_path}")

In [ ]:
import numpy as np
import pandas as pd

target_col = "avg_mdcr_stdzd_amt"
tail_flag_col = "is_top_1pct_avg_mdcr_stdzd_amt"
group_cols = ["provider_type", "HCPCS_Cd", "Place_Of_Srvc"]
pt_pos_state_cols = ["provider_type", "Place_Of_Srvc", "state"]
pt_pos_cols = ["provider_type", "Place_Of_Srvc"]
pt_cols = ["provider_type"]

# 1) Full 2023 test set (all rows, lag and no-lag)
test_df_full_2023 = eda_hcpcs_full_df.loc[eda_hcpcs_full_df["Year"] == 2023].copy()

# 2) Choose which years define the backoff means
# If you want to match the package that trained on 2021-2022 only:
train_years_for_means = [2021, 2022]
# If you intentionally want more history for means, use:
# train_years_for_means = [2020, 2021, 2022]

train_df_for_means = eda_hcpcs_full_df.loc[
    eda_hcpcs_full_df["Year"].isin(train_years_for_means)
].copy()

# 3) Compute backoff tables
train_means_grp = (
    train_df_for_means.groupby(group_cols)[target_col]
    .mean()
    .rename("mean_train_grp")
    .reset_index()
)

train_means_hcpcs = (
    train_df_for_means.groupby(["HCPCS_Cd"])[target_col]
    .mean()
    .rename("mean_train_hcpcs")
    .reset_index()
)

train_means_pt_pos_state = (
    train_df_for_means.groupby(pt_pos_state_cols)[target_col]
    .mean()
    .rename("mean_train_pt_pos_state")
    .reset_index()
)

train_medians_pt_pos_state = (
    train_df_for_means.groupby(pt_pos_state_cols)[target_col]
    .median()
    .rename("median_train_pt_pos_state")
    .reset_index()
)

train_means_pt_pos = (
    train_df_for_means.groupby(pt_pos_cols)[target_col]
    .mean()
    .rename("mean_train_pt_pos")
    .reset_index()
)

train_means_pt = (
    train_df_for_means.groupby(pt_cols)[target_col]
    .mean()
    .rename("mean_train_pt")
    .reset_index()
)

global_mean = float(train_df_for_means[target_col].mean())

# 4) Attach means to the FULL 2023 test set (preserve original index)
test_with_means = test_df_full_2023.copy()
test_with_means["_orig_idx"] = test_with_means.index

test_with_means = test_with_means.merge(train_means_grp, on=group_cols, how="left")
test_with_means = test_with_means.merge(train_means_hcpcs, on="HCPCS_Cd", how="left")
test_with_means = test_with_means.merge(train_means_pt_pos_state, on=pt_pos_state_cols, how="left")
test_with_means = test_with_means.merge(train_medians_pt_pos_state, on=pt_pos_state_cols, how="left")
test_with_means = test_with_means.merge(train_means_pt_pos, on=pt_pos_cols, how="left")
test_with_means = test_with_means.merge(train_means_pt, on=pt_cols, how="left")

# Restore original index
test_with_means = test_with_means.set_index("_orig_idx", drop=True)
test_with_means = test_with_means.loc[test_df_full_2023.index]  # keep same order

# 5) Split lag vs no-lag for residual model
mask_2023_lag = test_with_means["lag1_avg_amt"].notna() & (test_with_means["lag1_avg_amt"] >= 0)
test_2023_lag = test_with_means.loc[mask_2023_lag].copy()
test_2023_nolag = test_with_means.loc[~mask_2023_lag].copy()

# 6) Residual model prediction for lag-present rows
feature_cols = forecast_pkg_wt10_without_lags_no_2020_delta_trgt_rs50.feature_cols
X_2023_lag = test_2023_lag[feature_cols]

log_delta_hat = best_forecast_model_wt10_without_lags_no_2020_delta_trgt_rs50.predict(X_2023_lag)

pred_level_lag = np.expm1(np.log1p(test_2023_lag["lag1_avg_amt"].values) + log_delta_hat)

# 7) Backoff prediction for no-lag rows
pred_level_nolag = test_2023_nolag["mean_train_grp"].copy()
pred_level_nolag = pred_level_nolag.fillna(test_2023_nolag["mean_train_hcpcs"])
pred_level_nolag = pred_level_nolag.fillna(test_2023_nolag["mean_train_pt_pos_state"])
pred_level_nolag = pred_level_nolag.fillna(test_2023_nolag["median_train_pt_pos_state"])
pred_level_nolag = pred_level_nolag.fillna(test_2023_nolag["mean_train_pt_pos"])
pred_level_nolag = pred_level_nolag.fillna(test_2023_nolag["mean_train_pt"])
pred_level_nolag = pred_level_nolag.fillna(global_mean)
pred_level_nolag = pred_level_nolag.to_numpy()

# 8) Combine into a full prediction vector aligned to original 2023 test index
pred_full = pd.Series(index=test_df_full_2023.index, dtype=float)
pred_full.loc[test_2023_lag.index] = pred_level_lag
pred_full.loc[test_2023_nolag.index] = pred_level_nolag

# Rebuild pred_backoff with the same updated ladder
pred_backoff = test_with_means["lag1_avg_amt"].copy()
pred_backoff = pred_backoff.fillna(test_with_means["mean_train_grp"])
pred_backoff = pred_backoff.fillna(test_with_means["mean_train_hcpcs"])
pred_backoff = pred_backoff.fillna(test_with_means["mean_train_pt_pos_state"])
pred_backoff = pred_backoff.fillna(test_with_means["median_train_pt_pos_state"])
pred_backoff = pred_backoff.fillna(test_with_means["mean_train_pt_pos"])
pred_backoff = pred_backoff.fillna(test_with_means["mean_train_pt"])
pred_backoff = pred_backoff.fillna(global_mean)

In [ ]:
from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# Inputs you already have from your hybrid prediction cell:
# - test_df_full_2023  (full 2023 rows)
# - pred_full          (full 2023 predictions aligned to test_df_full_2023.index)
# ------------------------------------------------------------

target_col = "avg_mdcr_stdzd_amt"
tail_flag_col = "is_top_1pct_avg_mdcr_stdzd_amt"

y_true = test_df_full_2023[target_col].astype(float)
pred_test = pred_full.astype(float)

# Basic sanity check
assert pred_test.index.equals(test_df_full_2023.index)

# ------------------------------------------------------------
# 1) Overall metrics (2023 full)
# ------------------------------------------------------------
r2_test = r2_score(y_true, pred_test)
mae = mean_absolute_error(y_true, pred_test)
rmse = root_mean_squared_error(y_true, pred_test)

# "log space" metrics on LEVELS (not on delta), if you still want them:
r2_log_true_test = r2_score(np.log1p(y_true), np.log1p(pred_test.clip(lower=0)))
mae_log_true = mean_absolute_error(np.log1p(y_true), np.log1p(pred_test.clip(lower=0)))
rmse_log_true = root_mean_squared_error(np.log1p(y_true), np.log1p(pred_test.clip(lower=0)))

# ------------------------------------------------------------
# 2) Weighted (tail-emphasized) metrics on 2023 full
#    Use the SAME w_tail you used for weighting (10 here)
# ------------------------------------------------------------
w_tail = 10
is_tail_test = test_df_full_2023[tail_flag_col].astype(int).to_numpy()
sample_w_test = (1 + (w_tail - 1) * is_tail_test).astype(float)

r2_test_w = r2_score(y_true, pred_test, sample_weight=sample_w_test)
mae_w = mean_absolute_error(y_true, pred_test, sample_weight=sample_w_test)
rmse_w = root_mean_squared_error(y_true, pred_test, sample_weight=sample_w_test)

# ------------------------------------------------------------
# 3) Tail analysis table (same style as before)
# ------------------------------------------------------------
test_eval = test_df_full_2023.copy()
test_eval["pred"] = pred_test
test_eval["abs_err"] = (test_eval[target_col] - test_eval["pred"]).abs()
test_eval["sq_err"] = (test_eval[target_col] - test_eval["pred"]) ** 2

non_tail_pred_to_y_ratio = (
    test_eval.loc[~test_eval[tail_flag_col], "pred"].mean()
    / test_eval.loc[~test_eval[tail_flag_col], target_col].mean()
)
tail_pred_to_y_ratio = (
    test_eval.loc[test_eval[tail_flag_col], "pred"].mean()
    / test_eval.loc[test_eval[tail_flag_col], target_col].mean()
)

non_tail_sse = test_eval.loc[~test_eval[tail_flag_col], "sq_err"].sum()
tail_sse = test_eval.loc[test_eval[tail_flag_col], "sq_err"].sum()

non_tail_share_of_total_sse = non_tail_sse / (non_tail_sse + tail_sse)
tail_share_of_total_sse = tail_sse / (non_tail_sse + tail_sse)

non_tail_under_prediction_rate = (
    (test_eval.loc[~test_eval[tail_flag_col], "pred"] < test_eval.loc[~test_eval[tail_flag_col], target_col]).mean()
)
tail_under_prediction_rate = (
    (test_eval.loc[test_eval[tail_flag_col], "pred"] < test_eval.loc[test_eval[tail_flag_col], target_col]).mean()
)

non_tail_bias = (
    test_eval.loc[~test_eval[tail_flag_col], "pred"].mean()
    - test_eval.loc[~test_eval[tail_flag_col], target_col].mean()
)
tail_bias = (
    test_eval.loc[test_eval[tail_flag_col], "pred"].mean()
    - test_eval.loc[test_eval[tail_flag_col], target_col].mean()
)

forecast_hybrid_metrics_2023_without_lags_rs50 = pd.DataFrame({
    "w_tail":[w_tail],
    "r2_test": [r2_test],
    "mae": [mae],
    "rmse": [rmse],
    "r2_test_log": [r2_log_true_test],
    "mae_log": [mae_log_true],
    "rmse_log": [rmse_log_true],
    "r2_test_w": [r2_test_w],
    "mae_w": [mae_w],
    "rmse_w": [rmse_w],
    "non_tail_pred_to_y_ratio": [non_tail_pred_to_y_ratio],
    "tail_pred_to_y_ratio": [tail_pred_to_y_ratio],
    "non_tail_sse": [non_tail_sse],
    "tail_sse": [tail_sse],
    "non_tail_share_of_total_sse": [non_tail_share_of_total_sse],
    "tail_share_of_total_sse": [tail_share_of_total_sse],
    "non_tail_under_prediction_rate": [non_tail_under_prediction_rate],
    "tail_under_prediction_rate": [tail_under_prediction_rate],
    "non_tail_bias": [non_tail_bias],
    "tail_bias": [tail_bias],
})

forecast_hybrid_metrics_2023_without_lags_rs50

> Conclusion: Across multiple forecasting variants, the strongest and most reliable approach remained the simple lag-based baseline with structured fallback ladder. Residual XGBoost models, including both lag-aware and no-lag variants, did not deliver sufficiently consistent gains over this baseline to justify added complexity. Based on these results, I am stopping forecast-model iteration here and moving on to the new-provider modeling task.

# 2. New provider modeling: Training a model that predicts the average cost per service for new providers 

### 2.A. `GridSearchCV` + `target_col = "avc_mdcr_stdzd_amt"` + `w_tail=1` + `include_lags=True`

In [ ]:
from pathlib import Path
from joblib import dump, load

# 1. Setup Paths
ROOT = Path.cwd().resolve()
MODELS = (ROOT / "models").resolve()
MODELS.mkdir(parents=True, exist_ok=True)

# We now point to a package file instead of just the model file
pkg_path = MODELS / "newprov_pkg_wt1_with_lags.joblib"

# 2. Check/Load or Train/Save
if pkg_path.exists():
    print(f"Loading {pkg_path.name} from disk... (Skipping data prep and training)")
    # Load the entire package (data + fitted search object)
    newprov_pkg_wt1_with_lags = load(pkg_path)
    
    # Extract fitted best estimator for downstream use
    best_newprov_model_wt1_with_lags = newprov_pkg_wt1_with_lags.search.best_estimator_

else:
    print(f"File not found. Preparing data and training...")

    # Prepare grouped new-provider package: group holdout + GroupKFold (by NPI)
    newprov_pkg_wt1_with_lags = prepare_xgb_gridsearch(
        eda_hcpcs_df,
        approach="new_providers",
        target_col="avg_mdcr_stdzd_amt",
        w_tail=1,
        include_lags=True,
        test_size=0.2,
        random_state=0,
        n_splits=3,
    )

    # Train grouped CV search
    newprov_pkg_wt1_with_lags.search.fit(
        newprov_pkg_wt1_with_lags.X_train,
        newprov_pkg_wt1_with_lags.y_train,
        groups=newprov_pkg_wt1_with_lags.groups_train,
        model__sample_weight=newprov_pkg_wt1_with_lags.sample_w_train,
    )
    
    best_newprov_model_wt1_with_lags = newprov_pkg_wt1_with_lags.search.best_estimator_

    # 3. Save the ENTIRE package
    dump(newprov_pkg_wt1_with_lags, pkg_path)
    print(f"Training complete. Package and model saved to {pkg_path}")

### Model diagnostics (weighted training including lag variables)

In [ ]:
# Run diagnostics

# get train and test predictions and clip test predictions 
pred_test = best_newprov_model_wt1_with_lags.predict(newprov_pkg_wt1_with_lags.X_test)
pred_train = best_newprov_model_wt1_with_lags.predict(newprov_pkg_wt1_with_lags.X_train)

# get weighted-train unweighted/average-case metrics
r2_train = best_newprov_model_wt1_with_lags.score(newprov_pkg_wt1_with_lags.X_train, newprov_pkg_wt1_with_lags.y_train)
r2_test = best_newprov_model_wt1_with_lags.score(newprov_pkg_wt1_with_lags.X_test, newprov_pkg_wt1_with_lags.y_test)
mae = mean_absolute_error(newprov_pkg_wt1_with_lags.y_test, pred_test)
rmse = root_mean_squared_error(newprov_pkg_wt1_with_lags.y_test, pred_test)

# get weighted-train unweighted/average-case metrics in log space
log_pred_train = best_newprov_model_wt1_with_lags.regressor_.predict(newprov_pkg_wt1_with_lags.X_train)  # predictions in transformed target space
log_pred_test = best_newprov_model_wt1_with_lags.regressor_.predict(newprov_pkg_wt1_with_lags.X_test)  # predictions in transformed target space

r2_log_true_train = r2_score(np.log1p(newprov_pkg_wt1_with_lags.y_train), log_pred_train)
r2_log_true_test = r2_score(np.log1p(newprov_pkg_wt1_with_lags.y_test), log_pred_test)
mae_log_true = mean_absolute_error(np.log1p(newprov_pkg_wt1_with_lags.y_test), log_pred_test)
rmse_log_true = root_mean_squared_error(np.log1p(newprov_pkg_wt1_with_lags.y_test), log_pred_test)

# get weighted-train weighted/eval metrics

r2_train_w = r2_score(newprov_pkg_wt1_with_lags.y_train, pred_train, sample_weight=newprov_pkg_wt1_with_lags.sample_w_train)
r2_test_w = r2_score(newprov_pkg_wt1_with_lags.y_test, pred_test, sample_weight=newprov_pkg_wt1_with_lags.sample_w_test)
mae_w = mean_absolute_error(newprov_pkg_wt1_with_lags.y_test, pred_test, sample_weight=newprov_pkg_wt1_with_lags.sample_w_test)
rmse_w = root_mean_squared_error(newprov_pkg_wt1_with_lags.y_test, pred_test, sample_weight=newprov_pkg_wt1_with_lags.sample_w_test)

# Calculate tail evals
target_col = "avg_mdcr_stdzd_amt"
test_eval = newprov_pkg_wt1_with_lags.test_df.copy()

test_eval["pred"] = pred_test
test_eval["abs_err"] = (test_eval[target_col] - test_eval["pred"]).abs()
test_eval["sq_err"]  = (test_eval[target_col] - test_eval["pred"])**2

non_tail_pred_to_y_ratio = test_eval.loc[~test_eval["is_top_1pct_avg_mdcr_stdzd_amt"],"pred"].mean() / test_eval.loc[~test_eval["is_top_1pct_avg_mdcr_stdzd_amt"],target_col].mean()
tail_pred_to_y_ratio = test_eval.loc[test_eval["is_top_1pct_avg_mdcr_stdzd_amt"],"pred"].mean() / test_eval.loc[test_eval["is_top_1pct_avg_mdcr_stdzd_amt"],target_col].mean()

non_tail_sse = test_eval.loc[~test_eval["is_top_1pct_avg_mdcr_stdzd_amt"],"sq_err"].sum()
tail_sse = test_eval.loc[test_eval["is_top_1pct_avg_mdcr_stdzd_amt"],"sq_err"].sum()

non_tail_share_of_total_sse = non_tail_sse / (non_tail_sse + tail_sse)
tail_share_of_total_sse = tail_sse / (non_tail_sse + tail_sse)

non_tail_under_prediction_rate = ((test_eval.loc[~test_eval["is_top_1pct_avg_mdcr_stdzd_amt"],"pred"]) < test_eval.loc[~test_eval["is_top_1pct_avg_mdcr_stdzd_amt"],target_col]).mean()
tail_under_prediction_rate = ((test_eval.loc[test_eval["is_top_1pct_avg_mdcr_stdzd_amt"],"pred"]) < test_eval.loc[test_eval["is_top_1pct_avg_mdcr_stdzd_amt"],target_col]).mean()

non_tail_bias = test_eval.loc[~test_eval["is_top_1pct_avg_mdcr_stdzd_amt"],"pred"].mean() - test_eval.loc[~test_eval["is_top_1pct_avg_mdcr_stdzd_amt"],target_col].mean()
tail_bias = test_eval.loc[test_eval["is_top_1pct_avg_mdcr_stdzd_amt"],"pred"].mean() - test_eval.loc[test_eval["is_top_1pct_avg_mdcr_stdzd_amt"],target_col].mean()

newprov_weight1_lag_train_performance_metrics_test_eval = pd.DataFrame({
    "train_sample_weights":[1],
    "r2_train": r2_train,
    "r2_test": r2_test,
    "mae": mae,
    "rmse": rmse,
    "r2_train_log": r2_log_true_train,
    "r2_test_log": r2_log_true_test,
    "mae_log": mae_log_true,
    "rmse_log": rmse_log_true,
    "r2_train_w": r2_train_w,
    "r2_test_w": r2_test_w,
    "mae_w": mae_w,
    "rmse_w": rmse_w,
    "non_tail_pred_to_y_ratio": non_tail_pred_to_y_ratio,
    "tail_pred_to_y_ratio": tail_pred_to_y_ratio,
    "non_tail_sse": non_tail_sse,
    "tail_sse": tail_sse,
    "non_tail_share_of_total_sse": non_tail_share_of_total_sse,
    "tail_share_of_total_sse": tail_share_of_total_sse,
    "non_tail_under_prediction_rate": non_tail_under_prediction_rate,
    "tail_under_prediction_rate": tail_under_prediction_rate,
    "non_tail_bias": non_tail_bias,
    "tail_bias": tail_bias
})

newprov_weight1_lag_train_performance_metrics_test_eval

In [ ]:
with pd.option_context('display.max_columns', None):
    display(newprov_weight1_lag_train_performance_metrics_test_eval) # or print(df)

### 2.B. `GridSearchCV` + `target_col = "avc_mdcr_stdzd_amt"` + `w_tail=1` + `include_lags=False`

In [ ]:
from pathlib import Path
from joblib import dump, load

# 1. Setup Paths
ROOT = Path.cwd().resolve()
MODELS = (ROOT / "models").resolve()
MODELS.mkdir(parents=True, exist_ok=True)

# We now point to a package file instead of just the model file
pkg_path = MODELS / "newprov_pkg_wt1_without_lags.joblib"

# 2. Check/Load or Train/Save
if pkg_path.exists():
    print(f"Loading {pkg_path.name} from disk... (Skipping data prep and training)")
    # Load the entire package (data + fitted search object)
    newprov_pkg_wt1_without_lags = load(pkg_path)
    
    # Extract fitted best estimator for downstream use
    best_newprov_model_wt1_without_lags = newprov_pkg_wt1_without_lags.search.best_estimator_

else:
    print(f"File not found. Preparing data and training...")

    # Prepare grouped new-provider package: group holdout + GroupKFold (by NPI)
    newprov_pkg_wt1_without_lags = prepare_xgb_gridsearch(
        eda_hcpcs_df,
        approach="new_providers",
        target_col="avg_mdcr_stdzd_amt",
        w_tail=1,
        include_lags=False,
        test_size=0.2,
        random_state=0,
        n_splits=3,
    )

    # Train grouped CV search
    newprov_pkg_wt1_without_lags.search.fit(
        newprov_pkg_wt1_without_lags.X_train,
        newprov_pkg_wt1_without_lags.y_train,
        groups=newprov_pkg_wt1_without_lags.groups_train,
        model__sample_weight=newprov_pkg_wt1_without_lags.sample_w_train,
    )
    
    best_newprov_model_wt1_without_lags = newprov_pkg_wt1_without_lags.search.best_estimator_

    # 3. Save the ENTIRE package
    dump(newprov_pkg_wt1_without_lags, pkg_path)
    print(f"Training complete. Package and model saved to {pkg_path}")

### Model diagnostics (weighted training including lag variables)

In [ ]:
# Run diagnostics

# get train and test predictions and clip test predictions 
pred_test = best_newprov_model_wt1_without_lags.predict(newprov_pkg_wt1_without_lags.X_test)
pred_train = best_newprov_model_wt1_without_lags.predict(newprov_pkg_wt1_without_lags.X_train)

# get weighted-train unweighted/average-case metrics
r2_train = best_newprov_model_wt1_without_lags.score(newprov_pkg_wt1_without_lags.X_train, newprov_pkg_wt1_without_lags.y_train)
r2_test = best_newprov_model_wt1_without_lags.score(newprov_pkg_wt1_without_lags.X_test, newprov_pkg_wt1_without_lags.y_test)
mae = mean_absolute_error(newprov_pkg_wt1_without_lags.y_test, pred_test)
rmse = root_mean_squared_error(newprov_pkg_wt1_without_lags.y_test, pred_test)

# get weighted-train unweighted/average-case metrics in log space
log_pred_train = best_newprov_model_wt1_without_lags.regressor_.predict(newprov_pkg_wt1_without_lags.X_train)  # predictions in transformed target space
log_pred_test = best_newprov_model_wt1_without_lags.regressor_.predict(newprov_pkg_wt1_without_lags.X_test)  # predictions in transformed target space

r2_log_true_train = r2_score(np.log1p(newprov_pkg_wt1_without_lags.y_train), log_pred_train)
r2_log_true_test = r2_score(np.log1p(newprov_pkg_wt1_without_lags.y_test), log_pred_test)
mae_log_true = mean_absolute_error(np.log1p(newprov_pkg_wt1_without_lags.y_test), log_pred_test)
rmse_log_true = root_mean_squared_error(np.log1p(newprov_pkg_wt1_without_lags.y_test), log_pred_test)

# get weighted-train weighted/eval metrics

r2_train_w = r2_score(newprov_pkg_wt1_without_lags.y_train, pred_train, sample_weight=newprov_pkg_wt1_without_lags.sample_w_train)
r2_test_w = r2_score(newprov_pkg_wt1_without_lags.y_test, pred_test, sample_weight=newprov_pkg_wt1_without_lags.sample_w_test)
mae_w = mean_absolute_error(newprov_pkg_wt1_without_lags.y_test, pred_test, sample_weight=newprov_pkg_wt1_without_lags.sample_w_test)
rmse_w = root_mean_squared_error(newprov_pkg_wt1_without_lags.y_test, pred_test, sample_weight=newprov_pkg_wt1_without_lags.sample_w_test)

# Calculate tail evals
target_col = "avg_mdcr_stdzd_amt"
test_eval = newprov_pkg_wt1_without_lags.test_df.copy()

test_eval["pred"] = pred_test
test_eval["abs_err"] = (test_eval[target_col] - test_eval["pred"]).abs()
test_eval["sq_err"]  = (test_eval[target_col] - test_eval["pred"])**2

non_tail_pred_to_y_ratio = test_eval.loc[~test_eval["is_top_1pct_avg_mdcr_stdzd_amt"],"pred"].mean() / test_eval.loc[~test_eval["is_top_1pct_avg_mdcr_stdzd_amt"],target_col].mean()
tail_pred_to_y_ratio = test_eval.loc[test_eval["is_top_1pct_avg_mdcr_stdzd_amt"],"pred"].mean() / test_eval.loc[test_eval["is_top_1pct_avg_mdcr_stdzd_amt"],target_col].mean()

non_tail_sse = test_eval.loc[~test_eval["is_top_1pct_avg_mdcr_stdzd_amt"],"sq_err"].sum()
tail_sse = test_eval.loc[test_eval["is_top_1pct_avg_mdcr_stdzd_amt"],"sq_err"].sum()

non_tail_share_of_total_sse = non_tail_sse / (non_tail_sse + tail_sse)
tail_share_of_total_sse = tail_sse / (non_tail_sse + tail_sse)

non_tail_under_prediction_rate = ((test_eval.loc[~test_eval["is_top_1pct_avg_mdcr_stdzd_amt"],"pred"]) < test_eval.loc[~test_eval["is_top_1pct_avg_mdcr_stdzd_amt"],target_col]).mean()
tail_under_prediction_rate = ((test_eval.loc[test_eval["is_top_1pct_avg_mdcr_stdzd_amt"],"pred"]) < test_eval.loc[test_eval["is_top_1pct_avg_mdcr_stdzd_amt"],target_col]).mean()

non_tail_bias = test_eval.loc[~test_eval["is_top_1pct_avg_mdcr_stdzd_amt"],"pred"].mean() - test_eval.loc[~test_eval["is_top_1pct_avg_mdcr_stdzd_amt"],target_col].mean()
tail_bias = test_eval.loc[test_eval["is_top_1pct_avg_mdcr_stdzd_amt"],"pred"].mean() - test_eval.loc[test_eval["is_top_1pct_avg_mdcr_stdzd_amt"],target_col].mean()

newprov_weight1_no_lag_train_performance_metrics_test_eval = pd.DataFrame({
    "train_sample_weights":[1],
    "r2_train": r2_train,
    "r2_test": r2_test,
    "mae": mae,
    "rmse": rmse,
    "r2_train_log": r2_log_true_train,
    "r2_test_log": r2_log_true_test,
    "mae_log": mae_log_true,
    "rmse_log": rmse_log_true,
    "r2_train_w": r2_train_w,
    "r2_test_w": r2_test_w,
    "mae_w": mae_w,
    "rmse_w": rmse_w,
    "non_tail_pred_to_y_ratio": non_tail_pred_to_y_ratio,
    "tail_pred_to_y_ratio": tail_pred_to_y_ratio,
    "non_tail_sse": non_tail_sse,
    "tail_sse": tail_sse,
    "non_tail_share_of_total_sse": non_tail_share_of_total_sse,
    "tail_share_of_total_sse": tail_share_of_total_sse,
    "non_tail_under_prediction_rate": non_tail_under_prediction_rate,
    "tail_under_prediction_rate": tail_under_prediction_rate,
    "non_tail_bias": non_tail_bias,
    "tail_bias": tail_bias
})

newprov_weight1_no_lag_train_performance_metrics_test_eval

In [ ]:
with pd.option_context('display.max_columns', None):
    display(newprov_weight1_no_lag_train_performance_metrics_test_eval) # or print(df)

### 2.C. `GridSearchCV` + `target_col = "avc_mdcr_stdzd_amt"` + `w_tail=10` + `include_lags=True`

In [ ]:
from pathlib import Path
from joblib import dump, load

# 1. Setup Paths
ROOT = Path.cwd().resolve()
MODELS = (ROOT / "models").resolve()
MODELS.mkdir(parents=True, exist_ok=True)

# We now point to a package file instead of just the model file
pkg_path = MODELS / "newprov_pkg_wt10_with_lags.joblib"

# 2. Check/Load or Train/Save
if pkg_path.exists():
    print(f"Loading {pkg_path.name} from disk... (Skipping data prep and training)")
    # Load the entire package (data + fitted search object)
    newprov_pkg_wt10_with_lags = load(pkg_path)
    
    # Extract fitted best estimator for downstream use
    best_newprov_model_wt10_with_lags = newprov_pkg_wt10_with_lags.search.best_estimator_

else:
    print(f"File not found. Preparing data and training...")

    # Prepare grouped new-provider package: group holdout + GroupKFold (by NPI)
    newprov_pkg_wt10_with_lags = prepare_xgb_gridsearch(
        eda_hcpcs_df,
        approach="new_providers",
        target_col="avg_mdcr_stdzd_amt",
        w_tail=10,
        include_lags=True,
        test_size=0.2,
        random_state=0,
        n_splits=3,
    )

    # Train grouped CV search
    newprov_pkg_wt10_with_lags.search.fit(
        newprov_pkg_wt10_with_lags.X_train,
        newprov_pkg_wt10_with_lags.y_train,
        groups=newprov_pkg_wt10_with_lags.groups_train,
        model__sample_weight=newprov_pkg_wt10_with_lags.sample_w_train,
    )
    
    best_newprov_model_wt10_with_lags = newprov_pkg_wt10_with_lags.search.best_estimator_

    # Save the ENTIRE package
    dump(newprov_pkg_wt10_with_lags, pkg_path)
    print(f"Training complete. Package and model saved to {pkg_path}")

### Model diagnostics (weighted training including lag variables)

In [ ]:
# Run diagnostics

# get train and test predictions and clip test predictions 
pred_test = best_newprov_model_wt10_with_lags.predict(newprov_pkg_wt10_with_lags.X_test)
pred_train = best_newprov_model_wt10_with_lags.predict(newprov_pkg_wt10_with_lags.X_train)

# get weighted-train unweighted/average-case metrics
r2_train = best_newprov_model_wt10_with_lags.score(newprov_pkg_wt10_with_lags.X_train, newprov_pkg_wt10_with_lags.y_train)
r2_test = best_newprov_model_wt10_with_lags.score(newprov_pkg_wt10_with_lags.X_test, newprov_pkg_wt10_with_lags.y_test)
mae = mean_absolute_error(newprov_pkg_wt10_with_lags.y_test, pred_test)
rmse = root_mean_squared_error(newprov_pkg_wt10_with_lags.y_test, pred_test)

# get weighted-train unweighted/average-case metrics in log space
log_pred_train = best_newprov_model_wt10_with_lags.regressor_.predict(newprov_pkg_wt10_with_lags.X_train)  # predictions in transformed target space
log_pred_test = best_newprov_model_wt10_with_lags.regressor_.predict(newprov_pkg_wt10_with_lags.X_test)  # predictions in transformed target space

r2_log_true_train = r2_score(np.log1p(newprov_pkg_wt10_with_lags.y_train), log_pred_train)
r2_log_true_test = r2_score(np.log1p(newprov_pkg_wt10_with_lags.y_test), log_pred_test)
mae_log_true = mean_absolute_error(np.log1p(newprov_pkg_wt10_with_lags.y_test), log_pred_test)
rmse_log_true = root_mean_squared_error(np.log1p(newprov_pkg_wt10_with_lags.y_test), log_pred_test)

# get weighted-train weighted/eval metrics

r2_train_w = r2_score(newprov_pkg_wt10_with_lags.y_train, pred_train, sample_weight=newprov_pkg_wt10_with_lags.sample_w_train)
r2_test_w = r2_score(newprov_pkg_wt10_with_lags.y_test, pred_test, sample_weight=newprov_pkg_wt10_with_lags.sample_w_test)
mae_w = mean_absolute_error(newprov_pkg_wt10_with_lags.y_test, pred_test, sample_weight=newprov_pkg_wt10_with_lags.sample_w_test)
rmse_w = root_mean_squared_error(newprov_pkg_wt10_with_lags.y_test, pred_test, sample_weight=newprov_pkg_wt10_with_lags.sample_w_test)

# Calculate tail evals
target_col = "avg_mdcr_stdzd_amt"
test_eval = newprov_pkg_wt10_with_lags.test_df.copy()

test_eval["pred"] = pred_test
test_eval["abs_err"] = (test_eval[target_col] - test_eval["pred"]).abs()
test_eval["sq_err"]  = (test_eval[target_col] - test_eval["pred"])**2

non_tail_pred_to_y_ratio = test_eval.loc[~test_eval["is_top_1pct_avg_mdcr_stdzd_amt"],"pred"].mean() / test_eval.loc[~test_eval["is_top_1pct_avg_mdcr_stdzd_amt"],target_col].mean()
tail_pred_to_y_ratio = test_eval.loc[test_eval["is_top_1pct_avg_mdcr_stdzd_amt"],"pred"].mean() / test_eval.loc[test_eval["is_top_1pct_avg_mdcr_stdzd_amt"],target_col].mean()

non_tail_sse = test_eval.loc[~test_eval["is_top_1pct_avg_mdcr_stdzd_amt"],"sq_err"].sum()
tail_sse = test_eval.loc[test_eval["is_top_1pct_avg_mdcr_stdzd_amt"],"sq_err"].sum()

non_tail_share_of_total_sse = non_tail_sse / (non_tail_sse + tail_sse)
tail_share_of_total_sse = tail_sse / (non_tail_sse + tail_sse)

non_tail_under_prediction_rate = ((test_eval.loc[~test_eval["is_top_1pct_avg_mdcr_stdzd_amt"],"pred"]) < test_eval.loc[~test_eval["is_top_1pct_avg_mdcr_stdzd_amt"],target_col]).mean()
tail_under_prediction_rate = ((test_eval.loc[test_eval["is_top_1pct_avg_mdcr_stdzd_amt"],"pred"]) < test_eval.loc[test_eval["is_top_1pct_avg_mdcr_stdzd_amt"],target_col]).mean()

non_tail_bias = test_eval.loc[~test_eval["is_top_1pct_avg_mdcr_stdzd_amt"],"pred"].mean() - test_eval.loc[~test_eval["is_top_1pct_avg_mdcr_stdzd_amt"],target_col].mean()
tail_bias = test_eval.loc[test_eval["is_top_1pct_avg_mdcr_stdzd_amt"],"pred"].mean() - test_eval.loc[test_eval["is_top_1pct_avg_mdcr_stdzd_amt"],target_col].mean()

newprov_weight10_lag_train_performance_metrics_test_eval = pd.DataFrame({
    "train_sample_weights":[10],
    "r2_train": r2_train,
    "r2_test": r2_test,
    "mae": mae,
    "rmse": rmse,
    "r2_train_log": r2_log_true_train,
    "r2_test_log": r2_log_true_test,
    "mae_log": mae_log_true,
    "rmse_log": rmse_log_true,
    "r2_train_w": r2_train_w,
    "r2_test_w": r2_test_w,
    "mae_w": mae_w,
    "rmse_w": rmse_w,
    "non_tail_pred_to_y_ratio": non_tail_pred_to_y_ratio,
    "tail_pred_to_y_ratio": tail_pred_to_y_ratio,
    "non_tail_sse": non_tail_sse,
    "tail_sse": tail_sse,
    "non_tail_share_of_total_sse": non_tail_share_of_total_sse,
    "tail_share_of_total_sse": tail_share_of_total_sse,
    "non_tail_under_prediction_rate": non_tail_under_prediction_rate,
    "tail_under_prediction_rate": tail_under_prediction_rate,
    "non_tail_bias": non_tail_bias,
    "tail_bias": tail_bias
})

newprov_weight10_lag_train_performance_metrics_test_eval

In [ ]:
with pd.option_context('display.max_columns', None):
    display(newprov_weight10_lag_train_performance_metrics_test_eval) # or print(df)

### 2.D. `GridSearchCV` + `target_col = "avc_mdcr_stdzd_amt"` + `w_tail=10` + `include_lags=False`

In [ ]:
from pathlib import Path
from joblib import dump, load

# 1. Setup Paths
ROOT = Path.cwd().resolve()
MODELS = (ROOT / "models").resolve()
MODELS.mkdir(parents=True, exist_ok=True)

# We now point to a package file instead of just the model file
pkg_path = MODELS / "newprov_pkg_wt10_without_lags.joblib"

# 2. Check/Load or Train/Save
if pkg_path.exists():
    print(f"Loading {pkg_path.name} from disk... (Skipping data prep and training)")
    # Load the entire package (data + fitted search object)
    newprov_pkg_wt10_without_lags = load(pkg_path)
    
    # Extract fitted best estimator for downstream use
    best_newprov_model_wt10_without_lags = newprov_pkg_wt10_without_lags.search.best_estimator_

else:
    print(f"File not found. Preparing data and training...")

    # Prepare grouped new-provider package: group holdout + GroupKFold (by NPI)
    newprov_pkg_wt10_without_lags = prepare_xgb_gridsearch(
        eda_hcpcs_df,
        approach="new_providers",
        target_col="avg_mdcr_stdzd_amt",
        w_tail=10,
        include_lags=False,
        test_size=0.2,
        random_state=0,
        n_splits=3,
    )

    # Train grouped CV search
    newprov_pkg_wt10_without_lags.search.fit(
        newprov_pkg_wt10_without_lags.X_train,
        newprov_pkg_wt10_without_lags.y_train,
        groups=newprov_pkg_wt10_without_lags.groups_train,
        model__sample_weight=newprov_pkg_wt10_without_lags.sample_w_train,
    )
    
    best_newprov_model_wt10_without_lags = newprov_pkg_wt10_without_lags.search.best_estimator_

    # Save the ENTIRE package
    dump(newprov_pkg_wt10_without_lags, pkg_path)
    print(f"Training complete. Package and model saved to {pkg_path}")

### Model diagnostics (weighted training including lag variables)

In [ ]:
# Run diagnostics

# get train and test predictions and clip test predictions 
pred_test = best_newprov_model_wt10_without_lags.predict(newprov_pkg_wt10_without_lags.X_test)
pred_train = best_newprov_model_wt10_without_lags.predict(newprov_pkg_wt10_without_lags.X_train)

# get weighted-train unweighted/average-case metrics
r2_train = best_newprov_model_wt10_without_lags.score(newprov_pkg_wt10_without_lags.X_train, newprov_pkg_wt10_without_lags.y_train)
r2_test = best_newprov_model_wt10_without_lags.score(newprov_pkg_wt10_without_lags.X_test, newprov_pkg_wt10_without_lags.y_test)
mae = mean_absolute_error(newprov_pkg_wt10_without_lags.y_test, pred_test)
rmse = root_mean_squared_error(newprov_pkg_wt10_without_lags.y_test, pred_test)

# get weighted-train unweighted/average-case metrics in log space
log_pred_train = best_newprov_model_wt10_without_lags.regressor_.predict(newprov_pkg_wt10_without_lags.X_train)  # predictions in transformed target space
log_pred_test = best_newprov_model_wt10_without_lags.regressor_.predict(newprov_pkg_wt10_without_lags.X_test)  # predictions in transformed target space

r2_log_true_train = r2_score(np.log1p(newprov_pkg_wt10_without_lags.y_train), log_pred_train)
r2_log_true_test = r2_score(np.log1p(newprov_pkg_wt10_without_lags.y_test), log_pred_test)
mae_log_true = mean_absolute_error(np.log1p(newprov_pkg_wt10_without_lags.y_test), log_pred_test)
rmse_log_true = root_mean_squared_error(np.log1p(newprov_pkg_wt10_without_lags.y_test), log_pred_test)

# get weighted-train weighted/eval metrics

r2_train_w = r2_score(newprov_pkg_wt10_without_lags.y_train, pred_train, sample_weight=newprov_pkg_wt10_without_lags.sample_w_train)
r2_test_w = r2_score(newprov_pkg_wt10_without_lags.y_test, pred_test, sample_weight=newprov_pkg_wt10_without_lags.sample_w_test)
mae_w = mean_absolute_error(newprov_pkg_wt10_without_lags.y_test, pred_test, sample_weight=newprov_pkg_wt10_without_lags.sample_w_test)
rmse_w = root_mean_squared_error(newprov_pkg_wt10_without_lags.y_test, pred_test, sample_weight=newprov_pkg_wt10_without_lags.sample_w_test)

# Calculate tail evals
target_col = "avg_mdcr_stdzd_amt"
test_eval = newprov_pkg_wt10_without_lags.test_df.copy()

test_eval["pred"] = pred_test
test_eval["abs_err"] = (test_eval[target_col] - test_eval["pred"]).abs()
test_eval["sq_err"]  = (test_eval[target_col] - test_eval["pred"])**2

non_tail_pred_to_y_ratio = test_eval.loc[~test_eval["is_top_1pct_avg_mdcr_stdzd_amt"],"pred"].mean() / test_eval.loc[~test_eval["is_top_1pct_avg_mdcr_stdzd_amt"],target_col].mean()
tail_pred_to_y_ratio = test_eval.loc[test_eval["is_top_1pct_avg_mdcr_stdzd_amt"],"pred"].mean() / test_eval.loc[test_eval["is_top_1pct_avg_mdcr_stdzd_amt"],target_col].mean()

non_tail_sse = test_eval.loc[~test_eval["is_top_1pct_avg_mdcr_stdzd_amt"],"sq_err"].sum()
tail_sse = test_eval.loc[test_eval["is_top_1pct_avg_mdcr_stdzd_amt"],"sq_err"].sum()

non_tail_share_of_total_sse = non_tail_sse / (non_tail_sse + tail_sse)
tail_share_of_total_sse = tail_sse / (non_tail_sse + tail_sse)

non_tail_under_prediction_rate = ((test_eval.loc[~test_eval["is_top_1pct_avg_mdcr_stdzd_amt"],"pred"]) < test_eval.loc[~test_eval["is_top_1pct_avg_mdcr_stdzd_amt"],target_col]).mean()
tail_under_prediction_rate = ((test_eval.loc[test_eval["is_top_1pct_avg_mdcr_stdzd_amt"],"pred"]) < test_eval.loc[test_eval["is_top_1pct_avg_mdcr_stdzd_amt"],target_col]).mean()

non_tail_bias = test_eval.loc[~test_eval["is_top_1pct_avg_mdcr_stdzd_amt"],"pred"].mean() - test_eval.loc[~test_eval["is_top_1pct_avg_mdcr_stdzd_amt"],target_col].mean()
tail_bias = test_eval.loc[test_eval["is_top_1pct_avg_mdcr_stdzd_amt"],"pred"].mean() - test_eval.loc[test_eval["is_top_1pct_avg_mdcr_stdzd_amt"],target_col].mean()

newprov_weight10_no_lag_train_performance_metrics_test_eval = pd.DataFrame({
    "train_sample_weights":[10],
    "r2_train": r2_train,
    "r2_test": r2_test,
    "mae": mae,
    "rmse": rmse,
    "r2_train_log": r2_log_true_train,
    "r2_test_log": r2_log_true_test,
    "mae_log": mae_log_true,
    "rmse_log": rmse_log_true,
    "r2_train_w": r2_train_w,
    "r2_test_w": r2_test_w,
    "mae_w": mae_w,
    "rmse_w": rmse_w,
    "non_tail_pred_to_y_ratio": non_tail_pred_to_y_ratio,
    "tail_pred_to_y_ratio": tail_pred_to_y_ratio,
    "non_tail_sse": non_tail_sse,
    "tail_sse": tail_sse,
    "non_tail_share_of_total_sse": non_tail_share_of_total_sse,
    "tail_share_of_total_sse": tail_share_of_total_sse,
    "non_tail_under_prediction_rate": non_tail_under_prediction_rate,
    "tail_under_prediction_rate": tail_under_prediction_rate,
    "non_tail_bias": non_tail_bias,
    "tail_bias": tail_bias
})

newprov_weight10_no_lag_train_performance_metrics_test_eval

In [ ]:
with pd.option_context('display.max_columns', None):
    display(newprov_weight10_no_lag_train_performance_metrics_test_eval) # or print(df)

## New Provider `GridSearchCV` Modeling Summary (2A-2D)

### Objective
The second modeling task asked a different question from the forecast exercise: **can we predict average Medicare standardized amount per service for providers that were not seen during training?** To simulate that setting, the modeling workflow used provider-disjoint splits by `Rndrng_NPI`, so providers in the test set never appeared in the training folds.

This makes the task meaningfully harder than the forecast problem. In the forecast setting, the model could exploit continuity within the same provider over time. In the new-provider setting, the model must generalize across providers using cross-sectional structure rather than relying on direct provider-specific familiarity.

---

### Experimental Setup
We evaluated four baseline XGBoost pipelines using the grouped new-provider setup:
* **Unweighted, with lag features** (`w_tail=1`, `include_lags=True`)
* **Unweighted, without lag features** (`w_tail=1`, `include_lags=False`)
* **Tail-weighted, with lag features** (`w_tail=10`, `include_lags=True`)
* **Tail-weighted, without lag features** (`w_tail=10`, `include_lags=False`)

All models used:
* `GroupShuffleSplit` for provider-disjoint train/test partitioning
* `GroupKFold(n_splits=3)` for grouped cross-validation inside tuning
* `TransformedTargetRegressor(log1p/expm1)` for the target
* The same preprocessing framework used in the forecast workflow

> **QC Check:** I confirmed that the corrected *without-lag* packages truly excluded lag variables from `feature_cols`, `num_features`, `X_train`, and fitted preprocessing feature names. The comparisons below are now internally consistent and trustworthy.

---

### Performance Summary

#### 1) Unweighted model with lag features
`newprov_pkg_wt1_with_lags`

* **R² test:** 0.769
* **MAE:** 12.33
* **RMSE:** 100.24
* **R² test (log space):** 0.9788
* **Tail prediction-to-true ratio:** 0.810
* **Tail underprediction rate:** 0.798
* **Tail bias:** -321.11

**Interpretation:** This is a solid average-case baseline. The model performs reasonably well overall and has strong log-space fit, which means it captures broad multiplicative structure in the data. However, it still compresses the upper end of the outcome distribution too aggressively. Tail rows dominate the error structure, with about **92.1% of total SSE** coming from tail observations. The model systematically underpredicts expensive providers.

In practical terms, this model is decent for ordinary rows but not reliable enough if the goal is to better distinguish very high-cost providers.

#### 2) Unweighted model without lag features
`newprov_pkg_wt1_without_lags`

* **R² test:** 0.761
* **MAE:** 14.99
* **RMSE:** 101.98
* **R² test (log space):** 0.9794
* **Tail prediction-to-true ratio:** 0.754
* **Tail underprediction rate:** 0.885
* **Tail bias:** -416.65

**Interpretation:** This is the cleanest approximation of a harder cold-start scenario because the model does not use provider lag history. Once the lag features were correctly removed, performance declined relative to the unweighted lag model. The difference is not huge on overall R², but it is materially worse in the expensive regime.

This model shows the strongest tendency to miss upward on high-cost providers. It has the **worst tail bias** and **worst tail underprediction rate** among the four corrected baselines. That makes it the weakest option if identifying or approximating expensive providers matters.

#### 3) Tail-weighted model with lag features
`newprov_pkg_wt10_with_lags`

* **R² test:** 0.805
* **MAE:** 17.69
* **RMSE:** 92.15
* **R² test (log space):** 0.9523
* **Tail prediction-to-true ratio:** 0.869
* **Tail underprediction rate:** 0.678
* **Tail bias:** -222.06

**Interpretation:** Upweighting the tail from 1 to 10 substantially changed model behavior. Compared with the unweighted lag model, this version improved overall R², RMSE, tail calibration, tail bias, and tail underprediction rate. 

The tradeoff is that **MAE got worse** and log-space performance declined. That pattern is expected. By emphasizing expensive rows during training, the model gives up some average-case smoothness in exchange for better handling of the upper tail. This model is a strong candidate if the business goal prioritizes better handling of costly providers, even at the expense of somewhat worse typical-row absolute error.

#### 4) Tail-weighted model without lag features
`newprov_pkg_wt10_without_lags`

* **R² test:** 0.8869
* **MAE:** 13.87
* **RMSE:** 70.14
* **R² test (log space):** 0.9775
* **Tail prediction-to-true ratio:** 0.923
* **Tail underprediction rate:** 0.766
* **Tail bias:** -130.63

**Interpretation:** This was the **strongest overall performer** among the four corrected baselines. It achieved the best test R², the best RMSE, the best tail mean calibration, the smallest tail bias, and the most balanced split between tail and non-tail SSE.

The result is especially interesting because this model **does not use lag features**. That suggests that, for the provider-disjoint new-provider problem, broader cross-sectional structure generalized better than provider-history lag variables. In other words, lag history was not necessary to get the best grouped generalization once the model was encouraged to care about the expensive regime. It appears to offer the best balance between overall fit and upper-tail robustness.

---

### Cross-Model Interpretation

A few high-level conclusions stand out clearly:

* **Tail weighting helped a lot:** Moving from `w_tail=1` to `w_tail=10` improved performance substantially in the new-provider task. The weighted models achieved higher test R², lower RMSE, better tail calibration, and smaller tail bias. This indicates that the expensive rows were not being fit adequately under ordinary training and benefited from explicit reweighting.
* **Lag features did not dominate the new-provider task:** In the forecast problem, lag features were naturally very useful because the same provider was being followed through time. In the new-provider problem, the test providers are disjoint from training providers by design. The corrected results suggest that with `w_tail=10`, the *without-lag* model actually performed best overall. Transportable cross-sectional variables may matter more than provider-history lag features.
* **Tail errors still matter even in the strongest model:** Even the best model still underpredicts the tail on average. The strongest model reduced that problem a lot, but did not eliminate it completely. That means tail-aware refinement is still possible, especially if downstream use cases are highly sensitive to underestimating high-cost providers.

---

### Best Baseline at This Stage

**Top Baseline:** `newprov_pkg_wt10_without_lags`

Why this one stands out:
* Highest R² test
* Lowest RMSE
* Strongest tail calibration
* Smallest tail bias
* Balanced error distribution between tail and non-tail rows

This makes it the natural baseline to beat in the next step when exploring `RandomizedSearchCV()`. 

A strong secondary comparator is `newprov_pkg_wt10_with_lags`. This model is still useful as a contrast because it had a somewhat better tail underprediction rate, even though it lost on most other top-level metrics.

### Practical Takeaway
The new-provider task behaves differently from the forecast task. In forecast modeling, lag structure was central. In provider-disjoint generalization, the most successful baseline so far is the **tail-weighted model without lag features**, which suggests that transportable service-, provider-, and geography-level structure may be more important than provider-history features when predicting outcomes for unseen NPIs. 

That result gives a strong foundation for the next stage of tuning with `RandomizedSearchCV()`.

---

### Corrected Baseline Performance Snapshot

| Model | Tail Weight | Lags? | R² test | MAE | RMSE | R² test log | Tail Pred/True Ratio | Tail Underprediction Rate | Tail Bias |
| :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- |
| `newprov_pkg_wt1_with_lags` | 1 | Yes | 0.7690 | 12.33 | 100.24 | 0.9788 | 0.8102 | 0.7977 | -321.11 |
| `newprov_pkg_wt1_without_lags` | 1 | No | 0.7609 | 14.99 | 101.98 | 0.9794 | 0.7537 | 0.8848 | -416.65 |
| `newprov_pkg_wt10_with_lags` | 10 | Yes | 0.8048 | 17.69 | 92.15 | 0.9523 | 0.8687 | 0.6780 | -222.06 |
| `newprov_pkg_wt10_without_lags` | 10 | No | 0.8869 | 13.87 | 70.14 | 0.9775 | 0.9228 | 0.7656 | -130.63 |

### 2.E. `RandomizedSearchCV` + `target_col = "avg_mdcr_stdzd_amt"` + `w_tail=10` + `include_lags=True`

In [ ]:
from pathlib import Path
from joblib import dump, load

# 1. Setup Paths
ROOT = Path.cwd().resolve()
MODELS = (ROOT / "models").resolve()
MODELS.mkdir(parents=True, exist_ok=True)

pkg_path = MODELS / "newprov_pkg_wt10_with_lags_rs50.joblib"

# 2. Check/Load or Train/Save
if pkg_path.exists():
    print(f"Loading {pkg_path.name} from disk... (Skipping data prep and training)")
    newprov_pkg_wt10_with_lags_rs50 = load(pkg_path)
    best_newprov_model_wt10_with_lags_rs50 = newprov_pkg_wt10_with_lags_rs50.search.best_estimator_

else:
    print(f"File not found. Preparing data and training...")

    # --- PREPARE DATA ---
    newprov_pkg_wt10_with_lags_rs50 = prepare_xgb_randomizedsearch(
        eda_hcpcs_df,  # Assumes base df, change to lags_present_df if required
        approach="new_providers",
        target_col="avg_mdcr_stdzd_amt",
        w_tail=10,
        include_lags=True,
        test_size=0.2,
        random_state=0,
        n_splits=3,
        n_iter=50
    )

    # --- TRAIN ---
    if newprov_pkg_wt10_with_lags_rs50.groups_train is None:
        newprov_pkg_wt10_with_lags_rs50.search.fit(
            newprov_pkg_wt10_with_lags_rs50.X_train,
            newprov_pkg_wt10_with_lags_rs50.y_train,
            model__sample_weight=newprov_pkg_wt10_with_lags_rs50.sample_w_train,
        )
    else:
        newprov_pkg_wt10_with_lags_rs50.search.fit(
            newprov_pkg_wt10_with_lags_rs50.X_train,
            newprov_pkg_wt10_with_lags_rs50.y_train,
            groups=newprov_pkg_wt10_with_lags_rs50.groups_train,
            model__sample_weight=newprov_pkg_wt10_with_lags_rs50.sample_w_train,
        )
        
    best_newprov_model_wt10_with_lags_rs50 = newprov_pkg_wt10_with_lags_rs50.search.best_estimator_

    # --- SAVE ---
    dump(newprov_pkg_wt10_with_lags_rs50, pkg_path)
    print(f"Training complete. Package and model saved to {pkg_path}")

In [ ]:
# Run diagnostics

# get train and test predictions and clip test predictions 
pred_test = best_newprov_model_wt10_with_lags_rs50.predict(newprov_pkg_wt10_with_lags_rs50.X_test)
pred_train = best_newprov_model_wt10_with_lags_rs50.predict(newprov_pkg_wt10_with_lags_rs50.X_train)

# get weighted-train unweighted/average-case metrics
r2_train = best_newprov_model_wt10_with_lags_rs50.score(newprov_pkg_wt10_with_lags_rs50.X_train, newprov_pkg_wt10_with_lags_rs50.y_train)
r2_test = best_newprov_model_wt10_with_lags_rs50.score(newprov_pkg_wt10_with_lags_rs50.X_test, newprov_pkg_wt10_with_lags_rs50.y_test)
mae = mean_absolute_error(newprov_pkg_wt10_with_lags_rs50.y_test, pred_test)
rmse = root_mean_squared_error(newprov_pkg_wt10_with_lags_rs50.y_test, pred_test)

# get weighted-train unweighted/average-case metrics in log space
log_pred_train = best_newprov_model_wt10_with_lags_rs50.regressor_.predict(newprov_pkg_wt10_with_lags_rs50.X_train)  # predictions in transformed target space
log_pred_test = best_newprov_model_wt10_with_lags_rs50.regressor_.predict(newprov_pkg_wt10_with_lags_rs50.X_test)  # predictions in transformed target space

r2_log_true_train = r2_score(np.log1p(newprov_pkg_wt10_with_lags_rs50.y_train), log_pred_train)
r2_log_true_test = r2_score(np.log1p(newprov_pkg_wt10_with_lags_rs50.y_test), log_pred_test)
mae_log_true = mean_absolute_error(np.log1p(newprov_pkg_wt10_with_lags_rs50.y_test), log_pred_test)
rmse_log_true = root_mean_squared_error(np.log1p(newprov_pkg_wt10_with_lags_rs50.y_test), log_pred_test)

# get weighted-train weighted/eval metrics

r2_train_w = r2_score(newprov_pkg_wt10_with_lags_rs50.y_train, pred_train, sample_weight=newprov_pkg_wt10_with_lags_rs50.sample_w_train)
r2_test_w = r2_score(newprov_pkg_wt10_with_lags_rs50.y_test, pred_test, sample_weight=newprov_pkg_wt10_with_lags_rs50.sample_w_test)
mae_w = mean_absolute_error(newprov_pkg_wt10_with_lags_rs50.y_test, pred_test, sample_weight=newprov_pkg_wt10_with_lags_rs50.sample_w_test)
rmse_w = root_mean_squared_error(newprov_pkg_wt10_with_lags_rs50.y_test, pred_test, sample_weight=newprov_pkg_wt10_with_lags_rs50.sample_w_test)

# Calculate tail evals
target_col = "avg_mdcr_stdzd_amt"
test_eval = newprov_pkg_wt10_with_lags_rs50.test_df.copy()

test_eval["pred"] = pred_test
test_eval["abs_err"] = (test_eval[target_col] - test_eval["pred"]).abs()
test_eval["sq_err"]  = (test_eval[target_col] - test_eval["pred"])**2

non_tail_pred_to_y_ratio = test_eval.loc[~test_eval["is_top_1pct_avg_mdcr_stdzd_amt"],"pred"].mean() / test_eval.loc[~test_eval["is_top_1pct_avg_mdcr_stdzd_amt"],target_col].mean()
tail_pred_to_y_ratio = test_eval.loc[test_eval["is_top_1pct_avg_mdcr_stdzd_amt"],"pred"].mean() / test_eval.loc[test_eval["is_top_1pct_avg_mdcr_stdzd_amt"],target_col].mean()

non_tail_sse = test_eval.loc[~test_eval["is_top_1pct_avg_mdcr_stdzd_amt"],"sq_err"].sum()
tail_sse = test_eval.loc[test_eval["is_top_1pct_avg_mdcr_stdzd_amt"],"sq_err"].sum()

non_tail_share_of_total_sse = non_tail_sse / (non_tail_sse + tail_sse)
tail_share_of_total_sse = tail_sse / (non_tail_sse + tail_sse)

non_tail_under_prediction_rate = ((test_eval.loc[~test_eval["is_top_1pct_avg_mdcr_stdzd_amt"],"pred"]) < test_eval.loc[~test_eval["is_top_1pct_avg_mdcr_stdzd_amt"],target_col]).mean()
tail_under_prediction_rate = ((test_eval.loc[test_eval["is_top_1pct_avg_mdcr_stdzd_amt"],"pred"]) < test_eval.loc[test_eval["is_top_1pct_avg_mdcr_stdzd_amt"],target_col]).mean()

non_tail_bias = test_eval.loc[~test_eval["is_top_1pct_avg_mdcr_stdzd_amt"],"pred"].mean() - test_eval.loc[~test_eval["is_top_1pct_avg_mdcr_stdzd_amt"],target_col].mean()
tail_bias = test_eval.loc[test_eval["is_top_1pct_avg_mdcr_stdzd_amt"],"pred"].mean() - test_eval.loc[test_eval["is_top_1pct_avg_mdcr_stdzd_amt"],target_col].mean()

newprov_weight10_lag_train_rs50_performance_metrics_test_eval = pd.DataFrame({
    "train_sample_weights":[10],
    "r2_train": r2_train,
    "r2_test": r2_test,
    "mae": mae,
    "rmse": rmse,
    "r2_train_log": r2_log_true_train,
    "r2_test_log": r2_log_true_test,
    "mae_log": mae_log_true,
    "rmse_log": rmse_log_true,
    "r2_train_w": r2_train_w,
    "r2_test_w": r2_test_w,
    "mae_w": mae_w,
    "rmse_w": rmse_w,
    "non_tail_pred_to_y_ratio": non_tail_pred_to_y_ratio,
    "tail_pred_to_y_ratio": tail_pred_to_y_ratio,
    "non_tail_sse": non_tail_sse,
    "tail_sse": tail_sse,
    "non_tail_share_of_total_sse": non_tail_share_of_total_sse,
    "tail_share_of_total_sse": tail_share_of_total_sse,
    "non_tail_under_prediction_rate": non_tail_under_prediction_rate,
    "tail_under_prediction_rate": tail_under_prediction_rate,
    "non_tail_bias": non_tail_bias,
    "tail_bias": tail_bias
})

newprov_weight10_lag_train_rs50_performance_metrics_test_eval

In [ ]:
with pd.option_context('display.max_columns', None):
    display(newprov_weight10_lag_train_rs50_performance_metrics_test_eval) # or print(df)

### 2.F. `RandomizedSearchCV` + `target_col = "avg_mdcr_stdzd_amt"` + `w_tail=10` + `include_lags=False`

In [ ]:
from pathlib import Path
from joblib import dump, load

# 1. Setup Paths
ROOT = Path.cwd().resolve()
MODELS = (ROOT / "models").resolve()
MODELS.mkdir(parents=True, exist_ok=True)

pkg_path = MODELS / "newprov_pkg_wt10_without_lags_rs50.joblib"

# 2. Check/Load or Train/Save
if pkg_path.exists():
    print(f"Loading {pkg_path.name} from disk... (Skipping data prep and training)")
    newprov_pkg_wt10_without_lags_rs50 = load(pkg_path)
    best_newprov_model_wt10_without_lags_rs50 = newprov_pkg_wt10_without_lags_rs50.search.best_estimator_

else:
    print(f"File not found. Preparing data and training...")

    # --- PREPARE DATA ---
    newprov_pkg_wt10_without_lags_rs50 = prepare_xgb_randomizedsearch(
        eda_hcpcs_df,
        approach="new_providers",
        target_col="avg_mdcr_stdzd_amt",
        w_tail=10,
        include_lags=False,
        test_size=0.2,
        random_state=0,
        n_splits=3,
        n_iter=50
    )

    # --- TRAIN ---
    if newprov_pkg_wt10_without_lags_rs50.groups_train is None:
        newprov_pkg_wt10_without_lags_rs50.search.fit(
            newprov_pkg_wt10_without_lags_rs50.X_train,
            newprov_pkg_wt10_without_lags_rs50.y_train,
            model__sample_weight=newprov_pkg_wt10_without_lags_rs50.sample_w_train,
        )
    else:
        newprov_pkg_wt10_without_lags_rs50.search.fit(
            newprov_pkg_wt10_without_lags_rs50.X_train,
            newprov_pkg_wt10_without_lags_rs50.y_train,
            groups=newprov_pkg_wt10_without_lags_rs50.groups_train,
            model__sample_weight=newprov_pkg_wt10_without_lags_rs50.sample_w_train,
        )
        
    best_newprov_model_wt10_without_lags_rs50 = newprov_pkg_wt10_without_lags_rs50.search.best_estimator_

    # --- SAVE ---
    dump(newprov_pkg_wt10_without_lags_rs50, pkg_path)
    print(f"Training complete. Package and model saved to {pkg_path}")

In [ ]:
# Run diagnostics

# get train and test predictions and clip test predictions 
pred_test = best_newprov_model_wt10_without_lags_rs50.predict(newprov_pkg_wt10_without_lags_rs50.X_test)
pred_train = best_newprov_model_wt10_without_lags_rs50.predict(newprov_pkg_wt10_without_lags_rs50.X_train)

# get weighted-train unweighted/average-case metrics
r2_train = best_newprov_model_wt10_without_lags_rs50.score(newprov_pkg_wt10_without_lags_rs50.X_train, newprov_pkg_wt10_without_lags_rs50.y_train)
r2_test = best_newprov_model_wt10_without_lags_rs50.score(newprov_pkg_wt10_without_lags_rs50.X_test, newprov_pkg_wt10_without_lags_rs50.y_test)
mae = mean_absolute_error(newprov_pkg_wt10_without_lags_rs50.y_test, pred_test)
rmse = root_mean_squared_error(newprov_pkg_wt10_without_lags_rs50.y_test, pred_test)

# get weighted-train unweighted/average-case metrics in log space
log_pred_train = best_newprov_model_wt10_without_lags_rs50.regressor_.predict(newprov_pkg_wt10_without_lags_rs50.X_train)  # predictions in transformed target space
log_pred_test = best_newprov_model_wt10_without_lags_rs50.regressor_.predict(newprov_pkg_wt10_without_lags_rs50.X_test)  # predictions in transformed target space

r2_log_true_train = r2_score(np.log1p(newprov_pkg_wt10_without_lags_rs50.y_train), log_pred_train)
r2_log_true_test = r2_score(np.log1p(newprov_pkg_wt10_without_lags_rs50.y_test), log_pred_test)
mae_log_true = mean_absolute_error(np.log1p(newprov_pkg_wt10_without_lags_rs50.y_test), log_pred_test)
rmse_log_true = root_mean_squared_error(np.log1p(newprov_pkg_wt10_without_lags_rs50.y_test), log_pred_test)

# get weighted-train weighted/eval metrics

r2_train_w = r2_score(newprov_pkg_wt10_without_lags_rs50.y_train, pred_train, sample_weight=newprov_pkg_wt10_without_lags_rs50.sample_w_train)
r2_test_w = r2_score(newprov_pkg_wt10_without_lags_rs50.y_test, pred_test, sample_weight=newprov_pkg_wt10_without_lags_rs50.sample_w_test)
mae_w = mean_absolute_error(newprov_pkg_wt10_without_lags_rs50.y_test, pred_test, sample_weight=newprov_pkg_wt10_without_lags_rs50.sample_w_test)
rmse_w = root_mean_squared_error(newprov_pkg_wt10_without_lags_rs50.y_test, pred_test, sample_weight=newprov_pkg_wt10_without_lags_rs50.sample_w_test)

# Calculate tail evals
target_col = "avg_mdcr_stdzd_amt"
test_eval = newprov_pkg_wt10_without_lags_rs50.test_df.copy()

test_eval["pred"] = pred_test
test_eval["abs_err"] = (test_eval[target_col] - test_eval["pred"]).abs()
test_eval["sq_err"]  = (test_eval[target_col] - test_eval["pred"])**2

non_tail_pred_to_y_ratio = test_eval.loc[~test_eval["is_top_1pct_avg_mdcr_stdzd_amt"],"pred"].mean() / test_eval.loc[~test_eval["is_top_1pct_avg_mdcr_stdzd_amt"],target_col].mean()
tail_pred_to_y_ratio = test_eval.loc[test_eval["is_top_1pct_avg_mdcr_stdzd_amt"],"pred"].mean() / test_eval.loc[test_eval["is_top_1pct_avg_mdcr_stdzd_amt"],target_col].mean()

non_tail_sse = test_eval.loc[~test_eval["is_top_1pct_avg_mdcr_stdzd_amt"],"sq_err"].sum()
tail_sse = test_eval.loc[test_eval["is_top_1pct_avg_mdcr_stdzd_amt"],"sq_err"].sum()

non_tail_share_of_total_sse = non_tail_sse / (non_tail_sse + tail_sse)
tail_share_of_total_sse = tail_sse / (non_tail_sse + tail_sse)

non_tail_under_prediction_rate = ((test_eval.loc[~test_eval["is_top_1pct_avg_mdcr_stdzd_amt"],"pred"]) < test_eval.loc[~test_eval["is_top_1pct_avg_mdcr_stdzd_amt"],target_col]).mean()
tail_under_prediction_rate = ((test_eval.loc[test_eval["is_top_1pct_avg_mdcr_stdzd_amt"],"pred"]) < test_eval.loc[test_eval["is_top_1pct_avg_mdcr_stdzd_amt"],target_col]).mean()

non_tail_bias = test_eval.loc[~test_eval["is_top_1pct_avg_mdcr_stdzd_amt"],"pred"].mean() - test_eval.loc[~test_eval["is_top_1pct_avg_mdcr_stdzd_amt"],target_col].mean()
tail_bias = test_eval.loc[test_eval["is_top_1pct_avg_mdcr_stdzd_amt"],"pred"].mean() - test_eval.loc[test_eval["is_top_1pct_avg_mdcr_stdzd_amt"],target_col].mean()

newprov_weight10_no_lag_train_rs50_performance_metrics_test_eval = pd.DataFrame({
    "train_sample_weights":[10],
    "r2_train": r2_train,
    "r2_test": r2_test,
    "mae": mae,
    "rmse": rmse,
    "r2_train_log": r2_log_true_train,
    "r2_test_log": r2_log_true_test,
    "mae_log": mae_log_true,
    "rmse_log": rmse_log_true,
    "r2_train_w": r2_train_w,
    "r2_test_w": r2_test_w,
    "mae_w": mae_w,
    "rmse_w": rmse_w,
    "non_tail_pred_to_y_ratio": non_tail_pred_to_y_ratio,
    "tail_pred_to_y_ratio": tail_pred_to_y_ratio,
    "non_tail_sse": non_tail_sse,
    "tail_sse": tail_sse,
    "non_tail_share_of_total_sse": non_tail_share_of_total_sse,
    "tail_share_of_total_sse": tail_share_of_total_sse,
    "non_tail_under_prediction_rate": non_tail_under_prediction_rate,
    "tail_under_prediction_rate": tail_under_prediction_rate,
    "non_tail_bias": non_tail_bias,
    "tail_bias": tail_bias
})

newprov_weight10_no_lag_train_rs50_performance_metrics_test_eval

In [ ]:
with pd.option_context('display.max_columns', None):
    display(newprov_weight10_no_lag_train_rs50_performance_metrics_test_eval) # or print(df)

## RandomizedSearchCV Follow-up for New-Provider Modeling

As a final check, I ran two additional `RandomizedSearchCV` models for the weighted new-provider setting (`w_tail=10`) to test whether a broader hyperparameter search could improve on the earlier grouped `GridSearchCV` results. I evaluated:

* A weighted new-provider model **with lag features** (`include_lags=True`)
* A weighted new-provider model **without lag features** (`include_lags=False`)



The goal was to see whether randomized search could outperform the earlier grid-search baselines under the same provider-disjoint evaluation framework.

---

### 1. Weighted new-provider model with lag features: randomized search did not improve performance
The randomized-search model with lag features underperformed the earlier weighted grid-search model on essentially every important metric.

Compared with the earlier `GridSearchCV` version, the randomized-search version showed:
* **Lower overall fit**, with test R² dropping from **0.8048** to **0.7260**
* **Worse absolute error**, with MAE increasing from **17.69** to **21.17**
* **Worse large-error behavior**, with RMSE increasing from **92.15** to **109.18**
* **Weaker log-scale fit**, with test log-R² dropping from **0.9523** to **0.9142**
* **Worse tail calibration**, with `tail_pred_to_y_ratio` falling from **0.8687** to **0.7883**
* **More severe underprediction of the tail**, with tail bias worsening from **-222.1** to **-358.1**

**Interpretation:** This indicates that the random-search model moved into a clearly worse region of hyperparameter space for this grouped generalization problem. In practical terms, the earlier weighted grid-search model with lags remained decisively better.

---

### 2. Weighted new-provider model without lag features: randomized search also underperformed
The same pattern appeared in the no-lag weighted setting.

Compared with the earlier `GridSearchCV` model without lag features, the randomized-search version showed:
* **Lower overall fit**, with test R² dropping from **0.8869** to **0.7893**
* **Higher MAE**, increasing from **13.87** to **19.29**
* **Higher RMSE**, increasing from **70.14** to **95.74**
* **Worse log-scale fit**, with test log-R² dropping from **0.9775** to **0.9583**
* **Worse tail calibration**, with `tail_pred_to_y_ratio` falling from **0.9228** to **0.8646**
* **More negative tail bias**, worsening from **-130.6** to **-229.1**

**Interpretation:** This is especially important because the no-lag weighted grid-search model was already the strongest corrected new-provider model. The randomized search failed to improve it and instead produced a meaningfully worse result.

---

### 3. Interpretation: the original grouped grid-search models were already strong
These follow-up experiments suggest that the earlier grouped `GridSearchCV` models were already sitting in a good part of the hyperparameter space.

A few conclusions stand out:
* The original grid-search ranges were likely already well targeted for this problem.
* A broader random search over 50 draws did **not** uncover a better region.
* In this grouped provider-holdout setting, performance appears fairly sensitive to hyperparameter choice, and random search likely sampled many combinations that were too weak or poorly calibrated for the tail.
* Tail-weighted training appears to require a more constrained and well-targeted tuning strategy than a relatively broad randomized search.

In other words, the randomized-search follow-up was useful not because it found a better model, but because it confirmed that the earlier weighted grid-search models were already stronger.

---

### 4. Final takeaway from the randomized-search follow-up
The `RandomizedSearchCV` follow-up did **not** improve on the earlier grouped `GridSearchCV` baselines for the weighted new-provider task.

Across both lag and no-lag specifications, randomized search produced:
* Lower test R²
* Higher MAE and RMSE
* Weaker log-scale fit
* Worse tail calibration
* More severe tail underprediction

As a result, the preferred weighted new-provider models remain the original grouped grid-search versions.

---

### 5. Preferred new-provider models after all grouped experiments
At this stage, the strongest grouped new-provider models are:

* **Best overall corrected model:** `newprov_pkg_wt10_without_lags`
* **Best weighted model with lag features:** `newprov_pkg_wt10_with_lags`

Among all corrected grouped new-provider models tested so far, the best performance still comes from the **weighted no-lag grid-search model**, which achieved:

* **Test R²:** 0.8869
* **MAE:** 13.87
* **RMSE:** 70.14
* **Test log-R²:** 0.9775
* **Tail prediction-to-true ratio:** 0.9228
* **Tail bias:** -130.6

This remains the strongest benchmark as I move forward to the next stage of new-provider modeling.

## Delta cost modeling: 

In [ ]:
# Keep full data intact
eda_hcpcs_full_df = eda_hcpcs_df.copy()

# Lag-present mask
mask_lag = (
    eda_hcpcs_full_df["lag1_avg_amt"].notna()
    & (eda_hcpcs_full_df["lag1_avg_amt"] >= 0)
)

# Subset only for residual-model training
eda_hcpcs_lags_present_df = eda_hcpcs_full_df.loc[mask_lag].copy()

# Residual target (log space delta)
eda_hcpcs_lags_present_df["log_delta_cost"] = (
    np.log1p(eda_hcpcs_lags_present_df["avg_mdcr_stdzd_amt"])
    - np.log1p(eda_hcpcs_lags_present_df["lag1_avg_amt"])
)

In [ ]:
from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error
import numpy as np
import pandas as pd


def evaluate_delta_target_model_same_schema(
    pkg,
    best_model,
    *,
    train_sample_weights_label: int,
    level_target_col: str = "avg_mdcr_stdzd_amt",
    delta_target_col: str = "log_delta_cost",
    lag_col: str = "lag1_avg_amt",
    tail_flag_col: str = "is_top_1pct_avg_mdcr_stdzd_amt",
):
    """
    Evaluate a delta-target model (predicting log_delta_cost) using the SAME output
    schema as the earlier level-model tables.

    Column meanings:
    - r2_train / r2_test / mae / rmse / weighted versions:
        computed in reconstructed LEVEL space
    - r2_train_log / r2_test_log / mae_log / rmse_log:
        computed in DELTA space (because target is already log_delta_cost)
    - tail metrics:
        computed in reconstructed LEVEL space
    """

    # ------------------------------------------------------------
    # 1) Delta-space predictions
    # ------------------------------------------------------------
    pred_test_delta = best_model.predict(pkg.X_test)
    pred_train_delta = best_model.predict(pkg.X_train)

    y_test_delta = pkg.y_test.astype(float).to_numpy()
    y_train_delta = pkg.y_train.astype(float).to_numpy()

    # ------------------------------------------------------------
    # 2) Reconstruct LEVEL predictions from lag + predicted delta
    #    pred_level = expm1(log1p(lag) + predicted_log_delta)
    # ------------------------------------------------------------
    lag_train = pkg.train_df[lag_col].astype(float).to_numpy()
    lag_test = pkg.test_df[lag_col].astype(float).to_numpy()

    pred_train = np.expm1(np.log1p(lag_train) + pred_train_delta)
    pred_test = np.expm1(np.log1p(lag_test) + pred_test_delta)

    # True values in LEVEL space
    y_train = pkg.train_df[level_target_col].astype(float).to_numpy()
    y_test = pkg.test_df[level_target_col].astype(float).to_numpy()

    # ------------------------------------------------------------
    # 3) Level-space overall metrics
    # ------------------------------------------------------------
    r2_train = r2_score(y_train, pred_train)
    r2_test = r2_score(y_test, pred_test)
    mae = mean_absolute_error(y_test, pred_test)
    rmse = root_mean_squared_error(y_test, pred_test)

    # ------------------------------------------------------------
    # 4) Delta-space metrics
    #    Keep old column names for schema compatibility
    # ------------------------------------------------------------
    r2_train_log = r2_score(y_train_delta, pred_train_delta)
    r2_test_log = r2_score(y_test_delta, pred_test_delta)
    mae_log_true = mean_absolute_error(y_test_delta, pred_test_delta)
    rmse_log_true = root_mean_squared_error(y_test_delta, pred_test_delta)

    # ------------------------------------------------------------
    # 5) Weighted level-space metrics
    # ------------------------------------------------------------
    r2_train_w = r2_score(y_train, pred_train, sample_weight=pkg.sample_w_train)
    r2_test_w = r2_score(y_test, pred_test, sample_weight=pkg.sample_w_test)
    mae_w = mean_absolute_error(y_test, pred_test, sample_weight=pkg.sample_w_test)
    rmse_w = root_mean_squared_error(y_test, pred_test, sample_weight=pkg.sample_w_test)

    # ------------------------------------------------------------
    # 6) Tail evaluation in LEVEL space
    # ------------------------------------------------------------
    test_eval = pkg.test_df.copy()
    test_eval["pred"] = pred_test
    test_eval["abs_err"] = (test_eval[level_target_col] - test_eval["pred"]).abs()
    test_eval["sq_err"] = (test_eval[level_target_col] - test_eval["pred"]) ** 2

    non_tail_pred_to_y_ratio = (
        test_eval.loc[~test_eval[tail_flag_col], "pred"].mean()
        / test_eval.loc[~test_eval[tail_flag_col], level_target_col].mean()
    )
    tail_pred_to_y_ratio = (
        test_eval.loc[test_eval[tail_flag_col], "pred"].mean()
        / test_eval.loc[test_eval[tail_flag_col], level_target_col].mean()
    )

    non_tail_sse = test_eval.loc[~test_eval[tail_flag_col], "sq_err"].sum()
    tail_sse = test_eval.loc[test_eval[tail_flag_col], "sq_err"].sum()

    non_tail_share_of_total_sse = non_tail_sse / (non_tail_sse + tail_sse)
    tail_share_of_total_sse = tail_sse / (non_tail_sse + tail_sse)

    non_tail_under_prediction_rate = (
        (
            test_eval.loc[~test_eval[tail_flag_col], "pred"]
            < test_eval.loc[~test_eval[tail_flag_col], level_target_col]
        ).mean()
    )
    tail_under_prediction_rate = (
        (
            test_eval.loc[test_eval[tail_flag_col], "pred"]
            < test_eval.loc[test_eval[tail_flag_col], level_target_col]
        ).mean()
    )

    non_tail_bias = (
        test_eval.loc[~test_eval[tail_flag_col], "pred"].mean()
        - test_eval.loc[~test_eval[tail_flag_col], level_target_col].mean()
    )
    tail_bias = (
        test_eval.loc[test_eval[tail_flag_col], "pred"].mean()
        - test_eval.loc[test_eval[tail_flag_col], level_target_col].mean()
    )

    # ------------------------------------------------------------
    # 7) Match previous schema exactly
    # ------------------------------------------------------------
    out = pd.DataFrame({
        "train_sample_weights": [train_sample_weights_label],
        "r2_train": [r2_train],
        "r2_test": [r2_test],
        "mae": [mae],
        "rmse": [rmse],
        "r2_train_log": [r2_train_log],
        "r2_test_log": [r2_test_log],
        "mae_log": [mae_log_true],
        "rmse_log": [rmse_log_true],
        "r2_train_w": [r2_train_w],
        "r2_test_w": [r2_test_w],
        "mae_w": [mae_w],
        "rmse_w": [rmse_w],
        "non_tail_pred_to_y_ratio": [non_tail_pred_to_y_ratio],
        "tail_pred_to_y_ratio": [tail_pred_to_y_ratio],
        "non_tail_sse": [non_tail_sse],
        "tail_sse": [tail_sse],
        "non_tail_share_of_total_sse": [non_tail_share_of_total_sse],
        "tail_share_of_total_sse": [tail_share_of_total_sse],
        "non_tail_under_prediction_rate": [non_tail_under_prediction_rate],
        "tail_under_prediction_rate": [tail_under_prediction_rate],
        "non_tail_bias": [non_tail_bias],
        "tail_bias": [tail_bias],
    })

    return out

### 2.G. `GridSearchCV` + `target_col="log_delta_cost"` + `w_tail=10` + `include_lags=True` + `use_ttr=False`

In [ ]:
from pathlib import Path
from joblib import dump, load

# 1. Setup Paths
ROOT = Path.cwd().resolve()
MODELS = (ROOT / "models").resolve()
MODELS.mkdir(parents=True, exist_ok=True)

# We now point to a package file instead of just the model file
pkg_path = MODELS / "newprov_pkg_wt10_with_lags_delta_trgt.joblib"

# 2. Check/Load or Train/Save
if pkg_path.exists():
    print(f"Loading {pkg_path.name} from disk... (Skipping data prep and training)")
    # Load the entire package (data + fitted search object)
    newprov_pkg_wt10_with_lags_delta_trgt = load(pkg_path)
    
    # Extract the model so downstream code works as expected
    best_newprov_model_wt10_with_lags_delta_trgt = newprov_pkg_wt10_with_lags_delta_trgt.search.best_estimator_

else:
    print(f"File not found. Preparing data and training...")

    # --- PREPARE DATA ---
    newprov_pkg_wt10_with_lags_delta_trgt = prepare_xgb_gridsearch(
        eda_hcpcs_lags_present_df,
        approach="new_providers",
        target_col="log_delta_cost",
        w_tail=10,
        include_lags=True,
        use_ttr=False,
        test_size=0.2,
        random_state=0,
        n_splits=3,
    )

    # --- TRAIN ---
    if newprov_pkg_wt10_with_lags_delta_trgt.groups_train is None:
        newprov_pkg_wt10_with_lags_delta_trgt.search.fit(
            newprov_pkg_wt10_with_lags_delta_trgt.X_train,
            newprov_pkg_wt10_with_lags_delta_trgt.y_train,
            model__sample_weight=newprov_pkg_wt10_with_lags_delta_trgt.sample_w_train,
        )
    else:
        newprov_pkg_wt10_with_lags_delta_trgt.search.fit(
            newprov_pkg_wt10_with_lags_delta_trgt.X_train,
            newprov_pkg_wt10_with_lags_delta_trgt.y_train,
            groups=newprov_pkg_wt10_with_lags_delta_trgt.groups_train,
            model__sample_weight=newprov_pkg_wt10_with_lags_delta_trgt.sample_w_train,
        )
        
    best_newprov_model_wt10_with_lags_delta_trgt = newprov_pkg_wt10_with_lags_delta_trgt.search.best_estimator_

    # --- SAVE ---
    # 3. Save the ENTIRE package
    dump(newprov_pkg_wt10_with_lags_delta_trgt, pkg_path)
    print(f"Training complete. Package and model saved to {pkg_path}")

In [ ]:
newprov_weight10_lag_delta_trgt_performance_metrics_test_eval = (
    evaluate_delta_target_model_same_schema(
        pkg=newprov_pkg_wt10_with_lags_delta_trgt,
        best_model=best_newprov_model_wt10_with_lags_delta_trgt,
        train_sample_weights_label=10,
        level_target_col="avg_mdcr_stdzd_amt",
        delta_target_col="log_delta_cost",
        lag_col="lag1_avg_amt",
        tail_flag_col="is_top_1pct_avg_mdcr_stdzd_amt",
    )
)

newprov_weight10_lag_delta_trgt_performance_metrics_test_eval

In [ ]:
with pd.option_context('display.max_columns', None):
    display(newprov_weight10_lag_delta_trgt_performance_metrics_test_eval) # or print(df)

### 2.H. `RandomizedSearchCV` + `target_col="log_delta_cost"` + `w_tail=10` + `include_lags=True` _ `use_ttr=False`

In [ ]:
from pathlib import Path
from joblib import dump, load

# 1. Setup Paths
ROOT = Path.cwd().resolve()
MODELS = (ROOT / "models").resolve()
MODELS.mkdir(parents=True, exist_ok=True)

pkg_path = MODELS / "newprov_pkg_wt10_with_lags_delta_trgt_rs50.joblib"

# 2. Check/Load or Train/Save
if pkg_path.exists():
    print(f"Loading {pkg_path.name} from disk... (Skipping data prep and training)")
    newprov_pkg_wt10_with_lags_delta_trgt_rs50 = load(pkg_path)
    best_newprov_model_wt10_with_lags_delta_trgt_rs50 = newprov_pkg_wt10_with_lags_delta_trgt_rs50.search.best_estimator_

else:
    print(f"File not found. Preparing data and training...")

    # --- PREPARE DATA ---
    newprov_pkg_wt10_with_lags_delta_trgt_rs50 = prepare_xgb_randomizedsearch(
        eda_hcpcs_lags_present_df,
        approach="new_providers",
        target_col="log_delta_cost",
        w_tail=10,
        include_lags=True,
        test_size=0.2,
        random_state=0,
        n_splits=3,
        n_iter=50,
        use_ttr=False
    )

    # --- TRAIN ---
    if newprov_pkg_wt10_with_lags_delta_trgt_rs50.groups_train is None:
        newprov_pkg_wt10_with_lags_delta_trgt_rs50.search.fit(
            newprov_pkg_wt10_with_lags_delta_trgt_rs50.X_train,
            newprov_pkg_wt10_with_lags_delta_trgt_rs50.y_train,
            model__sample_weight=newprov_pkg_wt10_with_lags_delta_trgt_rs50.sample_w_train,
        )
    else:
        newprov_pkg_wt10_with_lags_delta_trgt_rs50.search.fit(
            newprov_pkg_wt10_with_lags_delta_trgt_rs50.X_train,
            newprov_pkg_wt10_with_lags_delta_trgt_rs50.y_train,
            groups=newprov_pkg_wt10_with_lags_delta_trgt_rs50.groups_train,
            model__sample_weight=newprov_pkg_wt10_with_lags_delta_trgt_rs50.sample_w_train,
        )
        
    best_newprov_model_wt10_with_lags_delta_trgt_rs50 = newprov_pkg_wt10_with_lags_delta_trgt_rs50.search.best_estimator_

    # --- SAVE ---
    dump(newprov_pkg_wt10_with_lags_delta_trgt_rs50, pkg_path)
    print(f"Training complete. Package and model saved to {pkg_path}")

In [ ]:
newprov_weight10_lag_delta_trgt_rs50_performance_metrics_test_eval = (
    evaluate_delta_target_model_same_schema(
        pkg=newprov_pkg_wt10_with_lags_delta_trgt_rs50,
        best_model=best_newprov_model_wt10_with_lags_delta_trgt_rs50,
        train_sample_weights_label=10,
        level_target_col="avg_mdcr_stdzd_amt",
        delta_target_col="log_delta_cost",
        lag_col="lag1_avg_amt",
        tail_flag_col="is_top_1pct_avg_mdcr_stdzd_amt",
    )
)

newprov_weight10_lag_delta_trgt_rs50_performance_metrics_test_eval

In [ ]:
with pd.option_context('display.max_columns', None):
    display(newprov_weight10_lag_delta_trgt_rs50_performance_metrics_test_eval) # or print(df)

### 2.I. `RandomizedSearchCV` + `target_col="log_delta_cost"` + `w_tail=10` + `include_lags=False` + `use_ttr=False`

In [ ]:
from pathlib import Path
from joblib import dump, load

# 1. Setup Paths
ROOT = Path.cwd().resolve()
MODELS = (ROOT / "models").resolve()
MODELS.mkdir(parents=True, exist_ok=True)

pkg_path = MODELS / "newprov_pkg_wt10_without_lags_delta_trgt_rs50.joblib"

# 2. Check/Load or Train/Save
if pkg_path.exists():
    print(f"Loading {pkg_path.name} from disk... (Skipping data prep and training)")
    newprov_pkg_wt10_without_lags_delta_trgt_rs50 = load(pkg_path)
    best_newprov_model_wt10_without_lags_delta_trgt_rs50 = newprov_pkg_wt10_without_lags_delta_trgt_rs50.search.best_estimator_

else:
    print(f"File not found. Preparing data and training...")

    # --- PREPARE DATA ---
    newprov_pkg_wt10_without_lags_delta_trgt_rs50 = prepare_xgb_randomizedsearch(
        eda_hcpcs_lags_present_df,
        approach="new_providers",
        target_col="log_delta_cost",
        w_tail=10,
        include_lags=False,
        test_size=0.2,
        random_state=0,
        n_splits=3,
        n_iter=50,
        use_ttr=False
    )

    # --- TRAIN ---
    if newprov_pkg_wt10_without_lags_delta_trgt_rs50.groups_train is None:
        newprov_pkg_wt10_without_lags_delta_trgt_rs50.search.fit(
            newprov_pkg_wt10_without_lags_delta_trgt_rs50.X_train,
            newprov_pkg_wt10_without_lags_delta_trgt_rs50.y_train,
            model__sample_weight=newprov_pkg_wt10_without_lags_delta_trgt_rs50.sample_w_train,
        )
    else:
        newprov_pkg_wt10_without_lags_delta_trgt_rs50.search.fit(
            newprov_pkg_wt10_without_lags_delta_trgt_rs50.X_train,
            newprov_pkg_wt10_without_lags_delta_trgt_rs50.y_train,
            groups=newprov_pkg_wt10_without_lags_delta_trgt_rs50.groups_train,
            model__sample_weight=newprov_pkg_wt10_without_lags_delta_trgt_rs50.sample_w_train,
        )
        
    best_newprov_model_wt10_without_lags_delta_trgt_rs50 = newprov_pkg_wt10_without_lags_delta_trgt_rs50.search.best_estimator_

    # --- SAVE ---
    dump(newprov_pkg_wt10_without_lags_delta_trgt_rs50, pkg_path)
    print(f"Training complete. Package and model saved to {pkg_path}")

In [ ]:
newprov_weight10_no_lag_delta_trgt_rs50_performance_metrics_test_eval = (
    evaluate_delta_target_model_same_schema(
        pkg=newprov_pkg_wt10_without_lags_delta_trgt_rs50,
        best_model=best_newprov_model_wt10_without_lags_delta_trgt_rs50,
        train_sample_weights_label=10,
        level_target_col="avg_mdcr_stdzd_amt",
        delta_target_col="log_delta_cost",
        lag_col="lag1_avg_amt",
        tail_flag_col="is_top_1pct_avg_mdcr_stdzd_amt",
    )
)

newprov_weight10_no_lag_delta_trgt_rs50_performance_metrics_test_eval

In [ ]:
with pd.option_context('display.max_columns', None):
    display(newprov_weight10_no_lag_delta_trgt_rs50_performance_metrics_test_eval) # or print(df)

## New-Provider Delta-Target Modeling Summary

I next evaluated whether reframing the new-provider problem as **delta-cost prediction** would improve performance relative to directly modeling raw average standardized Medicare amount per service.

To do this, I created a lag-present subset and defined the residual target as:
`log_delta_cost = log1p(avg_mdcr_stdzd_amt) - log1p(lag1_avg_amt)`

This formulation asks the model to learn the **change relative to prior cost**, rather than the raw cost level itself. After prediction, I reconstructed level-space predictions by combining the predicted delta with `lag1_avg_amt`.



---

### Delta-Target Dataset Construction
I first preserved the full HCPCS-level dataset and then created a lag-present subset for delta modeling:
* `eda_hcpcs_full_df` retained the full analytic dataset.
* `eda_hcpcs_lags_present_df` was restricted to rows where `lag1_avg_amt` was non-missing and non-negative.
* `log_delta_cost` was then added only to this lag-present subset.

This was the correct setup for delta modeling because the target itself depends on lag being present.

### QC Confirmation
The package QC checks confirmed the models were built correctly:
* **2G and 2H** were true *with-lag* delta-target models:
  * `target_col = "log_delta_cost"`
  * Lag columns were present in both `train_df` and `X_train`.
* **2I** was a true *without-lag* delta-target model:
  * `target_col = "log_delta_cost"`
  * Lag columns remained available in `train_df` for level reconstruction.
  * But lag columns were **absent from** `X_train`, so the model itself could not use lag as a predictor.
* All three packages had finite `y_train` and `y_test`.

That means the delta-target comparisons below are valid and interpretable.

---

### 2G. GridSearchCV, `target_col="log_delta_cost"`, `w_tail=10`, `include_lags=True`, `use_ttr=False`
*This was the strongest of the delta-target new-provider models.*

**Level-space performance**
* **R² test:** 0.9334
* **MAE:** 3.49
* **RMSE:** 43.01

> **Note:** These are excellent results. The model explained roughly **93.3% of the variance** in reconstructed level-space cost on unseen providers and achieved the lowest MAE and RMSE among the three delta-target models.

**Delta-space performance**
* **R² test (log space):** 0.5659
* **MAE (log space):** 0.0405
* **RMSE (log space):** 0.1065

These values show that the model was only moderately strong in delta space itself, but that was still enough to produce very strong level-space predictions after combining the predicted delta with prior cost. This reinforces an important modeling lesson: **delta-space fit does not need to be perfect for level-space reconstruction to be excellent, as long as lag already carries most of the signal.**

**Calibration and tail behavior**
* **Non-tail prediction-to-true ratio:** 1.0003
* **Tail prediction-to-true ratio:** 0.9959
* **Non-tail bias:** 0.02
* **Tail bias:** -5.89

This is extremely strong calibration. Non-tail predictions were essentially unbiased, and tail predictions were only slightly underpredicted on average.

**Tail error distribution**
* **Non-tail share of total SSE:** 0.8126
* **Tail share of total SSE:** 0.1874

Most SSE came from the non-tail group, which is encouraging because it suggests the model substantially reduced the usual concentration of catastrophic tail error.

**Interpretation:** This model appears to be the best-performing new-provider model so far. It combines strong global fit, low average error, strong calibration, and relatively mild tail underprediction.

---

### 2H. RandomizedSearchCV, `target_col="log_delta_cost"`, `w_tail=10`, `include_lags=True`, `use_ttr=False`
*This model was also strong, but it did not outperform 2G.*

**Level-space performance**
* **R² test:** 0.9303
* **MAE:** 3.94
* **RMSE:** 44.01
*(Very good results, but slightly worse than 2G across the main metrics.)*

**Delta-space performance**
* **R² test (log space):** 0.5145
* **MAE (log space):** 0.0466
* **RMSE (log space):** 0.1127
*(Weaker than 2G in delta space, aligning with the slightly weaker reconstructed level-space performance.)*

**Calibration and tail behavior**
* **Non-tail prediction-to-true ratio:** 1.0010
* **Tail prediction-to-true ratio:** 0.9929
* **Non-tail bias:** 0.07
* **Tail bias:** -10.15
*(Calibration remained good overall, but tail underprediction was a bit more pronounced than in 2G.)*

**Tail error distribution**
* **Non-tail share of total SSE:** 0.8030
* **Tail share of total SSE:** 0.1970
*(Favorable, but again slightly worse than 2G.)*

**Interpretation:** Randomized search did not find a better solution than the grid-searched with-lag delta model. It produced a strong model, but one that was clearly a small downgrade from 2G.

---

### 2I. RandomizedSearchCV, `target_col="log_delta_cost"`, `w_tail=10`, `include_lags=False`, `use_ttr=False`
*This model removed lag features from `X_train`, while still evaluating on the same lag-present delta-target dataset.*

**Level-space performance**
* **R² test:** 0.9041
* **MAE:** 4.63
* **RMSE:** 51.62
*(Performance was still respectable, but clearly worse than the with-lag delta models.)*

**Delta-space performance**
* **R² test (log space):** 0.2333
* **MAE (log space):** 0.0565
* **RMSE (log space):** 0.1416
*(This is the clearest sign that removing lag features made the delta problem much harder. The model was substantially weaker at learning `log_delta_cost` when it could not directly use lag variables as predictors.)*

**Calibration and tail behavior**
* **Non-tail prediction-to-true ratio:** 1.0048
* **Tail prediction-to-true ratio:** 0.9996
* **Non-tail bias:** 0.35
* **Tail bias:** -0.63
*(Interestingly, despite weaker overall fit, this model remained quite well calibrated on average. That suggests its weakness comes less from systematic directional bias and more from increased dispersion and loss of row-level precision.)*

**Tail error distribution**
* **Non-tail share of total SSE:** 0.8641
* **Tail share of total SSE:** 0.1359
*(Indicates more of the remaining error shifted into the non-tail group, consistent with a model that is broadly reasonable but less precise without direct lag information.)*

**Interpretation:** This model shows that delta-target modeling still helps even when lag predictors are removed, but performance drops meaningfully. The lag variables clearly add major predictive value in the new-provider setting.

---

### Overall Interpretation
These three runs strongly support the conclusion that **delta-target modeling is superior to direct level-target modeling** for new-provider prediction when lag information is available.



**Ranking of Delta-Target Models:**
1. **2G.** GridSearchCV + delta target + with lags
2. **2H.** RandomizedSearchCV + delta target + with lags
3. **2I.** RandomizedSearchCV + delta target + without lags

**Main Lessons:**
* Predicting change relative to prior cost is easier than predicting raw cost directly.
* Lag features materially improve delta-target performance.
* Grid search outperformed randomized search in this case.
* **2G** is the strongest new-provider model explored so far.

---

### Practical Takeaway
At this stage, the best-performing new-provider specification is:
* `target_col="log_delta_cost"`
* `w_tail=10`
* `include_lags=True`
* `use_ttr=False`
* **`GridSearchCV`**

This model achieved the strongest combination of highest test R², lowest MAE, lowest RMSE, near-zero non-tail bias, and only mild tail underprediction. That makes it the leading candidate for the new-provider modeling task.

# Establish baselines for new provider modeling

Below are 4 standalone code chunks, one for each baseline, written to fit your new provider setup.

These will assume:
- `eda_hcpcs_df` is your full modeling dataframe
- `avg_mdcr_stdzd_amt` is the level target
- `lag1_avg_amt` exists in the dataframe
- we want to evaluate on the same group-holdout split logic used for new-provider modeling
- `Rndrng_NPI` is the provider group column


### Baseline A: Lag only

> Compare Baseline A to new-provider models with lag features

In [ ]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# Baseline A: Lag only
# New provider setting, with lag available
# ------------------------------------------------------------

target_col = "avg_mdcr_stdzd_amt"
lag_col = "lag1_avg_amt"
tail_flag_col = "is_top_1pct_avg_mdcr_stdzd_amt"
group_col = "Rndrng_NPI"

test_size = 0.20
random_state = 0
w_tail = 10

# 1) Build same new-provider split
groups_all = eda_hcpcs_df[group_col].to_numpy()
gss = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=random_state)
train_idx, test_idx = next(gss.split(eda_hcpcs_df, groups=groups_all))

train_df = eda_hcpcs_df.iloc[train_idx].copy()
test_df = eda_hcpcs_df.iloc[test_idx].copy()

# 2) Restrict to rows where lag exists, because this baseline requires lag
mask_test_lag = test_df[lag_col].notna() & (test_df[lag_col] >= 0)
test_eval = test_df.loc[mask_test_lag].copy()

# 3) Predict
pred_test = test_eval[lag_col].astype(float)
y_true = test_eval[target_col].astype(float)

# 4) Weights
is_tail_test = test_eval[tail_flag_col].astype(int).to_numpy()
sample_w_test = (1 + (w_tail - 1) * is_tail_test).astype(float)

# 5) Metrics
r2_test = r2_score(y_true, pred_test)
mae = mean_absolute_error(y_true, pred_test)
rmse = root_mean_squared_error(y_true, pred_test)

pred_clip = pred_test.clip(lower=0)
r2_test_log = r2_score(np.log1p(y_true), np.log1p(pred_clip))
mae_log = mean_absolute_error(np.log1p(y_true), np.log1p(pred_clip))
rmse_log = root_mean_squared_error(np.log1p(y_true), np.log1p(pred_clip))

r2_test_w = r2_score(y_true, pred_test, sample_weight=sample_w_test)
mae_w = mean_absolute_error(y_true, pred_test, sample_weight=sample_w_test)
rmse_w = root_mean_squared_error(y_true, pred_test, sample_weight=sample_w_test)

test_eval["pred"] = pred_test
test_eval["abs_err"] = (test_eval[target_col] - test_eval["pred"]).abs()
test_eval["sq_err"] = (test_eval[target_col] - test_eval["pred"]) ** 2

non_tail_pred_to_y_ratio = (
    test_eval.loc[~test_eval[tail_flag_col], "pred"].mean()
    / test_eval.loc[~test_eval[tail_flag_col], target_col].mean()
)
tail_pred_to_y_ratio = (
    test_eval.loc[test_eval[tail_flag_col], "pred"].mean()
    / test_eval.loc[test_eval[tail_flag_col], target_col].mean()
)

non_tail_sse = test_eval.loc[~test_eval[tail_flag_col], "sq_err"].sum()
tail_sse = test_eval.loc[test_eval[tail_flag_col], "sq_err"].sum()

non_tail_share_of_total_sse = non_tail_sse / (non_tail_sse + tail_sse)
tail_share_of_total_sse = tail_sse / (non_tail_sse + tail_sse)

non_tail_under_prediction_rate = (
    (test_eval.loc[~test_eval[tail_flag_col], "pred"] < test_eval.loc[~test_eval[tail_flag_col], target_col]).mean()
)
tail_under_prediction_rate = (
    (test_eval.loc[test_eval[tail_flag_col], "pred"] < test_eval.loc[test_eval[tail_flag_col], target_col]).mean()
)

non_tail_bias = (
    test_eval.loc[~test_eval[tail_flag_col], "pred"].mean()
    - test_eval.loc[~test_eval[tail_flag_col], target_col].mean()
)
tail_bias = (
    test_eval.loc[test_eval[tail_flag_col], "pred"].mean()
    - test_eval.loc[test_eval[tail_flag_col], target_col].mean()
)

baseline_A_lag_only_metrics = pd.DataFrame({
    "train_sample_weights": [np.nan],
    "r2_train": [np.nan],
    "r2_test": [r2_test],
    "mae": [mae],
    "rmse": [rmse],
    "r2_train_log": [np.nan],
    "r2_test_log": [r2_test_log],
    "mae_log": [mae_log],
    "rmse_log": [rmse_log],
    "r2_train_w": [np.nan],
    "r2_test_w": [r2_test_w],
    "mae_w": [mae_w],
    "rmse_w": [rmse_w],
    "non_tail_pred_to_y_ratio": [non_tail_pred_to_y_ratio],
    "tail_pred_to_y_ratio": [tail_pred_to_y_ratio],
    "non_tail_sse": [non_tail_sse],
    "tail_sse": [tail_sse],
    "non_tail_share_of_total_sse": [non_tail_share_of_total_sse],
    "tail_share_of_total_sse": [tail_share_of_total_sse],
    "non_tail_under_prediction_rate": [non_tail_under_prediction_rate],
    "tail_under_prediction_rate": [tail_under_prediction_rate],
    "non_tail_bias": [non_tail_bias],
    "tail_bias": [tail_bias],
})

baseline_A_lag_only_metrics

In [ ]:
with pd.option_context('display.max_columns', None):
    display(baseline_A_lag_only_metrics) # or print(df)

### Baseline B: Lag plus backoff

> Compare Baseline B to new-provider models with lag features

In [ ]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# Baseline B: Lag + backoff mean
# New provider setting
# ------------------------------------------------------------

target_col = "avg_mdcr_stdzd_amt"
lag_col = "lag1_avg_amt"
tail_flag_col = "is_top_1pct_avg_mdcr_stdzd_amt"
group_col = "Rndrng_NPI"

group_cols = ["provider_type", "HCPCS_Cd", "Place_Of_Srvc"]
test_size = 0.20
random_state = 0
w_tail = 10

# 1) Build same new-provider split
groups_all = eda_hcpcs_df[group_col].to_numpy()
gss = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=random_state)
train_idx, test_idx = next(gss.split(eda_hcpcs_df, groups=groups_all))

train_df = eda_hcpcs_df.iloc[train_idx].copy()
test_df = eda_hcpcs_df.iloc[test_idx].copy()

# 2) Build backoff means from TRAIN only
train_means_grp = (
    train_df.groupby(group_cols)[target_col]
    .mean()
    .rename("mean_train_grp")
    .reset_index()
)

train_means_hcpcs = (
    train_df.groupby(["HCPCS_Cd"])[target_col]
    .mean()
    .rename("mean_train_hcpcs")
    .reset_index()
)

global_mean = float(train_df[target_col].mean())

# 3) Attach means to TEST while preserving original index
test_with_means = test_df.copy()
test_with_means["_orig_idx"] = test_with_means.index

test_with_means = test_with_means.merge(train_means_grp, on=group_cols, how="left")
test_with_means = test_with_means.merge(train_means_hcpcs, on="HCPCS_Cd", how="left")

test_with_means = test_with_means.set_index("_orig_idx", drop=True)
test_with_means = test_with_means.loc[test_df.index]

# 4) Predict: lag if present, else grp mean, else HCPCS mean, else global mean
pred_test = test_with_means[lag_col].copy()
pred_test = pred_test.fillna(test_with_means["mean_train_grp"])
pred_test = pred_test.fillna(test_with_means["mean_train_hcpcs"])
pred_test = pred_test.fillna(global_mean)

y_true = test_with_means[target_col].astype(float)
pred_test = pred_test.astype(float)

# 5) Weights
is_tail_test = test_with_means[tail_flag_col].astype(int).to_numpy()
sample_w_test = (1 + (w_tail - 1) * is_tail_test).astype(float)

# 6) Metrics
r2_test = r2_score(y_true, pred_test)
mae = mean_absolute_error(y_true, pred_test)
rmse = root_mean_squared_error(y_true, pred_test)

pred_clip = pred_test.clip(lower=0)
r2_test_log = r2_score(np.log1p(y_true), np.log1p(pred_clip))
mae_log = mean_absolute_error(np.log1p(y_true), np.log1p(pred_clip))
rmse_log = root_mean_squared_error(np.log1p(y_true), np.log1p(pred_clip))

r2_test_w = r2_score(y_true, pred_test, sample_weight=sample_w_test)
mae_w = mean_absolute_error(y_true, pred_test, sample_weight=sample_w_test)
rmse_w = root_mean_squared_error(y_true, pred_test, sample_weight=sample_w_test)

test_eval = test_with_means.copy()
test_eval["pred"] = pred_test
test_eval["abs_err"] = (test_eval[target_col] - test_eval["pred"]).abs()
test_eval["sq_err"] = (test_eval[target_col] - test_eval["pred"]) ** 2

non_tail_pred_to_y_ratio = (
    test_eval.loc[~test_eval[tail_flag_col], "pred"].mean()
    / test_eval.loc[~test_eval[tail_flag_col], target_col].mean()
)
tail_pred_to_y_ratio = (
    test_eval.loc[test_eval[tail_flag_col], "pred"].mean()
    / test_eval.loc[test_eval[tail_flag_col], target_col].mean()
)

non_tail_sse = test_eval.loc[~test_eval[tail_flag_col], "sq_err"].sum()
tail_sse = test_eval.loc[test_eval[tail_flag_col], "sq_err"].sum()

non_tail_share_of_total_sse = non_tail_sse / (non_tail_sse + tail_sse)
tail_share_of_total_sse = tail_sse / (non_tail_sse + tail_sse)

non_tail_under_prediction_rate = (
    (test_eval.loc[~test_eval[tail_flag_col], "pred"] < test_eval.loc[~test_eval[tail_flag_col], target_col]).mean()
)
tail_under_prediction_rate = (
    (test_eval.loc[test_eval[tail_flag_col], "pred"] < test_eval.loc[test_eval[tail_flag_col], target_col]).mean()
)

non_tail_bias = (
    test_eval.loc[~test_eval[tail_flag_col], "pred"].mean()
    - test_eval.loc[~test_eval[tail_flag_col], target_col].mean()
)
tail_bias = (
    test_eval.loc[test_eval[tail_flag_col], "pred"].mean()
    - test_eval.loc[test_eval[tail_flag_col], target_col].mean()
)

baseline_B_lag_plus_backoff_metrics = pd.DataFrame({
    "train_sample_weights": [np.nan],
    "r2_train": [np.nan],
    "r2_test": [r2_test],
    "mae": [mae],
    "rmse": [rmse],
    "r2_train_log": [np.nan],
    "r2_test_log": [r2_test_log],
    "mae_log": [mae_log],
    "rmse_log": [rmse_log],
    "r2_train_w": [np.nan],
    "r2_test_w": [r2_test_w],
    "mae_w": [mae_w],
    "rmse_w": [rmse_w],
    "non_tail_pred_to_y_ratio": [non_tail_pred_to_y_ratio],
    "tail_pred_to_y_ratio": [tail_pred_to_y_ratio],
    "non_tail_sse": [non_tail_sse],
    "tail_sse": [tail_sse],
    "non_tail_share_of_total_sse": [non_tail_share_of_total_sse],
    "tail_share_of_total_sse": [tail_share_of_total_sse],
    "non_tail_under_prediction_rate": [non_tail_under_prediction_rate],
    "tail_under_prediction_rate": [tail_under_prediction_rate],
    "non_tail_bias": [non_tail_bias],
    "tail_bias": [tail_bias],
})

baseline_B_lag_plus_backoff_metrics

In [ ]:
with pd.option_context('display.max_columns', None):
    display(baseline_B_lag_plus_backoff_metrics) # or print(df)

### Baseline C. Provider-type + POS + state mean

> Compare Baseline C to new-provider models without lag features

In [ ]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# Baseline C: Provider-type + POS + state mean
# New provider cold-start baseline
# ------------------------------------------------------------

target_col = "avg_mdcr_stdzd_amt"
tail_flag_col = "is_top_1pct_avg_mdcr_stdzd_amt"
group_col = "Rndrng_NPI"

pt_pos_state_cols = ["provider_type", "Place_Of_Srvc", "state"]
test_size = 0.20
random_state = 0
w_tail = 10

# 1) Build same new-provider split
groups_all = eda_hcpcs_df[group_col].to_numpy()
gss = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=random_state)
train_idx, test_idx = next(gss.split(eda_hcpcs_df, groups=groups_all))

train_df = eda_hcpcs_df.iloc[train_idx].copy()
test_df = eda_hcpcs_df.iloc[test_idx].copy()

# 2) Build means from TRAIN only
train_means_pt_pos_state = (
    train_df.groupby(pt_pos_state_cols)[target_col]
    .mean()
    .rename("mean_train_pt_pos_state")
    .reset_index()
)

global_mean = float(train_df[target_col].mean())

# 3) Attach to TEST
test_eval = test_df.copy()
test_eval["_orig_idx"] = test_eval.index

test_eval = test_eval.merge(train_means_pt_pos_state, on=pt_pos_state_cols, how="left")

test_eval = test_eval.set_index("_orig_idx", drop=True)
test_eval = test_eval.loc[test_df.index]

# 4) Predict
pred_test = test_eval["mean_train_pt_pos_state"].fillna(global_mean).astype(float)
y_true = test_eval[target_col].astype(float)

# 5) Weights
is_tail_test = test_eval[tail_flag_col].astype(int).to_numpy()
sample_w_test = (1 + (w_tail - 1) * is_tail_test).astype(float)

# 6) Metrics
r2_test = r2_score(y_true, pred_test)
mae = mean_absolute_error(y_true, pred_test)
rmse = root_mean_squared_error(y_true, pred_test)

pred_clip = pred_test.clip(lower=0)
r2_test_log = r2_score(np.log1p(y_true), np.log1p(pred_clip))
mae_log = mean_absolute_error(np.log1p(y_true), np.log1p(pred_clip))
rmse_log = root_mean_squared_error(np.log1p(y_true), np.log1p(pred_clip))

r2_test_w = r2_score(y_true, pred_test, sample_weight=sample_w_test)
mae_w = mean_absolute_error(y_true, pred_test, sample_weight=sample_w_test)
rmse_w = root_mean_squared_error(y_true, pred_test, sample_weight=sample_w_test)

test_eval["pred"] = pred_test
test_eval["abs_err"] = (test_eval[target_col] - test_eval["pred"]).abs()
test_eval["sq_err"] = (test_eval[target_col] - test_eval["pred"]) ** 2

non_tail_pred_to_y_ratio = (
    test_eval.loc[~test_eval[tail_flag_col], "pred"].mean()
    / test_eval.loc[~test_eval[tail_flag_col], target_col].mean()
)
tail_pred_to_y_ratio = (
    test_eval.loc[test_eval[tail_flag_col], "pred"].mean()
    / test_eval.loc[test_eval[tail_flag_col], target_col].mean()
)

non_tail_sse = test_eval.loc[~test_eval[tail_flag_col], "sq_err"].sum()
tail_sse = test_eval.loc[test_eval[tail_flag_col], "sq_err"].sum()

non_tail_share_of_total_sse = non_tail_sse / (non_tail_sse + tail_sse)
tail_share_of_total_sse = tail_sse / (non_tail_sse + tail_sse)

non_tail_under_prediction_rate = (
    (test_eval.loc[~test_eval[tail_flag_col], "pred"] < test_eval.loc[~test_eval[tail_flag_col], target_col]).mean()
)
tail_under_prediction_rate = (
    (test_eval.loc[test_eval[tail_flag_col], "pred"] < test_eval.loc[test_eval[tail_flag_col], target_col]).mean()
)

non_tail_bias = (
    test_eval.loc[~test_eval[tail_flag_col], "pred"].mean()
    - test_eval.loc[~test_eval[tail_flag_col], target_col].mean()
)
tail_bias = (
    test_eval.loc[test_eval[tail_flag_col], "pred"].mean()
    - test_eval.loc[test_eval[tail_flag_col], target_col].mean()
)

baseline_C_pt_pos_state_mean_metrics = pd.DataFrame({
    "train_sample_weights": [np.nan],
    "r2_train": [np.nan],
    "r2_test": [r2_test],
    "mae": [mae],
    "rmse": [rmse],
    "r2_train_log": [np.nan],
    "r2_test_log": [r2_test_log],
    "mae_log": [mae_log],
    "rmse_log": [rmse_log],
    "r2_train_w": [np.nan],
    "r2_test_w": [r2_test_w],
    "mae_w": [mae_w],
    "rmse_w": [rmse_w],
    "non_tail_pred_to_y_ratio": [non_tail_pred_to_y_ratio],
    "tail_pred_to_y_ratio": [tail_pred_to_y_ratio],
    "non_tail_sse": [non_tail_sse],
    "tail_sse": [tail_sse],
    "non_tail_share_of_total_sse": [non_tail_share_of_total_sse],
    "tail_share_of_total_sse": [tail_share_of_total_sse],
    "non_tail_under_prediction_rate": [non_tail_under_prediction_rate],
    "tail_under_prediction_rate": [tail_under_prediction_rate],
    "non_tail_bias": [non_tail_bias],
    "tail_bias": [tail_bias],
})

baseline_C_pt_pos_state_mean_metrics

In [ ]:
with pd.option_context('display.max_columns', None):
    display(baseline_C_pt_pos_state_mean_metrics) # or print(df)

### Baseline D. Full backoff ladder

Compare Baseline D to new-provider models without lag features

In [ ]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# Baseline D: Full backoff ladder
# New provider cold-start baseline
# ------------------------------------------------------------

target_col = "avg_mdcr_stdzd_amt"
tail_flag_col = "is_top_1pct_avg_mdcr_stdzd_amt"
group_col = "Rndrng_NPI"

group_cols = ["provider_type", "HCPCS_Cd", "Place_Of_Srvc", "state"]
grp_no_state_cols = ["provider_type", "HCPCS_Cd", "Place_Of_Srvc"]
hcpcs_cols = ["HCPCS_Cd"]
pt_pos_state_cols = ["provider_type", "Place_Of_Srvc", "state"]
pt_pos_cols = ["provider_type", "Place_Of_Srvc"]
pt_cols = ["provider_type"]

test_size = 0.20
random_state = 0
w_tail = 10

# 1) Build same new-provider split
groups_all = eda_hcpcs_df[group_col].to_numpy()
gss = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=random_state)
train_idx, test_idx = next(gss.split(eda_hcpcs_df, groups=groups_all))

train_df = eda_hcpcs_df.iloc[train_idx].copy()
test_df = eda_hcpcs_df.iloc[test_idx].copy()

# 2) Build all lookup tables from TRAIN only
mean_grp_state = (
    train_df.groupby(group_cols)[target_col]
    .mean()
    .rename("mean_grp_state")
    .reset_index()
)

mean_grp = (
    train_df.groupby(grp_no_state_cols)[target_col]
    .mean()
    .rename("mean_grp")
    .reset_index()
)

mean_hcpcs = (
    train_df.groupby(hcpcs_cols)[target_col]
    .mean()
    .rename("mean_hcpcs")
    .reset_index()
)

mean_pt_pos_state = (
    train_df.groupby(pt_pos_state_cols)[target_col]
    .mean()
    .rename("mean_pt_pos_state")
    .reset_index()
)

mean_pt_pos = (
    train_df.groupby(pt_pos_cols)[target_col]
    .mean()
    .rename("mean_pt_pos")
    .reset_index()
)

mean_pt = (
    train_df.groupby(pt_cols)[target_col]
    .mean()
    .rename("mean_pt")
    .reset_index()
)

global_mean = float(train_df[target_col].mean())

# 3) Attach to TEST while preserving index
test_eval = test_df.copy()
test_eval["_orig_idx"] = test_eval.index

test_eval = test_eval.merge(mean_grp_state, on=group_cols, how="left")
test_eval = test_eval.merge(mean_grp, on=grp_no_state_cols, how="left")
test_eval = test_eval.merge(mean_hcpcs, on=hcpcs_cols, how="left")
test_eval = test_eval.merge(mean_pt_pos_state, on=pt_pos_state_cols, how="left")
test_eval = test_eval.merge(mean_pt_pos, on=pt_pos_cols, how="left")
test_eval = test_eval.merge(mean_pt, on=pt_cols, how="left")

test_eval = test_eval.set_index("_orig_idx", drop=True)
test_eval = test_eval.loc[test_df.index]

# 4) Backoff ladder
pred_test = test_eval["mean_grp_state"]
pred_test = pred_test.fillna(test_eval["mean_grp"])
pred_test = pred_test.fillna(test_eval["mean_hcpcs"])
pred_test = pred_test.fillna(test_eval["mean_pt_pos_state"])
pred_test = pred_test.fillna(test_eval["mean_pt_pos"])
pred_test = pred_test.fillna(test_eval["mean_pt"])
pred_test = pred_test.fillna(global_mean).astype(float)

y_true = test_eval[target_col].astype(float)

# 5) Optional label for which backoff source got used
test_eval["backoff_source"] = np.select(
    [
        test_eval["mean_grp_state"].notna(),
        test_eval["mean_grp"].notna(),
        test_eval["mean_hcpcs"].notna(),
        test_eval["mean_pt_pos_state"].notna(),
        test_eval["mean_pt_pos"].notna(),
        test_eval["mean_pt"].notna(),
    ],
    [
        "grp_state",
        "grp_no_state",
        "hcpcs",
        "pt_pos_state",
        "pt_pos",
        "pt",
    ],
    default="global"
)

# 6) Weights
is_tail_test = test_eval[tail_flag_col].astype(int).to_numpy()
sample_w_test = (1 + (w_tail - 1) * is_tail_test).astype(float)

# 7) Metrics
r2_test = r2_score(y_true, pred_test)
mae = mean_absolute_error(y_true, pred_test)
rmse = root_mean_squared_error(y_true, pred_test)

pred_clip = pred_test.clip(lower=0)
r2_test_log = r2_score(np.log1p(y_true), np.log1p(pred_clip))
mae_log = mean_absolute_error(np.log1p(y_true), np.log1p(pred_clip))
rmse_log = root_mean_squared_error(np.log1p(y_true), np.log1p(pred_clip))

r2_test_w = r2_score(y_true, pred_test, sample_weight=sample_w_test)
mae_w = mean_absolute_error(y_true, pred_test, sample_weight=sample_w_test)
rmse_w = root_mean_squared_error(y_true, pred_test, sample_weight=sample_w_test)

test_eval["pred"] = pred_test
test_eval["abs_err"] = (test_eval[target_col] - test_eval["pred"]).abs()
test_eval["sq_err"] = (test_eval[target_col] - test_eval["pred"]) ** 2

non_tail_pred_to_y_ratio = (
    test_eval.loc[~test_eval[tail_flag_col], "pred"].mean()
    / test_eval.loc[~test_eval[tail_flag_col], target_col].mean()
)
tail_pred_to_y_ratio = (
    test_eval.loc[test_eval[tail_flag_col], "pred"].mean()
    / test_eval.loc[test_eval[tail_flag_col], target_col].mean()
)

non_tail_sse = test_eval.loc[~test_eval[tail_flag_col], "sq_err"].sum()
tail_sse = test_eval.loc[test_eval[tail_flag_col], "sq_err"].sum()

non_tail_share_of_total_sse = non_tail_sse / (non_tail_sse + tail_sse)
tail_share_of_total_sse = tail_sse / (non_tail_sse + tail_sse)

non_tail_under_prediction_rate = (
    (test_eval.loc[~test_eval[tail_flag_col], "pred"] < test_eval.loc[~test_eval[tail_flag_col], target_col]).mean()
)
tail_under_prediction_rate = (
    (test_eval.loc[test_eval[tail_flag_col], "pred"] < test_eval.loc[test_eval[tail_flag_col], target_col]).mean()
)

non_tail_bias = (
    test_eval.loc[~test_eval[tail_flag_col], "pred"].mean()
    - test_eval.loc[~test_eval[tail_flag_col], target_col].mean()
)
tail_bias = (
    test_eval.loc[test_eval[tail_flag_col], "pred"].mean()
    - test_eval.loc[test_eval[tail_flag_col], target_col].mean()
)

baseline_D_full_backoff_ladder_metrics = pd.DataFrame({
    "train_sample_weights": [np.nan],
    "r2_train": [np.nan],
    "r2_test": [r2_test],
    "mae": [mae],
    "rmse": [rmse],
    "r2_train_log": [np.nan],
    "r2_test_log": [r2_test_log],
    "mae_log": [mae_log],
    "rmse_log": [rmse_log],
    "r2_train_w": [np.nan],
    "r2_test_w": [r2_test_w],
    "mae_w": [mae_w],
    "rmse_w": [rmse_w],
    "non_tail_pred_to_y_ratio": [non_tail_pred_to_y_ratio],
    "tail_pred_to_y_ratio": [tail_pred_to_y_ratio],
    "non_tail_sse": [non_tail_sse],
    "tail_sse": [tail_sse],
    "non_tail_share_of_total_sse": [non_tail_share_of_total_sse],
    "tail_share_of_total_sse": [tail_share_of_total_sse],
    "non_tail_under_prediction_rate": [non_tail_under_prediction_rate],
    "tail_under_prediction_rate": [tail_under_prediction_rate],
    "non_tail_bias": [non_tail_bias],
    "tail_bias": [tail_bias],
})

baseline_D_full_backoff_ladder_metrics

In [ ]:
with pd.option_context('display.max_columns', None):
    display(baseline_D_full_backoff_ladder_metrics) # or print(df)

## New-provider baselines A to D: detailed interpretation

### First, what the baselines are trying to tell us
These baselines answer four increasingly nuanced questions:
* **Baseline A. Lag only:** If I know last period’s cost for this provider-service row, how far can I get by simply carrying it forward?
* **Baseline B. Lag + backoff:** If lag exists, use it. If lag is missing, fall back to train-set means.
* **Baseline C. Provider-type + POS + state mean:** In a cold-start setting with no provider history, can broad contextual averaging do the job?
* **Baseline D. Full backoff ladder:** If I intelligently back off through progressively broader train-set groupings, how strong can a pure lookup baseline become?

These are essential because they define the “floor” that sophisticated models must beat.

---

### Baseline A. Lag only

**Reported metrics**
* **r2_test:** 0.942565
* **mae:** 5.707927
* **rmse:** 38.946548
* **r2_test_log:** 0.990242
* **mae_log:** 0.061641
* **rmse_log:** 0.148831
* **r2_test_w:** 0.929433
* **mae_w:** 17.683063
* **rmse_w:** 109.379102
* **non_tail_pred_to_y_ratio:** 1.00035
* **tail_pred_to_y_ratio:** 1.078064
* **non_tail_sse:** 3.555263e+07
* **tail_sse:** 1.845035e+08
* **non_tail_share_of_total_sse:** 0.161562
* **tail_share_of_total_sse:** 0.838438
* **non_tail_under_prediction_rate:** 0.409546
* **tail_under_prediction_rate:** 0.337296
* **non_tail_bias:** 0.025847
* **tail_bias:** 111.758282

#### Line-by-line interpretation

* **`r2_test = 0.942565`**: This is extremely strong. On the subset where lag exists, simply using prior cost explains about 94.3 percent of the variation in current cost. That tells you that cost persistence is very high for rows where lag is available.
* **`mae = 5.707927`**: On average, absolute prediction error is about 5.71 dollars per service. That is excellent, especially given the scale and heterogeneity of provider-service data.
* **`rmse = 38.946548`**: RMSE is meaningfully larger than MAE, which tells you a smaller number of larger misses are present. In these medical cost problems, that usually means the tail contributes disproportionately to squared error.
* **`r2_test_log = 0.990242`**: In log space, the lag-only baseline is even stronger. That means relative cost movements are very stable when lag exists. The model is very good at capturing multiplicative scale, even if some large absolute misses remain.
* **`mae_log = 0.061641` and `rmse_log = 0.148831`**: These are very low. Again, this supports the story that lag is a very strong predictor of future cost levels when the same provider-service relationship already has historical information.
* **`r2_test_w = 0.929433`**: Once tail rows are emphasized 10x, performance drops a bit. That is expected. The tail is harder. But 0.929 is still very strong, which means lag remains a formidable baseline even when rare expensive rows matter more.
* **`mae_w = 17.683063` and `rmse_w = 109.379102`**: These weighted errors are much larger than the unweighted ones. This means that even though lag is strong overall, it struggles much more on expensive tail rows.
* **`non_tail_pred_to_y_ratio = 1.00035`**: For non-tail rows, the prediction mean is almost perfectly calibrated. This is nearly ideal.
* **`tail_pred_to_y_ratio = 1.078064`**: For tail rows, lag-only overpredicts on average by about 7.8 percent. So tail costs this year tend to be somewhat lower than last year, at least in aggregate for this subset, or lag is capturing a previous spike.
* **`non_tail_sse` vs `tail_sse`**: Tail SSE (1.85e+08) is dramatically larger than non-tail SSE (3.56e+07).
* **`tail_share_of_total_sse = 0.838438`**: About 83.8 percent of total squared error comes from tail rows. That is a huge result. It means this baseline is excellent for typical rows, but most damage comes from the expensive minority.
* **`non_tail_under_prediction_rate = 0.409546`**: For non-tail rows, fewer than half of predictions are under the truth. So there is a mild tendency to overpredict.
* **`tail_under_prediction_rate = 0.337296`**: For tail rows, only about one third are underpredictions. This is consistent with the positive tail bias.
* **`non_tail_bias = 0.025847`**: Essentially unbiased for non-tail rows.
* **`tail_bias = 111.758282`**: Tail predictions are high by about 112 dollars per service on average. That is a sizable overprediction, even though the tail still dominates SSE.

**Bottom line for Baseline A:**
This is a **very powerful warm-start baseline**. If lag exists, simple carry-forward is hard to beat. It is especially good on non-tail rows and still very strong overall, though not fully robust in the tail.

---

### Baseline B. Lag + backoff mean

**Reported metrics**
* **r2_test:** 0.930421
* **mae:** 8.61911
* **rmse:** 55.014137
* **r2_test_log:** 0.98354
* **mae_log:** 0.077874
* **rmse_log:** 0.193377
* **r2_test_w:** 0.941156
* **mae_w:** 27.599187
* **rmse_w:** 136.10913
* **non_tail_pred_to_y_ratio:** 1.013726
* **tail_pred_to_y_ratio:** 0.975051
* **non_tail_sse:** 2.663388e+08
* **tail_sse:** 4.467903e+08
* **non_tail_share_of_total_sse:** 0.373479
* **tail_share_of_total_sse:** 0.626521
* **non_tail_under_prediction_rate:** 0.461677
* **tail_under_prediction_rate:** 0.539295
* **non_tail_bias:** 1.01244
* **tail_bias:** -42.206414

#### Line-by-line interpretation

* **`r2_test = 0.930421`**: This is still strong, but lower than Baseline A. That tells you that once you extend evaluation to all rows, including no-lag cases handled by mean backoff, performance declines.
* **`mae = 8.61911` and `rmse = 55.014137`**: These are materially worse than Baseline A. Again, the no-lag rows are introducing harder cases, and the mean-based fallback is less precise than direct lag.
* **`r2_test_log = 0.98354`**: Still excellent in log space. This indicates the backoff still captures general scale reasonably well.
* **`r2_test_w = 0.941156`**: Interesting point: weighted R² is a bit higher than Baseline A’s weighted R². That does not mean it is better overall in the tail. You need to interpret it alongside weighted MAE/RMSE and bias. Weighted R² can sometimes reward variance capture differently than direct error size metrics.
* **`mae_w = 27.599187` and `rmse_w = 136.10913`**: These are much worse than Baseline A. So in direct weighted error terms, this baseline is weaker on important high-cost rows.
* **`non_tail_pred_to_y_ratio = 1.013726`**: Slight overprediction in non-tail.
* **`tail_pred_to_y_ratio = 0.975051`**: Slight underprediction in tail.
* **`non_tail_share_of_total_sse = 0.373479`, `tail_share_of_total_sse = 0.626521`**: Tail still dominates error, though less extremely than in Baseline A. This is because no-lag fallback increases error outside the tail as well.
* **`non_tail_under_prediction_rate = 0.461677`**: Near balanced, but slightly more overprediction overall.
* **`tail_under_prediction_rate = 0.539295`**: Now the tail more often gets underpredicted than overpredicted.
* **`non_tail_bias = 1.01244`**: Small positive bias.
* **`tail_bias = -42.206414`**: Moderate underprediction in tail on average.

**Bottom line for Baseline B:**
This is a **practical operational baseline** for all rows, but it is noticeably weaker than lag-only on lag-available rows. It is important because it defines a realistic full-coverage benchmark, not because it is the strongest performer.

---

### Baseline C. Provider-type + POS + state mean

**Reported metrics**
* **r2_test:** 0.058866
* **mae:** 72.994691
* **rmse:** 202.329669
* **r2_test_log:** -0.091179
* **mae_log:** 1.167284
* **rmse_log:** 1.574487
* **r2_test_w:** -0.00719
* **mae_w:** 187.663518
* **rmse_w:** 563.107099
* **non_tail_pred_to_y_ratio:** 1.20858
* **tail_pred_to_y_ratio:** 0.087527
* **non_tail_sse:** 1.713983e+09
* **tail_sse:** 7.931827e+09
* **non_tail_share_of_total_sse:** 0.177692
* **tail_share_of_total_sse:** 0.822308
* **non_tail_under_prediction_rate:** 0.309811
* **tail_under_prediction_rate:** 1.0
* **non_tail_bias:** 15.385176
* **tail_bias:** -1543.616931

#### Line-by-line interpretation

This baseline is telling you what happens in a very coarse cold-start setting.

* **`r2_test = 0.058866`**: Essentially no meaningful predictive power on the level scale.
* **`mae = 72.99`, `rmse = 202.33`**: Huge errors relative to the better baselines and models.
* **`r2_test_log = -0.091179`**: Negative log-space R² means this baseline is worse than a trivial constant predictor in log space.
* **`mae_log = 1.167284`, `rmse_log = 1.574487`**: These are very poor.
* **`r2_test_w = -0.00719`**: Once the tail matters, performance is effectively useless.
* **`mae_w = 187.663518`, `rmse_w = 563.107099`**: This baseline collapses badly under weighted evaluation.
* **`non_tail_pred_to_y_ratio = 1.20858`**: It overpredicts non-tail by about 21 percent on average.
* **`tail_pred_to_y_ratio = 0.087527`**: This is catastrophic underprediction for tail rows. It predicts less than 9 percent of the true tail mean on average.
* **`tail_share_of_total_sse = 0.822308`**: Tail dominates the damage.
* **`tail_under_prediction_rate = 1.0`**: Every tail row is underpredicted. That is one of the clearest diagnostic results in your entire notebook.
* **`tail_bias = -1543.616931`**: On average, tail rows are underpredicted by more than 1,500 dollars per service. This is an enormous miss.

**Bottom line for Baseline C:**
This is a **weak cold-start baseline**. It is useful conceptually because it shows that broad contextual averaging alone is nowhere near enough.

---

### Baseline D. Full backoff ladder

**Reported metrics**
* **r2_test:** 0.931978
* **mae:** 9.889925
* **rmse:** 54.394893
* **r2_test_log:** 0.98007
* **mae_log:** 0.085787
* **rmse_log:** 0.212786
* **r2_test_w:** 0.950689
* **mae_w:** 30.051029
* **rmse_w:** 124.596617
* **non_tail_pred_to_y_ratio:** 1.016807
* **tail_pred_to_y_ratio:** 0.920057
* **non_tail_sse:** 3.338238e+08
* **tail_sse:** 3.633415e+08
* **non_tail_share_of_total_sse:** 0.47883
* **tail_share_of_total_sse:** 0.52117
* **non_tail_under_prediction_rate:** 0.501817
* **tail_under_prediction_rate:** 0.682927
* **non_tail_bias:** 1.239743
* **tail_bias:** -135.238878

#### Line-by-line interpretation

This is a very important baseline.

* **`r2_test = 0.931978`**: Surprisingly strong. This means a well-designed lookup ladder from train-set averages can explain a lot of variation.
* **`mae = 9.889925`, `rmse = 54.394893`**: Worse than Baseline A, but much better than Baseline C. This tells you the ladder is doing something genuinely useful.
* **`r2_test_log = 0.98007`**: Excellent in log space. So this ladder captures broad scale quite well.
* **`r2_test_w = 0.950689`**: Very strong weighted R². In fact, stronger than several model families.
* **`mae_w = 30.051029`, `rmse_w = 124.596617`**: These are still sizable, but much better than Baseline C and comparable to some trained models.
* **`non_tail_pred_to_y_ratio = 1.016807`**: Slight overprediction for non-tail.
* **`tail_pred_to_y_ratio = 0.920057`**: Moderate underprediction for tail.
* **`tail_share_of_total_sse = 0.52117`**: The tail still slightly dominates total SSE, but error is much more balanced across tail and non-tail than in Baselines A, B, or C.
* **`tail_under_prediction_rate = 0.682927`**: Tail rows are underpredicted about 68 percent of the time.
* **`tail_bias = -135.238878`**: The ladder is still biased downward in the tail, but far less catastrophically than Baseline C.

**Bottom line for Baseline D:**
This is the **strongest cold-start baseline by far**. It is a serious benchmark. Any “without lag” model needs to beat this, not just beat Baseline C.

---

### Baseline Summary Table

| Baseline | Core idea | R² test | MAE | RMSE | R² test weighted | MAE weighted | RMSE weighted | Tail bias |
| :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- |
| **A** | Lag only | 0.942565 | 5.7079 | 38.9465 | 0.929433 | 17.6831 | 109.3791 | 111.7583 |
| **B** | Lag + backoff | 0.930421 | 8.6191 | 55.0141 | 0.941156 | 27.5992 | 136.1091 | -42.2064 |
| **C** | PT + POS + state mean | 0.058866 | 72.9947 | 202.3297 | -0.007190 | 187.6635 | 563.1071 | -1543.6169 |
| **D** | Full backoff ladder | 0.931978 | 9.8899 | 54.3949 | 0.950689 | 30.0510 | 124.5966 | -135.2389 |

## Models A to I: Detailed Interpretation

I will use your labels A to I exactly as you provided them.

---

### Model A. `wt=1`, with lags, level target, GridSearchCV

**Metrics**
* **r2_train:** 0.715032
* **r2_test:** 0.768979
* **mae:** 12.333052
* **rmse:** 100.244215
* **r2_train_log:** 0.979366
* **r2_test_log:** 0.978827
* **mae_log:** 0.124418
* **rmse_log:** 0.219321
* **r2_train_w:** 0.715032
* **r2_test_w:** 0.768979
* **mae_w:** 12.333052
* **rmse_w:** 100.244215
* **tail_pred_to_y_ratio:** 0.810185
* **tail_share_of_total_sse:** 0.920688
* **tail_under_prediction_rate:** 0.797651
* **tail_bias:** -321.107964

**Interpretation**
*This is an unweighted level model with lag features.*

* **Train and test R² are identical at about 0.769 on test:** This is decent but nowhere near Baseline A. So even with lag features, the model is not exploiting them as effectively as simply copying lag.
* **Log-space R² is very high, around 0.979:** This shows the model preserves rank and relative scale fairly well, but level errors are still substantial.
* **Tail behavior is weak:** Tail mean prediction is only about 81 percent of truth. Tail underprediction happens nearly 80 percent of the time. Tail bias is about -321. So this model systematically undershoots expensive rows.
* **Tail dominates error:** More than 92 percent of SSE comes from tail rows. This is very poor tail robustness.

**Bottom line:** Model A is inferior to lag-based baselines and weak in the tail.

---

### Model B. `wt=1`, without lags, level target, GridSearchCV

**Metrics**
* **r2_test:** 0.760929
* **mae:** 14.993705
* **rmse:** 101.975945
* **r2_test_log:** 0.979429
* **mae_log:** 0.126904
* **rmse_log:** 0.216183
* **r2_test_w:** 0.760929
* **mae_w:** 14.993705
* **rmse_w:** 101.975945
* **tail_pred_to_y_ratio:** 0.753707
* **tail_share_of_total_sse:** 0.892789
* **tail_under_prediction_rate:** 0.884824
* **tail_bias:** -416.650689

**Interpretation**
*This is the unweighted cold-start level model without lag features.*

* **Compared with Model A:** slightly worse test R², substantially worse MAE, worse tail calibration, and worse tail bias. This is expected. Removing lag features makes the task harder.
* **Tail prediction is only about 75 percent of truth:** This is strong systematic underprediction.
* **Tail underprediction rate is 88.5 percent:** Almost all expensive rows are underestimated.

**Bottom line:** Model B is a weak cold-start model and clearly not competitive with the stronger baselines or weighted models.

---

### Model C. `wt=10`, with lags, level target, GridSearchCV

**Metrics**
* **r2_test:** 0.804760
* **mae:** 17.688952
* **rmse:** 92.154814
* **r2_test_log:** 0.952343
* **mae_log:** 0.209681
* **rmse_log:** 0.329044
* **r2_test_w:** 0.798161
* **mae_w:** 41.802460
* **rmse_w:** 252.079967
* **tail_pred_to_y_ratio:** 0.868736
* **tail_share_of_total_sse:** 0.790573
* **tail_under_prediction_rate:** 0.677958
* **tail_bias:** -222.057183

**Interpretation**
*This is a weighted level model with lag features.*

* **R² improves over unweighted lag model:** From 0.769 to 0.805. So weighting helps variance explanation.
* **But MAE gets much worse:** 17.69 versus 12.33. This means the model is making larger average errors on the typical row in order to try to serve the tail better.
* **Weighted test R² improves modestly:** 0.798 versus 0.769. But weighted MAE and RMSE are still quite large.
* **Tail improves vs Model A:** Tail prediction rises from 81 percent to 86.9 percent of truth. Tail bias becomes less negative. Tail underprediction rate drops from 79.8 percent to 67.8 percent. So weighting helps the tail somewhat, but not enough.

**Bottom line:** Model C is better than unweighted lag model, but still substantially weaker than simple lag baselines.

---

### Model D. `wt=10`, without lags, level target, GridSearchCV

**Metrics**
* **r2_test:** 0.886899
* **mae:** 13.867930
* **rmse:** 70.140176
* **r2_test_log:** 0.977456
* **mae_log:** 0.133327
* **rmse_log:** 0.226314
* **r2_test_w:** 0.920303
* **mae_w:** 30.880678
* **rmse_w:** 158.400588
* **tail_pred_to_y_ratio:** 0.922783
* **tail_share_of_total_sse:** 0.503491
* **tail_under_prediction_rate:** 0.765583
* **tail_bias:** -130.626763

**Interpretation**
*This is the most important level cold-start model among A to D.*

* **Excellent test R² for a no-lag model:** 0.887 is very strong, especially for unseen-provider prediction without lag features.
* **RMSE = 70.14:** Much better than Models A, B, and C.
* **Weighted R² = 0.920303:** Very strong. It is competitive with Baseline D.
* **Tail calibration improves a lot:** Tail predictions average about 92.3 percent of truth. Tail bias is -130.6, much better than Models B and C, and very close to Baseline D.
* **Error is more balanced:** Tail share of total SSE is 50.3 percent, almost a perfect balance between tail and non-tail error contribution. This is a strong robustness signal.

**Bottom line:** Model D is the best level-target cold-start model among the first four. It is a serious contender versus Baseline D.

---

### Model E. `wt=10`, with lags, level target, RandomizedSearchCV

**Metrics**
* **r2_test:** 0.725958
* **mae:** 21.165937
* **rmse:** 109.179882
* **r2_test_log:** 0.914207
* **mae_log:** 0.277166
* **rmse_log:** 0.441486
* **r2_test_w:** 0.708240
* **mae_w:** 50.903769
* **rmse_w:** 303.073671
* **tail_pred_to_y_ratio:** 0.788339
* **tail_share_of_total_sse:** 0.817481
* **tail_under_prediction_rate:** 0.793586
* **tail_bias:** -358.064019

**Interpretation**
*This is clearly worse than Model C.*

* Randomized search here found an inferior configuration relative to the grid search.
* **Everything worsens:** lower R², higher MAE, higher RMSE, poorer tail calibration, larger negative tail bias.

**Bottom line:** Model E is dominated by Model C and should be discarded.

---

### Model F. `wt=10`, without lags, level target, RandomizedSearchCV

**Metrics**
* **r2_test:** 0.789285
* **mae:** 19.294453
* **rmse:** 95.737380
* **r2_test_log:** 0.958325
* **mae_log:** 0.210326
* **rmse_log:** 0.307700
* **r2_test_w:** 0.804227
* **mae_w:** 43.422870
* **rmse_w:** 248.262741
* **tail_pred_to_y_ratio:** 0.864591
* **tail_share_of_total_sse:** 0.699241
* **tail_under_prediction_rate:** 0.794038
* **tail_bias:** -229.068779

**Interpretation**
*This is also clearly worse than Model D.*

* **It has:** much lower test R², worse MAE and RMSE, worse weighted metrics, more negative tail bias, and poorer tail capture.

**Bottom line:** Model F is dominated by Model D.

---

### Model G. `wt=10`, with lags, delta target, GridSearchCV

**Metrics**
* **r2_train:** 0.995563
* **r2_test:** 0.933416
* **mae:** 3.493940
* **rmse:** 43.006421
* **r2_train_log:** 0.699577
* **r2_test_log:** 0.565856
* **mae_log:** 0.040481
* **rmse_log:** 0.106547
* **r2_train_w:** 0.997749
* **r2_test_w:** 0.975015
* **mae_w:** 6.752066
* **rmse_w:** 67.723853
* **tail_pred_to_y_ratio:** 0.995876
* **tail_share_of_total_sse:** 0.187444
* **tail_under_prediction_rate:** 0.510561
* **tail_bias:** -5.894139

**Interpretation**
*This is one of the most impressive models in your whole notebook.*

*Important nuance: although the target is delta, the evaluation is back on the level scale.*

* **`r2_test = 0.933416`:** Very strong. Close to the best lag-based baselines, but now with model-driven adjustment instead of naive carry-forward.
* **`mae = 3.49394`:** This is outstanding. Better than every baseline and every level-target model you listed.
* **`rmse = 43.006421`:** Very competitive. Slightly worse than Baseline A’s 38.95, but far better than most learned models.
* **`r2_test_w = 0.975015`:** This is extraordinary. Weighted tail-sensitive performance is dramatically better than every level-target model and every baseline.
* **`mae_w = 6.752066`, `rmse_w = 67.723853`:** Again, excellent. This is the first model that truly looks tail-robust.
* **`tail_pred_to_y_ratio = 0.995876`:** Almost perfectly calibrated in the tail.
* **`tail_bias = -5.894139`:** Essentially unbiased in the tail.
* **`tail_share_of_total_sse = 0.187444`:** Only about 18.7 percent of squared error comes from the tail. That is a dramatic improvement relative to almost all other methods.

**Bottom line:** Model G is a major breakthrough. It is the strongest lag-enabled learned model by a wide margin and the best overall performer in many important dimensions, especially tail-sensitive ones.

---

### Model H. `wt=10`, with lags, delta target, RandomizedSearchCV

**Metrics**
* **r2_test:** 0.930269
* **mae:** 3.941185
* **rmse:** 44.010877
* **r2_test_w:** 0.972997
* **mae_w:** 7.393144
* **rmse_w:** 70.405245
* **tail_pred_to_y_ratio:** 0.992898
* **tail_share_of_total_sse:** 0.196993
* **tail_under_prediction_rate:** 0.512017
* **tail_bias:** -10.149489

**Interpretation**
*This is extremely strong, but slightly weaker than Model G.*

* **Everything says:** still excellent, still highly tail-aware, still nearly unbiased in the tail, but slightly worse than the grid-searched delta model.

**Bottom line:** Model H is very strong, but Model G is better.

---

### Model I. `wt=10`, without lags, delta target, RandomizedSearchCV

**Metrics**
* **r2_test:** 0.904079
* **mae:** 4.632643
* **rmse:** 51.618434
* **r2_test_w:** 0.970215
* **mae_w:** 8.527295
* **rmse_w:** 73.942416
* **tail_pred_to_y_ratio:** 0.999560
* **tail_share_of_total_sse:** 0.135939
* **tail_under_prediction_rate:** 0.468318
* **tail_bias:** -0.628872

**Interpretation**
*This is the best no-lag model in the entire set.*

*This is remarkable because it excludes lag features from `X_train`, yet still predicts the delta target and reconstructs level predictions using lag in the final formula. That structure appears to regularize the problem effectively.*

* **`r2_test = 0.904079`:** Excellent for a no-lag-feature model.
* **`mae = 4.632643`, `rmse = 51.618434`:** Much better than Model D and much better than Baseline D.
* **`r2_test_w = 0.970215`:** Stunning weighted performance.
* **`tail_pred_to_y_ratio = 0.99956`:** Almost perfect tail calibration.
* **`tail_bias = -0.628872`:** Essentially no tail bias.
* **`tail_share_of_total_sse = 0.135939`:** Only 13.6 percent of squared error comes from the tail. That is the best tail concentration result in the whole set.

**Bottom line:** Model I is an exceptionally strong no-lag-feature model and arguably the best cold-start-style learned model here.

---

### Model Summary Table

| Model | Description | R² test | MAE | RMSE | R² test weighted | MAE weighted | RMSE weighted | Tail pred / y | Tail bias |
| :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- |
| **A** | wt1, with lags, level, grid | 0.768979 | 12.3331 | 100.2442 | 0.768979 | 12.3331 | 100.2442 | 0.810185 | -321.1080 |
| **B** | wt1, without lags, level, grid | 0.760929 | 14.9937 | 101.9759 | 0.760929 | 14.9937 | 101.9759 | 0.753707 | -416.6507 |
| **C** | wt10, with lags, level, grid | 0.804760 | 17.6890 | 92.1548 | 0.798161 | 41.8025 | 252.0800 | 0.868736 | -222.0572 |
| **D** | wt10, without lags, level, grid | 0.886899 | 13.8679 | 70.1402 | 0.920303 | 30.8807 | 158.4006 | 0.922783 | -130.6268 |
| **E** | wt10, with lags, level, rs50 | 0.725958 | 21.1659 | 109.1799 | 0.708240 | 50.9038 | 303.0737 | 0.788339 | -358.0640 |
| **F** | wt10, without lags, level, rs50 | 0.789285 | 19.2945 | 95.7374 | 0.804227 | 43.4229 | 248.2627 | 0.864591 | -229.0688 |
| **G** | wt10, with lags, delta, grid | 0.933416 | 3.4939 | 43.0064 | 0.975015 | 6.7521 | 67.7239 | 0.995876 | -5.8941 |
| **H** | wt10, with lags, delta, rs50 | 0.930269 | 3.9412 | 44.0109 | 0.972997 | 7.3931 | 70.4052 | 0.992898 | -10.1495 |
| **I** | wt10, without lags, delta, rs50 | 0.904079 | 4.6326 | 51.6184 | 0.970215 | 8.5273 | 73.9424 | 0.999560 | -0.6289 |

### Comparison 1: Baselines A and B versus models with lag features

The **lag-feature models** are:
* Model A
* Model C
* Model E
* Model G
* Model H

The relevant **lag baselines** are:
* Baseline A: lag only
* Baseline B: lag + backoff

#### Baseline A versus lag-feature level models
Baseline A crushes the **level-target lag-feature models** A, C, and E on standard level metrics:
* **Baseline A R²** = 0.9426 vs Model A 0.7690, Model C 0.8048, Model E 0.7260
* **Baseline A MAE** = 5.71 vs 12.33, 17.69, 21.17
* **Baseline A RMSE** = 38.95 vs 100.24, 92.15, 109.18

**This tells you something very important:**
If the target is level cost and lag is available, simple carry-forward is stronger than the learned level models you trained.

That is exactly the same kind of conclusion you reached in the forecasting work. Lag is such a powerful signal that a model can easily underperform it if it smooths too aggressively or compromises to fit the tail.

#### Baseline A versus delta-target lag-feature models G and H
This is where things get interesting.

Compared with Baseline A:
* Baseline A has slightly better RMSE: 38.95 vs 43.01 for G, 44.01 for H
* But Model G and H are **dramatically better** on weighted metrics and tail calibration

For example:
* **Baseline A r2_test_w** = 0.9294 vs **G** 0.9750
* **Baseline A mae_w** = 17.68 vs **G** 6.75
* **Baseline A rmse_w** = 109.38 vs **G** 67.72
* **Baseline A tail_bias** = +111.76 vs **G** -5.89
* **Baseline A tail_pred_to_y_ratio** = 1.078 vs **G** 0.996

So the comparison is:
* Baseline A wins or nearly wins on simple unweighted warm-start forecasting
* Model G wins decisively on tail-aware, calibrated, weighted performance

That is a huge modeling insight.

#### Baseline B versus lag-feature models
Baseline B is more of a full-coverage operational benchmark. It is weaker than Baseline A, but still useful.

Compared with Models A, C, and E:
* Model C beats Baseline B on test R² but not on MAE
* Models A and E are not clearly better overall
* Weighted errors of Model C and E remain large

Compared with G and H:
* G and H are decisively better on almost every important metric, especially MAE, weighted MAE, weighted RMSE, tail bias, and tail calibration.

#### Take-home on lag side
If lag exists:
* **Best naive baseline:** Baseline A
* **Best learned model:** Model G
* **Interpretation:** The only learned models that truly justify their complexity are the **delta-target lag models**, especially Model G.

---

### Comparison 2: Baselines C and D versus models without lag features

The **no-lag-feature models** are:
* Model B
* Model D
* Model F
* Model I

The relevant **baselines** are:
* Baseline C
* Baseline D

#### Baseline C versus no-lag models
Baseline C is weak, and every no-lag model beats it badly. 

That tells you broad contextual averaging alone is insufficient.

#### Baseline D versus no-lag level models B, D, F
Baseline D is actually a very strong benchmark.

Compare:
* **Baseline D:** r2_test = 0.9320, mae = 9.89, rmse = 54.39, r2_test_w = 0.9507
* **Model D:** 0.8869, 13.87, 70.14, 0.9203
* **Model F:** 0.7893, 19.29, 95.74, 0.8042
* **Model B:** much worse still

So for **level-target no-lag models**, Baseline D is stronger than all of them.

**That is a very important conclusion:** Your clever lookup ladder outperforms your no-lag level-target machine learning models.

#### Baseline D versus no-lag delta model I
Now compare Baseline D with Model I:
* **Baseline D r2_test** = 0.9320 vs **Model I** 0.9041
* **Baseline D mae** = 9.89 vs **Model I** 4.63
* **Baseline D rmse** = 54.39 vs **Model I** 51.62
* **Baseline D r2_test_w** = 0.9507 vs **Model I** 0.9702
* **Baseline D mae_w** = 30.05 vs **Model I** 8.53
* **Baseline D rmse_w** = 124.60 vs **Model I** 73.94
* **Baseline D tail_bias** = -135.24 vs **Model I** -0.63
* **Baseline D tail_pred_to_y_ratio** = 0.9201 vs **Model I** 0.9996

This is fascinating. Baseline D has slightly higher plain R², but Model I is clearly better on MAE, RMSE, weighted metrics, tail calibration, tail bias, and tail SSE share. So Model I appears to be a more robust and useful no-lag learned model, even if plain R² alone does not capture that fully.

#### Take-home on no-lag side
If lag features are excluded from the feature set:
* **Best baseline:** Baseline D
* **Best learned model:** Model I
* **Interpretation:** Level-target no-lag ML does not beat the lookup ladder, but delta-target no-lag modeling does.

---

### All Baselines and Models in One Table



| Label | Type | Description | R² test | MAE | RMSE | R² test weighted | MAE weighted | RMSE weighted | Tail pred / y | Tail bias |
| :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- |
| **Baseline A** | Baseline | Lag only | 0.942565 | 5.707927 | 38.946548 | 0.929433 | 17.683063 | 109.379102 | 1.078064 | 111.758282 |
| **Baseline B** | Baseline | Lag + backoff | 0.930421 | 8.619110 | 55.014137 | 0.941156 | 27.599187 | 136.109130 | 0.975051 | -42.206414 |
| **Baseline C** | Baseline | PT + POS + state mean | 0.058866 | 72.994691 | 202.329669 | -0.007190 | 187.663518 | 563.107099 | 0.087527 | -1543.616931 |
| **Baseline D** | Baseline | Full backoff ladder | 0.931978 | 9.889925 | 54.394893 | 0.950689 | 30.051029 | 124.596617 | 0.920057 | -135.238878 |
| **Model A** | Model | wt1 with lags, level, grid | 0.768979 | 12.333052 | 100.244215 | 0.768979 | 12.333052 | 100.244215 | 0.810185 | -321.107964 |
| **Model B** | Model | wt1 without lags, level, grid | 0.760929 | 14.993705 | 101.975945 | 0.760929 | 14.993705 | 101.975945 | 0.753707 | -416.650689 |
| **Model C** | Model | wt10 with lags, level, grid | 0.804760 | 17.688952 | 92.154814 | 0.798161 | 41.802460 | 252.079967 | 0.868736 | -222.057183 |
| **Model D** | Model | wt10 without lags, level, grid | 0.886899 | 13.867930 | 70.140176 | 0.920303 | 30.880678 | 158.400588 | 0.922783 | -130.626763 |
| **Model E** | Model | wt10 with lags, level, rs50 | 0.725958 | 21.165937 | 109.179882 | 0.708240 | 50.903769 | 303.073671 | 0.788339 | -358.064019 |
| **Model F** | Model | wt10 without lags, level, rs50 | 0.789285 | 19.294453 | 95.737380 | 0.804227 | 43.422870 | 248.262741 | 0.864591 | -229.068779 |
| **Model G** | Model | wt10 with lags, delta, grid | 0.933416 | 3.493940 | 43.006421 | 0.975015 | 6.752066 | 67.723853 | 0.995876 | -5.894139 |
| **Model H** | Model | wt10 with lags, delta, rs50 | 0.930269 | 3.941185 | 44.010877 | 0.972997 | 7.393144 | 70.405245 | 0.992898 | -10.149489 |
| **Model I** | Model | wt10 without lags, delta, rs50 | 0.904079 | 4.632643 | 51.618434 | 0.970215 | 8.527295 | 73.942416 | 0.999560 | -0.628872 |

# Benchmarking Version 1

## 1. Unified expected-cost scoring pipeline

In [ ]:
import numpy as np
import pandas as pd

# ============================================================
# Unified expected-cost scoring pipeline
# Hot start  -> Model G
# Cold start -> Model I
# Includes benchmarking signals, support metadata, and flags
# ============================================================

# ------------------------------------------------------------
# Required objects assumed to already exist in memory
# ------------------------------------------------------------
# - eda_hcpcs_df
# - best_newprov_model_wt10_with_lags_delta_trgt
# - newprov_pkg_wt10_with_lags_delta_trgt
# - best_newprov_model_wt10_without_lags_delta_trgt_rs50
# - newprov_pkg_wt10_without_lags_delta_trgt_rs50

# ------------------------------------------------------------
# Core column names
# ------------------------------------------------------------
target_col = "avg_mdcr_stdzd_amt"
lag_col = "lag1_avg_amt"
tail_flag_col = "is_top_1pct_avg_mdcr_stdzd_amt"

group_cols = ["provider_type", "HCPCS_Cd", "Place_Of_Srvc"]
pt_pos_state_cols = ["provider_type", "Place_Of_Srvc", "state"]
pt_pos_cols = ["provider_type", "Place_Of_Srvc"]
pt_cols = ["provider_type"]

# ------------------------------------------------------------
# 1) Start from full row-level source data
# ------------------------------------------------------------
benchmark_df = eda_hcpcs_df.copy()
benchmark_df["_orig_idx"] = benchmark_df.index

# ------------------------------------------------------------
# 2) Explicit identity/context columns
#    These are already in the dataframe, but this section makes
#    the intended benchmark inputs crystal clear.
# ------------------------------------------------------------
identity_cols = [
    "Rndrng_NPI",
    "HCPCS_Cd",
    "provider_type",
    "Place_Of_Srvc",
    "state",
    "Year",
]

# Optional support/context columns we want to keep visible
support_context_cols = [
    lag_col,
    "services",
    "benes",
    "bene_day_services",
]

# ------------------------------------------------------------
# 3) Define routing: hot start vs cold start
# ------------------------------------------------------------
benchmark_df["has_lag"] = (
    benchmark_df[lag_col].notna()
    & (benchmark_df[lag_col] >= 0)
)

mask_hot = benchmark_df["has_lag"]
mask_cold = ~mask_hot

benchmark_df["expected_cost_source"] = np.where(
    mask_hot,
    "hot_start_model_G",
    "cold_start_model_I"
)

# ------------------------------------------------------------
# 4) Hot-start expected cost using Model G
#    Model G predicts log_delta_cost
# ------------------------------------------------------------
feature_cols_hot = newprov_pkg_wt10_with_lags_delta_trgt.feature_cols
X_hot = benchmark_df.loc[mask_hot, feature_cols_hot].copy()

log_delta_hat_hot = best_newprov_model_wt10_with_lags_delta_trgt.predict(X_hot)

benchmark_df.loc[mask_hot, "pred_log_delta_hot"] = log_delta_hat_hot

expected_cost_hot = np.expm1(
    np.log1p(benchmark_df.loc[mask_hot, lag_col].astype(float).to_numpy())
    + log_delta_hat_hot
)

# ------------------------------------------------------------
# 5) Build cold-start anchor ladder
#    This gives Model I a level anchor for converting predicted
#    log-delta into expected cost.
# ------------------------------------------------------------
train_years_for_anchor = [2021, 2022]

anchor_train_df = benchmark_df.loc[
    benchmark_df["Year"].isin(train_years_for_anchor)
].copy()

global_mean = float(anchor_train_df[target_col].mean())

anchor_mean_grp = (
    anchor_train_df.groupby(group_cols)[target_col]
    .mean()
    .rename("anchor_mean_grp")
    .reset_index()
)

anchor_mean_hcpcs = (
    anchor_train_df.groupby(["HCPCS_Cd"])[target_col]
    .mean()
    .rename("anchor_mean_hcpcs")
    .reset_index()
)

anchor_mean_pt_pos_state = (
    anchor_train_df.groupby(pt_pos_state_cols)[target_col]
    .mean()
    .rename("anchor_mean_pt_pos_state")
    .reset_index()
)

anchor_median_pt_pos_state = (
    anchor_train_df.groupby(pt_pos_state_cols)[target_col]
    .median()
    .rename("anchor_median_pt_pos_state")
    .reset_index()
)

anchor_mean_pt_pos = (
    anchor_train_df.groupby(pt_pos_cols)[target_col]
    .mean()
    .rename("anchor_mean_pt_pos")
    .reset_index()
)

anchor_mean_pt = (
    anchor_train_df.groupby(pt_cols)[target_col]
    .mean()
    .rename("anchor_mean_pt")
    .reset_index()
)

# Merge anchor tables back to full scoring dataframe
benchmark_df = benchmark_df.merge(anchor_mean_grp, on=group_cols, how="left")
benchmark_df = benchmark_df.merge(anchor_mean_hcpcs, on="HCPCS_Cd", how="left")
benchmark_df = benchmark_df.merge(anchor_mean_pt_pos_state, on=pt_pos_state_cols, how="left")
benchmark_df = benchmark_df.merge(anchor_median_pt_pos_state, on=pt_pos_state_cols, how="left")
benchmark_df = benchmark_df.merge(anchor_mean_pt_pos, on=pt_pos_cols, how="left")
benchmark_df = benchmark_df.merge(anchor_mean_pt, on=pt_cols, how="left")

# ------------------------------------------------------------
# 6) Build explicit cold-start anchor and anchor source
# ------------------------------------------------------------
cold_start_anchor = benchmark_df["anchor_mean_grp"].copy()
cold_start_anchor_source = pd.Series(index=benchmark_df.index, dtype="object")

cold_start_anchor_source[cold_start_anchor.notna()] = "grp_mean"

cold_start_anchor = cold_start_anchor.fillna(benchmark_df["anchor_mean_hcpcs"])
cold_start_anchor_source[
    cold_start_anchor_source.isna() & benchmark_df["anchor_mean_hcpcs"].notna()
] = "hcpcs_mean"

cold_start_anchor = cold_start_anchor.fillna(benchmark_df["anchor_mean_pt_pos_state"])
cold_start_anchor_source[
    cold_start_anchor_source.isna() & benchmark_df["anchor_mean_pt_pos_state"].notna()
] = "pt_pos_state_mean"

cold_start_anchor = cold_start_anchor.fillna(benchmark_df["anchor_median_pt_pos_state"])
cold_start_anchor_source[
    cold_start_anchor_source.isna() & benchmark_df["anchor_median_pt_pos_state"].notna()
] = "pt_pos_state_median"

cold_start_anchor = cold_start_anchor.fillna(benchmark_df["anchor_mean_pt_pos"])
cold_start_anchor_source[
    cold_start_anchor_source.isna() & benchmark_df["anchor_mean_pt_pos"].notna()
] = "pt_pos_mean"

cold_start_anchor = cold_start_anchor.fillna(benchmark_df["anchor_mean_pt"])
cold_start_anchor_source[
    cold_start_anchor_source.isna() & benchmark_df["anchor_mean_pt"].notna()
] = "pt_mean"

cold_start_anchor = cold_start_anchor.fillna(global_mean)
cold_start_anchor_source[cold_start_anchor_source.isna()] = "global_mean"

benchmark_df["cold_start_anchor"] = cold_start_anchor.astype(float)
benchmark_df["cold_start_anchor_source"] = cold_start_anchor_source

# ------------------------------------------------------------
# 7) Cold-start expected cost using Model I
#    Model I predicts log_delta_cost without lag features
# ------------------------------------------------------------
feature_cols_cold = newprov_pkg_wt10_without_lags_delta_trgt_rs50.feature_cols
X_cold = benchmark_df.loc[mask_cold, feature_cols_cold].copy()

log_delta_hat_cold = best_newprov_model_wt10_without_lags_delta_trgt_rs50.predict(X_cold)

benchmark_df.loc[mask_cold, "pred_log_delta_cold"] = log_delta_hat_cold

expected_cost_cold = np.expm1(
    np.log1p(benchmark_df.loc[mask_cold, "cold_start_anchor"].astype(float).to_numpy())
    + log_delta_hat_cold
)

# ------------------------------------------------------------
# 8) Combine into one expected_cost column
# ------------------------------------------------------------
benchmark_df["expected_cost"] = np.nan

benchmark_df.loc[mask_hot, "expected_cost"] = expected_cost_hot
benchmark_df.loc[mask_cold, "expected_cost"] = expected_cost_cold

benchmark_df["expected_cost"] = benchmark_df["expected_cost"].clip(lower=0)

# ------------------------------------------------------------
# 9) Explicit observed and benchmarking columns
# ------------------------------------------------------------
benchmark_df["observed_cost"] = benchmark_df[target_col].astype(float)

benchmark_df["residual"] = benchmark_df["observed_cost"] - benchmark_df["expected_cost"]
benchmark_df["abs_residual"] = benchmark_df["residual"].abs()

benchmark_df["oe_ratio"] = (
    benchmark_df["observed_cost"] / (benchmark_df["expected_cost"] + 1e-9)
)

benchmark_df["pct_diff"] = (
    benchmark_df["residual"] / (benchmark_df["expected_cost"] + 1e-9)
)

benchmark_df["log_oe"] = (
    np.log1p(benchmark_df["observed_cost"])
    - np.log1p(benchmark_df["expected_cost"])
)

# ------------------------------------------------------------
# 10) Support / confidence tier
#     This is intentionally explicit and interpretable.
# ------------------------------------------------------------
benchmark_df["expected_cost_support_tier"] = np.select(
    [
        benchmark_df["has_lag"],

        (~benchmark_df["has_lag"]) & benchmark_df["cold_start_anchor_source"].isin([
            "grp_mean", "hcpcs_mean"
        ]),

        (~benchmark_df["has_lag"]) & benchmark_df["cold_start_anchor_source"].isin([
            "pt_pos_state_mean", "pt_pos_state_median", "pt_pos_mean"
        ]),

        (~benchmark_df["has_lag"]) & benchmark_df["cold_start_anchor_source"].isin([
            "pt_mean", "global_mean"
        ]),
    ],
    [
        "high",
        "medium_high",
        "medium",
        "low",
    ],
    default="unknown"
)

# ------------------------------------------------------------
# 11) Optional anomaly / prioritization flags
#     Thresholds are simple and explicit so they are easy to
#     inspect and later refine.
# ------------------------------------------------------------

# High positive residual: observed materially exceeds expected
benchmark_df["high_positive_residual"] = (
    benchmark_df["residual"] > benchmark_df["residual"].quantile(0.99)
)

# High O/E ratio
benchmark_df["high_oe"] = (
    benchmark_df["oe_ratio"] > benchmark_df["oe_ratio"].quantile(0.99)
)

# Extreme observed cost outlier
benchmark_df["extreme_cost_outlier"] = (
    benchmark_df["observed_cost"] > benchmark_df["observed_cost"].quantile(0.99)
)

# High-confidence anomaly candidate:
# high support + large positive residual + high O/E
benchmark_df["high_confidence_anomaly_candidate"] = (
    benchmark_df["expected_cost_support_tier"].isin(["high", "medium_high"])
    & benchmark_df["high_positive_residual"]
    & benchmark_df["high_oe"]
)

# ------------------------------------------------------------
# 12) Final column ordering for benchmarking table
# ------------------------------------------------------------
final_cols = [
    # Identity/context
    "Rndrng_NPI",
    "HCPCS_Cd",
    "provider_type",
    "Place_Of_Srvc",
    "state",
    "Year",

    # Observed / expected
    "observed_cost",
    "expected_cost",
    "expected_cost_source",

    # Benchmarking signals
    "residual",
    "abs_residual",
    "oe_ratio",
    "pct_diff",
    "log_oe",

    # Support / reliability metadata
    "lag1_avg_amt",
    "has_lag",
    "services",
    "benes",
    "bene_day_services",
    "cold_start_anchor",
    "cold_start_anchor_source",
    "expected_cost_support_tier",

    # Optional flags
    "high_positive_residual",
    "high_oe",
    "extreme_cost_outlier",
    "high_confidence_anomaly_candidate",
]

# Keep only columns that actually exist
final_cols = [c for c in final_cols if c in benchmark_df.columns]

# Reorder so the benchmark table is easy to inspect
other_cols = [c for c in benchmark_df.columns if c not in final_cols]
benchmark_df = benchmark_df[final_cols + other_cols]

# ------------------------------------------------------------
# 13) Sanity checks
# ------------------------------------------------------------
print("Rows:", len(benchmark_df))
print("Expected cost NaN rate:", benchmark_df["expected_cost"].isna().mean())
print("Expected cost negative rate:", (benchmark_df["expected_cost"] < 0).mean())

print("\nExpected-cost source breakdown:")
print(benchmark_df["expected_cost_source"].value_counts(dropna=False))

print("\nSupport tier breakdown:")
print(benchmark_df["expected_cost_support_tier"].value_counts(dropna=False))

print("\nCold-start anchor breakdown:")
print(benchmark_df.loc[~benchmark_df["has_lag"], "cold_start_anchor_source"].value_counts(dropna=False))

benchmark_df.head()

In [ ]:
with pd.option_context('display.max_columns', None):
    display(benchmark_df.head())

## 2. A compact evaluation block for the unified engine

In [ ]:
from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error
import numpy as np
import pandas as pd

target_col = "observed_cost"
pred_col = "expected_cost"
tail_flag_col = "is_top_1pct_avg_mdcr_stdzd_amt"
w_tail = 10

eval_df = benchmark_df.copy()

y_true = eval_df[target_col].astype(float)
pred = eval_df[pred_col].astype(float)

is_tail = eval_df[tail_flag_col].astype(bool)
is_non_tail = ~is_tail

sample_w = (1 + (w_tail - 1) * eval_df[tail_flag_col].astype(int).to_numpy()).astype(float)

eval_df["abs_err"] = (y_true - pred).abs()
eval_df["sq_err"] = (y_true - pred) ** 2

r2_test = r2_score(y_true, pred)
mae = mean_absolute_error(y_true, pred)
rmse = root_mean_squared_error(y_true, pred)

pred_clip = pred.clip(lower=0)
r2_test_log = r2_score(np.log1p(y_true), np.log1p(pred_clip))
mae_log = mean_absolute_error(np.log1p(y_true), np.log1p(pred_clip))
rmse_log = root_mean_squared_error(np.log1p(y_true), np.log1p(pred_clip))

r2_test_w = r2_score(y_true, pred, sample_weight=sample_w)
mae_w = mean_absolute_error(y_true, pred, sample_weight=sample_w)
rmse_w = root_mean_squared_error(y_true, pred, sample_weight=sample_w)

non_tail_pred_to_y_ratio = eval_df.loc[is_non_tail, pred_col].mean() / eval_df.loc[is_non_tail, target_col].mean()
tail_pred_to_y_ratio = eval_df.loc[is_tail, pred_col].mean() / eval_df.loc[is_tail, target_col].mean()

non_tail_sse = eval_df.loc[is_non_tail, "sq_err"].sum()
tail_sse = eval_df.loc[is_tail, "sq_err"].sum()

non_tail_share_of_total_sse = non_tail_sse / (non_tail_sse + tail_sse)
tail_share_of_total_sse = tail_sse / (non_tail_sse + tail_sse)

non_tail_under_prediction_rate = (
    (eval_df.loc[is_non_tail, pred_col] < eval_df.loc[is_non_tail, target_col]).mean()
)
tail_under_prediction_rate = (
    (eval_df.loc[is_tail, pred_col] < eval_df.loc[is_tail, target_col]).mean()
)

non_tail_bias = eval_df.loc[is_non_tail, pred_col].mean() - eval_df.loc[is_non_tail, target_col].mean()
tail_bias = eval_df.loc[is_tail, pred_col].mean() - eval_df.loc[is_tail, target_col].mean()

unified_expected_cost_metrics = pd.DataFrame({
    "w_tail": [w_tail],
    "r2_test": [r2_test],
    "mae": [mae],
    "rmse": [rmse],
    "r2_test_log": [r2_test_log],
    "mae_log": [mae_log],
    "rmse_log": [rmse_log],
    "r2_test_w": [r2_test_w],
    "mae_w": [mae_w],
    "rmse_w": [rmse_w],
    "non_tail_pred_to_y_ratio": [non_tail_pred_to_y_ratio],
    "tail_pred_to_y_ratio": [tail_pred_to_y_ratio],
    "non_tail_sse": [non_tail_sse],
    "tail_sse": [tail_sse],
    "non_tail_share_of_total_sse": [non_tail_share_of_total_sse],
    "tail_share_of_total_sse": [tail_share_of_total_sse],
    "non_tail_under_prediction_rate": [non_tail_under_prediction_rate],
    "tail_under_prediction_rate": [tail_under_prediction_rate],
    "non_tail_bias": [non_tail_bias],
    "tail_bias": [tail_bias],
})

with pd.option_context("display.max_columns", None):
    display(unified_expected_cost_metrics)

## 3. Useful ranking views for downstream benchmarking

#### A. Top rows above expected by dollar residual

In [ ]:
top_over_expected_by_residual = (
    benchmark_df.sort_values("residual", ascending=False)
    [[
        "Rndrng_NPI", "Year", "HCPCS_Cd", "provider_type", "Place_Of_Srvc", "state",
        "expected_cost_source", "cold_start_anchor_source",
        "observed_cost", "expected_cost", "residual", "oe_ratio", "pct_diff"
    ]]
    .head(50)
)

top_over_expected_by_residual

#### B. Top rows above expected by O/E ratio

In [ ]:
top_over_expected_by_oe = (
    benchmark_df.loc[benchmark_df["expected_cost"] > 0]
    .sort_values("oe_ratio", ascending=False)
    [[
        "Rndrng_NPI", "Year", "HCPCS_Cd", "provider_type", "Place_Of_Srvc", "state",
        "expected_cost_source", "cold_start_anchor_source",
        "observed_cost", "expected_cost", "residual", "oe_ratio", "pct_diff"
    ]]
    .head(50)
)

top_over_expected_by_oe

#### C. Provider-level aggregation

In [ ]:
provider_benchmark_summary = (
    benchmark_df.groupby("Rndrng_NPI", as_index=False)
    .agg(
        n_rows=("Rndrng_NPI", "size"),
        observed_total=("observed_cost", "sum"),
        expected_total=("expected_cost", "sum"),
        residual_total=("residual", "sum"),
        abs_residual_total=("abs_residual", "sum"),
        mean_oe_ratio=("oe_ratio", "mean")
    )
)

provider_benchmark_summary["provider_oe_ratio"] = (
    provider_benchmark_summary["observed_total"]
    / (provider_benchmark_summary["expected_total"] + 1e-9)
)

provider_benchmark_summary.sort_values("residual_total", ascending=False).head(50)

# Performances of Model G on lag-present and Model I on lag-present and lag-absent rows of benchmarking dataset

## 1) Rebuild comparable predictions on benchmark_df

Run this first. It creates explicit prediction columns for:
- Model G on hot rows
- Model I on hot rows
- Model I on cold rows

In [ ]:
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# Assumes benchmark_df already exists from your unified pipeline
# and includes:
# - has_lag
# - lag1_avg_amt
# - cold_start_anchor
# - avg_mdcr_stdzd_amt
# - is_top_1pct_avg_mdcr_stdzd_amt
# ------------------------------------------------------------

target_col = "avg_mdcr_stdzd_amt"
lag_col = "lag1_avg_amt"
tail_flag_col = "is_top_1pct_avg_mdcr_stdzd_amt"

mask_hot = benchmark_df["has_lag"].astype(bool)
mask_cold = ~mask_hot

# ------------------------------------------------------------
# A) Model G predictions on hot rows
# ------------------------------------------------------------
feature_cols_G = newprov_pkg_wt10_with_lags_delta_trgt.feature_cols
X_hot_G = benchmark_df.loc[mask_hot, feature_cols_G].copy()

pred_log_delta_G_hot = best_newprov_model_wt10_with_lags_delta_trgt.predict(X_hot_G)

pred_level_G_hot = np.expm1(
    np.log1p(benchmark_df.loc[mask_hot, lag_col].astype(float).to_numpy())
    + pred_log_delta_G_hot
)

benchmark_df.loc[mask_hot, "pred_log_delta_G_hot"] = pred_log_delta_G_hot
benchmark_df.loc[mask_hot, "pred_level_G_hot"] = pred_level_G_hot

# ------------------------------------------------------------
# B) Model I predictions on hot rows
#    Important: Model I does not use lag as an input feature,
#    but for this comparison we still reconstruct with TRUE lag
#    because these rows do have lag.
# ------------------------------------------------------------
feature_cols_I = newprov_pkg_wt10_without_lags_delta_trgt_rs50.feature_cols
X_hot_I = benchmark_df.loc[mask_hot, feature_cols_I].copy()

pred_log_delta_I_hot = best_newprov_model_wt10_without_lags_delta_trgt_rs50.predict(X_hot_I)

pred_level_I_hot = np.expm1(
    np.log1p(benchmark_df.loc[mask_hot, lag_col].astype(float).to_numpy())
    + pred_log_delta_I_hot
)

benchmark_df.loc[mask_hot, "pred_log_delta_I_hot"] = pred_log_delta_I_hot
benchmark_df.loc[mask_hot, "pred_level_I_hot"] = pred_level_I_hot

# ------------------------------------------------------------
# C) Model I predictions on cold rows
#    True production-style cold-start reconstruction:
#    anchor ladder + predicted delta
# ------------------------------------------------------------
X_cold_I = benchmark_df.loc[mask_cold, feature_cols_I].copy()

pred_log_delta_I_cold = best_newprov_model_wt10_without_lags_delta_trgt_rs50.predict(X_cold_I)

pred_level_I_cold = np.expm1(
    np.log1p(benchmark_df.loc[mask_cold, "cold_start_anchor"].astype(float).to_numpy())
    + pred_log_delta_I_cold
)

benchmark_df.loc[mask_cold, "pred_log_delta_I_cold"] = pred_log_delta_I_cold
benchmark_df.loc[mask_cold, "pred_level_I_cold"] = pred_level_I_cold

print("Hot rows:", mask_hot.sum())
print("Cold rows:", mask_cold.sum())

## 2) Helper to evaluate any prediction column with your familiar schema

In [ ]:
from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error
import numpy as np
import pandas as pd

def evaluate_level_predictions_same_schema(
    df,
    pred_col,
    *,
    train_sample_weights_label=np.nan,
    target_col="avg_mdcr_stdzd_amt",
    tail_flag_col="is_top_1pct_avg_mdcr_stdzd_amt",
    w_tail=10,
):
    eval_df = df.copy()

    y_true = eval_df[target_col].astype(float)
    pred = eval_df[pred_col].astype(float)

    is_tail = eval_df[tail_flag_col].astype(bool)
    is_non_tail = ~is_tail

    sample_w = (1 + (w_tail - 1) * eval_df[tail_flag_col].astype(int).to_numpy()).astype(float)

    eval_df["pred"] = pred
    eval_df["abs_err"] = (y_true - pred).abs()
    eval_df["sq_err"] = (y_true - pred) ** 2

    r2_test = r2_score(y_true, pred)
    mae = mean_absolute_error(y_true, pred)
    rmse = root_mean_squared_error(y_true, pred)

    pred_clip = pred.clip(lower=0)
    r2_test_log = r2_score(np.log1p(y_true), np.log1p(pred_clip))
    mae_log = mean_absolute_error(np.log1p(y_true), np.log1p(pred_clip))
    rmse_log = root_mean_squared_error(np.log1p(y_true), np.log1p(pred_clip))

    r2_test_w = r2_score(y_true, pred, sample_weight=sample_w)
    mae_w = mean_absolute_error(y_true, pred, sample_weight=sample_w)
    rmse_w = root_mean_squared_error(y_true, pred, sample_weight=sample_w)

    non_tail_pred_to_y_ratio = (
        eval_df.loc[is_non_tail, "pred"].mean()
        / eval_df.loc[is_non_tail, target_col].mean()
    )
    tail_pred_to_y_ratio = (
        eval_df.loc[is_tail, "pred"].mean()
        / eval_df.loc[is_tail, target_col].mean()
    )

    non_tail_sse = eval_df.loc[is_non_tail, "sq_err"].sum()
    tail_sse = eval_df.loc[is_tail, "sq_err"].sum()

    non_tail_share_of_total_sse = non_tail_sse / (non_tail_sse + tail_sse)
    tail_share_of_total_sse = tail_sse / (non_tail_sse + tail_sse)

    non_tail_under_prediction_rate = (
        (eval_df.loc[is_non_tail, "pred"] < eval_df.loc[is_non_tail, target_col]).mean()
    )
    tail_under_prediction_rate = (
        (eval_df.loc[is_tail, "pred"] < eval_df.loc[is_tail, target_col]).mean()
    )

    non_tail_bias = (
        eval_df.loc[is_non_tail, "pred"].mean()
        - eval_df.loc[is_non_tail, target_col].mean()
    )
    tail_bias = (
        eval_df.loc[is_tail, "pred"].mean()
        - eval_df.loc[is_tail, target_col].mean()
    )

    return pd.DataFrame({
        "train_sample_weights": [train_sample_weights_label],
        "rows_eval": [len(eval_df)],
        "r2_train": [np.nan],
        "r2_test": [r2_test],
        "mae": [mae],
        "rmse": [rmse],
        "r2_train_log": [np.nan],
        "r2_test_log": [r2_test_log],
        "mae_log": [mae_log],
        "rmse_log": [rmse_log],
        "r2_train_w": [np.nan],
        "r2_test_w": [r2_test_w],
        "mae_w": [mae_w],
        "rmse_w": [rmse_w],
        "non_tail_pred_to_y_ratio": [non_tail_pred_to_y_ratio],
        "tail_pred_to_y_ratio": [tail_pred_to_y_ratio],
        "non_tail_sse": [non_tail_sse],
        "tail_sse": [tail_sse],
        "non_tail_share_of_total_sse": [non_tail_share_of_total_sse],
        "tail_share_of_total_sse": [tail_share_of_total_sse],
        "non_tail_under_prediction_rate": [non_tail_under_prediction_rate],
        "tail_under_prediction_rate": [tail_under_prediction_rate],
        "non_tail_bias": [non_tail_bias],
        "tail_bias": [tail_bias],
    })

## 3) Evaluate the three comparisons 

In [ ]:
# ------------------------------------------------------------
# 1) Model G on lag-present rows
# ------------------------------------------------------------
metrics_G_on_hot = evaluate_level_predictions_same_schema(
    benchmark_df.loc[benchmark_df["has_lag"]].copy(),
    pred_col="pred_level_G_hot",
    train_sample_weights_label=10,
    target_col="avg_mdcr_stdzd_amt",
    tail_flag_col="is_top_1pct_avg_mdcr_stdzd_amt",
    w_tail=10,
)
metrics_G_on_hot.insert(0, "evaluation_name", "Model_G_on_hot_rows")

# ------------------------------------------------------------
# 2) Model I on lag-present rows
# ------------------------------------------------------------
metrics_I_on_hot = evaluate_level_predictions_same_schema(
    benchmark_df.loc[benchmark_df["has_lag"]].copy(),
    pred_col="pred_level_I_hot",
    train_sample_weights_label=10,
    target_col="avg_mdcr_stdzd_amt",
    tail_flag_col="is_top_1pct_avg_mdcr_stdzd_amt",
    w_tail=10,
)
metrics_I_on_hot.insert(0, "evaluation_name", "Model_I_on_hot_rows")

# ------------------------------------------------------------
# 3) Model I on lag-absent rows
# ------------------------------------------------------------
metrics_I_on_cold = evaluate_level_predictions_same_schema(
    benchmark_df.loc[~benchmark_df["has_lag"]].copy(),
    pred_col="pred_level_I_cold",
    train_sample_weights_label=10,
    target_col="avg_mdcr_stdzd_amt",
    tail_flag_col="is_top_1pct_avg_mdcr_stdzd_amt",
    w_tail=10,
)
metrics_I_on_cold.insert(0, "evaluation_name", "Model_I_on_cold_rows")

# ------------------------------------------------------------
# Combine for direct comparison
# ------------------------------------------------------------
model_GI_benchmark_segment_comparison = pd.concat(
    [metrics_G_on_hot, metrics_I_on_hot, metrics_I_on_cold],
    ignore_index=True,
)

with pd.option_context("display.max_columns", None):
    display(model_GI_benchmark_segment_comparison)

## 4) Add direct deltas for the hot-row comparison

This is the cleanest way to compare Model G vs Model I on the same lag-present rows.

In [ ]:
compare_hot = model_GI_benchmark_segment_comparison.set_index("evaluation_name")

hot_delta_summary = pd.DataFrame({
    "metric": [
        "r2_test",
        "mae",
        "rmse",
        "r2_test_w",
        "mae_w",
        "rmse_w",
        "tail_pred_to_y_ratio",
        "tail_under_prediction_rate",
        "tail_bias",
    ],
    "Model_G_minus_Model_I_on_hot": [
        compare_hot.loc["Model_G_on_hot_rows", "r2_test"] - compare_hot.loc["Model_I_on_hot_rows", "r2_test"],
        compare_hot.loc["Model_G_on_hot_rows", "mae"] - compare_hot.loc["Model_I_on_hot_rows", "mae"],
        compare_hot.loc["Model_G_on_hot_rows", "rmse"] - compare_hot.loc["Model_I_on_hot_rows", "rmse"],
        compare_hot.loc["Model_G_on_hot_rows", "r2_test_w"] - compare_hot.loc["Model_I_on_hot_rows", "r2_test_w"],
        compare_hot.loc["Model_G_on_hot_rows", "mae_w"] - compare_hot.loc["Model_I_on_hot_rows", "mae_w"],
        compare_hot.loc["Model_G_on_hot_rows", "rmse_w"] - compare_hot.loc["Model_I_on_hot_rows", "rmse_w"],
        compare_hot.loc["Model_G_on_hot_rows", "tail_pred_to_y_ratio"] - compare_hot.loc["Model_I_on_hot_rows", "tail_pred_to_y_ratio"],
        compare_hot.loc["Model_G_on_hot_rows", "tail_under_prediction_rate"] - compare_hot.loc["Model_I_on_hot_rows", "tail_under_prediction_rate"],
        compare_hot.loc["Model_G_on_hot_rows", "tail_bias"] - compare_hot.loc["Model_I_on_hot_rows", "tail_bias"],
    ]
})

with pd.option_context("display.max_columns", None):
    display(hot_delta_summary)

## 5) Compare source-specific performance inside cold rows

This is especially useful because our cold-start rows are not all equally difficult. Rows anchored by `grp_mean` are much easier than rows anchored by `pt_pos_state_mean` or anything weaker.

In [ ]:
cold_eval = benchmark_df.loc[~benchmark_df["has_lag"]].copy()

cold_anchor_breakdown = (
    cold_eval.groupby("cold_start_anchor_source")
    .apply(lambda g: pd.Series({
        "n": len(g),
        "mae": (g["avg_mdcr_stdzd_amt"] - g["pred_level_I_cold"]).abs().mean(),
        "rmse": np.sqrt(((g["avg_mdcr_stdzd_amt"] - g["pred_level_I_cold"]) ** 2).mean()),
        "mean_true": g["avg_mdcr_stdzd_amt"].mean(),
        "mean_pred": g["pred_level_I_cold"].mean(),
        "tail_rate": g["is_top_1pct_avg_mdcr_stdzd_amt"].mean(),
    }))
    .sort_values("n", ascending=False)
)

with pd.option_context("display.max_columns", None):
    display(cold_anchor_breakdown)

# Failure analysis

## A. Identify catastrophic predictions

In [ ]:
import numpy as np
import pandas as pd

# ============================================================
# A. Identify catastrophic prediction buckets
# ============================================================

# ------------------------------------------------------------
# 1) Start from benchmark_df
# ------------------------------------------------------------
failure_df = benchmark_df.copy()

# ------------------------------------------------------------
# 2) Define key thresholds
#    These are intentionally explicit and easy to inspect.
#    You can tighten or loosen them later.
# ------------------------------------------------------------

# Quantile-based thresholds
abs_resid_q99 = failure_df["abs_residual"].quantile(0.99)
oe_q99 = failure_df["oe_ratio"].quantile(0.99)
resid_pos_q99 = failure_df["residual"].quantile(0.99)
resid_neg_q01 = failure_df["residual"].quantile(0.01)

print("99th percentile abs_residual:", abs_resid_q99)
print("99th percentile oe_ratio:", oe_q99)
print("99th percentile residual:", resid_pos_q99)
print("1st percentile residual:", resid_neg_q01)

# ------------------------------------------------------------
# 3) Add intuitive boolean flags for catastrophic buckets
# ------------------------------------------------------------

# Very high absolute error
failure_df["cat_abs_residual_q99"] = (
    failure_df["abs_residual"] >= abs_resid_q99
)

# Very high O/E
failure_df["cat_oe_q99"] = (
    failure_df["oe_ratio"] >= oe_q99
)

# Very large positive residual
failure_df["cat_large_positive_residual"] = (
    failure_df["residual"] >= resid_pos_q99
)

# Very large negative residual
failure_df["cat_large_negative_residual"] = (
    failure_df["residual"] <= resid_neg_q01
)

# Large error on high-support rows
failure_df["cat_large_error_high_support"] = (
    failure_df["expected_cost_support_tier"].isin(["high", "medium_high"])
    & (failure_df["abs_residual"] >= abs_resid_q99)
)

# Large error within tail rows
failure_df["cat_large_error_tail_row"] = (
    failure_df["is_top_1pct_avg_mdcr_stdzd_amt"]
    & (failure_df["abs_residual"] >= abs_resid_q99)
)

# ------------------------------------------------------------
# 4) More operational / narrative buckets
# ------------------------------------------------------------

# Catastrophic underprediction:
# observed is very large and expected is far too low
failure_df["cat_underprediction"] = (
    (failure_df["observed_cost"] >= failure_df["observed_cost"].quantile(0.99))
    & (failure_df["oe_ratio"] >= 3.0)
)

# Catastrophic overprediction:
# expected is very large relative to observed
failure_df["cat_overprediction"] = (
    (failure_df["expected_cost"] >= failure_df["expected_cost"].quantile(0.99))
    & (failure_df["oe_ratio"] <= 1 / 3.0)
)

# High-confidence anomaly candidate:
# already defined in your benchmark_df, but we keep a dedicated catastrophic flag too
failure_df["cat_high_conf_anomaly"] = (
    failure_df["high_confidence_anomaly_candidate"]
)

# Anchor failure:
# cold-start row + weaker support tier + large miss
failure_df["cat_anchor_failure"] = (
    (~failure_df["has_lag"])
    & failure_df["expected_cost_support_tier"].isin(["medium", "low"])
    & (failure_df["abs_residual"] >= abs_resid_q99)
)

# ------------------------------------------------------------
# 5) Create one broad catastrophic flag
#    This is useful for downstream filtering
# ------------------------------------------------------------
cat_cols = [
    "cat_abs_residual_q99",
    "cat_oe_q99",
    "cat_large_positive_residual",
    "cat_large_negative_residual",
    "cat_large_error_high_support",
    "cat_large_error_tail_row",
    "cat_underprediction",
    "cat_overprediction",
    "cat_high_conf_anomaly",
    "cat_anchor_failure",
]

failure_df["is_any_catastrophic"] = failure_df[cat_cols].any(axis=1)

# ------------------------------------------------------------
# 6) Quick bucket counts
# ------------------------------------------------------------
catastrophic_bucket_counts = pd.DataFrame({
    "bucket": cat_cols + ["is_any_catastrophic"],
    "n_rows": [failure_df[c].sum() for c in cat_cols + ["is_any_catastrophic"]],
    "share_of_all_rows": [failure_df[c].mean() for c in cat_cols + ["is_any_catastrophic"]],
})

with pd.option_context("display.max_rows", None, "display.max_columns", None):
    display(catastrophic_bucket_counts.sort_values("n_rows", ascending=False))

# ------------------------------------------------------------
# 7) Preview catastrophic rows
# ------------------------------------------------------------
catastrophic_preview = failure_df.loc[
    failure_df["is_any_catastrophic"],
    [
        "Rndrng_NPI", "Year", "HCPCS_Cd", "provider_type", "Place_Of_Srvc", "state",
        "observed_cost", "expected_cost", "residual", "abs_residual", "oe_ratio",
        "expected_cost_source", "cold_start_anchor_source", "expected_cost_support_tier",
        "high_confidence_anomaly_candidate"
    ] + cat_cols
].sort_values("abs_residual", ascending=False)

with pd.option_context("display.max_columns", None):
    display(catastrophic_preview.head(50))

## B. Split failures by route

In [ ]:
import numpy as np
import pandas as pd

# ============================================================
# B. Split catastrophic failures by route
# ============================================================

# ------------------------------------------------------------
# 1) Add route labels
# ------------------------------------------------------------
failure_df = failure_df.copy()

failure_df["route"] = np.where(
    failure_df["has_lag"],
    "hot_start",
    "cold_start"
)

# For hot rows, anchor source is not the actual prediction route
# but we keep it for context if present.
failure_df["cold_anchor_route"] = np.where(
    failure_df["route"] == "cold_start",
    failure_df["cold_start_anchor_source"],
    np.nan
)

# ------------------------------------------------------------
# 2) Restrict to catastrophic rows
# ------------------------------------------------------------
cat_df = failure_df.loc[failure_df["is_any_catastrophic"]].copy()

print("Total catastrophic rows:", len(cat_df))

# ------------------------------------------------------------
# 3) Split by hot-start vs cold-start
# ------------------------------------------------------------
cat_by_route = (
    cat_df.groupby("route", dropna=False)
    .agg(
        n_rows=("route", "size"),
        mean_abs_residual=("abs_residual", "mean"),
        mean_oe_ratio=("oe_ratio", "mean"),
        tail_rate=("is_top_1pct_avg_mdcr_stdzd_amt", "mean"),
    )
    .reset_index()
)

cat_by_route["share_within_catastrophic"] = (
    cat_by_route["n_rows"] / cat_by_route["n_rows"].sum()
)

with pd.option_context("display.max_columns", None):
    display(cat_by_route.sort_values("n_rows", ascending=False))

# ------------------------------------------------------------
# 4) Within cold-start, split by anchor source
# ------------------------------------------------------------
cat_cold_by_anchor = (
    cat_df.loc[cat_df["route"] == "cold_start"]
    .groupby("cold_start_anchor_source", dropna=False)
    .agg(
        n_rows=("cold_start_anchor_source", "size"),
        mean_abs_residual=("abs_residual", "mean"),
        median_abs_residual=("abs_residual", "median"),
        mean_oe_ratio=("oe_ratio", "mean"),
        tail_rate=("is_top_1pct_avg_mdcr_stdzd_amt", "mean"),
    )
    .reset_index()
)

if len(cat_cold_by_anchor) > 0:
    cat_cold_by_anchor["share_within_cold_catastrophic"] = (
        cat_cold_by_anchor["n_rows"] / cat_cold_by_anchor["n_rows"].sum()
    )

with pd.option_context("display.max_columns", None):
    display(cat_cold_by_anchor.sort_values("n_rows", ascending=False))

# ------------------------------------------------------------
# 5) Cross-tab: route x support tier
# ------------------------------------------------------------
cat_route_support = pd.crosstab(
    cat_df["route"],
    cat_df["expected_cost_support_tier"],
    margins=True
)

display(cat_route_support)

# ------------------------------------------------------------
# 6) Cross-tab: route x specific catastrophic bucket
# ------------------------------------------------------------
cat_route_bucket_summary = (
    cat_df.groupby("route")[cat_cols]
    .sum()
    .T
)

display(cat_route_bucket_summary)

## C. Look at peers of catastrophic rows

In [ ]:
import numpy as np
import pandas as pd

# ============================================================
# C. Build peer-set context for catastrophic rows (VECTORIZED)
# ============================================================

# ------------------------------------------------------------
# 1) Define reference history table
# ------------------------------------------------------------
peer_history_df = eda_hcpcs_df.copy()
peer_history_df["_peer_row_id"] = peer_history_df.index

# ------------------------------------------------------------
# 2) Take catastrophic rows only
# ------------------------------------------------------------
cat_df = failure_df.loc[failure_df["is_any_catastrophic"]].copy()
cat_df["_peer_row_id"] = cat_df.index

print("Catastrophic rows to profile:", len(cat_df))

# ------------------------------------------------------------
# 3) Base info to carry through every peer view
# ------------------------------------------------------------
base_cols = [
    "_peer_row_id",
    "Rndrng_NPI",
    "Year",
    "HCPCS_Cd",
    "provider_type",
    "Place_Of_Srvc",
    "state",
    "observed_cost",
    "expected_cost",
    "residual",
    "abs_residual",
    "oe_ratio",
    "expected_cost_source",
    "cold_start_anchor_source",
    "expected_cost_support_tier",
]

# ------------------------------------------------------------
# 4) Define peer sets
# ------------------------------------------------------------
peer_key_sets = [
    ["HCPCS_Cd"],
    ["provider_type", "HCPCS_Cd", "Place_Of_Srvc"],
    ["provider_type", "Place_Of_Srvc", "state"],
    ["provider_type"],
]

# ------------------------------------------------------------
# 5) Helper: build one peer summary table via groupby + merge
# ------------------------------------------------------------
def build_peer_summary_vectorized(
    cat_df: pd.DataFrame,
    peer_history_df: pd.DataFrame,
    key_cols: list[str],
    target_col: str = "avg_mdcr_stdzd_amt",
) -> pd.DataFrame:
    """
    Vectorized peer summary:
    - compute group stats once on full peer_history_df
    - merge onto catastrophic rows
    - exclude self-row approximately by subtracting 1 from peer_n
      when the catastrophic row is also present in peer_history_df
    """

    # Group-level summaries from full peer history
    grp = (
        peer_history_df.groupby(key_cols)[target_col]
        .agg(
            peer_n_raw="size",
            peer_mean="mean",
            peer_std="std",
            peer_min="min",
            peer_q25=lambda s: s.quantile(0.25),
            peer_median="median",
            peer_q75=lambda s: s.quantile(0.75),
            peer_max="max",
        )
        .reset_index()
    )

    # De-duplicate columns while preserving order
    select_cols = list(dict.fromkeys(base_cols + key_cols))

    # Start from catastrophic rows
    out = cat_df[select_cols].copy()

    # Merge group summaries onto catastrophic rows
    out = out.merge(grp, on=key_cols, how="left")

    # Did this catastrophic row itself exist in peer_history_df?
    out["self_in_history"] = out["_peer_row_id"].isin(peer_history_df["_peer_row_id"])

    # Exclude the row itself when possible
    out["peer_n"] = out["peer_n_raw"] - out["self_in_history"].astype(int)

    # If excluding self leaves zero peers, blank the summary stats
    stat_cols = [
        "peer_mean",
        "peer_std",
        "peer_min",
        "peer_q25",
        "peer_median",
        "peer_q75",
        "peer_max",
    ]

    no_peers_mask = out["peer_n"] <= 0
    out.loc[no_peers_mask, stat_cols] = np.nan
    out.loc[no_peers_mask, "peer_n"] = 0

    # Add peer key label
    out["peer_key"] = " + ".join(key_cols)

    # Keep final columns tidy
    out = out[
        base_cols
        + ["peer_key", "peer_n"]
        + stat_cols
    ]

    return out

# ------------------------------------------------------------
# 6) Build all peer summaries and stack together
# ------------------------------------------------------------
peer_summary_tables = []

for key_cols in peer_key_sets:
    peer_summary_tables.append(
        build_peer_summary_vectorized(
            cat_df=cat_df,
            peer_history_df=peer_history_df,
            key_cols=key_cols,
            target_col="avg_mdcr_stdzd_amt",
        )
    )

cat_peer_context = pd.concat(peer_summary_tables, ignore_index=True)

with pd.option_context("display.max_columns", None):
    display(cat_peer_context.head(20))

## D. Compare catastrophic rows against peer distributions

#### D1. Safe vectorized peer-distribution summary for all catastrophic rows

In [ ]:
import numpy as np
import pandas as pd

# ============================================================
# D1. Compare catastrophic rows against peer distributions
#     SAFE VECTORIZE VERSION (NO FULL PERCENTILE JOIN)
# ============================================================

peer_history_df = eda_hcpcs_df.copy()
peer_history_df["_peer_row_id"] = peer_history_df.index

cat_df = failure_df.loc[failure_df["is_any_catastrophic"]].copy()
cat_df["_peer_row_id"] = cat_df.index

print("Catastrophic rows to profile:", len(cat_df))

base_cols = [
    "_peer_row_id",
    "Rndrng_NPI",
    "Year",
    "HCPCS_Cd",
    "provider_type",
    "Place_Of_Srvc",
    "state",
    "observed_cost",
    "expected_cost",
    "residual",
    "abs_residual",
    "oe_ratio",
    "expected_cost_source",
    "cold_start_anchor_source",
    "expected_cost_support_tier",
]

peer_key_sets = [
    ["HCPCS_Cd"],
    ["provider_type", "HCPCS_Cd", "Place_Of_Srvc"],
    ["provider_type", "Place_Of_Srvc", "state"],
    ["provider_type"],
]

def build_peer_distribution_context_safe(
    cat_df: pd.DataFrame,
    peer_history_df: pd.DataFrame,
    key_cols: list[str],
    target_col: str = "avg_mdcr_stdzd_amt",
) -> pd.DataFrame:
    # Group summary stats on full history
    grp_stats = (
        peer_history_df.groupby(key_cols)[target_col]
        .agg(
            peer_n_raw="size",
            peer_mean="mean",
            peer_median="median",
            peer_q25=lambda s: s.quantile(0.25),
            peer_q75=lambda s: s.quantile(0.75),
            peer_p90=lambda s: s.quantile(0.90),
            peer_p95=lambda s: s.quantile(0.95),
            peer_p99=lambda s: s.quantile(0.99),
        )
        .reset_index()
    )

    grp_stats["peer_iqr"] = grp_stats["peer_q75"] - grp_stats["peer_q25"]

    select_cols = list(dict.fromkeys(base_cols + key_cols))
    out = cat_df[select_cols].copy()

    out = out.merge(grp_stats, on=key_cols, how="left")

    # approximate leave-one-out peer count
    out["self_in_history"] = out["_peer_row_id"].isin(peer_history_df["_peer_row_id"])
    out["peer_n"] = out["peer_n_raw"] - out["self_in_history"].astype(int)

    no_peers_mask = out["peer_n"] <= 0

    stat_cols = [
        "peer_mean",
        "peer_median",
        "peer_q25",
        "peer_q75",
        "peer_iqr",
        "peer_p90",
        "peer_p95",
        "peer_p99",
    ]

    out.loc[no_peers_mask, stat_cols] = np.nan
    out.loc[no_peers_mask, "peer_n"] = 0

    out["obs_to_peer_median_ratio"] = (
        out["observed_cost"] / (out["peer_median"] + 1e-9)
    )
    out["exp_to_peer_median_ratio"] = (
        out["expected_cost"] / (out["peer_median"] + 1e-9)
    )

    out.loc[no_peers_mask, ["obs_to_peer_median_ratio", "exp_to_peer_median_ratio"]] = np.nan

    out["peer_key"] = " + ".join(key_cols)

    out = out[
        base_cols
        + [
            "peer_key",
            "peer_n",
            "peer_mean",
            "peer_median",
            "peer_q25",
            "peer_q75",
            "peer_iqr",
            "peer_p90",
            "peer_p95",
            "peer_p99",
            "obs_to_peer_median_ratio",
            "exp_to_peer_median_ratio",
        ]
    ]

    return out

peer_tables = []
for key_cols in peer_key_sets:
    peer_tables.append(
        build_peer_distribution_context_safe(
            cat_df=cat_df,
            peer_history_df=peer_history_df,
            key_cols=key_cols,
            target_col="avg_mdcr_stdzd_amt",
        )
    )

cat_peer_distribution_context = pd.concat(peer_tables, ignore_index=True)

with pd.option_context("display.max_columns", None):
    display(cat_peer_distribution_context.head(20))

#### D2. Rank useful cases from the safe table

In [ ]:
with pd.option_context("display.max_columns", None):
    print("Top rows by observed / peer median ratio")
    display(
        cat_peer_distribution_context
        .sort_values("obs_to_peer_median_ratio", ascending=False)
        .head(25)
    )

    print("Top rows by expected / peer median ratio (lowest first)")
    display(
        cat_peer_distribution_context
        .sort_values("exp_to_peer_median_ratio", ascending=True)
        .head(25)
    )

#### D3. Exact percentile calculation for only the top-N interesting cases


In [ ]:
import numpy as np
import pandas as pd

# ============================================================
# D3. Exact percentile within peers for TOP-N only
# ============================================================

# Choose a small subset to avoid memory blowups
topN = 100

top_cases = (
    cat_peer_distribution_context
    .query("peer_key == 'HCPCS_Cd'")
    .sort_values("obs_to_peer_median_ratio", ascending=False)
    .head(topN)
    .copy()
)

def compute_exact_peer_percentile_small(
    top_cases: pd.DataFrame,
    peer_history_df: pd.DataFrame,
    key_cols: list[str],
    target_col: str = "avg_mdcr_stdzd_amt",
) -> pd.DataFrame:
    results = []

    for _, row in top_cases.iterrows():
        peers = peer_history_df.copy()
        for col in key_cols:
            peers = peers.loc[peers[col] == row[col]]

        peers = peers.loc[peers["_peer_row_id"] != row["_peer_row_id"]]

        vals = peers[target_col].astype(float)

        if len(vals) == 0:
            pctile = np.nan
        else:
            pctile = (vals <= row["observed_cost"]).mean()

        results.append({
            "_peer_row_id": row["_peer_row_id"],
            "peer_key": " + ".join(key_cols),
            "row_pctile_within_peers": pctile,
        })

    return pd.DataFrame(results)

top_cases_pctile = compute_exact_peer_percentile_small(
    top_cases=top_cases,
    peer_history_df=peer_history_df,
    key_cols=["HCPCS_Cd"],
    target_col="avg_mdcr_stdzd_amt",
)

top_cases_with_pctile = top_cases.merge(
    top_cases_pctile,
    on=["_peer_row_id", "peer_key"],
    how="left",
)

with pd.option_context("display.max_columns", None):
    display(top_cases_with_pctile.head(25))

## E. Examine support tier and anchor source jointly

In [ ]:
import numpy as np
import pandas as pd

# ============================================================
# E. Examine support tier and anchor source jointly
# ============================================================

# ------------------------------------------------------------
# 1) Catastrophic rows only
# ------------------------------------------------------------
cat_df = failure_df.loc[failure_df["is_any_catastrophic"]].copy()

# ------------------------------------------------------------
# 2) Summary by expected_cost_source
# ------------------------------------------------------------
cat_by_expected_source = (
    cat_df.groupby("expected_cost_source", dropna=False)
    .agg(
        n_rows=("expected_cost_source", "size"),
        mean_abs_residual=("abs_residual", "mean"),
        mean_oe_ratio=("oe_ratio", "mean"),
        tail_rate=("is_top_1pct_avg_mdcr_stdzd_amt", "mean"),
    )
    .reset_index()
)

cat_by_expected_source["share_within_catastrophic"] = (
    cat_by_expected_source["n_rows"] / cat_by_expected_source["n_rows"].sum()
)

with pd.option_context("display.max_columns", None):
    display(cat_by_expected_source.sort_values("n_rows", ascending=False))

# ------------------------------------------------------------
# 3) Summary by support tier
# ------------------------------------------------------------
cat_by_support_tier = (
    cat_df.groupby("expected_cost_support_tier", dropna=False)
    .agg(
        n_rows=("expected_cost_support_tier", "size"),
        mean_abs_residual=("abs_residual", "mean"),
        mean_oe_ratio=("oe_ratio", "mean"),
        tail_rate=("is_top_1pct_avg_mdcr_stdzd_amt", "mean"),
    )
    .reset_index()
)

cat_by_support_tier["share_within_catastrophic"] = (
    cat_by_support_tier["n_rows"] / cat_by_support_tier["n_rows"].sum()
)

with pd.option_context("display.max_columns", None):
    display(cat_by_support_tier.sort_values("n_rows", ascending=False))

# ------------------------------------------------------------
# 4) Cold-start catastrophic rows by anchor source + support tier
# ------------------------------------------------------------
cat_cold_df = cat_df.loc[cat_df["expected_cost_source"] == "cold_start_model_I"].copy()

anchor_support_crosstab = pd.crosstab(
    cat_cold_df["cold_start_anchor_source"],
    cat_cold_df["expected_cost_support_tier"],
    margins=True
)

display(anchor_support_crosstab)

# ------------------------------------------------------------
# 5) Share of catastrophic rows by support tier in the full data
#    This helps distinguish prevalence from raw counts
# ------------------------------------------------------------
full_support_counts = (
    failure_df.groupby("expected_cost_support_tier")
    .size()
    .rename("n_all")
)

cat_support_counts = (
    cat_df.groupby("expected_cost_support_tier")
    .size()
    .rename("n_cat")
)

support_cat_rate = pd.concat([full_support_counts, cat_support_counts], axis=1).fillna(0)
support_cat_rate["catastrophic_rate_within_tier"] = (
    support_cat_rate["n_cat"] / support_cat_rate["n_all"]
)

display(support_cat_rate.reset_index())

# ------------------------------------------------------------
# 6) Cold-start catastrophic rate by anchor source
# ------------------------------------------------------------
full_cold_anchor_counts = (
    failure_df.loc[failure_df["expected_cost_source"] == "cold_start_model_I"]
    .groupby("cold_start_anchor_source")
    .size()
    .rename("n_all_cold")
)

cat_cold_anchor_counts = (
    cat_cold_df.groupby("cold_start_anchor_source")
    .size()
    .rename("n_cat_cold")
)

cold_anchor_cat_rate = pd.concat(
    [full_cold_anchor_counts, cat_cold_anchor_counts],
    axis=1
).fillna(0)

cold_anchor_cat_rate["catastrophic_rate_within_anchor_source"] = (
    cold_anchor_cat_rate["n_cat_cold"] / cold_anchor_cat_rate["n_all_cold"]
)

display(cold_anchor_cat_rate.reset_index())

## F. Study problematic HCPCS families

In [ ]:
import numpy as np
import pandas as pd

# ============================================================
# F. Study problematic HCPCS families
# ============================================================

# ------------------------------------------------------------
# 1) Catastrophic rows only
# ------------------------------------------------------------
cat_df = failure_df.loc[failure_df["is_any_catastrophic"]].copy()

# ------------------------------------------------------------
# 2) HCPCS frequency among catastrophic rows
# ------------------------------------------------------------
cat_hcpcs_counts = (
    cat_df.groupby(["HCPCS_Cd", "hcpcs_desc"], dropna=False)
    .agg(
        n_rows=("HCPCS_Cd", "size"),
        mean_abs_residual=("abs_residual", "mean"),
        median_abs_residual=("abs_residual", "median"),
        mean_oe_ratio=("oe_ratio", "mean"),
        tail_rate=("is_top_1pct_avg_mdcr_stdzd_amt", "mean"),
    )
    .reset_index()
    .sort_values("n_rows", ascending=False)
)

with pd.option_context("display.max_columns", None):
    display(cat_hcpcs_counts.head(50))

# ------------------------------------------------------------
# 3) Compare catastrophic prevalence by HCPCS
#    This helps identify codes that are not just frequent,
#    but disproportionately problematic.
# ------------------------------------------------------------
all_hcpcs_counts = (
    failure_df.groupby("HCPCS_Cd")
    .size()
    .rename("n_all")
)

cat_hcpcs_only_counts = (
    cat_df.groupby("HCPCS_Cd")
    .size()
    .rename("n_cat")
)

hcpcs_cat_rate = pd.concat([all_hcpcs_counts, cat_hcpcs_only_counts], axis=1).fillna(0)
hcpcs_cat_rate["catastrophic_rate"] = hcpcs_cat_rate["n_cat"] / hcpcs_cat_rate["n_all"]

hcpcs_desc_lookup = (
    failure_df[["HCPCS_Cd", "hcpcs_desc"]]
    .drop_duplicates()
)

hcpcs_cat_rate = (
    hcpcs_cat_rate.reset_index()
    .merge(hcpcs_desc_lookup, on="HCPCS_Cd", how="left")
    .sort_values(["catastrophic_rate", "n_cat"], ascending=[False, False])
)

with pd.option_context("display.max_columns", None):
    display(hcpcs_cat_rate.head(50))

# ------------------------------------------------------------
# 4) Explicitly inspect known troublesome codes/families
# ------------------------------------------------------------
problem_codes = ["J9999", "J3490", "J3590"]

problem_code_rows = failure_df.loc[
    failure_df["HCPCS_Cd"].isin(problem_codes)
].copy()

problem_code_summary = (
    problem_code_rows.groupby("HCPCS_Cd")
    .agg(
        n_all=("HCPCS_Cd", "size"),
        n_cat=("is_any_catastrophic", "sum"),
        catastrophic_rate=("is_any_catastrophic", "mean"),
        mean_abs_residual=("abs_residual", "mean"),
        mean_oe_ratio=("oe_ratio", "mean"),
        tail_rate=("is_top_1pct_avg_mdcr_stdzd_amt", "mean"),
    )
    .reset_index()
)

with pd.option_context("display.max_columns", None):
    display(problem_code_summary)

# ------------------------------------------------------------
# 5) Simple heuristic flags for potentially problematic families
# ------------------------------------------------------------
failure_df["is_misc_unclassified_drug_code"] = failure_df["HCPCS_Cd"].isin(["J9999", "J3490", "J3590"])

# Very rough radiopharmaceutical heuristic.
# Adjust if you later want a better curated list.
failure_df["is_potential_radiopharm_code"] = failure_df["HCPCS_Cd"].astype(str).str.startswith("A9")

family_summary = pd.DataFrame({
    "family": [
        "misc_unclassified_drug_code",
        "potential_radiopharm_code",
    ],
    "n_all": [
        failure_df["is_misc_unclassified_drug_code"].sum(),
        failure_df["is_potential_radiopharm_code"].sum(),
    ],
    "n_cat": [
        failure_df.loc[failure_df["is_misc_unclassified_drug_code"], "is_any_catastrophic"].sum(),
        failure_df.loc[failure_df["is_potential_radiopharm_code"], "is_any_catastrophic"].sum(),
    ],
})

family_summary["catastrophic_rate"] = family_summary["n_cat"] / family_summary["n_all"]

display(family_summary)

# ------------------------------------------------------------
# 6) Drill into catastrophic rows for known problematic families
# ------------------------------------------------------------
problem_family_rows = failure_df.loc[
    failure_df["is_any_catastrophic"]
    & (
        failure_df["is_misc_unclassified_drug_code"]
        | failure_df["is_potential_radiopharm_code"]
    )
].copy()

problem_family_view = problem_family_rows[[
    "Rndrng_NPI", "Year", "HCPCS_Cd", "hcpcs_desc",
    "provider_type", "Place_Of_Srvc", "state",
    "observed_cost", "expected_cost", "residual", "abs_residual", "oe_ratio",
    "expected_cost_source", "cold_start_anchor_source", "expected_cost_support_tier"
]].sort_values("abs_residual", ascending=False)

with pd.option_context("display.max_columns", None):
    display(problem_family_view.head(100))

# Important Revelation! 

When we constructed Model I, it did not include lag features during training, however, when reconstructing the cost level predictions, it used lag cost. This gave us the illusion that it performs really well. But these predictions were on "hot rows" (lag cost-present rows). When we tested Model I on "cold rows" (lag cost-absent rows) of unified benchmark dataframe, we realized it performed much worse. This was because for "cold rows" it had to realy on calculating the anchor using a fallback ladder. In other words, it needed to get the base cost to add the predicted log delta cost to. Because the fallback ladder was a weak predictor, adding the correction (i.e., predicted delta cost) did not meaningfully bring the final prediction close to the actual cost. 

When we build direct cost prediction models that did not use lag features during training, Model D was notably the best at predicting direct cost for mixed hot and cold rows (`avg_mdcr_stdzd_amt` in `newprov_pkg_wt10_without_lags.test_df` has `NaN`s, i.e. cold rows). 

Therefore, we need to compare performances of Model D and Model I on the cold rows of the unified benchmarking dataframe to see which one would perform better on truly cold rows. 

### 1. Generate Model D predictions on truly cold rows of benchmark_df


In [ ]:
import numpy as np
import pandas as pd

# ============================================================
# Model D on truly cold rows of benchmark_df
# Direct-cost model, no lag features
# ============================================================

# ------------------------------------------------------------
# 1) Basic setup
# ------------------------------------------------------------
target_col = "avg_mdcr_stdzd_amt"

# Truly cold rows only
mask_cold = ~benchmark_df["has_lag"].astype(bool)

print("Cold rows:", mask_cold.sum())

# ------------------------------------------------------------
# 2) Build feature matrix using Model D feature columns
#    Model D = no-lag direct-cost model
# ------------------------------------------------------------
feature_cols_D = newprov_pkg_wt10_without_lags.feature_cols
X_cold_D = benchmark_df.loc[mask_cold, feature_cols_D].copy()

# ------------------------------------------------------------
# 3) Predict level cost directly
# ------------------------------------------------------------
pred_level_D_cold = best_newprov_model_wt10_without_lags.predict(X_cold_D)

# Optional safety clip
pred_level_D_cold = np.clip(pred_level_D_cold, a_min=0, a_max=None)

# ------------------------------------------------------------
# 4) Store predictions on benchmark_df
# ------------------------------------------------------------
benchmark_df.loc[mask_cold, "pred_level_D_cold"] = pred_level_D_cold

print("Model D cold prediction NaN rate:",
      benchmark_df.loc[mask_cold, "pred_level_D_cold"].isna().mean())

### 2. Evaluate Model D on truly cold rows using the same schema as Model I cold-row comparison


In [ ]:
# ------------------------------------------------------------
# Model D on lag-absent rows
# ------------------------------------------------------------
metrics_D_on_cold = evaluate_level_predictions_same_schema(
    benchmark_df.loc[~benchmark_df["has_lag"]].copy(),
    pred_col="pred_level_D_cold",
    train_sample_weights_label=10,
    target_col="avg_mdcr_stdzd_amt",
    tail_flag_col="is_top_1pct_avg_mdcr_stdzd_amt",
    w_tail=10,
)

metrics_D_on_cold.insert(0, "evaluation_name", "Model_D_on_cold_rows")

with pd.option_context("display.max_columns", None):
    display(metrics_D_on_cold)

### 3. Compare Model D vs Model I on truly cold rows


In [ ]:
cold_head_to_head = pd.concat(
    [
        metrics_D_on_cold,
        metrics_I_on_cold,
    ],
    ignore_index=True,
)

with pd.option_context("display.max_columns", None):
    display(cold_head_to_head)

### 4. Add direct deltas for Model D vs Model I on cold rows

This gives us the same style of explicit comparison we made for hot rows.

In [ ]:
compare_cold = cold_head_to_head.set_index("evaluation_name")

cold_delta_summary = pd.DataFrame({
    "metric": [
        "r2_test",
        "mae",
        "rmse",
        "r2_test_w",
        "mae_w",
        "rmse_w",
        "non_tail_pred_to_y_ratio",
        "tail_pred_to_y_ratio",
        "tail_under_prediction_rate",
        "tail_bias",
    ],
    "Model_D_minus_Model_I_on_cold": [
        compare_cold.loc["Model_D_on_cold_rows", "r2_test"] - compare_cold.loc["Model_I_on_cold_rows", "r2_test"],
        compare_cold.loc["Model_D_on_cold_rows", "mae"] - compare_cold.loc["Model_I_on_cold_rows", "mae"],
        compare_cold.loc["Model_D_on_cold_rows", "rmse"] - compare_cold.loc["Model_I_on_cold_rows", "rmse"],
        compare_cold.loc["Model_D_on_cold_rows", "r2_test_w"] - compare_cold.loc["Model_I_on_cold_rows", "r2_test_w"],
        compare_cold.loc["Model_D_on_cold_rows", "mae_w"] - compare_cold.loc["Model_I_on_cold_rows", "mae_w"],
        compare_cold.loc["Model_D_on_cold_rows", "rmse_w"] - compare_cold.loc["Model_I_on_cold_rows", "rmse_w"],
        compare_cold.loc["Model_D_on_cold_rows", "non_tail_pred_to_y_ratio"] - compare_cold.loc["Model_I_on_cold_rows", "non_tail_pred_to_y_ratio"],
        compare_cold.loc["Model_D_on_cold_rows", "tail_pred_to_y_ratio"] - compare_cold.loc["Model_I_on_cold_rows", "tail_pred_to_y_ratio"],
        compare_cold.loc["Model_D_on_cold_rows", "tail_under_prediction_rate"] - compare_cold.loc["Model_I_on_cold_rows", "tail_under_prediction_rate"],
        compare_cold.loc["Model_D_on_cold_rows", "tail_bias"] - compare_cold.loc["Model_I_on_cold_rows", "tail_bias"],
    ]
})

with pd.option_context("display.max_columns", None):
    display(cold_delta_summary)

## Cold-Start Model Deep Dive: Model D vs. Model I

On truly cold rows, **Model D beat Model I clearly and decisively**.

This is not a subtle result. The gap is large across the most important metrics, especially the ones we care about for hard cases and tails.

---

### The Core Conclusion

We originally chose Model I as the cold-start winner based on the earlier evaluation framework. But that earlier framework did **not** evaluate a true cold-start scenario. It evaluated Model I inside the package setup where the model was trained on lag-present rows and then reconstructed predictions in a way that still benefited from lag during evaluation.

Once we forced the comparison onto **truly lag-absent rows only** from the unified benchmark dataframe, the story changed. That is the more operationally honest test for cold-start use.

And under that test:
* Model D is better than Model I
* not just on average-case metrics
* but also on weighted and tail-sensitive metrics

So our instinct here was right.

---

### Why Model D Wins Here

**Model D** is the *direct-cost, no-lag model*.

**Model I** is the *no-lag delta model*. To turn its prediction into a dollar cost, it needs an anchor:
$$\text{expected cost} = \exp\big(\log(1+\text{anchor}) + \widehat{\log\_delta}\big) - 1$$

That means Model I has **two chances to go wrong** on cold rows:
1. The **anchor** can be wrong.
2. The **predicted delta** can be wrong.

Model D skips that whole dependency chain and directly predicts the cost level from the no-lag features. So on true cold rows, Model D is often easier to optimize and more stable because it does not inherit anchor error. That is exactly what our results now show.

---

### Head-to-Head Interpretation

#### 1. Overall Fit
* **Model D R² test:** 0.891
* **Model I R² test:** 0.798
* **Difference:** +0.093 for Model D
*(A big improvement. Model D explains substantially more of the variance in cold-row cost.)*

#### 2. Average Absolute Error
* **Model D MAE:** 16.09
* **Model I MAE:** 18.97
* **Difference:** Model D is lower by 2.88
*(A meaningful improvement at the row level.)*

#### 3. RMSE
* **Model D RMSE:** 93.57
* **Model I RMSE:** 127.45
* **Difference:** Model D is lower by 33.87
*(RMSE punishes large misses heavily. This tells us Model D is much better at controlling big cold-start failures.)*

---

### Weighted Metrics Matter Even More Here

Because we care about expensive and difficult cases, the weighted metrics are very important.

* **Weighted R²:** Model D (0.906) vs. Model I (0.791) — **+0.115 for Model D**
* **Weighted MAE:** Model D (41.70) vs. Model I (83.13) — **Model D better by 41.42**
* **Weighted RMSE:** Model D (242.24) vs. Model I (361.65) — **Model D better by 119.41**

*(This is one of the strongest signals in the whole comparison. In the tail-emphasized regime, Model D is far more robust.)*

---

### Tail Behavior

This is where the earlier combined benchmark made us uneasy, and now we can see why.

* **Tail predicted-to-true ratio:** Model D (0.870) vs. Model I (0.653).
*(Model I was missing the tail badly on cold rows, predicting only ~65% of the true average tail cost. Model D gets closer at ~87%.)*

* **Tail bias:** Model D (-271.7) vs. Model I (-724.4) — **Model D better by +452.7**
*(That is huge. Both underpredict, but Model I underpredicts the tail much more aggressively.)*

* **Tail underprediction rate:** Model D (0.871) vs. Model I (0.896).
*(Model D still underpredicts most tail rows, but slightly less often. The more important fact is that when both underpredict, Model D misses by much less.)*

---

### Non-Tail Behavior

* **Non-tail predicted-to-true ratio:** Model D (0.983) vs. Model I (1.051).

So Model D is slightly conservative on non-tail rows, while Model I slightly overpredicts non-tail rows. That alone is not enough to choose a winner, but it fits the broader pattern:
* Model I tends to drift upward on non-tail rows and still badly miss the real cold-start tails.
* Model D is more balanced and far more stable overall.

---

### Why the Earlier Choice of Model I Made Sense at the Time

It was not a foolish choice. It was just based on a **different evaluation target**.

Earlier, Model I looked strong because it excluded lag features from training, but when evaluated in its package framework, it still reconstructed level predictions using lag where available.

So it was really answering: *"How well does this no-lag delta model behave on a test set that still contains lag-present rows?"*
That is not the same as: *"How well does this model work for true cold-start rows with no lag available at prediction time?"*

Those are different questions. Once we asked the true cold-start question, Model D won. That is exactly the right scientific correction to make.

---

### Conceptual Logic & Recommendation

Let's revise the conclusion like this:
* **Hot-start winner:** Model G
* **Cold-start winner:** Model D

**Why this makes conceptual sense:**
* **For hot-start rows:** The lag is real, specific, row-level history. Delta modeling is powerful because it predicts deviation from a strong anchor.
* **For cold-start rows:** There is no real lag; the anchor becomes synthetic. A synthetic anchor + predicted delta can be less stable than just predicting the cost directly.

So the modeling logic becomes:
* **Real anchor present** → delta model shines
* **Real anchor absent** → direct level model shines

That is a very clean and interpretable result.

#### Recommendation
Let's update the benchmarking pipeline to use:
* **Model G** for hot rows
* **Model D** for cold rows

Then rerun the unified benchmark evaluation and failure analysis with that revised routing. That will give us the true best version of your expected-cost engine.

A clean naming convention could be:
* `hot_start_model` = Model G
* `cold_start_model` = Model D
* `unified_expected_cost_v2` = G_for_hot + D_for_cold

That would likely improve the combined benchmark materially, especially on cold-row tails.

# Performances of Model G on lag-present and Model D on lag-absent rows of benchmarking dataset

## 1) Rebuild comparable predictions on benchmark_df (V2)

This version creates explicit prediction columns for:
- pred_level_G_hot
- pred_level_D_hot
- pred_level_D_cold


In [ ]:
import numpy as np
import pandas as pd

# ============================================================
# V2. Rebuild comparable predictions on benchmark_df
# Uses:
# - Model G for hot rows (lag-present)
# - Model D for cold rows (lag-absent)
# Also computes Model D on hot rows just for comparison
# ============================================================

# ------------------------------------------------------------
# Assumes benchmark_df already exists and includes:
# - has_lag
# - lag1_avg_amt
# - avg_mdcr_stdzd_amt
# - is_top_1pct_avg_mdcr_stdzd_amt
#
# Also assumes these model objects already exist:
# - best_newprov_model_wt10_with_lags_delta_trgt
# - newprov_pkg_wt10_with_lags_delta_trgt
# - best_newprov_model_wt10_without_lags
# - newprov_pkg_wt10_without_lags
# ------------------------------------------------------------

target_col = "avg_mdcr_stdzd_amt"
lag_col = "lag1_avg_amt"
tail_flag_col = "is_top_1pct_avg_mdcr_stdzd_amt"

mask_hot = benchmark_df["has_lag"].astype(bool)
mask_cold = ~mask_hot

# ------------------------------------------------------------
# A) Model G predictions on hot rows
#    Model G predicts log_delta_cost, then reconstructs level
#    using the TRUE lag on hot rows.
# ------------------------------------------------------------
feature_cols_G = newprov_pkg_wt10_with_lags_delta_trgt.feature_cols
X_hot_G = benchmark_df.loc[mask_hot, feature_cols_G].copy()

pred_log_delta_G_hot = best_newprov_model_wt10_with_lags_delta_trgt.predict(X_hot_G)

pred_level_G_hot = np.expm1(
    np.log1p(benchmark_df.loc[mask_hot, lag_col].astype(float).to_numpy())
    + pred_log_delta_G_hot
)

benchmark_df.loc[mask_hot, "pred_log_delta_G_hot"] = pred_log_delta_G_hot
benchmark_df.loc[mask_hot, "pred_level_G_hot"] = pred_level_G_hot

# ------------------------------------------------------------
# B) Model D predictions on hot rows
#    This is only for curiosity / comparison.
#    Model D is a direct-cost no-lag model, so it predicts
#    the level directly even when lag exists.
# ------------------------------------------------------------
feature_cols_D = newprov_pkg_wt10_without_lags.feature_cols
X_hot_D = benchmark_df.loc[mask_hot, feature_cols_D].copy()

pred_level_D_hot = best_newprov_model_wt10_without_lags.predict(X_hot_D)
pred_level_D_hot = np.clip(pred_level_D_hot, a_min=0, a_max=None)

benchmark_df.loc[mask_hot, "pred_level_D_hot"] = pred_level_D_hot

# ------------------------------------------------------------
# C) Model D predictions on cold rows
#    This is the true production-style cold-start prediction.
#    No lag is used. Model D predicts the level directly.
# ------------------------------------------------------------
X_cold_D = benchmark_df.loc[mask_cold, feature_cols_D].copy()

pred_level_D_cold = best_newprov_model_wt10_without_lags.predict(X_cold_D)
pred_level_D_cold = np.clip(pred_level_D_cold, a_min=0, a_max=None)

benchmark_df.loc[mask_cold, "pred_level_D_cold"] = pred_level_D_cold

print("Hot rows:", mask_hot.sum())
print("Cold rows:", mask_cold.sum())

print("NaN rate, pred_level_G_hot:", benchmark_df.loc[mask_hot, "pred_level_G_hot"].isna().mean())
print("NaN rate, pred_level_D_hot:", benchmark_df.loc[mask_hot, "pred_level_D_hot"].isna().mean())
print("NaN rate, pred_level_D_cold:", benchmark_df.loc[mask_cold, "pred_level_D_cold"].isna().mean())

## 2) Helper to evaluate any prediction column with our familiar schema

We do not need to change this helper. It is already appropriate for evaluating any level prediction column.


In [ ]:
from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error
import numpy as np
import pandas as pd

def evaluate_level_predictions_same_schema(
    df,
    pred_col,
    *,
    train_sample_weights_label=np.nan,
    target_col="avg_mdcr_stdzd_amt",
    tail_flag_col="is_top_1pct_avg_mdcr_stdzd_amt",
    w_tail=10,
):
    eval_df = df.copy()

    y_true = eval_df[target_col].astype(float)
    pred = eval_df[pred_col].astype(float)

    is_tail = eval_df[tail_flag_col].astype(bool)
    is_non_tail = ~is_tail

    sample_w = (1 + (w_tail - 1) * eval_df[tail_flag_col].astype(int).to_numpy()).astype(float)

    eval_df["pred"] = pred
    eval_df["abs_err"] = (y_true - pred).abs()
    eval_df["sq_err"] = (y_true - pred) ** 2

    r2_test = r2_score(y_true, pred)
    mae = mean_absolute_error(y_true, pred)
    rmse = root_mean_squared_error(y_true, pred)

    pred_clip = pred.clip(lower=0)
    r2_test_log = r2_score(np.log1p(y_true), np.log1p(pred_clip))
    mae_log = mean_absolute_error(np.log1p(y_true), np.log1p(pred_clip))
    rmse_log = root_mean_squared_error(np.log1p(y_true), np.log1p(pred_clip))

    r2_test_w = r2_score(y_true, pred, sample_weight=sample_w)
    mae_w = mean_absolute_error(y_true, pred, sample_weight=sample_w)
    rmse_w = root_mean_squared_error(y_true, pred, sample_weight=sample_w)

    non_tail_pred_to_y_ratio = (
        eval_df.loc[is_non_tail, "pred"].mean()
        / eval_df.loc[is_non_tail, target_col].mean()
    )
    tail_pred_to_y_ratio = (
        eval_df.loc[is_tail, "pred"].mean()
        / eval_df.loc[is_tail, target_col].mean()
    )

    non_tail_sse = eval_df.loc[is_non_tail, "sq_err"].sum()
    tail_sse = eval_df.loc[is_tail, "sq_err"].sum()

    non_tail_share_of_total_sse = non_tail_sse / (non_tail_sse + tail_sse)
    tail_share_of_total_sse = tail_sse / (non_tail_sse + tail_sse)

    non_tail_under_prediction_rate = (
        (eval_df.loc[is_non_tail, "pred"] < eval_df.loc[is_non_tail, target_col]).mean()
    )
    tail_under_prediction_rate = (
        (eval_df.loc[is_tail, "pred"] < eval_df.loc[is_tail, target_col]).mean()
    )

    non_tail_bias = (
        eval_df.loc[is_non_tail, "pred"].mean()
        - eval_df.loc[is_non_tail, target_col].mean()
    )
    tail_bias = (
        eval_df.loc[is_tail, "pred"].mean()
        - eval_df.loc[is_tail, target_col].mean()
    )

    return pd.DataFrame({
        "train_sample_weights": [train_sample_weights_label],
        "rows_eval": [len(eval_df)],
        "r2_train": [np.nan],
        "r2_test": [r2_test],
        "mae": [mae],
        "rmse": [rmse],
        "r2_train_log": [np.nan],
        "r2_test_log": [r2_test_log],
        "mae_log": [mae_log],
        "rmse_log": [rmse_log],
        "r2_train_w": [np.nan],
        "r2_test_w": [r2_test_w],
        "mae_w": [mae_w],
        "rmse_w": [rmse_w],
        "non_tail_pred_to_y_ratio": [non_tail_pred_to_y_ratio],
        "tail_pred_to_y_ratio": [tail_pred_to_y_ratio],
        "non_tail_sse": [non_tail_sse],
        "tail_sse": [tail_sse],
        "non_tail_share_of_total_sse": [non_tail_share_of_total_sse],
        "tail_share_of_total_sse": [tail_share_of_total_sse],
        "non_tail_under_prediction_rate": [non_tail_under_prediction_rate],
        "tail_under_prediction_rate": [tail_under_prediction_rate],
        "non_tail_bias": [non_tail_bias],
        "tail_bias": [tail_bias],
    })

## 3) Evaluate the three comparisons (V2)

This should now be:
- Model G on hot rows
- Model D on hot rows
- Model D on cold rows


In [ ]:
# ============================================================
# V2. Evaluate the three comparisons
# ============================================================

# ------------------------------------------------------------
# 1) Model G on lag-present rows
# ------------------------------------------------------------
metrics_G_on_hot = evaluate_level_predictions_same_schema(
    benchmark_df.loc[benchmark_df["has_lag"]].copy(),
    pred_col="pred_level_G_hot",
    train_sample_weights_label=10,
    target_col="avg_mdcr_stdzd_amt",
    tail_flag_col="is_top_1pct_avg_mdcr_stdzd_amt",
    w_tail=10,
)
metrics_G_on_hot.insert(0, "evaluation_name", "Model_G_on_hot_rows")

# ------------------------------------------------------------
# 2) Model D on lag-present rows
#    Curiosity comparison only
# ------------------------------------------------------------
metrics_D_on_hot = evaluate_level_predictions_same_schema(
    benchmark_df.loc[benchmark_df["has_lag"]].copy(),
    pred_col="pred_level_D_hot",
    train_sample_weights_label=10,
    target_col="avg_mdcr_stdzd_amt",
    tail_flag_col="is_top_1pct_avg_mdcr_stdzd_amt",
    w_tail=10,
)
metrics_D_on_hot.insert(0, "evaluation_name", "Model_D_on_hot_rows")

# ------------------------------------------------------------
# 3) Model D on lag-absent rows
#    True cold-start route in V2
# ------------------------------------------------------------
metrics_D_on_cold = evaluate_level_predictions_same_schema(
    benchmark_df.loc[~benchmark_df["has_lag"]].copy(),
    pred_col="pred_level_D_cold",
    train_sample_weights_label=10,
    target_col="avg_mdcr_stdzd_amt",
    tail_flag_col="is_top_1pct_avg_mdcr_stdzd_amt",
    w_tail=10,
)
metrics_D_on_cold.insert(0, "evaluation_name", "Model_D_on_cold_rows")

# ------------------------------------------------------------
# Combine for direct comparison
# ------------------------------------------------------------
model_GD_benchmark_segment_comparison = pd.concat(
    [metrics_G_on_hot, metrics_D_on_hot, metrics_D_on_cold],
    ignore_index=True,
)

with pd.option_context("display.max_columns", None):
    display(model_GD_benchmark_segment_comparison)

## 4) Add direct deltas for the hot-row comparison (V2)

This is still useful. It now compares Model G vs Model D on the same hot rows.


In [ ]:
# ============================================================
# V2. Direct hot-row comparison: Model G vs Model D
# ============================================================

compare_hot = model_GD_benchmark_segment_comparison.set_index("evaluation_name")

hot_delta_summary = pd.DataFrame({
    "metric": [
        "r2_test",
        "mae",
        "rmse",
        "r2_test_w",
        "mae_w",
        "rmse_w",
        "tail_pred_to_y_ratio",
        "tail_under_prediction_rate",
        "tail_bias",
    ],
    "Model_G_minus_Model_D_on_hot": [
        compare_hot.loc["Model_G_on_hot_rows", "r2_test"] - compare_hot.loc["Model_D_on_hot_rows", "r2_test"],
        compare_hot.loc["Model_G_on_hot_rows", "mae"] - compare_hot.loc["Model_D_on_hot_rows", "mae"],
        compare_hot.loc["Model_G_on_hot_rows", "rmse"] - compare_hot.loc["Model_D_on_hot_rows", "rmse"],
        compare_hot.loc["Model_G_on_hot_rows", "r2_test_w"] - compare_hot.loc["Model_D_on_hot_rows", "r2_test_w"],
        compare_hot.loc["Model_G_on_hot_rows", "mae_w"] - compare_hot.loc["Model_D_on_hot_rows", "mae_w"],
        compare_hot.loc["Model_G_on_hot_rows", "rmse_w"] - compare_hot.loc["Model_D_on_hot_rows", "rmse_w"],
        compare_hot.loc["Model_G_on_hot_rows", "tail_pred_to_y_ratio"] - compare_hot.loc["Model_D_on_hot_rows", "tail_pred_to_y_ratio"],
        compare_hot.loc["Model_G_on_hot_rows", "tail_under_prediction_rate"] - compare_hot.loc["Model_D_on_hot_rows", "tail_under_prediction_rate"],
        compare_hot.loc["Model_G_on_hot_rows", "tail_bias"] - compare_hot.loc["Model_D_on_hot_rows", "tail_bias"],
    ]
})

with pd.option_context("display.max_columns", None):
    display(hot_delta_summary)

# Benchmarking Version 2

## 1) Unified expected-cost scoring pipeline (V2)

In [ ]:
import numpy as np
import pandas as pd

# ============================================================
# Unified expected-cost scoring pipeline (V2)
# Hot start  -> Model G
# Cold start -> Model D
# Includes benchmarking signals, support metadata, and flags
# ============================================================

# ------------------------------------------------------------
# Required objects assumed to already exist in memory
# ------------------------------------------------------------
# - eda_hcpcs_df
# - best_newprov_model_wt10_with_lags_delta_trgt
# - newprov_pkg_wt10_with_lags_delta_trgt
# - best_newprov_model_wt10_without_lags
# - newprov_pkg_wt10_without_lags

# ------------------------------------------------------------
# Core column names
# ------------------------------------------------------------
target_col = "avg_mdcr_stdzd_amt"
lag_col = "lag1_avg_amt"
tail_flag_col = "is_top_1pct_avg_mdcr_stdzd_amt"

group_cols = ["provider_type", "HCPCS_Cd", "Place_Of_Srvc"]
pt_pos_state_cols = ["provider_type", "Place_Of_Srvc", "state"]
pt_pos_cols = ["provider_type", "Place_Of_Srvc"]
pt_cols = ["provider_type"]

# ------------------------------------------------------------
# 1) Start from full row-level source data
# ------------------------------------------------------------
benchmark_df = eda_hcpcs_df.copy()
benchmark_df["_orig_idx"] = benchmark_df.index

# ------------------------------------------------------------
# 2) Explicit identity/context columns
# ------------------------------------------------------------
identity_cols = [
    "Rndrng_NPI",
    "HCPCS_Cd",
    "provider_type",
    "Place_Of_Srvc",
    "state",
    "Year",
]

support_context_cols = [
    lag_col,
    "services",
    "benes",
    "bene_day_services",
]

# ------------------------------------------------------------
# 3) Define routing: hot start vs cold start
# ------------------------------------------------------------
benchmark_df["has_lag"] = (
    benchmark_df[lag_col].notna()
    & (benchmark_df[lag_col] >= 0)
)

mask_hot = benchmark_df["has_lag"]
mask_cold = ~mask_hot

benchmark_df["expected_cost_source"] = np.where(
    mask_hot,
    "hot_start_model_G",
    "cold_start_model_D"
)

# ------------------------------------------------------------
# 4) Hot-start expected cost using Model G
#    Model G predicts log_delta_cost and reconstructs level
#    using the true lag on hot rows.
# ------------------------------------------------------------
feature_cols_hot = newprov_pkg_wt10_with_lags_delta_trgt.feature_cols
X_hot = benchmark_df.loc[mask_hot, feature_cols_hot].copy()

log_delta_hat_hot = best_newprov_model_wt10_with_lags_delta_trgt.predict(X_hot)

benchmark_df.loc[mask_hot, "pred_log_delta_hot"] = log_delta_hat_hot

expected_cost_hot = np.expm1(
    np.log1p(benchmark_df.loc[mask_hot, lag_col].astype(float).to_numpy())
    + log_delta_hat_hot
)

# ------------------------------------------------------------
# 5) Build cold-start support ladder metadata
#    IMPORTANT:
#    Model D does NOT use this ladder to predict.
#    We keep it for:
#    - interpretability
#    - support tier labeling
#    - downstream diagnostics / failure analysis
# ------------------------------------------------------------
train_years_for_anchor = [2021, 2022]

anchor_train_df = benchmark_df.loc[
    benchmark_df["Year"].isin(train_years_for_anchor)
].copy()

global_mean = float(anchor_train_df[target_col].mean())

anchor_mean_grp = (
    anchor_train_df.groupby(group_cols)[target_col]
    .mean()
    .rename("anchor_mean_grp")
    .reset_index()
)

anchor_mean_hcpcs = (
    anchor_train_df.groupby(["HCPCS_Cd"])[target_col]
    .mean()
    .rename("anchor_mean_hcpcs")
    .reset_index()
)

anchor_mean_pt_pos_state = (
    anchor_train_df.groupby(pt_pos_state_cols)[target_col]
    .mean()
    .rename("anchor_mean_pt_pos_state")
    .reset_index()
)

anchor_median_pt_pos_state = (
    anchor_train_df.groupby(pt_pos_state_cols)[target_col]
    .median()
    .rename("anchor_median_pt_pos_state")
    .reset_index()
)

anchor_mean_pt_pos = (
    anchor_train_df.groupby(pt_pos_cols)[target_col]
    .mean()
    .rename("anchor_mean_pt_pos")
    .reset_index()
)

anchor_mean_pt = (
    anchor_train_df.groupby(pt_cols)[target_col]
    .mean()
    .rename("anchor_mean_pt")
    .reset_index()
)

benchmark_df = benchmark_df.merge(anchor_mean_grp, on=group_cols, how="left")
benchmark_df = benchmark_df.merge(anchor_mean_hcpcs, on="HCPCS_Cd", how="left")
benchmark_df = benchmark_df.merge(anchor_mean_pt_pos_state, on=pt_pos_state_cols, how="left")
benchmark_df = benchmark_df.merge(anchor_median_pt_pos_state, on=pt_pos_state_cols, how="left")
benchmark_df = benchmark_df.merge(anchor_mean_pt_pos, on=pt_pos_cols, how="left")
benchmark_df = benchmark_df.merge(anchor_mean_pt, on=pt_cols, how="left")

# ------------------------------------------------------------
# 6) Build explicit cold-start anchor metadata and source label
#    Again: this is metadata, not the Model D prediction route.
# ------------------------------------------------------------
cold_start_anchor = benchmark_df["anchor_mean_grp"].copy()
cold_start_anchor_source = pd.Series(index=benchmark_df.index, dtype="object")

cold_start_anchor_source[cold_start_anchor.notna()] = "grp_mean"

cold_start_anchor = cold_start_anchor.fillna(benchmark_df["anchor_mean_hcpcs"])
cold_start_anchor_source[
    cold_start_anchor_source.isna() & benchmark_df["anchor_mean_hcpcs"].notna()
] = "hcpcs_mean"

cold_start_anchor = cold_start_anchor.fillna(benchmark_df["anchor_mean_pt_pos_state"])
cold_start_anchor_source[
    cold_start_anchor_source.isna() & benchmark_df["anchor_mean_pt_pos_state"].notna()
] = "pt_pos_state_mean"

cold_start_anchor = cold_start_anchor.fillna(benchmark_df["anchor_median_pt_pos_state"])
cold_start_anchor_source[
    cold_start_anchor_source.isna() & benchmark_df["anchor_median_pt_pos_state"].notna()
] = "pt_pos_state_median"

cold_start_anchor = cold_start_anchor.fillna(benchmark_df["anchor_mean_pt_pos"])
cold_start_anchor_source[
    cold_start_anchor_source.isna() & benchmark_df["anchor_mean_pt_pos"].notna()
] = "pt_pos_mean"

cold_start_anchor = cold_start_anchor.fillna(benchmark_df["anchor_mean_pt"])
cold_start_anchor_source[
    cold_start_anchor_source.isna() & benchmark_df["anchor_mean_pt"].notna()
] = "pt_mean"

cold_start_anchor = cold_start_anchor.fillna(global_mean)
cold_start_anchor_source[cold_start_anchor_source.isna()] = "global_mean"

benchmark_df["cold_start_anchor"] = cold_start_anchor.astype(float)
benchmark_df["cold_start_anchor_source"] = cold_start_anchor_source

# ------------------------------------------------------------
# 7) Cold-start expected cost using Model D
#    Model D predicts level cost directly without lag features.
# ------------------------------------------------------------
feature_cols_cold = newprov_pkg_wt10_without_lags.feature_cols
X_cold = benchmark_df.loc[mask_cold, feature_cols_cold].copy()

expected_cost_cold = best_newprov_model_wt10_without_lags.predict(X_cold)
expected_cost_cold = np.clip(expected_cost_cold, a_min=0, a_max=None)

benchmark_df.loc[mask_cold, "pred_level_cold_D"] = expected_cost_cold

# ------------------------------------------------------------
# 8) Combine into one expected_cost column
# ------------------------------------------------------------
benchmark_df["expected_cost"] = np.nan

benchmark_df.loc[mask_hot, "expected_cost"] = expected_cost_hot
benchmark_df.loc[mask_cold, "expected_cost"] = expected_cost_cold

benchmark_df["expected_cost"] = benchmark_df["expected_cost"].clip(lower=0)

# ------------------------------------------------------------
# 9) Explicit observed and benchmarking columns
# ------------------------------------------------------------
benchmark_df["observed_cost"] = benchmark_df[target_col].astype(float)

benchmark_df["residual"] = benchmark_df["observed_cost"] - benchmark_df["expected_cost"]
benchmark_df["abs_residual"] = benchmark_df["residual"].abs()

benchmark_df["oe_ratio"] = (
    benchmark_df["observed_cost"] / (benchmark_df["expected_cost"] + 1e-9)
)

benchmark_df["pct_diff"] = (
    benchmark_df["residual"] / (benchmark_df["expected_cost"] + 1e-9)
)

benchmark_df["log_oe"] = (
    np.log1p(benchmark_df["observed_cost"])
    - np.log1p(benchmark_df["expected_cost"])
)

# ------------------------------------------------------------
# 10) Support / confidence tier
#     We still define this from lag presence + support ladder
#     because it remains useful as interpretability metadata.
# ------------------------------------------------------------
benchmark_df["expected_cost_support_tier"] = np.select(
    [
        benchmark_df["has_lag"],

        (~benchmark_df["has_lag"]) & benchmark_df["cold_start_anchor_source"].isin([
            "grp_mean", "hcpcs_mean"
        ]),

        (~benchmark_df["has_lag"]) & benchmark_df["cold_start_anchor_source"].isin([
            "pt_pos_state_mean", "pt_pos_state_median", "pt_pos_mean"
        ]),

        (~benchmark_df["has_lag"]) & benchmark_df["cold_start_anchor_source"].isin([
            "pt_mean", "global_mean"
        ]),
    ],
    [
        "high",
        "medium_high",
        "medium",
        "low",
    ],
    default="unknown"
)

# ------------------------------------------------------------
# 11) Optional anomaly / prioritization flags
# ------------------------------------------------------------
benchmark_df["high_positive_residual"] = (
    benchmark_df["residual"] > benchmark_df["residual"].quantile(0.99)
)

benchmark_df["high_oe"] = (
    benchmark_df["oe_ratio"] > benchmark_df["oe_ratio"].quantile(0.99)
)

benchmark_df["extreme_cost_outlier"] = (
    benchmark_df["observed_cost"] > benchmark_df["observed_cost"].quantile(0.99)
)

benchmark_df["high_confidence_anomaly_candidate"] = (
    benchmark_df["expected_cost_support_tier"].isin(["high", "medium_high"])
    & benchmark_df["high_positive_residual"]
    & benchmark_df["high_oe"]
)

# ------------------------------------------------------------
# 12) Final column ordering for benchmarking table
# ------------------------------------------------------------
final_cols = [
    # Identity/context
    "Rndrng_NPI",
    "HCPCS_Cd",
    "provider_type",
    "Place_Of_Srvc",
    "state",
    "Year",

    # Observed / expected
    "observed_cost",
    "expected_cost",
    "expected_cost_source",

    # Benchmarking signals
    "residual",
    "abs_residual",
    "oe_ratio",
    "pct_diff",
    "log_oe",

    # Support / reliability metadata
    "lag1_avg_amt",
    "has_lag",
    "services",
    "benes",
    "bene_day_services",
    "cold_start_anchor",
    "cold_start_anchor_source",
    "expected_cost_support_tier",

    # Optional flags
    "high_positive_residual",
    "high_oe",
    "extreme_cost_outlier",
    "high_confidence_anomaly_candidate",
]

final_cols = [c for c in final_cols if c in benchmark_df.columns]
other_cols = [c for c in benchmark_df.columns if c not in final_cols]
benchmark_df = benchmark_df[final_cols + other_cols]

# ------------------------------------------------------------
# 13) Sanity checks
# ------------------------------------------------------------
print("Rows:", len(benchmark_df))
print("Expected cost NaN rate:", benchmark_df["expected_cost"].isna().mean())
print("Expected cost negative rate:", (benchmark_df["expected_cost"] < 0).mean())

print("\nExpected-cost source breakdown:")
print(benchmark_df["expected_cost_source"].value_counts(dropna=False))

print("\nSupport tier breakdown:")
print(benchmark_df["expected_cost_support_tier"].value_counts(dropna=False))

print("\nCold-start anchor breakdown (metadata only):")
print(benchmark_df.loc[~benchmark_df["has_lag"], "cold_start_anchor_source"].value_counts(dropna=False))

with pd.option_context("display.max_columns", None):
    display(benchmark_df.head())

## 2) A compact evaluation block for the unified engine (V2)

This block does not need structural changes. It will automatically evaluate the new V2 engine because `benchmark_df["expected_cost"]` now uses Model G + Model D.

In [ ]:
from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error
import numpy as np
import pandas as pd

target_col = "observed_cost"
pred_col = "expected_cost"
tail_flag_col = "is_top_1pct_avg_mdcr_stdzd_amt"
w_tail = 10

eval_df = benchmark_df.copy()

y_true = eval_df[target_col].astype(float)
pred = eval_df[pred_col].astype(float)

is_tail = eval_df[tail_flag_col].astype(bool)
is_non_tail = ~is_tail

sample_w = (1 + (w_tail - 1) * eval_df[tail_flag_col].astype(int).to_numpy()).astype(float)

eval_df["abs_err"] = (y_true - pred).abs()
eval_df["sq_err"] = (y_true - pred) ** 2

r2_test = r2_score(y_true, pred)
mae = mean_absolute_error(y_true, pred)
rmse = root_mean_squared_error(y_true, pred)

pred_clip = pred.clip(lower=0)
r2_test_log = r2_score(np.log1p(y_true), np.log1p(pred_clip))
mae_log = mean_absolute_error(np.log1p(y_true), np.log1p(pred_clip))
rmse_log = root_mean_squared_error(np.log1p(y_true), np.log1p(pred_clip))

r2_test_w = r2_score(y_true, pred, sample_weight=sample_w)
mae_w = mean_absolute_error(y_true, pred, sample_weight=sample_w)
rmse_w = root_mean_squared_error(y_true, pred, sample_weight=sample_w)

non_tail_pred_to_y_ratio = eval_df.loc[is_non_tail, pred_col].mean() / eval_df.loc[is_non_tail, target_col].mean()
tail_pred_to_y_ratio = eval_df.loc[is_tail, pred_col].mean() / eval_df.loc[is_tail, target_col].mean()

non_tail_sse = eval_df.loc[is_non_tail, "sq_err"].sum()
tail_sse = eval_df.loc[is_tail, "sq_err"].sum()

non_tail_share_of_total_sse = non_tail_sse / (non_tail_sse + tail_sse)
tail_share_of_total_sse = tail_sse / (non_tail_sse + tail_sse)

non_tail_under_prediction_rate = (
    (eval_df.loc[is_non_tail, pred_col] < eval_df.loc[is_non_tail, target_col]).mean()
)
tail_under_prediction_rate = (
    (eval_df.loc[is_tail, pred_col] < eval_df.loc[is_tail, target_col]).mean()
)

non_tail_bias = eval_df.loc[is_non_tail, pred_col].mean() - eval_df.loc[is_non_tail, target_col].mean()
tail_bias = eval_df.loc[is_tail, pred_col].mean() - eval_df.loc[is_tail, target_col].mean()

unified_expected_cost_metrics_v2 = pd.DataFrame({
    "w_tail": [w_tail],
    "r2_test": [r2_test],
    "mae": [mae],
    "rmse": [rmse],
    "r2_test_log": [r2_test_log],
    "mae_log": [mae_log],
    "rmse_log": [rmse_log],
    "r2_test_w": [r2_test_w],
    "mae_w": [mae_w],
    "rmse_w": [rmse_w],
    "non_tail_pred_to_y_ratio": [non_tail_pred_to_y_ratio],
    "tail_pred_to_y_ratio": [tail_pred_to_y_ratio],
    "non_tail_sse": [non_tail_sse],
    "tail_sse": [tail_sse],
    "non_tail_share_of_total_sse": [non_tail_share_of_total_sse],
    "tail_share_of_total_sse": [tail_share_of_total_sse],
    "non_tail_under_prediction_rate": [non_tail_under_prediction_rate],
    "tail_under_prediction_rate": [tail_under_prediction_rate],
    "non_tail_bias": [non_tail_bias],
    "tail_bias": [tail_bias],
})

with pd.option_context("display.max_columns", None):
    display(unified_expected_cost_metrics_v2)

## 3.A) Top rows above expected by dollar residual (V2)

This also works with the same structure. I would just rename the output object for clarity.

In [ ]:
top_over_expected_by_residual_v2 = (
    benchmark_df.sort_values("residual", ascending=False)
    [[
        "Rndrng_NPI", "Year", "HCPCS_Cd", "provider_type", "Place_Of_Srvc", "state",
        "expected_cost_source", "cold_start_anchor_source",
        "observed_cost", "expected_cost", "residual", "oe_ratio", "pct_diff"
    ]]
    .head(50)
)

top_over_expected_by_residual_v2

## 3.B) Top rows above expected by O/E ratio (V2)

Same logic. Again, just rename for clarity.

In [ ]:
top_over_expected_by_oe_v2 = (
    benchmark_df.loc[benchmark_df["expected_cost"] > 0]
    .sort_values("oe_ratio", ascending=False)
    [[
        "Rndrng_NPI", "Year", "HCPCS_Cd", "provider_type", "Place_Of_Srvc", "state",
        "expected_cost_source", "cold_start_anchor_source",
        "observed_cost", "expected_cost", "residual", "oe_ratio", "pct_diff"
    ]]
    .head(50)
)

top_over_expected_by_oe_v2

## 3.C) Provider-level aggregation (V2)

Same structure. Rename the output object.

In [ ]:
provider_benchmark_summary_v2 = (
    benchmark_df.groupby("Rndrng_NPI", as_index=False)
    .agg(
        n_rows=("Rndrng_NPI", "size"),
        observed_total=("observed_cost", "sum"),
        expected_total=("expected_cost", "sum"),
        residual_total=("residual", "sum"),
        abs_residual_total=("abs_residual", "sum"),
        mean_oe_ratio=("oe_ratio", "mean")
    )
)

provider_benchmark_summary_v2["provider_oe_ratio"] = (
    provider_benchmark_summary_v2["observed_total"]
    / (provider_benchmark_summary_v2["expected_total"] + 1e-9)
)

provider_benchmark_summary_v2.sort_values("residual_total", ascending=False).head(50)

## Optional but recommended: add a very small route audit block for V2

This is not one of our original chunks, but it is very useful right after building V2.

This helps us quickly verify that:
- hot rows are being scored by Model G
- cold rows are being scored by Model D
- the scale of expected costs looks sensible in each segment

In [ ]:
route_audit_v2 = pd.DataFrame({
    "segment": ["hot_rows", "cold_rows"],
    "n_rows": [benchmark_df["has_lag"].sum(), (~benchmark_df["has_lag"]).sum()],
    "mean_observed_cost": [
        benchmark_df.loc[benchmark_df["has_lag"], "observed_cost"].mean(),
        benchmark_df.loc[~benchmark_df["has_lag"], "observed_cost"].mean(),
    ],
    "mean_expected_cost": [
        benchmark_df.loc[benchmark_df["has_lag"], "expected_cost"].mean(),
        benchmark_df.loc[~benchmark_df["has_lag"], "expected_cost"].mean(),
    ],
    "tail_rate": [
        benchmark_df.loc[benchmark_df["has_lag"], "is_top_1pct_avg_mdcr_stdzd_amt"].mean(),
        benchmark_df.loc[~benchmark_df["has_lag"], "is_top_1pct_avg_mdcr_stdzd_amt"].mean(),
    ],
})

route_audit_v2

## Important conceptual note for V2

In this new V2 engine:
- `cold_start_anchor`
- `cold_start_anchor_source`
- `expected_cost_support_tier`

are now support and interpretability metadata, not part of the actual Model D prediction calculation.

That means:
- they are still very useful for benchmarking and failure analysis
- but they are no longer part of the cold prediction mechanism itself

So if later we do failure analysis for V2, we should phrase it as:
- “How does Model D behave across cold rows with different support tiers?”

not:
- “Which anchor source did Model D use?”

because Model D does not use the anchor ladder.


# Failure Analysis Version 2

## A. Identify catastrophic predictions (V2)

What changes from V1?
- `cat_anchor_failure` should be renamed conceptually, because the cold model is no longer anchor-based.
- We can replace it with something like `cat_cold_low_support_failure`.
- Everything else still makes sense.

In [ ]:
import numpy as np
import pandas as pd

# ============================================================
# A. Identify catastrophic prediction buckets (V2)
# Model G for hot rows, Model D for cold rows
# ============================================================

# ------------------------------------------------------------
# 1) Start from benchmark_df
# ------------------------------------------------------------
failure_df = benchmark_df.copy()

# ------------------------------------------------------------
# 2) Define key thresholds
#    Explicit and easy to inspect / tune later
# ------------------------------------------------------------
abs_resid_q99 = failure_df["abs_residual"].quantile(0.99)
oe_q99 = failure_df["oe_ratio"].quantile(0.99)
resid_pos_q99 = failure_df["residual"].quantile(0.99)
resid_neg_q01 = failure_df["residual"].quantile(0.01)

obs_q99 = failure_df["observed_cost"].quantile(0.99)
exp_q99 = failure_df["expected_cost"].quantile(0.99)

print("99th percentile abs_residual:", abs_resid_q99)
print("99th percentile oe_ratio:", oe_q99)
print("99th percentile residual:", resid_pos_q99)
print("1st percentile residual:", resid_neg_q01)
print("99th percentile observed_cost:", obs_q99)
print("99th percentile expected_cost:", exp_q99)

# ------------------------------------------------------------
# 3) Add intuitive boolean flags for catastrophic buckets
# ------------------------------------------------------------

# Very high absolute error
failure_df["cat_abs_residual_q99"] = (
    failure_df["abs_residual"] >= abs_resid_q99
)

# Very high O/E
failure_df["cat_oe_q99"] = (
    failure_df["oe_ratio"] >= oe_q99
)

# Very large positive residual
failure_df["cat_large_positive_residual"] = (
    failure_df["residual"] >= resid_pos_q99
)

# Very large negative residual
failure_df["cat_large_negative_residual"] = (
    failure_df["residual"] <= resid_neg_q01
)

# Large error on high-support rows
failure_df["cat_large_error_high_support"] = (
    failure_df["expected_cost_support_tier"].isin(["high", "medium_high"])
    & (failure_df["abs_residual"] >= abs_resid_q99)
)

# Large error within tail rows
failure_df["cat_large_error_tail_row"] = (
    failure_df["is_top_1pct_avg_mdcr_stdzd_amt"]
    & (failure_df["abs_residual"] >= abs_resid_q99)
)

# ------------------------------------------------------------
# 4) More operational / narrative buckets
# ------------------------------------------------------------

# Catastrophic underprediction:
# very high observed + expected much too low
failure_df["cat_underprediction"] = (
    (failure_df["observed_cost"] >= obs_q99)
    & (failure_df["oe_ratio"] >= 3.0)
)

# Catastrophic overprediction:
# very high expected + observed much lower than expected
failure_df["cat_overprediction"] = (
    (failure_df["expected_cost"] >= exp_q99)
    & (failure_df["oe_ratio"] <= (1 / 3.0))
)

# High-confidence anomaly candidate
failure_df["cat_high_conf_anomaly"] = (
    failure_df["high_confidence_anomaly_candidate"]
)

# Cold-row low-support failure:
# cold row + weaker contextual support + very large miss
failure_df["cat_cold_low_support_failure"] = (
    (~failure_df["has_lag"])
    & failure_df["expected_cost_support_tier"].isin(["medium", "low"])
    & (failure_df["abs_residual"] >= abs_resid_q99)
)

# Hot-row failure:
# hot row + very large miss
failure_df["cat_hot_failure"] = (
    failure_df["has_lag"]
    & (failure_df["abs_residual"] >= abs_resid_q99)
)

# Cold-row failure:
# cold row + very large miss
failure_df["cat_cold_failure"] = (
    (~failure_df["has_lag"])
    & (failure_df["abs_residual"] >= abs_resid_q99)
)

# ------------------------------------------------------------
# 5) Create one broad catastrophic flag
# ------------------------------------------------------------
cat_cols = [
    "cat_abs_residual_q99",
    "cat_oe_q99",
    "cat_large_positive_residual",
    "cat_large_negative_residual",
    "cat_large_error_high_support",
    "cat_large_error_tail_row",
    "cat_underprediction",
    "cat_overprediction",
    "cat_high_conf_anomaly",
    "cat_cold_low_support_failure",
    "cat_hot_failure",
    "cat_cold_failure",
]

failure_df["is_any_catastrophic"] = failure_df[cat_cols].any(axis=1)

# ------------------------------------------------------------
# 6) Quick bucket counts
# ------------------------------------------------------------
catastrophic_bucket_counts_v2 = pd.DataFrame({
    "bucket": cat_cols + ["is_any_catastrophic"],
    "n_rows": [failure_df[c].sum() for c in cat_cols + ["is_any_catastrophic"]],
    "share_of_all_rows": [failure_df[c].mean() for c in cat_cols + ["is_any_catastrophic"]],
})

with pd.option_context("display.max_rows", None, "display.max_columns", None):
    display(catastrophic_bucket_counts_v2.sort_values("n_rows", ascending=False))

# ------------------------------------------------------------
# 7) Preview catastrophic rows
# ------------------------------------------------------------
catastrophic_preview_v2 = failure_df.loc[
    failure_df["is_any_catastrophic"],
    [
        "Rndrng_NPI", "Year", "HCPCS_Cd", "provider_type", "Place_Of_Srvc", "state",
        "observed_cost", "expected_cost", "residual", "abs_residual", "oe_ratio",
        "expected_cost_source", "cold_start_anchor_source", "expected_cost_support_tier",
        "high_confidence_anomaly_candidate"
    ] + cat_cols
].sort_values("abs_residual", ascending=False)

with pd.option_context("display.max_columns", None):
    display(catastrophic_preview_v2.head(50))

## B. Split failures by route (V2)

This is one of the most important sections.

What changes from V1?
- We still split by `hot_start` vs `cold_start`
- But for cold rows, `cold_start_anchor_source` is now support metadata, not prediction route
- So the section should say “cold support context” rather than implying it was the scoring route

In [ ]:
import numpy as np
import pandas as pd

# ============================================================
# B. Split catastrophic failures by scoring route (V2)
# ============================================================

failure_df = failure_df.copy()

# ------------------------------------------------------------
# 1) Add route labels
# ------------------------------------------------------------
failure_df["route"] = np.where(
    failure_df["has_lag"],
    "hot_start",
    "cold_start"
)

# This is support/context metadata for cold rows, not the model route
failure_df["cold_support_context"] = np.where(
    failure_df["route"] == "cold_start",
    failure_df["cold_start_anchor_source"],
    np.nan
)

# ------------------------------------------------------------
# 2) Restrict to catastrophic rows
# ------------------------------------------------------------
cat_df = failure_df.loc[failure_df["is_any_catastrophic"]].copy()

print("Total catastrophic rows:", len(cat_df))

# ------------------------------------------------------------
# 3) Split by hot-start vs cold-start
# ------------------------------------------------------------
cat_by_route_v2 = (
    cat_df.groupby("route", dropna=False)
    .agg(
        n_rows=("route", "size"),
        mean_abs_residual=("abs_residual", "mean"),
        median_abs_residual=("abs_residual", "median"),
        mean_oe_ratio=("oe_ratio", "mean"),
        tail_rate=("is_top_1pct_avg_mdcr_stdzd_amt", "mean"),
    )
    .reset_index()
)

cat_by_route_v2["share_within_catastrophic"] = (
    cat_by_route_v2["n_rows"] / cat_by_route_v2["n_rows"].sum()
)

with pd.option_context("display.max_columns", None):
    display(cat_by_route_v2.sort_values("n_rows", ascending=False))

# ------------------------------------------------------------
# 4) Within cold-start, split by cold support context
#    Again: not the prediction route, just context metadata
# ------------------------------------------------------------
cat_cold_by_support_context_v2 = (
    cat_df.loc[cat_df["route"] == "cold_start"]
    .groupby("cold_support_context", dropna=False)
    .agg(
        n_rows=("cold_support_context", "size"),
        mean_abs_residual=("abs_residual", "mean"),
        median_abs_residual=("abs_residual", "median"),
        mean_oe_ratio=("oe_ratio", "mean"),
        tail_rate=("is_top_1pct_avg_mdcr_stdzd_amt", "mean"),
    )
    .reset_index()
)

if len(cat_cold_by_support_context_v2) > 0:
    cat_cold_by_support_context_v2["share_within_cold_catastrophic"] = (
        cat_cold_by_support_context_v2["n_rows"] / cat_cold_by_support_context_v2["n_rows"].sum()
    )

with pd.option_context("display.max_columns", None):
    display(cat_cold_by_support_context_v2.sort_values("n_rows", ascending=False))

# ------------------------------------------------------------
# 5) Cross-tab: route x support tier
# ------------------------------------------------------------
cat_route_support_v2 = pd.crosstab(
    cat_df["route"],
    cat_df["expected_cost_support_tier"],
    margins=True
)

display(cat_route_support_v2)

# ------------------------------------------------------------
# 6) Cross-tab: route x catastrophic bucket
# ------------------------------------------------------------
cat_route_bucket_summary_v2 = (
    cat_df.groupby("route")[cat_cols]
    .sum()
    .T
)

display(cat_route_bucket_summary_v2)

## C. Look at peers of catastrophic rows (V2)

Do we need changes? Only minor wording changes. The peer logic itself is still valid.

In [ ]:
import numpy as np
import pandas as pd

# ============================================================
# C. Build peer-set context for catastrophic rows (V2)
# ============================================================

# ------------------------------------------------------------
# 1) Define reference history table
# ------------------------------------------------------------
peer_history_df = eda_hcpcs_df.copy()
peer_history_df["_peer_row_id"] = peer_history_df.index

# ------------------------------------------------------------
# 2) Take catastrophic rows only
# ------------------------------------------------------------
cat_df = failure_df.loc[failure_df["is_any_catastrophic"]].copy()
cat_df["_peer_row_id"] = cat_df.index

print("Catastrophic rows to profile:", len(cat_df))

# ------------------------------------------------------------
# 3) Base info carried through every peer view
# ------------------------------------------------------------
base_cols = [
    "_peer_row_id",
    "Rndrng_NPI",
    "Year",
    "HCPCS_Cd",
    "provider_type",
    "Place_Of_Srvc",
    "state",
    "observed_cost",
    "expected_cost",
    "residual",
    "abs_residual",
    "oe_ratio",
    "expected_cost_source",
    "cold_start_anchor_source",
    "expected_cost_support_tier",
]

# ------------------------------------------------------------
# 4) Define peer sets
# ------------------------------------------------------------
peer_key_sets = [
    ["HCPCS_Cd"],
    ["provider_type", "HCPCS_Cd", "Place_Of_Srvc"],
    ["provider_type", "Place_Of_Srvc", "state"],
    ["provider_type"],
]

# ------------------------------------------------------------
# 5) Helper: build vectorized peer summary
# ------------------------------------------------------------
def build_peer_summary_vectorized(
    cat_df: pd.DataFrame,
    peer_history_df: pd.DataFrame,
    key_cols: list[str],
    target_col: str = "avg_mdcr_stdzd_amt",
) -> pd.DataFrame:
    grp = (
        peer_history_df.groupby(key_cols)[target_col]
        .agg(
            peer_n_raw="size",
            peer_mean="mean",
            peer_std="std",
            peer_min="min",
            peer_q25=lambda s: s.quantile(0.25),
            peer_median="median",
            peer_q75=lambda s: s.quantile(0.75),
            peer_max="max",
        )
        .reset_index()
    )

    select_cols = list(dict.fromkeys(base_cols + key_cols))
    out = cat_df[select_cols].copy()
    out = out.merge(grp, on=key_cols, how="left")

    out["self_in_history"] = out["_peer_row_id"].isin(peer_history_df["_peer_row_id"])
    out["peer_n"] = out["peer_n_raw"] - out["self_in_history"].astype(int)

    stat_cols = [
        "peer_mean",
        "peer_std",
        "peer_min",
        "peer_q25",
        "peer_median",
        "peer_q75",
        "peer_max",
    ]

    no_peers_mask = out["peer_n"] <= 0
    out.loc[no_peers_mask, stat_cols] = np.nan
    out.loc[no_peers_mask, "peer_n"] = 0

    out["peer_key"] = " + ".join(key_cols)

    out = out[
        base_cols
        + ["peer_key", "peer_n"]
        + stat_cols
    ]

    return out

# ------------------------------------------------------------
# 6) Build and stack all peer summaries
# ------------------------------------------------------------
peer_summary_tables_v2 = []

for key_cols in peer_key_sets:
    peer_summary_tables_v2.append(
        build_peer_summary_vectorized(
            cat_df=cat_df,
            peer_history_df=peer_history_df,
            key_cols=key_cols,
            target_col="avg_mdcr_stdzd_amt",
        )
    )

cat_peer_context_v2 = pd.concat(peer_summary_tables_v2, ignore_index=True)

with pd.option_context("display.max_columns", None):
    display(cat_peer_context_v2.head(20))

## D. Compare catastrophic rows against peer distributions (V2)

### D1. Safe vectorized peer-distribution summary

No structural change needed, just rename outputs.

In [ ]:
import numpy as np
import pandas as pd

# ============================================================
# D1. Compare catastrophic rows against peer distributions (V2)
# SAFE VECTORIZE VERSION
# ============================================================

peer_history_df = eda_hcpcs_df.copy()
peer_history_df["_peer_row_id"] = peer_history_df.index

cat_df = failure_df.loc[failure_df["is_any_catastrophic"]].copy()
cat_df["_peer_row_id"] = cat_df.index

print("Catastrophic rows to profile:", len(cat_df))

base_cols = [
    "_peer_row_id",
    "Rndrng_NPI",
    "Year",
    "HCPCS_Cd",
    "provider_type",
    "Place_Of_Srvc",
    "state",
    "observed_cost",
    "expected_cost",
    "residual",
    "abs_residual",
    "oe_ratio",
    "expected_cost_source",
    "cold_start_anchor_source",
    "expected_cost_support_tier",
]

peer_key_sets = [
    ["HCPCS_Cd"],
    ["provider_type", "HCPCS_Cd", "Place_Of_Srvc"],
    ["provider_type", "Place_Of_Srvc", "state"],
    ["provider_type"],
]

def build_peer_distribution_context_safe(
    cat_df: pd.DataFrame,
    peer_history_df: pd.DataFrame,
    key_cols: list[str],
    target_col: str = "avg_mdcr_stdzd_amt",
) -> pd.DataFrame:
    grp_stats = (
        peer_history_df.groupby(key_cols)[target_col]
        .agg(
            peer_n_raw="size",
            peer_mean="mean",
            peer_median="median",
            peer_q25=lambda s: s.quantile(0.25),
            peer_q75=lambda s: s.quantile(0.75),
            peer_p90=lambda s: s.quantile(0.90),
            peer_p95=lambda s: s.quantile(0.95),
            peer_p99=lambda s: s.quantile(0.99),
        )
        .reset_index()
    )

    grp_stats["peer_iqr"] = grp_stats["peer_q75"] - grp_stats["peer_q25"]

    select_cols = list(dict.fromkeys(base_cols + key_cols))
    out = cat_df[select_cols].copy()
    out = out.merge(grp_stats, on=key_cols, how="left")

    out["self_in_history"] = out["_peer_row_id"].isin(peer_history_df["_peer_row_id"])
    out["peer_n"] = out["peer_n_raw"] - out["self_in_history"].astype(int)

    no_peers_mask = out["peer_n"] <= 0

    stat_cols = [
        "peer_mean",
        "peer_median",
        "peer_q25",
        "peer_q75",
        "peer_iqr",
        "peer_p90",
        "peer_p95",
        "peer_p99",
    ]

    out.loc[no_peers_mask, stat_cols] = np.nan
    out.loc[no_peers_mask, "peer_n"] = 0

    out["obs_to_peer_median_ratio"] = (
        out["observed_cost"] / (out["peer_median"] + 1e-9)
    )
    out["exp_to_peer_median_ratio"] = (
        out["expected_cost"] / (out["peer_median"] + 1e-9)
    )

    out.loc[no_peers_mask, ["obs_to_peer_median_ratio", "exp_to_peer_median_ratio"]] = np.nan

    out["peer_key"] = " + ".join(key_cols)

    out = out[
        base_cols
        + [
            "peer_key",
            "peer_n",
            "peer_mean",
            "peer_median",
            "peer_q25",
            "peer_q75",
            "peer_iqr",
            "peer_p90",
            "peer_p95",
            "peer_p99",
            "obs_to_peer_median_ratio",
            "exp_to_peer_median_ratio",
        ]
    ]

    return out

peer_tables_v2 = []
for key_cols in peer_key_sets:
    peer_tables_v2.append(
        build_peer_distribution_context_safe(
            cat_df=cat_df,
            peer_history_df=peer_history_df,
            key_cols=key_cols,
            target_col="avg_mdcr_stdzd_amt",
        )
    )

cat_peer_distribution_context_v2 = pd.concat(peer_tables_v2, ignore_index=True)

with pd.option_context("display.max_columns", None):
    display(cat_peer_distribution_context_v2.head(20))

### D2. Rank useful cases from the safe table (V2)

In [ ]:
with pd.option_context("display.max_columns", None):
    print("Top rows by observed / peer median ratio")
    display(
        cat_peer_distribution_context_v2
        .sort_values("obs_to_peer_median_ratio", ascending=False)
        .head(25)
    )

    print("Top rows by expected / peer median ratio (lowest first)")
    display(
        cat_peer_distribution_context_v2
        .sort_values("exp_to_peer_median_ratio", ascending=True)
        .head(25)
    )

### D3. Exact percentile calculation for only the top-N interesting cases (V2)

In [ ]:
import numpy as np
import pandas as pd

# ============================================================
# D3. Exact percentile within peers for TOP-N only (V2)
# ============================================================

topN = 100

top_cases_v2 = (
    cat_peer_distribution_context_v2
    .query("peer_key == 'HCPCS_Cd'")
    .sort_values("obs_to_peer_median_ratio", ascending=False)
    .head(topN)
    .copy()
)

def compute_exact_peer_percentile_small(
    top_cases: pd.DataFrame,
    peer_history_df: pd.DataFrame,
    key_cols: list[str],
    target_col: str = "avg_mdcr_stdzd_amt",
) -> pd.DataFrame:
    results = []

    for _, row in top_cases.iterrows():
        peers = peer_history_df.copy()
        for col in key_cols:
            peers = peers.loc[peers[col] == row[col]]

        peers = peers.loc[peers["_peer_row_id"] != row["_peer_row_id"]]

        vals = peers[target_col].astype(float)

        if len(vals) == 0:
            pctile = np.nan
        else:
            pctile = (vals <= row["observed_cost"]).mean()

        results.append({
            "_peer_row_id": row["_peer_row_id"],
            "peer_key": " + ".join(key_cols),
            "row_pctile_within_peers": pctile,
        })

    return pd.DataFrame(results)

top_cases_pctile_v2 = compute_exact_peer_percentile_small(
    top_cases=top_cases_v2,
    peer_history_df=peer_history_df,
    key_cols=["HCPCS_Cd"],
    target_col="avg_mdcr_stdzd_amt",
)

top_cases_with_pctile_v2 = top_cases_v2.merge(
    top_cases_pctile_v2,
    on=["_peer_row_id", "peer_key"],
    how="left",
)

with pd.option_context("display.max_columns", None):
    display(top_cases_with_pctile_v2.head(25))

### E. Examine support tier and cold-row support context jointly (V2)

Do we need this anymore?

Yes, but rewritten.

We no longer need the old interpretation of this section as “anchor-route analysis.”
But it is still valuable to analyze:
- catastrophic failures by `expected_cost_source`
- catastrophic failures by `expected_cost_support_tier`
- catastrophic failures across cold-row `cold_start_anchor_source` as support context metadata

That gives us a strong interpretability story.

In [ ]:
import numpy as np
import pandas as pd

# ============================================================
# E. Examine support tier and cold-row support context jointly (V2)
# ============================================================

# ------------------------------------------------------------
# 1) Catastrophic rows only
# ------------------------------------------------------------
cat_df = failure_df.loc[failure_df["is_any_catastrophic"]].copy()

# ------------------------------------------------------------
# 2) Summary by expected_cost_source
# ------------------------------------------------------------
cat_by_expected_source_v2 = (
    cat_df.groupby("expected_cost_source", dropna=False)
    .agg(
        n_rows=("expected_cost_source", "size"),
        mean_abs_residual=("abs_residual", "mean"),
        median_abs_residual=("abs_residual", "median"),
        mean_oe_ratio=("oe_ratio", "mean"),
        tail_rate=("is_top_1pct_avg_mdcr_stdzd_amt", "mean"),
    )
    .reset_index()
)

cat_by_expected_source_v2["share_within_catastrophic"] = (
    cat_by_expected_source_v2["n_rows"] / cat_by_expected_source_v2["n_rows"].sum()
)

with pd.option_context("display.max_columns", None):
    display(cat_by_expected_source_v2.sort_values("n_rows", ascending=False))

# ------------------------------------------------------------
# 3) Summary by support tier
# ------------------------------------------------------------
cat_by_support_tier_v2 = (
    cat_df.groupby("expected_cost_support_tier", dropna=False)
    .agg(
        n_rows=("expected_cost_support_tier", "size"),
        mean_abs_residual=("abs_residual", "mean"),
        median_abs_residual=("abs_residual", "median"),
        mean_oe_ratio=("oe_ratio", "mean"),
        tail_rate=("is_top_1pct_avg_mdcr_stdzd_amt", "mean"),
    )
    .reset_index()
)

cat_by_support_tier_v2["share_within_catastrophic"] = (
    cat_by_support_tier_v2["n_rows"] / cat_by_support_tier_v2["n_rows"].sum()
)

with pd.option_context("display.max_columns", None):
    display(cat_by_support_tier_v2.sort_values("n_rows", ascending=False))

# ------------------------------------------------------------
# 4) Cold-start catastrophic rows by support context + support tier
#    IMPORTANT:
#    cold_start_anchor_source is now contextual metadata only
# ------------------------------------------------------------
cat_cold_df_v2 = cat_df.loc[cat_df["expected_cost_source"] == "cold_start_model_D"].copy()

cold_context_support_crosstab_v2 = pd.crosstab(
    cat_cold_df_v2["cold_start_anchor_source"],
    cat_cold_df_v2["expected_cost_support_tier"],
    margins=True
)

display(cold_context_support_crosstab_v2)

# ------------------------------------------------------------
# 5) Share of catastrophic rows by support tier in full data
# ------------------------------------------------------------
full_support_counts_v2 = (
    failure_df.groupby("expected_cost_support_tier")
    .size()
    .rename("n_all")
)

cat_support_counts_v2 = (
    cat_df.groupby("expected_cost_support_tier")
    .size()
    .rename("n_cat")
)

support_cat_rate_v2 = pd.concat(
    [full_support_counts_v2, cat_support_counts_v2],
    axis=1
).fillna(0)

support_cat_rate_v2["catastrophic_rate_within_tier"] = (
    support_cat_rate_v2["n_cat"] / support_cat_rate_v2["n_all"]
)

display(support_cat_rate_v2.reset_index())

# ------------------------------------------------------------
# 6) Cold-start catastrophic rate by cold-row support context
#    Again: this is NOT the prediction route, only context
# ------------------------------------------------------------
full_cold_context_counts_v2 = (
    failure_df.loc[failure_df["expected_cost_source"] == "cold_start_model_D"]
    .groupby("cold_start_anchor_source")
    .size()
    .rename("n_all_cold")
)

cat_cold_context_counts_v2 = (
    cat_cold_df_v2.groupby("cold_start_anchor_source")
    .size()
    .rename("n_cat_cold")
)

cold_context_cat_rate_v2 = pd.concat(
    [full_cold_context_counts_v2, cat_cold_context_counts_v2],
    axis=1
).fillna(0)

cold_context_cat_rate_v2["catastrophic_rate_within_context"] = (
    cold_context_cat_rate_v2["n_cat_cold"] / cold_context_cat_rate_v2["n_all_cold"]
)

display(cold_context_cat_rate_v2.reset_index())

### F. Study problematic HCPCS families (V2)

Do we need changes?

Only minimal changes. We will replace `cold_start_model_I` with `cold_start_model_D` wherever relevant by using the new `failure_df`.

In [ ]:
import numpy as np
import pandas as pd

# ============================================================
# F. Study problematic HCPCS families (V2)
# ============================================================

# ------------------------------------------------------------
# 1) Catastrophic rows only
# ------------------------------------------------------------
cat_df = failure_df.loc[failure_df["is_any_catastrophic"]].copy()

# ------------------------------------------------------------
# 2) HCPCS frequency among catastrophic rows
# ------------------------------------------------------------
cat_hcpcs_counts_v2 = (
    cat_df.groupby(["HCPCS_Cd", "hcpcs_desc"], dropna=False)
    .agg(
        n_rows=("HCPCS_Cd", "size"),
        mean_abs_residual=("abs_residual", "mean"),
        median_abs_residual=("abs_residual", "median"),
        mean_oe_ratio=("oe_ratio", "mean"),
        tail_rate=("is_top_1pct_avg_mdcr_stdzd_amt", "mean"),
    )
    .reset_index()
    .sort_values("n_rows", ascending=False)
)

with pd.option_context("display.max_columns", None):
    display(cat_hcpcs_counts_v2.head(50))

# ------------------------------------------------------------
# 3) Compare catastrophic prevalence by HCPCS
# ------------------------------------------------------------
all_hcpcs_counts_v2 = (
    failure_df.groupby("HCPCS_Cd")
    .size()
    .rename("n_all")
)

cat_hcpcs_only_counts_v2 = (
    cat_df.groupby("HCPCS_Cd")
    .size()
    .rename("n_cat")
)

hcpcs_cat_rate_v2 = pd.concat(
    [all_hcpcs_counts_v2, cat_hcpcs_only_counts_v2],
    axis=1
).fillna(0)

hcpcs_cat_rate_v2["catastrophic_rate"] = (
    hcpcs_cat_rate_v2["n_cat"] / hcpcs_cat_rate_v2["n_all"]
)

hcpcs_desc_lookup_v2 = (
    failure_df[["HCPCS_Cd", "hcpcs_desc"]]
    .drop_duplicates()
)

hcpcs_cat_rate_v2 = (
    hcpcs_cat_rate_v2.reset_index()
    .merge(hcpcs_desc_lookup_v2, on="HCPCS_Cd", how="left")
    .sort_values(["catastrophic_rate", "n_cat"], ascending=[False, False])
)

with pd.option_context("display.max_columns", None):
    display(hcpcs_cat_rate_v2.head(50))

# ------------------------------------------------------------
# 4) Explicitly inspect known troublesome codes/families
# ------------------------------------------------------------
problem_codes = ["J9999", "J3490", "J3590"]

problem_code_rows_v2 = failure_df.loc[
    failure_df["HCPCS_Cd"].isin(problem_codes)
].copy()

problem_code_summary_v2 = (
    problem_code_rows_v2.groupby("HCPCS_Cd")
    .agg(
        n_all=("HCPCS_Cd", "size"),
        n_cat=("is_any_catastrophic", "sum"),
        catastrophic_rate=("is_any_catastrophic", "mean"),
        mean_abs_residual=("abs_residual", "mean"),
        mean_oe_ratio=("oe_ratio", "mean"),
        tail_rate=("is_top_1pct_avg_mdcr_stdzd_amt", "mean"),
    )
    .reset_index()
)

with pd.option_context("display.max_columns", None):
    display(problem_code_summary_v2)

# ------------------------------------------------------------
# 5) Simple heuristic flags for potentially problematic families
# ------------------------------------------------------------
failure_df["is_misc_unclassified_drug_code"] = (
    failure_df["HCPCS_Cd"].isin(["J9999", "J3490", "J3590"])
)

failure_df["is_potential_radiopharm_code"] = (
    failure_df["HCPCS_Cd"].astype(str).str.startswith("A9")
)

family_summary_v2 = pd.DataFrame({
    "family": [
        "misc_unclassified_drug_code",
        "potential_radiopharm_code",
    ],
    "n_all": [
        failure_df["is_misc_unclassified_drug_code"].sum(),
        failure_df["is_potential_radiopharm_code"].sum(),
    ],
    "n_cat": [
        failure_df.loc[failure_df["is_misc_unclassified_drug_code"], "is_any_catastrophic"].sum(),
        failure_df.loc[failure_df["is_potential_radiopharm_code"], "is_any_catastrophic"].sum(),
    ],
})

family_summary_v2["catastrophic_rate"] = (
    family_summary_v2["n_cat"] / family_summary_v2["n_all"]
)

display(family_summary_v2)

# ------------------------------------------------------------
# 6) Drill into catastrophic rows for known problematic families
# ------------------------------------------------------------
problem_family_rows_v2 = failure_df.loc[
    failure_df["is_any_catastrophic"]
    & (
        failure_df["is_misc_unclassified_drug_code"]
        | failure_df["is_potential_radiopharm_code"]
    )
].copy()

problem_family_view_v2 = problem_family_rows_v2[[
    "Rndrng_NPI", "Year", "HCPCS_Cd", "hcpcs_desc",
    "provider_type", "Place_Of_Srvc", "state",
    "observed_cost", "expected_cost", "residual", "abs_residual", "oe_ratio",
    "expected_cost_source", "cold_start_anchor_source", "expected_cost_support_tier"
]].sort_values("abs_residual", ascending=False)

with pd.option_context("display.max_columns", None):
    display(problem_family_view_v2.head(100))

### V1 to V2 change:

The main conceptual change in V2 is this:
- In V1, cold_start_anchor_source was tied to the cold prediction mechanism.
- In V2, cold_start_anchor_source is not the prediction mechanism, but it is still a very informative support-context variable.

So the analysis should not say:
- “Which anchor source did the model use?”

It should say:
- “Among cold rows, do catastrophic failures cluster in weaker contextual support regimes?”

That is still a strong and interpretable story.